# Module 6 — Evidence Retrieval + Fact Verification

Module 5 is FROZEN. Every cell here READS its corpus and never writes to it.

**Each stage is its own cell and is independently resumable.** If Colab
disconnects, re-run the failed cell only — completed shards are skipped.

**Runtime:** set *Runtime > Change runtime type*.
GPU is needed only for stages 1, 3, 5, 6, 7. Stages 2, 4, 8 and verification
are CPU. Do NOT hold a GPU while running CPU-only stages — an idle GPU is what
triggers free-tier disconnection.

**Compute scope (documented limitation):** top-50 candidates (not 100),
seq-len 256 (not 512), 3 epochs (not 10), 50k stratified FEVER training claims
(not 228,090). Full-scale numbers are NOT claimed anywhere.

**Label spaces are never mapped.** FEVER = SUPPORTS/REFUTES/NOT ENOUGH INFO
(entailment). X-FACT = false/true/partly true-misleading (veracity). The
encoder transfers; the head is replaced.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import subprocess, sys
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")
assert PROJECT.exists(), f"not found: {PROJECT}"

try:
    import torch
    GPU = torch.cuda.is_available()
    print("accelerator :", torch.cuda.get_device_name(0) if GPU else "CPU only")
except Exception:
    GPU = False
    print("accelerator : CPU only (torch not importable)")
print("GPU available:", GPU)
print("project      :", PROJECT)

In [ ]:
# DIAGNOSTIC — run BEFORE any encoding stage. ~1 minute. Downloads LaBSE once.
#
# A runtime that dies right after "Loading weights" with no traceback was
# OOM-killed: the OS terminates the kernel, so Python never gets to raise.
# This measures where the memory actually goes instead of guessing.
import shutil, sys, time
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")
sys.path.insert(0, str(PROJECT))


def mem():
    try:
        with open("/proc/meminfo") as f:
            d = {}
            for line in f:
                k, v = line.split(":", 1)
                d[k] = float(v.split()[0]) / 1024.0
        return d.get("MemAvailable", 0.0), d.get("MemTotal", 0.0)
    except Exception:
        return 0.0, 0.0


avail, total = mem()
print("RAM  : %.0f MB available of %.0f MB total" % (avail, total))
du = shutil.disk_usage("/content")
print("disk : %.1f GB free of %.1f GB" % (du.free / 1e9, du.total / 1e9))
try:
    import torch
    print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
except Exception as e:
    print("torch: NOT IMPORTABLE", e)

from scripts.m6_common import load_config, load_corpus

cfg = load_config(PROJECT / "configs/module6_retrieval_verification.yaml")
corpus = load_corpus(cfg, "x_fact", root=PROJECT)
lens = sorted(len(t) for t in corpus["evidence_texts"].values())
n = len(lens)
print()
print("x_fact evidence: %d items" % n)
print("  chars  median=%d  p95=%d  max=%d  total=%.1f MB"
      % (lens[n // 2], lens[int(n * 0.95)], lens[-1], sum(lens) / 1e6))
if lens[-1] > 100000:
    print("  !! Some evidence items are very large. Tokenisation truncates them")
    print("     at max_seq_length, but they still inflate peak RAM while batching.")

a0, _ = mem()
t0 = time.perf_counter()
from sentence_transformers import SentenceTransformer

MODEL = "sentence-transformers/LaBSE"
print()
print("loading", MODEL, "...")
try:
    m = SentenceTransformer(MODEL, device="cpu")
    a1, _ = mem()
    print("  loaded in %.1fs, RAM consumed by load: %.0f MB (available now %.0f MB)"
          % (time.perf_counter() - t0, a0 - a1, a1))
    m.max_seq_length = 256
    t1 = time.perf_counter()
    v = m.encode(list(corpus["evidence_texts"].values())[:64], batch_size=8,
                 convert_to_numpy=True, show_progress_bar=False)
    a2, _ = mem()
    dt = max(time.perf_counter() - t1, 1e-6)
    print("  encoded 64 texts in %.1fs -> %s, available now %.0f MB" % (dt, v.shape, a2))
    rate = 64 / dt
    print("  throughput ~%.1f texts/sec  =>  x_fact 6,968 in ~%.0f min,"
          "  FEVER 67,388 in ~%.0f min" % (rate, 6968 / rate / 60, 67388 / rate / 60))
    if a2 < 1500:
        print()
        print("  !! Under 1.5 GB free after a 64-text batch: LaBSE does not fit")
        print("     this runtime. Use the smaller encoder (see below).")
except Exception as e:
    print("  FAILED:", type(e).__name__, e)
    print("  -> LaBSE cannot load here; use the smaller encoder.")

print()
print("If LaBSE is too heavy, re-run Stage 1 with the lighter multilingual model:")
print("  !python scripts/m6_01_encode_evidence.py --dataset x_fact --phase test \\")
print("     --model sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 \\")
print("     --batch-size 16 --max-seq-length 128 --shard-size 2000")


In [ ]:
PAYLOAD_JSON = r'''{"configs/module6_retrieval_verification.yaml": {"b64": "IyBNb2R1bGUgNiDigJQgRXZpZGVuY2UgUmV0cmlldmFsICsgRmFjdCBWZXJpZmljYXRpb24KIwojIFJlYWRzIE1vZHVsZSA1J3MgZnJvemVuIGNvcnB1cy4gTmV2ZXIgd3JpdGVzIHRvIGl0LgojCiMgVmFsdWVzIGhlcmUgZGVsaWJlcmF0ZWx5IERJVkVSR0UgZnJvbSBjb25maWdzL21vZHVsZTVfZmFjdF92ZXJpZmljYXRpb24ueWFtbAojIHdoZXJlIGZyZWUtQ29sYWIgY29tcHV0ZSBmb3JjZXMgYSByZWR1Y3Rpb24uIEV2ZXJ5IGRpdmVyZ2VuY2UgaXMgcmVjb3JkZWQKIyB1bmRlciBgY29tcHV0ZV9zY29wZWAgd2l0aCB0aGUgb3JpZ2luYWwgdmFsdWUsIHNvIHRoZSBsaW1pdGF0aW9uIGlzCiMgcmVwb3J0YWJsZSByYXRoZXIgdGhhbiBoaWRkZW4uCgptb2R1bGU6IDYKbmFtZTogZXZpZGVuY2VfcmV0cmlldmFsX2FuZF9mYWN0X3ZlcmlmaWNhdGlvbgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gaW5wdXRzCiMgTW9kdWxlIDUgb3V0cHV0cy4gUkVBRC1PTkxZLiBNb2R1bGUgNiBtdXN0IG5ldmVyIG9wZW4gdGhlc2UgZm9yIHdyaXRpbmcuCmlucHV0czoKICBjb3JwdXNfZGlyOiBkYXRhc2V0cy9mYWN0X3ZlcmlmaWNhdGlvbgogIGNsYWltc19maWxlbmFtZTogY2xhaW1zLmpzb25sCiAgZXZpZGVuY2VfZmlsZW5hbWU6IGV2aWRlbmNlLmpzb25sCiAgc3BsaXRzX2ZpbGVuYW1lOiBjbGFpbV9zcGxpdHMuanNvbgogIGRhdGFzZXRzOiBbZmV2ZXIsIHhfZmFjdF0KICBtb2R1bGU1X2NvbmZpZzogY29uZmlncy9tb2R1bGU1X2ZhY3RfdmVyaWZpY2F0aW9uLnlhbWwKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBsYWJlbCBzcGFjZXMKIyBGRVZFUiBhbmQgWC1GQUNUIGFyZSBOT1QgdGhlIHNhbWUgdGFzayBhbmQgYXJlIG5ldmVyIG1lcmdlZCBvciBtYXBwZWQuCiMgRkVWRVIgaXMgdGV4dHVhbCBlbnRhaWxtZW50IGFnYWluc3QgZXZpZGVuY2U7IFgtRkFDVCBpcyBhIGZhY3QtY2hlY2tlcidzCiMgdmVyYWNpdHkgdmVyZGljdC4gTWFwcGluZyAicGFydGx5IHRydWUvbWlzbGVhZGluZyIgb250byBhbnkgRkVWRVIgY2xhc3MKIyB3b3VsZCBiZSBhIGZhYnJpY2F0aW9uLCBzbyB0aGUgdHdvIGhlYWRzIHN0YXkgc2VwYXJhdGUgYW5kIGFyZSByZXBvcnRlZAojIHNlcGFyYXRlbHkuIFRoZSBlbmNvZGVyIGlzIHNoYXJlZCBzbyBGRVZFUiAtPiBUYW1pbCB0cmFuc2ZlciBpcyBtZWFzdXJhYmxlLgpsYWJlbF9zcGFjZXM6CiAgZmV2ZXI6CiAgICB0YXNrOiBlbnRhaWxtZW50CiAgICBsYWJlbHM6IFsiU1VQUE9SVFMiLCAiUkVGVVRFUyIsICJOT1QgRU5PVUdIIElORk8iXQogICAgc291cmNlX2ZpZWxkOiBsYWJlbAogICAgcm9sZTogcHJldHJhaW5pbmcKICB4X2ZhY3Q6CiAgICB0YXNrOiB2ZXJhY2l0eQogICAgbGFiZWxzOiBbImZhbHNlIiwgInRydWUiLCAicGFydGx5IHRydWUvbWlzbGVhZGluZyJdCiAgICBzb3VyY2VfZmllbGQ6IGxhYmVsCiAgICByb2xlOiB0cmFuc2Zlcl90YXJnZXQKICAgIG5vdGU6ID4KICAgICAgTGFiZWxzIGFyZSBwcmVzZXJ2ZWQgZXhhY3RseSBhcyBwdWJsaXNoZWQgYnkgWC1GQUNULiBUaGV5IGFyZSBOT1QKICAgICAgcmVtYXBwZWQgdG8gRkVWRVIncyBsYWJlbCBzcGFjZS4gVGhlIFRhbWlsIHN1YnNldCAoaGZfY29uZmlnICd0YScpCiAgICAgIG9ubHk7IHRoZXNlIGFyZSBub3Qgb3ZlcmFsbCBYLUZBQ1Qgc3RhdGlzdGljcy4KICBtYXBwaW5nX2JldHdlZW5fc3BhY2VzOiBmYWxzZSAgICAgICMgbXVzdCBzdGF5IGZhbHNlOyBhc3NlcnRlZCBieSB0ZXN0cwoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gcmV0cmlldmFsCnJldHJpZXZhbDoKICBiaV9lbmNvZGVyOiBzZW50ZW5jZS10cmFuc2Zvcm1lcnMvTGFCU0UKICBhbHRlcm5hdGl2ZV9iaV9lbmNvZGVyOiBpbnRmbG9hdC9tdWx0aWxpbmd1YWwtZTUtYmFzZQogIGVtYmVkZGluZ19kaW06IDc2OAogIG5vcm1hbGl6ZTogdHJ1ZSAgICAgICAgICAgICAgICAgICAjIHJlcXVpcmVkIGZvciBJbmRleEZsYXRJUCA9PSBjb3NpbmUKICBpbmRleDogZmFpc3NfZmxhdF9pcAogIHRvcF9rX2NhbmRpZGF0ZXM6IDUwICAgICAgICAgICAgICAjIE1vZHVsZSA1IGNvbmZpZyBzYWlkIDEwMCAoc2VlIGNvbXB1dGVfc2NvcGUpCiAgZW5jb2RlX2JhdGNoX3NpemU6IDY0CiAgbWF4X3NlcV9sZW5ndGg6IDI1NgogIHNoYXJkX3NpemU6IDEwMDAwCiAgZXZhbHVhdGVfc3BsaXRzOiBbdmFsaWRhdGlvbiwgdGVzdF0KCmJtMjU6CiAgZW5hYmxlZDogdHJ1ZQogIGsxOiAxLjUKICBiOiAwLjc1CiAgdG9wX2s6IDUwCiAgdG9rZW5pemVyOiB1bmljb2RlX3dvcmQgICAgICAgICAgICMgVGFtaWwtc2FmZTogbm8gTGF0aW4tb25seSBhc3N1bXB0aW9ucwoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIHJlcmFua2luZwpyZXJhbmtpbmc6CiAgY3Jvc3NfZW5jb2RlcjogeGxtLXJvYmVydGEtYmFzZQogIG1heF9sZW5ndGg6IDI1NiAgICAgICAgICAgICAgICAgICAjIE1vZHVsZSA1IGNvbmZpZyBzYWlkIDUxMiAoc2VlIGNvbXB1dGVfc2NvcGUpCiAgdG9wX25fYWZ0ZXJfcmVyYW5rOiA1CiAgYmF0Y2hfc2l6ZTogMzIKICByZXJhbmtfc3BsaXRzOiBbdmFsaWRhdGlvbiwgdGVzdF0KICB0cmFpbl9zdWJzZXRfc2l6ZTogMjAwMDAgICAgICAgICAgIyBzdHJhdGlmaWVkOyBkb2N1bWVudGVkLCBub3Qgc2lsZW50CiAgY2FuZGlkYXRlX3NldF9tdXN0X2JlX3ByZXNlcnZlZDogdHJ1ZSAgICMgYXNzZXJ0ZWQgYnkgdGVzdHMKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gdmVyaWZpZXIKdmVyaWZpZXI6CiAgZW5jb2RlcjogeGxtLXJvYmVydGEtYmFzZQogIG1heF9sZW5ndGg6IDI1NiAgICAgICAgICAgICAgICAgICAjIE1vZHVsZSA1IGNvbmZpZyBzYWlkIDUxMgogIGhpZGRlbl9kaW06IDUxMgogIGRyb3BvdXQ6IDAuMQogIGVwb2NoczogMyAgICAgICAgICAgICAgICAgICAgICAgICAjIE1vZHVsZSA1IGNvbmZpZyBzYWlkIDEwCiAgYmF0Y2hfc2l6ZTogMTYKICBncmFkaWVudF9hY2N1bXVsYXRpb25fc3RlcHM6IDIKICBsZWFybmluZ19yYXRlOiAyLjBlLTUKICB3ZWlnaHRfZGVjYXk6IDAuMDEKICB3YXJtdXBfcmF0aW86IDAuMDYKICBlYXJseV9zdG9wcGluZ19wYXRpZW5jZTogMgogIG1vbml0b3I6IHZhbF9tYWNyb19mMQogIGV2aWRlbmNlX3Blcl9jbGFpbTogNQogIGNoZWNrcG9pbnRfZXZlcnlfc3RlcHM6IDUwMAogIGZldmVyOgogICAgdHJhaW5fc3Vic2V0X3NpemU6IDUwMDAwICAgICAgICAjIHN0cmF0aWZpZWQgZnJvbSAyMjgsMDkwIChzZWUgY29tcHV0ZV9zY29wZSkKICAgIHN0cmF0aWZ5X2J5OiBbbGFiZWxdCiAgeF9mYWN0OgogICAgdHJhaW5fc3Vic2V0X3NpemU6IG51bGwgICAgICAgICAjIDEsMDk2IGNsYWltczogdXNlIGFsbAogICAgZnJlZXplX2VuY29kZXJfbGF5ZXJzOiAwCiAgICByZWluaXRpYWxpc2VfaGVhZDogdHJ1ZSAgICAgICAgICMgaGVhZCBpcyByZXBsYWNlZCwgbm90IHJldXNlZAoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBjYWxpYnJhdGlvbgpjYWxpYnJhdGlvbjoKICBlY2VfYmluczogMTAKICB0ZW1wZXJhdHVyZV9zY2FsaW5nOiB0cnVlCiAgZml0X3RlbXBlcmF0dXJlX29uOiB2YWxpZGF0aW9uICAgICMgbmV2ZXIgb24gdGVzdAogIHJlbGlhYmlsaXR5X2RpYWdyYW06IHRydWUKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBldmFsdWF0aW9uCmV2YWx1YXRpb246CiAgcmV0cmlldmFsX21ldHJpY3M6IFsicmVjYWxsQDEiLCAicmVjYWxsQDUiLCAicmVjYWxsQDEwIiwgInJlY2FsbEA1MCIsICJtcnIiLCAibmRjZ0AxMCJdCiAgdmVyaWZpY2F0aW9uX21ldHJpY3M6IFsiYWNjdXJhY3kiLCAicHJlY2lzaW9uX21hY3JvIiwgInJlY2FsbF9tYWNybyIsICJtYWNyb19mMSIsICJlY2UiXQogIG5kY2dfazogMTAKICBwZXJfbGFuZ3VhZ2U6IHRydWUKICBsYW5ndWFnZXM6IFtlbiwgdGFdCiAgcGVyX2RhdGFzZXQ6IHRydWUKICB0ZXN0X2V2YWx1YXRlZF9vbmNlOiB0cnVlCiAgZXhjbHVkZV9jbGFpbXNfd2l0aG91dF9nb2xkX2V2aWRlbmNlOiB0cnVlCiAgcmVwb3J0X2V4Y2x1ZGVkX2NvdW50czogdHJ1ZSAgICAgICMgdGhlIDQyNyBGRVZFUiBORUkgY2xhaW1zIG11c3QgYmUgY291bnRlZAoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGFibGF0aW9ucwphYmxhdGlvbnM6CiAgYm0yNV92c19kZW5zZTogdHJ1ZQogIHRvcF9rX3ZhbHVlczogWzEsIDUsIDEwLCAyMCwgNTBdCiAgY2xhaW1fb25seV92c19jbGFpbV9ldmlkZW5jZTogdHJ1ZQogIGVuZ2xpc2hfb25seV92c19tdWx0aWxpbmd1YWxfZXZpZGVuY2U6IHRydWUKICByZXRyaWV2YWxfb25seV92c19yZXRyaWV2YWxfcmVyYW5rOiB0cnVlCiAgb3JhY2xlX2dvbGRfZXZpZGVuY2U6IHRydWUKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gbGVha2FnZSBjb250cm9sCiMgU2NvcGUgaXMgZGVsZWdhdGVkIHRvIE1vZHVsZSA1J3MgYnVpbGRfaW5kZXhfc2NvcGUoKTsgTW9kdWxlIDYgb25seQojIHRyYW5zbGF0ZXMgcGVybWl0dGVkIENMQUlNIGluZGljZXMgaW50byBwZXJtaXR0ZWQgRVZJREVOQ0UgaWRzLgpsZWFrYWdlOgogIHVzZV9tb2R1bGU1X2luZGV4X3Njb3BlOiB0cnVlCiAgaW5kZXhfc2NvcGU6CiAgICB0cmFpbjogW3RyYWluXQogICAgdmFsaWRhdGlvbjogW3RyYWluLCB2YWxpZGF0aW9uXQogICAgdGVzdDogW3RyYWluLCB2YWxpZGF0aW9uLCB0ZXN0XQogIGFzc2VydF9ub19vdXRfb2Zfc2NvcGVfZXZpZGVuY2U6IHRydWUKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBkb2N1bWVudGVkIGNvbXB1dGUgc2NvcGUKIyBGcmVlIENvbGFiLCAwIGNvbXB1dGUgdW5pdHMuIEVhY2ggcmVkdWN0aW9uIGlzIHJlY29yZGVkIHdpdGggdGhlIG9yaWdpbmFsLgpjb21wdXRlX3Njb3BlOgogIHJlYXNvbjogPgogICAgRnJlZS10aWVyIEdvb2dsZSBDb2xhYiB3aXRoIG5vIHB1cmNoYXNlZCBjb21wdXRlIHVuaXRzLiBUaGUgb3JpZ2luYWxseQogICAgY29uZmlndXJlZCBwaXBlbGluZSB3YXMgbWVhc3VyZWQgYXMgaW5mZWFzaWJsZTogY3Jvc3MtZW5jb2RlciByZXJhbmtpbmcKICAgIGF0IHRvcC0xMDAvc2VxLTUxMiBvdmVyIGFsbCBjbGFpbXMgaXMgfjI2TSBmb3J3YXJkIHBhc3NlcyAofjYwIGggb24gYQogICAgVDQpLCBhbmQgdmVyaWZpZXIgdHJhaW5pbmcgYXQgMjI4LDA5MCBjbGFpbXMgeCAxMCBlcG9jaHMgaXMgfjEyIGguCiAgcmVkdWN0aW9uczoKICAgIC0ge2l0ZW06IHRvcF9rX2NhbmRpZGF0ZXMsICAgICAgICBvcmlnaW5hbDogMTAwLCAgICB1c2VkOiA1MH0KICAgIC0ge2l0ZW06IHJlcmFua2VyX21heF9sZW5ndGgsICAgICBvcmlnaW5hbDogNTEyLCAgICB1c2VkOiAyNTZ9CiAgICAtIHtpdGVtOiB2ZXJpZmllcl9tYXhfbGVuZ3RoLCAgICAgb3JpZ2luYWw6IDUxMiwgICAgdXNlZDogMjU2fQogICAgLSB7aXRlbTogdmVyaWZpZXJfZXBvY2hzLCAgICAgICAgIG9yaWdpbmFsOiAxMCwgICAgIHVzZWQ6IDN9CiAgICAtIHtpdGVtOiBmZXZlcl90cmFpbl9jbGFpbXMsICAgICAgb3JpZ2luYWw6IDIyODA5MCwgdXNlZDogNTAwMDB9CiAgICAtIHtpdGVtOiByZXJhbmtfdHJhaW5fc3Vic2V0LCAgICAgb3JpZ2luYWw6ICJhbGwiLCAgdXNlZDogMjAwMDB9CiAgZnVsbF9zY2FsZV9yZXN1bHRzX2NsYWltZWQ6IGZhbHNlCiAgbXVzdF9kb2N1bWVudF9pbl9yZXBvcnQ6IHRydWUKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIHJlcHJvZHVjaWJpbGl0eQpyZXByb2R1Y2liaWxpdHk6CiAgc2VlZDogNDIKICBzZWVkczogWzQyLCA0MywgNDRdCiAgZGV0ZXJtaW5pc3RpYzogdHJ1ZQoKcnVudGltZToKICBkZXZpY2U6IGF1dG8gICAgICAgICAgICAgICAgICAgICAgIyBub3RlYm9vayBwcmludHMgd2hhdCBpdCBkZXRlY3RlZAogIHVzZV9taXhlZF9wcmVjaXNpb246IHRydWUKICBncHVfc3RhZ2VzOiBbZW5jb2RlX2V2aWRlbmNlLCBlbmNvZGVfY2xhaW1zLCByZXJhbmssIHRyYWluX3ZlcmlmaWVyLCBldmFsdWF0ZV92ZXJpZmllcl0KICBjcHVfc3RhZ2VzOiBbYnVpbGRfaW5kZXgsIGJtMjUsIHJldHJpZXZlLCByZXRyaWV2YWxfbWV0cmljcywgYWJsYXRpb25zLCB2ZXJpZnlfb3V0cHV0c10KCm91dHB1dDoKICByb290OiBvdXRwdXRzL21vZHVsZTYKICBtb2RlbHNfZGlyOiBtb2RlbHMvbW9kdWxlNgogIGVtYmVkZGluZ3NfZGlyOiBlbWJlZGRpbmdzL21vZHVsZTYKICBzdGFnZXM6IFtlbmNvZGVfZXZpZGVuY2UsIGJ1aWxkX2luZGV4LCBlbmNvZGVfY2xhaW1zLCByZXRyaWV2ZSwgcmVyYW5rLAogICAgICAgICAgIHZlcmlmaWVyX3RyYWluaW5nLCBldmFsdWF0aW9uLCBhYmxhdGlvbnMsIG1hbmlmZXN0c10KICByZXBvcnRfZmlsZW5hbWU6IG1vZHVsZTZfcmVwb3J0Lm1kCgpsb2dnaW5nOgogIGxldmVsOiBJTkZPCg==", "sha256": "5374fc35d7af3fb3718c1b3c8a7f70fb35d833b76d78acfd4b964690ad3b667d", "size": 6772}, "src/retrieval/__init__.py": {"b64": "IiIiTW9kdWxlIDYg4oCUIGV2aWRlbmNlIHJldHJpZXZhbC4KClJlYWRzIE1vZHVsZSA1J3MgZnJvemVuIGNvcnB1czsgbmV2ZXIgd3JpdGVzIHRvIGl0LgoKICAgIGluZGV4X3Njb3BlICAgICAgICBjbGFpbS1sZXZlbCBzY29wZSAtPiBwZXJtaXR0ZWQgZXZpZGVuY2UgaWRzIChsZWFrYWdlIGdhdGUpCiAgICBlbmNvZGVycyAgICAgICAgICAgc2hhcmRlZCwgcmVzdW1hYmxlIExhQlNFIGVuY29kaW5nCiAgICBmYWlzc19pbmRleCAgICAgICAgZXhhY3QgZmxhdCBpbm5lci1wcm9kdWN0IGluZGV4IG92ZXIgbm9ybWFsaXNlZCB2ZWN0b3JzCiAgICBibTI1X2Jhc2VsaW5lICAgICAgVGFtaWwtc2FmZSBsZXhpY2FsIGJhc2VsaW5lCiAgICByZXJhbmtlciAgICAgICAgICAgWExNLVIgY3Jvc3MtZW5jb2RlciwgY2FuZGlkYXRlLXNldCBwcmVzZXJ2aW5nCiAgICByZXRyaWV2YWxfbWV0cmljcyAgUmVjYWxsQGsgLyBNUlIgLyBuRENHIGFnYWluc3QgcmVhbCBnb2xkIGV2aWRlbmNlCiIiIgoKZnJvbSBzcmMucmV0cmlldmFsLmluZGV4X3Njb3BlIGltcG9ydCAoICAjIG5vcWE6IEY0MDEKICAgIEV2aWRlbmNlU2NvcGUsCiAgICBhc3NlcnRfaW5kZXhfd2l0aGluX3Njb3BlLAogICAgYnVpbGRfZXZpZGVuY2Vfc2NvcGUsCiAgICBjbGFpbV9ldmlkZW5jZV9pZHMsCiAgICB2ZXJpZnlfcGhhc2VfaXNvbGF0aW9uLAopCmZyb20gc3JjLnJldHJpZXZhbC5yZXRyaWV2YWxfbWV0cmljcyBpbXBvcnQgKCAgIyBub3FhOiBGNDAxCiAgICBldmFsdWF0ZV9yZXRyaWV2YWwsCiAgICBuZGNnX2F0X2ssCiAgICByZWNhbGxfYXRfaywKICAgIHJlY2lwcm9jYWxfcmFuaywKKQoKX19hbGxfXyA9IFsKICAgICJFdmlkZW5jZVNjb3BlIiwgImJ1aWxkX2V2aWRlbmNlX3Njb3BlIiwgImNsYWltX2V2aWRlbmNlX2lkcyIsCiAgICAiYXNzZXJ0X2luZGV4X3dpdGhpbl9zY29wZSIsICJ2ZXJpZnlfcGhhc2VfaXNvbGF0aW9uIiwKICAgICJldmFsdWF0ZV9yZXRyaWV2YWwiLCAicmVjYWxsX2F0X2siLCAicmVjaXByb2NhbF9yYW5rIiwgIm5kY2dfYXRfayIsCl0K", "sha256": "aa39fe261e17e175300d1fe58cd4f5c92433a727cde44ce94fdf14f6d6fa009c", "size": 1041}, "src/retrieval/encoders.py": {"b64": "IiIiTW9kdWxlIDYg4oCUIHNoYXJkZWQsIHJlc3VtYWJsZSBMYUJTRS9FNSBlbmNvZGluZy4KCkRlc2lnbmVkIGFyb3VuZCBmcmVlIENvbGFiIGRpc2Nvbm5lY3RpbmcgbWlkLXJ1bi4gV29yayBpcyBzcGxpdCBpbnRvIHNoYXJkcwpvZiBgc2hhcmRfc2l6ZWAgdGV4dHM7IGVhY2ggZmluaXNoZWQgc2hhcmQgaXMgd3JpdHRlbiBgLnRtcC5ucHlgIHRoZW4KYXRvbWljYWxseSByZW5hbWVkLCBhbmQgcmVjb3JkZWQgaW4gYSBtYW5pZmVzdC4gUmUtcnVubmluZyBza2lwcyBjb21wbGV0ZWQKc2hhcmRzLCBzbyBhIGRpc2Nvbm5lY3QgY29zdHMgb25lIHNoYXJkLCBub3QgdGhlIHdob2xlIHN0YWdlLgoKVGhlIGAudG1wLm5weWAgc3VmZml4IGlzIG5vdCBjb3NtZXRpYy4gYG5wLnNhdmVgIGFwcGVuZHMgYC5ucHlgIHVubGVzcyB0aGUKcGF0aCBhbHJlYWR5IGVuZHMgaW4gaXQsIHNvIGEgdGVtcCBuYW1lIGxpa2UgYHNoYXJkLm5weS50bXBgIHNpbGVudGx5IGJlY29tZXMKYHNoYXJkLm5weS50bXAubnB5YCBhbmQgZXZlcnkgcmVuYW1lIGZhaWxzLiBUaGF0IGV4YWN0IGJ1ZyB3YXMgZm91bmQgYW5kCmZpeGVkIGluIE1vZHVsZSAzOyB0aGUgc2FtZSBjb252ZW50aW9uIGlzIHVzZWQgaGVyZSBkZWxpYmVyYXRlbHkuCgpNb2RlbCBsb2FkaW5nIGlzIGxhenk6IGltcG9ydGluZyB0aGlzIG1vZHVsZSBtdXN0IG5vdCBwdWxsIH4xLjggR0Igb2YgTGFCU0UKd2VpZ2h0cywgc28gdGVzdHMgYW5kIENQVSBzdGFnZXMgY2FuIGltcG9ydCBpdCBmcmVlbHkuIEVuY29kaW5nIGlzCmRldGVybWluaXN0aWMgLS0gZXZhbCBtb2RlLCBubyBkcm9wb3V0LCBubyBzYW1wbGluZywgZml4ZWQgc2hhcmQgb3JkZXIuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGdjCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCB0aW1lCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueSwgQ2FsbGFibGUsIFNlcXVlbmNlCgppbXBvcnQgbnVtcHkgYXMgbnAKCgpjbGFzcyBMb3dNZW1vcnlTdG9wKFJ1bnRpbWVFcnJvcik6CiAgICAiIiJSYWlzZWQgdG8gc3RvcCBjbGVhbmx5IEJFRk9SRSB0aGUgT1Mga2lsbHMgdGhlIHByb2Nlc3MuCgogICAgQSBrZXJuZWwgT09NLWtpbGwgcHJvZHVjZXMgbm8gdHJhY2ViYWNrIGFuZCBubyBtYW5pZmVzdCB1cGRhdGUsIHNvIHRoZQogICAgc2hhcmQgaW4gZmxpZ2h0IGlzIGxvc3QgYW5kIHRoZSBjYXVzZSBpcyBpbnZpc2libGUuIFN0b3BwaW5nIHZvbHVudGFyaWx5CiAgICBrZWVwcyBldmVyeSBjb21wbGV0ZWQgc2hhcmQgYW5kIHNheXMgd2h5LgogICAgIiIiCgoKZGVmIGF2YWlsYWJsZV9tYigpIC0+IGZsb2F0OgogICAgIiIiU3lzdGVtIE1lbUF2YWlsYWJsZSBpbiBNQjsgMC4wIHdoZXJlIC9wcm9jIGlzIHVuYXZhaWxhYmxlLiIiIgogICAgdHJ5OgogICAgICAgIHdpdGggb3BlbigiL3Byb2MvbWVtaW5mbyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgICAgIGZvciBsaW5lIGluIGY6CiAgICAgICAgICAgICAgICBpZiBsaW5lLnN0YXJ0c3dpdGgoIk1lbUF2YWlsYWJsZToiKToKICAgICAgICAgICAgICAgICAgICByZXR1cm4gZmxvYXQobGluZS5zcGxpdCgpWzFdKSAvIDEwMjQuMAogICAgZXhjZXB0IEV4Y2VwdGlvbjogICMgbm9xYTogQkxFMDAxCiAgICAgICAgcGFzcwogICAgcmV0dXJuIDAuMAoKCmRlZiBtZW1vcnlfcmVwb3J0KCkgLT4gZGljdFtzdHIsIGZsb2F0XToKICAgICIiIlByb2Nlc3MgUlNTIGFuZCBzeXN0ZW0tYXZhaWxhYmxlIFJBTSwgaW4gTUIuCgogICAgQSBDb2xhYiBydW50aW1lIHRoYXQgZGllcyBpbW1lZGlhdGVseSBhZnRlciBtb2RlbCBsb2FkIHdpdGggbm8gdHJhY2ViYWNrCiAgICBoYXMgYWxtb3N0IGNlcnRhaW5seSBiZWVuIE9PTS1raWxsZWQ6IHRoZSBrZXJuZWwgaXMgdGVybWluYXRlZCwgc28gUHl0aG9uCiAgICBuZXZlciBnZXRzIHRvIHJhaXNlLiBMb2dnaW5nIG1lbW9yeSBwZXIgc2hhcmQgdHVybnMgdGhhdCBzaWxlbnQgZGVhdGggaW50bwogICAgYSB2aXNpYmxlIHRyZW5kLgogICAgIiIiCiAgICBvdXQ6IGRpY3Rbc3RyLCBmbG9hdF0gPSB7fQogICAgdHJ5OgogICAgICAgIGltcG9ydCByZXNvdXJjZQogICAgICAgIHJ1ID0gcmVzb3VyY2UuZ2V0cnVzYWdlKHJlc291cmNlLlJVU0FHRV9TRUxGKS5ydV9tYXhyc3MKICAgICAgICAjIExpbnV4IHJlcG9ydHMga0IsIG1hY09TIGJ5dGVzLgogICAgICAgIG91dFsicGVha19yc3NfbWIiXSA9IHJvdW5kKHJ1IC8gMTAyNC4wLCAxKSBpZiBydSA+IDEgPDwgMjAgZWxzZSByb3VuZChydSAvIDEwMjQuMCwgMSkKICAgIGV4Y2VwdCBFeGNlcHRpb246ICAjIG5vcWE6IEJMRTAwMQogICAgICAgIHBhc3MKICAgIHRyeToKICAgICAgICB3aXRoIG9wZW4oIi9wcm9jL21lbWluZm8iLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgICAgICBpbmZvID0ge30KICAgICAgICAgICAgZm9yIGxpbmUgaW4gZjoKICAgICAgICAgICAgICAgIHBhcnRzID0gbGluZS5zcGxpdCgpCiAgICAgICAgICAgICAgICBpZiBsZW4ocGFydHMpID49IDI6CiAgICAgICAgICAgICAgICAgICAgaW5mb1twYXJ0c1swXS5yc3RyaXAoIjoiKV0gPSBmbG9hdChwYXJ0c1sxXSkKICAgICAgICBpZiAiTWVtQXZhaWxhYmxlIiBpbiBpbmZvOgogICAgICAgICAgICBvdXRbImF2YWlsYWJsZV9tYiJdID0gcm91bmQoaW5mb1siTWVtQXZhaWxhYmxlIl0gLyAxMDI0LjAsIDEpCiAgICAgICAgaWYgIk1lbVRvdGFsIiBpbiBpbmZvOgogICAgICAgICAgICBvdXRbInRvdGFsX21iIl0gPSByb3VuZChpbmZvWyJNZW1Ub3RhbCJdIC8gMTAyNC4wLCAxKQogICAgZXhjZXB0IEV4Y2VwdGlvbjogICMgbm9xYTogQkxFMDAxCiAgICAgICAgcGFzcwogICAgcmV0dXJuIG91dAoKCmRlZiB0ZXh0X2ZpbmdlcnByaW50KHRleHRzOiBTZXF1ZW5jZVtzdHJdKSAtPiBzdHI6CiAgICAiIiJIYXNoIG9mIHRoZSBleGFjdCBpbnB1dCBsaXN0LCBzbyBhIG1hbmlmZXN0IGNhbm5vdCBiZSByZXVzZWQgZm9yCiAgICBkaWZmZXJlbnQgZGF0YS4gT3JkZXItc2Vuc2l0aXZlIG9uIHB1cnBvc2U6IHNoYXJkIGkgbXVzdCBtZWFuIHRoZSBzYW1lCiAgICByb3dzIG9uIHJlc3VtZS4iIiIKICAgIGggPSBoYXNobGliLnNoYTI1NigpCiAgICBoLnVwZGF0ZShzdHIobGVuKHRleHRzKSkuZW5jb2RlKCkpCiAgICBmb3IgdCBpbiB0ZXh0czoKICAgICAgICBoLnVwZGF0ZShiIlx4MDAiKQogICAgICAgIGgudXBkYXRlKHQuZW5jb2RlKCJ1dGYtOCIpKQogICAgcmV0dXJuIGguaGV4ZGlnZXN0KCkKCgpAZGF0YWNsYXNzCmNsYXNzIFNoYXJkUGxhbjoKICAgIHNoYXJkX3NpemU6IGludAogICAgbl9pdGVtczogaW50CgogICAgQHByb3BlcnR5CiAgICBkZWYgbl9zaGFyZHMoc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiAoc2VsZi5uX2l0ZW1zICsgc2VsZi5zaGFyZF9zaXplIC0gMSkgLy8gc2VsZi5zaGFyZF9zaXplIGlmIHNlbGYubl9pdGVtcyBlbHNlIDAKCiAgICBkZWYgYm91bmRzKHNlbGYsIGk6IGludCkgLT4gdHVwbGVbaW50LCBpbnRdOgogICAgICAgIHN0YXJ0ID0gaSAqIHNlbGYuc2hhcmRfc2l6ZQogICAgICAgIHJldHVybiBzdGFydCwgbWluKHN0YXJ0ICsgc2VsZi5zaGFyZF9zaXplLCBzZWxmLm5faXRlbXMpCgoKY2xhc3MgU2hhcmRlZEVuY29kZXI6CiAgICAiIiJSZXN1bWFibGUgZW5jb2RlLXRvLWRpc2sgZHJpdmVyLgoKICAgIGBlbmNvZGVfZm5gIG1hcHMgYSBsaXN0IG9mIHN0cmluZ3MgdG8gYW4gKG4sIGRpbSkgZmxvYXQzMiBhcnJheS4gSW5qZWN0aW5nCiAgICBpdCBrZWVwcyB0aGlzIGNsYXNzIHRlc3RhYmxlIHdpdGhvdXQgZG93bmxvYWRpbmcgYW55IG1vZGVsLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG91dF9kaXI6IFBhdGgsIHNoYXJkX3NpemU6IGludCA9IDEwMDAwLAogICAgICAgICAgICAgICAgIGVuY29kZV9mbjogQ2FsbGFibGVbW1NlcXVlbmNlW3N0cl1dLCBucC5uZGFycmF5XSB8IE5vbmUgPSBOb25lLAogICAgICAgICAgICAgICAgIG1vZGVsX25hbWU6IHN0ciA9ICJzZW50ZW5jZS10cmFuc2Zvcm1lcnMvTGFCU0UiKSAtPiBOb25lOgogICAgICAgIHNlbGYub3V0X2RpciA9IFBhdGgob3V0X2RpcikKICAgICAgICBzZWxmLm91dF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIHNlbGYuc2hhcmRfc2l6ZSA9IGludChzaGFyZF9zaXplKQogICAgICAgIHNlbGYuZW5jb2RlX2ZuID0gZW5jb2RlX2ZuCiAgICAgICAgc2VsZi5tb2RlbF9uYW1lID0gbW9kZWxfbmFtZQogICAgICAgIHNlbGYubWFuaWZlc3RfcGF0aCA9IHNlbGYub3V0X2RpciAvICJtYW5pZmVzdC5qc29uIgoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIG1hbmlmZXN0CiAgICBkZWYgbG9hZF9tYW5pZmVzdChzZWxmKSAtPiBkaWN0W3N0ciwgQW55XToKICAgICAgICBpZiBzZWxmLm1hbmlmZXN0X3BhdGguaXNfZmlsZSgpOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICByZXR1cm4ganNvbi5sb2FkcyhzZWxmLm1hbmlmZXN0X3BhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgICAgICAgICBleGNlcHQganNvbi5KU09ORGVjb2RlRXJyb3I6CiAgICAgICAgICAgICAgICByZXR1cm4ge30KICAgICAgICByZXR1cm4ge30KCiAgICBkZWYgX3dyaXRlX21hbmlmZXN0KHNlbGYsIG06IGRpY3Rbc3RyLCBBbnldKSAtPiBOb25lOgogICAgICAgIHRtcCA9IHNlbGYubWFuaWZlc3RfcGF0aC53aXRoX3N1ZmZpeCgiLnRtcC5qc29uIikKICAgICAgICB0bXAud3JpdGVfdGV4dChqc29uLmR1bXBzKG0sIGluZGVudD0yLCBlbnN1cmVfYXNjaWk9RmFsc2UpLCBlbmNvZGluZz0idXRmLTgiKQogICAgICAgIHRtcC5yZXBsYWNlKHNlbGYubWFuaWZlc3RfcGF0aCkKCiAgICBkZWYgc2hhcmRfcGF0aChzZWxmLCBpOiBpbnQpIC0+IFBhdGg6CiAgICAgICAgcmV0dXJuIHNlbGYub3V0X2RpciAvIGYic2hhcmRfe2k6MDVkfS5ucHkiCgogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBlbmNvZGUKICAgIGRlZiBlbmNvZGUoc2VsZiwgdGV4dHM6IFNlcXVlbmNlW3N0cl0sIGlkczogU2VxdWVuY2Vbc3RyXSB8IE5vbmUgPSBOb25lLAogICAgICAgICAgICAgICBsb2c6IENhbGxhYmxlW1tzdHJdLCBOb25lXSA9IHByaW50LAogICAgICAgICAgICAgICBtaW5fYXZhaWxhYmxlX21iOiBmbG9hdCA9IDAuMCwKICAgICAgICAgICAgICAgc2V0dGluZ3M6IGRpY3Rbc3RyLCBBbnldIHwgTm9uZSA9IE5vbmUsCiAgICAgICAgICAgICAgIHRleHRfZm46IENhbGxhYmxlW1tpbnQsIGludF0sIGxpc3Rbc3RyXV0gfCBOb25lID0gTm9uZSkgLT4gZGljdFtzdHIsIEFueV06CiAgICAgICAgIiIiRW5jb2RlIHNoYXJkIGJ5IHNoYXJkLCBzYXZpbmcgZWFjaCBvbmUgYmVmb3JlIHN0YXJ0aW5nIHRoZSBuZXh0LgoKICAgICAgICBgbWluX2F2YWlsYWJsZV9tYmAgc3RvcHMgY2xlYW5seSBpZiBzeXN0ZW0gUkFNIGRyb3BzIGJlbG93IHRoZQogICAgICAgIHRocmVzaG9sZCwgaW5zdGVhZCBvZiBsZXR0aW5nIHRoZSBrZXJuZWwgT09NLWtpbGwgdGhlIHByb2Nlc3MgKHdoaWNoCiAgICAgICAgeWllbGRzIG5vIHRyYWNlYmFjayBhbmQgbG9zZXMgdGhlIGluLWZsaWdodCBzaGFyZCkuCgogICAgICAgIGBzZXR0aW5nc2AgKG1vZGVsIC8gYmF0Y2ggLyBzZXEgbGVuKSBpcyBzdG9yZWQgaW4gdGhlIG1hbmlmZXN0IGFuZAogICAgICAgIHJlLWNoZWNrZWQgb24gcmVzdW1lOiBmaW5pc2hpbmcgYSBjb3JwdXMgaGFsZiBhdCBzZXEtMTI4IGFuZCBoYWxmIGF0CiAgICAgICAgc2VxLTI1NiB3b3VsZCBzaWxlbnRseSBwcm9kdWNlIHR3byBpbmNvbXBhdGlibGUgZW1iZWRkaW5nIHNwYWNlcy4KCiAgICAgICAgYHRleHRfZm4obG8sIGhpKWAgc3VwcGxpZXMgb25lIHNoYXJkJ3MgdGV4dHMgbGF6aWx5LCBzbyB0aGUgZnVsbCB0ZXh0CiAgICAgICAgbGlzdCBuZXZlciBoYXMgdG8gYmUgbWF0ZXJpYWxpc2VkIGFsb25nc2lkZSB0aGUgY29ycHVzLgogICAgICAgICIiIgogICAgICAgIGlmIHNlbGYuZW5jb2RlX2ZuIGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigibm8gZW5jb2RlX2ZuOiBwYXNzIG9uZSwgb3IgY2FsbCBidWlsZF9sYWJzZV9lbmNvZGVfZm4oKSIpCiAgICAgICAgaWYgaWRzIGlzIG5vdCBOb25lIGFuZCBsZW4oaWRzKSAhPSBsZW4odGV4dHMpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYie2xlbihpZHMpfSBpZHMgYnV0IHtsZW4odGV4dHMpfSB0ZXh0cyIpCgogICAgICAgIHBsYW4gPSBTaGFyZFBsYW4oc2VsZi5zaGFyZF9zaXplLCBsZW4odGV4dHMpKQogICAgICAgIGZwID0gdGV4dF9maW5nZXJwcmludCh0ZXh0cykKICAgICAgICBtYW4gPSBzZWxmLmxvYWRfbWFuaWZlc3QoKQoKICAgICAgICBpZiBtYW4uZ2V0KCJmaW5nZXJwcmludCIpIGFuZCBtYW5bImZpbmdlcnByaW50Il0gIT0gZnA6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgICAgICJNYW5pZmVzdCBmaW5nZXJwcmludCBkb2VzIG5vdCBtYXRjaCB0aGVzZSBpbnB1dHMuIFRoZSBjb3JwdXMgY2hhbmdlZCAiCiAgICAgICAgICAgICAgICAic2luY2UgdGhlIGxhc3QgcnVuOyBkZWxldGUgdGhlIHN0YWdlIGRpcmVjdG9yeSB0byByZS1lbmNvZGUgcmF0aGVyICIKICAgICAgICAgICAgICAgICJ0aGFuIG1peGluZyBlbWJlZGRpbmdzIGZyb20gdHdvIGRpZmZlcmVudCBjb3Jwb3JhLiIpCgogICAgICAgIGlmIHNldHRpbmdzOgogICAgICAgICAgICBwcmV2ID0gbWFuLmdldCgic2V0dGluZ3MiKQogICAgICAgICAgICBpZiBwcmV2IGFuZCBwcmV2ICE9IHNldHRpbmdzOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICAgICAgICAgICAgIGYiRW5jb2RlciBzZXR0aW5ncyBjaGFuZ2VkIGJldHdlZW4gcnVuczoge3ByZXZ9IC0+IHtzZXR0aW5nc30uICIKICAgICAgICAgICAgICAgICAgICBmIlJlc3VtaW5nIHdvdWxkIG1peCBlbWJlZGRpbmdzIHByb2R1Y2VkIHVuZGVyIGRpZmZlcmVudCBzZXR0aW5ncy4gIgogICAgICAgICAgICAgICAgICAgIGYiRGVsZXRlIHtzZWxmLm91dF9kaXJ9IHRvIHJlLWVuY29kZSBmcm9tIHNjcmF0Y2gsIG9yIHJlc3RvcmUgdGhlICIKICAgICAgICAgICAgICAgICAgICBmInByZXZpb3VzIHNldHRpbmdzLiIpCiAgICAgICAgICAgIG1hblsic2V0dGluZ3MiXSA9IHNldHRpbmdzCgogICAgICAgIGNvbXBsZXRlZCA9IHNldChtYW4uZ2V0KCJjb21wbGV0ZWRfc2hhcmRzIiwgW10pKQogICAgICAgICMgVHJ1c3QgdGhlIGZpbGUsIG5vdCBqdXN0IHRoZSBtYW5pZmVzdDogYSBjcmFzaCBiZXR3ZWVuIHJlbmFtZSBhbmQKICAgICAgICAjIG1hbmlmZXN0IHdyaXRlIHdvdWxkIG90aGVyd2lzZSBsb3NlIGEgZ29vZCBzaGFyZC4KICAgICAgICBmb3IgaSBpbiByYW5nZShwbGFuLm5fc2hhcmRzKToKICAgICAgICAgICAgaWYgc2VsZi5zaGFyZF9wYXRoKGkpLmlzX2ZpbGUoKToKICAgICAgICAgICAgICAgIGNvbXBsZXRlZC5hZGQoaSkKCiAgICAgICAgbWFuLnVwZGF0ZSh7Im1vZGVsIjogc2VsZi5tb2RlbF9uYW1lLCAiZmluZ2VycHJpbnQiOiBmcCwgIm5faXRlbXMiOiBsZW4odGV4dHMpLAogICAgICAgICAgICAgICAgICAgICJzaGFyZF9zaXplIjogc2VsZi5zaGFyZF9zaXplLCAibl9zaGFyZHMiOiBwbGFuLm5fc2hhcmRzLAogICAgICAgICAgICAgICAgICAgICJjb21wbGV0ZWRfc2hhcmRzIjogc29ydGVkKGNvbXBsZXRlZCksICJzdGF0dXMiOiAiaW5fcHJvZ3Jlc3MifSkKICAgICAgICBzZWxmLl93cml0ZV9tYW5pZmVzdChtYW4pCgogICAgICAgIHQwID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIHJhbV9zZWVuOiBsaXN0W2Zsb2F0XSA9IFtdCiAgICAgICAgZm9yIGkgaW4gcmFuZ2UocGxhbi5uX3NoYXJkcyk6CiAgICAgICAgICAgIGlmIGkgaW4gY29tcGxldGVkOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgbG8sIGhpID0gcGxhbi5ib3VuZHMoaSkKCiAgICAgICAgICAgIGF2YWlsID0gYXZhaWxhYmxlX21iKCkKICAgICAgICAgICAgaWYgYXZhaWw6CiAgICAgICAgICAgICAgICByYW1fc2Vlbi5hcHBlbmQoYXZhaWwpCiAgICAgICAgICAgIGlmIG1pbl9hdmFpbGFibGVfbWIgYW5kIGF2YWlsIGFuZCBhdmFpbCA8IG1pbl9hdmFpbGFibGVfbWI6CiAgICAgICAgICAgICAgICBtYW5bInN0YXR1cyJdID0gInN0b3BwZWRfbG93X21lbW9yeSIKICAgICAgICAgICAgICAgIG1hblsic3RvcHBlZF9hdF9zaGFyZCJdID0gaQogICAgICAgICAgICAgICAgbWFuWyJhdmFpbGFibGVfbWJfYXRfc3RvcCJdID0gYXZhaWwKICAgICAgICAgICAgICAgIG1hblsibWluX2F2YWlsYWJsZV9tYl9vYnNlcnZlZCJdID0gbWluKHJhbV9zZWVuKSBpZiByYW1fc2VlbiBlbHNlIE5vbmUKICAgICAgICAgICAgICAgIHNlbGYuX3dyaXRlX21hbmlmZXN0KG1hbikKICAgICAgICAgICAgICAgIHJhaXNlIExvd01lbW9yeVN0b3AoCiAgICAgICAgICAgICAgICAgICAgZiJTdG9wcGluZyBiZWZvcmUgc2hhcmQge2l9OiBvbmx5IHthdmFpbDouMGZ9IE1CIFJBTSBhdmFpbGFibGUgIgogICAgICAgICAgICAgICAgICAgIGYiKHRocmVzaG9sZCB7bWluX2F2YWlsYWJsZV9tYjouMGZ9IE1CKS4gIgogICAgICAgICAgICAgICAgICAgIGYie2xlbihjb21wbGV0ZWQpfS97cGxhbi5uX3NoYXJkc30gc2hhcmRzIGFyZSBzYXZlZCBhbmQgd2lsbCBiZSAiCiAgICAgICAgICAgICAgICAgICAgZiJza2lwcGVkIG9uIHJlc3VtZS4gUmUtcnVuIHRoZSBzYW1lIGNvbW1hbmQgYWZ0ZXIgcmVzdGFydGluZyB0aGUgIgogICAgICAgICAgICAgICAgICAgIGYicnVudGltZSwgb3IgbG93ZXIgLS1iYXRjaC1zaXplIC8gLS1zaGFyZC1zaXplLiIpCgogICAgICAgICAgICBiYXRjaF90ZXh0cyA9IHRleHRfZm4obG8sIGhpKSBpZiB0ZXh0X2ZuIGVsc2UgbGlzdCh0ZXh0c1tsbzpoaV0pCiAgICAgICAgICAgIHZlY3MgPSBucC5hc2FycmF5KHNlbGYuZW5jb2RlX2ZuKGJhdGNoX3RleHRzKSwgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICAgICAgaWYgdmVjcy5zaGFwZVswXSAhPSBoaSAtIGxvOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYic2hhcmQge2l9OiBlbmNvZGVyIHJldHVybmVkIHt2ZWNzLnNoYXBlWzBdfSByb3dzICIKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmImZvciB7aGkgLSBsb30gdGV4dHMiKQogICAgICAgICAgICB0bXAgPSBzZWxmLm91dF9kaXIgLyBmInNoYXJkX3tpOjA1ZH0udG1wLm5weSIgICAjIC50bXAubnB5LCBub3QgLm5weS50bXAKICAgICAgICAgICAgbnAuc2F2ZSh0bXAsIHZlY3MpCiAgICAgICAgICAgIHRtcC5yZXBsYWNlKHNlbGYuc2hhcmRfcGF0aChpKSkKICAgICAgICAgICAgY29tcGxldGVkLmFkZChpKQogICAgICAgICAgICBtYW5bImNvbXBsZXRlZF9zaGFyZHMiXSA9IHNvcnRlZChjb21wbGV0ZWQpCiAgICAgICAgICAgIG1hblsibGFzdF9zaGFyZF9zZWNvbmRzIl0gPSByb3VuZCh0aW1lLnBlcmZfY291bnRlcigpIC0gdDAsIDIpCiAgICAgICAgICAgIG1hblsibWluX2F2YWlsYWJsZV9tYl9vYnNlcnZlZCJdID0gbWluKHJhbV9zZWVuKSBpZiByYW1fc2VlbiBlbHNlIE5vbmUKICAgICAgICAgICAgc2VsZi5fd3JpdGVfbWFuaWZlc3QobWFuKQoKICAgICAgICAgICAgIyBSZWxlYXNlIHRoZSBzaGFyZCdzIHRlbnNvcnMvYXJyYXlzIGJlZm9yZSB0aGUgbmV4dCBvbmUgc3RhcnRzLgogICAgICAgICAgICBkZWwgYmF0Y2hfdGV4dHMsIHZlY3MKICAgICAgICAgICAgZ2MuY29sbGVjdCgpCgogICAgICAgICAgICBhZnRlciA9IGF2YWlsYWJsZV9tYigpCiAgICAgICAgICAgIGxvZyhmIiAgc2hhcmQge2kgKyAxfS97cGxhbi5uX3NoYXJkc30gW3tsb306e2hpfV0gLT4ge3NlbGYuc2hhcmRfcGF0aChpKS5uYW1lfSIKICAgICAgICAgICAgICAgICsgKGYiICB8IFJBTSB7YXZhaWw6LjBmfS0+e2FmdGVyOi4wZn0gTUIiIGlmIGF2YWlsIGVsc2UgIiIpKQoKICAgICAgICBpZiBpZHMgaXMgbm90IE5vbmUgYW5kIG5vdCAoc2VsZi5vdXRfZGlyIC8gImlkcy5qc29uIikuaXNfZmlsZSgpOgogICAgICAgICAgICB0bXAgPSBzZWxmLm91dF9kaXIgLyAiaWRzLnRtcC5qc29uIgogICAgICAgICAgICB0bXAud3JpdGVfdGV4dChqc29uLmR1bXBzKGxpc3QoaWRzKSwgZW5zdXJlX2FzY2lpPUZhbHNlKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgICAgICAgICAgdG1wLnJlcGxhY2Uoc2VsZi5vdXRfZGlyIC8gImlkcy5qc29uIikKCiAgICAgICAgbWFuWyJzdGF0dXMiXSA9ICJjb21wbGV0ZSIKICAgICAgICBtYW5bInRvdGFsX3NlY29uZHMiXSA9IHJvdW5kKHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MCwgMikKICAgICAgICBzZWxmLl93cml0ZV9tYW5pZmVzdChtYW4pCiAgICAgICAgcmV0dXJuIG1hbgoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGxvYWQKICAgIGRlZiBsb2FkX2FsbChzZWxmKSAtPiBucC5uZGFycmF5OgogICAgICAgIG1hbiA9IHNlbGYubG9hZF9tYW5pZmVzdCgpCiAgICAgICAgbl9zaGFyZHMgPSBpbnQobWFuLmdldCgibl9zaGFyZHMiLCAwKSkKICAgICAgICBtaXNzaW5nID0gW2kgZm9yIGkgaW4gcmFuZ2Uobl9zaGFyZHMpIGlmIG5vdCBzZWxmLnNoYXJkX3BhdGgoaSkuaXNfZmlsZSgpXQogICAgICAgIGlmIG1pc3Npbmc6CiAgICAgICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYie2xlbihtaXNzaW5nKX0gc2hhcmQocykgbWlzc2luZywgZS5nLiB7bWlzc2luZ1s6NV19OyAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGYicmUtcnVuIHRoZSBlbmNvZGUgc3RhZ2UgdG8gcmVzdW1lIikKICAgICAgICBpZiBuX3NoYXJkcyA9PSAwOgogICAgICAgICAgICByZXR1cm4gbnAuemVyb3MoKDAsIDApLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgIHBhcnRzID0gW25wLmxvYWQoc2VsZi5zaGFyZF9wYXRoKGkpKSBmb3IgaSBpbiByYW5nZShuX3NoYXJkcyldCiAgICAgICAgb3V0ID0gbnAuY29uY2F0ZW5hdGUocGFydHMsIGF4aXM9MCkKICAgICAgICBpZiBvdXQuc2hhcGVbMF0gIT0gaW50KG1hbi5nZXQoIm5faXRlbXMiLCBvdXQuc2hhcGVbMF0pKToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYibG9hZGVkIHtvdXQuc2hhcGVbMF19IHJvd3MsIG1hbmlmZXN0IHNheXMge21hbi5nZXQoJ25faXRlbXMnKX0iKQogICAgICAgIHJldHVybiBvdXQKCiAgICBkZWYgbG9hZF9pZHMoc2VsZikgLT4gbGlzdFtzdHJdOgogICAgICAgIHAgPSBzZWxmLm91dF9kaXIgLyAiaWRzLmpzb24iCiAgICAgICAgcmV0dXJuIGpzb24ubG9hZHMocC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpIGlmIHAuaXNfZmlsZSgpIGVsc2UgW10KCgpkZWYgYnVpbGRfbGFic2VfZW5jb2RlX2ZuKG1vZGVsX25hbWU6IHN0ciA9ICJzZW50ZW5jZS10cmFuc2Zvcm1lcnMvTGFCU0UiLAogICAgICAgICAgICAgICAgICAgICAgICAgIGJhdGNoX3NpemU6IGludCA9IDY0LCBtYXhfc2VxX2xlbmd0aDogaW50ID0gMjU2LAogICAgICAgICAgICAgICAgICAgICAgICAgIGRldmljZTogc3RyIHwgTm9uZSA9IE5vbmUsIG51bV90aHJlYWRzOiBpbnQgfCBOb25lID0gTm9uZSwKICAgICAgICAgICAgICAgICAgICAgICAgICBnY19ldmVyeV9iYXRjaGVzOiBpbnQgPSA4LAogICAgICAgICAgICAgICAgICAgICAgICAgIGxvZzogQ2FsbGFibGVbW3N0cl0sIE5vbmVdIHwgTm9uZSA9IE5vbmUKICAgICAgICAgICAgICAgICAgICAgICAgICApIC0+IENhbGxhYmxlW1tTZXF1ZW5jZVtzdHJdXSwgbnAubmRhcnJheV06CiAgICAiIiJNZW1vcnktYm91bmRlZCBzZW50ZW5jZS10cmFuc2Zvcm1lcnMgZW5jb2Rlci4KCiAgICBUaGUgb3JpZ2luYWwgdmVyc2lvbiBoYW5kZWQgdGhlIHdob2xlIHNoYXJkIHRvIGBtb2RlbC5lbmNvZGUoKWAuIFdpdGggdGhlCiAgICBkZWZhdWx0IDEwLDAwMC1pdGVtIHNoYXJkIHRoYXQgbWVhbnQgb25lIGNhbGwgY292ZXJpbmcgdGhlIGVudGlyZSA2LDk2OC1pdGVtCiAgICBYLUZBQ1QgcG9vbDogc2VudGVuY2UtdHJhbnNmb3JtZXJzIGxlbmd0aC1zb3J0cyB0aGUgaW5wdXQsIGJ1aWxkcyBiYXRjaGVzIG9mCiAgICA2NCBhdCBzZXEtMjU2LCBhbmQgaG9sZHMgZXZlcnkgb3V0cHV0IGVtYmVkZGluZyBwbHVzIGludGVybmFsIGJ1ZmZlcnMgdW50aWwKICAgIHRoZSBjYWxsIHJldHVybnMuIE9uIGEgZnJlZSBDUFUgcnVudGltZSB0aGF0IHBlYWsgaXMgd2hhdCBnZXRzIHRoZSBwcm9jZXNzCiAgICBPT00ta2lsbGVkIC0tIHNpbGVudGx5LCBiZWNhdXNlIHRoZSBrZXJuZWwgdGVybWluYXRlcyBpdCBiZWZvcmUgUHl0aG9uIGNhbgogICAgcmFpc2UuCgogICAgSGVyZSB0aGUgc2hhcmQgaXMgd2Fsa2VkIGluIG1pY3JvLWJhdGNoZXMsIGVhY2ggcmVzdWx0IGNvcGllZCBpbnRvIGEKICAgIHByZS1hbGxvY2F0ZWQgb3V0cHV0IGFycmF5IGFuZCB0aGVuIHJlbGVhc2VkLCBzbyBwZWFrIG1lbW9yeSBpcyBzZXQgYnkKICAgIGBiYXRjaF9zaXplYCByYXRoZXIgdGhhbiBieSBzaGFyZCBzaXplLgoKICAgIGBudW1fdGhyZWFkc2AgbWF0dGVycyBtb3JlIHRoYW4gaXQgbG9va3M6IHRvcmNoIGRlZmF1bHRzIHRvIG9uZSB0aHJlYWQgcGVyCiAgICBjb3JlIGFuZCBlYWNoIGNhcnJpZXMgaXRzIG93biB3b3Jrc3BhY2UsIHNvIGFuIHVuYm91bmRlZCB0aHJlYWQgY291bnQKICAgIG11bHRpcGxpZXMgcGVhayBSQU0gb24gYSBzbWFsbCBjb250YWluZXIuIDIgaXMgYSBzYW5lIGRlZmF1bHQgZm9yIENvbGFiLgoKICAgIEVtYmVkZGluZ3MgYXJlIGJ5dGUtaWRlbnRpY2FsIHRvIHRoZSBwcmV2aW91cyBpbXBsZW1lbnRhdGlvbiBmb3IgdGhlIHNhbWUKICAgIG1vZGVsIGFuZCBtYXhfc2VxX2xlbmd0aCAtLSBtaWNyby1iYXRjaGluZyBjaGFuZ2VzIG9ubHkgd2hlbiBtZW1vcnkgaXMKICAgIGZyZWVkLCBub3Qgd2hhdCBpcyBjb21wdXRlZC4gTm9ybWFsaXNhdGlvbiBzdGlsbCBoYXBwZW5zIGluIHRoZSBpbmRleC4KICAgICIiIgogICAgaW1wb3J0IHRvcmNoCiAgICBmcm9tIHNlbnRlbmNlX3RyYW5zZm9ybWVycyBpbXBvcnQgU2VudGVuY2VUcmFuc2Zvcm1lciAgIyBsb2NhbCBpbXBvcnQgb24gcHVycG9zZQoKICAgIGlmIG51bV90aHJlYWRzOgogICAgICAgIHRvcmNoLnNldF9udW1fdGhyZWFkcyhpbnQobnVtX3RocmVhZHMpKQogICAgICAgIHRyeToKICAgICAgICAgICAgdG9yY2guc2V0X251bV9pbnRlcm9wX3RocmVhZHMoMSkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOiAgIyBub3FhOiBCTEUwMDEgLSBvbmx5IHNldHRhYmxlIGJlZm9yZSBwYXJhbGxlbCB3b3JrIHN0YXJ0cwogICAgICAgICAgICBwYXNzCgogICAgbW9kZWwgPSBTZW50ZW5jZVRyYW5zZm9ybWVyKG1vZGVsX25hbWUsIGRldmljZT1kZXZpY2Ugb3IgImNwdSIpCiAgICBtb2RlbC5tYXhfc2VxX2xlbmd0aCA9IGludChtYXhfc2VxX2xlbmd0aCkKICAgIG1vZGVsLmV2YWwoKQoKICAgIGRpbSA9IGludChtb2RlbC5nZXRfc2VudGVuY2VfZW1iZWRkaW5nX2RpbWVuc2lvbigpKQogICAgYnMgPSBtYXgoMSwgaW50KGJhdGNoX3NpemUpKQoKICAgIGRlZiBfZW5jb2RlKHRleHRzOiBTZXF1ZW5jZVtzdHJdKSAtPiBucC5uZGFycmF5OgogICAgICAgIGl0ZW1zID0gbGlzdCh0ZXh0cykKICAgICAgICBvdXQgPSBucC5lbXB0eSgobGVuKGl0ZW1zKSwgZGltKSwgZHR5cGU9bnAuZmxvYXQzMikgICAjIGFsbG9jYXRlZCBvbmNlCiAgICAgICAgbl9iYXRjaGVzID0gKGxlbihpdGVtcykgKyBicyAtIDEpIC8vIGJzCiAgICAgICAgd2l0aCB0b3JjaC5pbmZlcmVuY2VfbW9kZSgpOiAgICAgICAgICAgICAgICAgICAgICAgICAgIyBubyBhdXRvZ3JhZCBncmFwaAogICAgICAgICAgICBmb3IgYiBpbiByYW5nZShuX2JhdGNoZXMpOgogICAgICAgICAgICAgICAgbG8gPSBiICogYnMKICAgICAgICAgICAgICAgIGhpID0gbWluKGxvICsgYnMsIGxlbihpdGVtcykpCiAgICAgICAgICAgICAgICB2ZWNzID0gbW9kZWwuZW5jb2RlKGl0ZW1zW2xvOmhpXSwgYmF0Y2hfc2l6ZT1icywgY29udmVydF90b19udW1weT1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBub3JtYWxpemVfZW1iZWRkaW5ncz1GYWxzZSwgc2hvd19wcm9ncmVzc19iYXI9RmFsc2UpCiAgICAgICAgICAgICAgICBvdXRbbG86aGldID0gbnAuYXNhcnJheSh2ZWNzLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgICAgICAgICAgZGVsIHZlY3MKICAgICAgICAgICAgICAgIGlmIGdjX2V2ZXJ5X2JhdGNoZXMgYW5kIChiICsgMSkgJSBnY19ldmVyeV9iYXRjaGVzID09IDA6CiAgICAgICAgICAgICAgICAgICAgZ2MuY29sbGVjdCgpCiAgICAgICAgICAgICAgICAgICAgaWYgbG9nIGFuZCAoYiArIDEpICUgKGdjX2V2ZXJ5X2JhdGNoZXMgKiA0KSA9PSAwOgogICAgICAgICAgICAgICAgICAgICAgICBsb2coZiIgICAgYmF0Y2gge2IgKyAxfS97bl9iYXRjaGVzfSAgUkFNIHthdmFpbGFibGVfbWIoKTouMGZ9IE1CIikKICAgICAgICBnYy5jb2xsZWN0KCkKICAgICAgICByZXR1cm4gb3V0CgogICAgcmV0dXJuIF9lbmNvZGUK", "sha256": "baf0c9032cd3fdbd548759f768ba041733f035381ae8e4d8c2bbf5c7316cc422", "size": 14745}, "src/retrieval/faiss_index.py": {"b64": "IiIiTW9kdWxlIDYg4oCUIEZBSVNTIGZsYXQgaW5uZXItcHJvZHVjdCBpbmRleCBvdmVyIG5vcm1hbGlzZWQgZW1iZWRkaW5ncy4KCkluZGV4RmxhdElQIG9uIEwyLW5vcm1hbGlzZWQgdmVjdG9ycyBJUyBleGFjdCBjb3NpbmUgc2ltaWxhcml0eSwgYW5kICJmbGF0IgptZWFucyBleGhhdXN0aXZlOiBubyBhcHByb3hpbWF0aW9uLCBubyByZWNhbGwgbG9zcywgbm8gdHJhaW5pbmcgc3RlcCwgYW5kCnRoZXJlZm9yZSBub3RoaW5nIHN0b2NoYXN0aWMgdG8gcmVwcm9kdWNlLiBBdCB0aGlzIGNvcnB1cyBzaXplICg2NywzODggRkVWRVIKYW5kIDYsOTY4IFgtRkFDVCBldmlkZW5jZSBpdGVtcykgYW4gYXBwcm94aW1hdGUgaW5kZXggd291bGQgYnV5IGxpdHRsZSBhbmQKY29zdCBleGFjdG5lc3MsIHNvIHRoZSBmbGF0IGluZGV4IGlzIHRoZSByaWdodCB0cmFkZSBhbmQga2VlcHMgdGhlIHJldHJpZXZhbApudW1iZXJzIGF1ZGl0YWJsZS4KCkRldGVybWluaXNtOiB2ZWN0b3JzIGFyZSBpbmRleGVkIGluIGEgZml4ZWQsIGNhbGxlci1zdXBwbGllZCBpZCBvcmRlciwgYW5kCnRpZXMgaW4gdGhlIHNjb3JlIGFyZSBicm9rZW4gYnkgdGhhdCBvcmRlci4gYHNlYXJjaCgpYCBpcyB0aGVyZWZvcmUKcmVwcm9kdWNpYmxlIGFjcm9zcyBydW5zIGFuZCBtYWNoaW5lcy4KCkEgcHVyZS1OdW1QeSBmYWxsYmFjayBpcyB1c2VkIHdoZW4gZmFpc3MgaXMgdW5hdmFpbGFibGUsIHNvIHRlc3RzIGFuZCBDUFUKc3RhZ2VzIHJ1biBhbnl3aGVyZS4gYHRlc3RzL3Rlc3RfcmV0cmlldmFsLnB5YCBhc3NlcnRzIHRoZSB0d28gYWdyZWUuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIFNlcXVlbmNlCgppbXBvcnQgbnVtcHkgYXMgbnAKCnRyeTogICMgcHJhZ21hOiBubyBjb3ZlciAtIGF2YWlsYWJpbGl0eSBkaWZmZXJzIGJ5IGVudmlyb25tZW50CiAgICBpbXBvcnQgZmFpc3MgICMgdHlwZTogaWdub3JlCiAgICBfSEFTX0ZBSVNTID0gVHJ1ZQpleGNlcHQgRXhjZXB0aW9uOiAgIyBub3FhOiBCTEUwMDEKICAgIGZhaXNzID0gTm9uZSAgIyB0eXBlOiBpZ25vcmUKICAgIF9IQVNfRkFJU1MgPSBGYWxzZQoKCmRlZiBmYWlzc19hdmFpbGFibGUoKSAtPiBib29sOgogICAgcmV0dXJuIF9IQVNfRkFJU1MKCgpkZWYgbDJfbm9ybWFsaXplKHg6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJSb3ctd2lzZSBMMiBub3JtYWxpc2F0aW9uLiBaZXJvIHJvd3Mgc3RheSB6ZXJvIHJhdGhlciB0aGFuIGJlY29taW5nIE5hTi4iIiIKICAgIHggPSBucC5hc2FycmF5KHgsIGR0eXBlPW5wLmZsb2F0MzIpCiAgICBub3JtcyA9IG5wLmxpbmFsZy5ub3JtKHgsIGF4aXM9MSwga2VlcGRpbXM9VHJ1ZSkKICAgIHJldHVybiB4IC8gbnAubWF4aW11bShub3JtcywgMWUtMTIpCgoKY2xhc3MgRXZpZGVuY2VJbmRleDoKICAgICIiIkV4YWN0IGlubmVyLXByb2R1Y3QgaW5kZXggd2l0aCBhbiBleHBsaWNpdCBpZCBvcmRlcmluZy4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgZXZpZGVuY2VfaWRzOiBTZXF1ZW5jZVtzdHJdLCB2ZWN0b3JzOiBucC5uZGFycmF5LAogICAgICAgICAgICAgICAgIG5vcm1hbGl6ZTogYm9vbCA9IFRydWUsIHVzZV9mYWlzczogYm9vbCB8IE5vbmUgPSBOb25lLAogICAgICAgICAgICAgICAgIHZlY3RvcnNfcHJlbm9ybWFsaXplZDogYm9vbCA9IEZhbHNlKSAtPiBOb25lOgogICAgICAgICIiImBub3JtYWxpemVgIGdvdmVybnMgQk9USCBzdG9yZWQgdmVjdG9ycyBhbmQgaW5jb21pbmcgcXVlcmllcy4KCiAgICAgICAgYHZlY3RvcnNfcHJlbm9ybWFsaXplZGAgc2F5cyB0aGUgdmVjdG9ycyBhcmUgYWxyZWFkeSB1bml0LWxlbmd0aCAoYXMKICAgICAgICB3aGVuIHJlbG9hZGluZyBmcm9tIGRpc2spIHNvIHRoZXkgYXJlIG5vdCBub3JtYWxpc2VkIHR3aWNlIC0tIGJ1dAogICAgICAgIHF1ZXJpZXMgc3RpbGwgYXJlLiBDb25mbGF0aW5nIHRoZXNlIHR3byBpcyBhIHN1YnRsZSBidWc6IGFuIGluZGV4CiAgICAgICAgcmVsb2FkZWQgd2l0aCBgbm9ybWFsaXplPUZhbHNlYCB3b3VsZCBzdG9wIG5vcm1hbGlzaW5nIHF1ZXJpZXMgYW5kCiAgICAgICAgc2lsZW50bHkgcmV0dXJuIGlubmVyIHByb2R1Y3RzIGluc3RlYWQgb2YgY29zaW5lcy4KICAgICAgICAiIiIKICAgICAgICBpZiBsZW4oZXZpZGVuY2VfaWRzKSAhPSB2ZWN0b3JzLnNoYXBlWzBdOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYie2xlbihldmlkZW5jZV9pZHMpfSBpZHMgYnV0IHt2ZWN0b3JzLnNoYXBlWzBdfSB2ZWN0b3JzIikKICAgICAgICBpZiBsZW4oc2V0KGV2aWRlbmNlX2lkcykpICE9IGxlbihldmlkZW5jZV9pZHMpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJkdXBsaWNhdGUgZXZpZGVuY2UgaWRzOiBpbmRleCBvcmRlciB3b3VsZCBiZSBhbWJpZ3VvdXMiKQogICAgICAgIHNlbGYuZXZpZGVuY2VfaWRzID0gbGlzdChldmlkZW5jZV9pZHMpCiAgICAgICAgc2VsZi5kaW0gPSBpbnQodmVjdG9ycy5zaGFwZVsxXSkKICAgICAgICBzZWxmLm5vcm1hbGl6ZWQgPSBub3JtYWxpemUKICAgICAgICB2ZWNzID0gKG5wLmFzYXJyYXkodmVjdG9ycywgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICAgICAgICAgIGlmICh2ZWN0b3JzX3ByZW5vcm1hbGl6ZWQgb3Igbm90IG5vcm1hbGl6ZSkgZWxzZSBsMl9ub3JtYWxpemUodmVjdG9ycykpCiAgICAgICAgc2VsZi52ZWN0b3JzID0gbnAuYXNjb250aWd1b3VzYXJyYXkodmVjcywgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICBzZWxmLnVzZV9mYWlzcyA9IF9IQVNfRkFJU1MgaWYgdXNlX2ZhaXNzIGlzIE5vbmUgZWxzZSAodXNlX2ZhaXNzIGFuZCBfSEFTX0ZBSVNTKQogICAgICAgIHNlbGYuX2luZGV4ID0gTm9uZQogICAgICAgIGlmIHNlbGYudXNlX2ZhaXNzOgogICAgICAgICAgICBzZWxmLl9pbmRleCA9IGZhaXNzLkluZGV4RmxhdElQKHNlbGYuZGltKQogICAgICAgICAgICBzZWxmLl9pbmRleC5hZGQoc2VsZi52ZWN0b3JzKQoKICAgIGRlZiBfX2xlbl9fKHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gbGVuKHNlbGYuZXZpZGVuY2VfaWRzKQoKICAgIGRlZiBzZWFyY2goc2VsZiwgcXVlcmllczogbnAubmRhcnJheSwgdG9wX2s6IGludCkgLT4gdHVwbGVbbGlzdFtsaXN0W3N0cl1dLCBucC5uZGFycmF5XToKICAgICAgICAiIiJUb3AtayBldmlkZW5jZSBpZHMgYW5kIHNjb3JlcyBwZXIgcXVlcnkgcm93LgoKICAgICAgICBgdG9wX2tgIGlzIGNsYW1wZWQgdG8gdGhlIGluZGV4IHNpemUgc28gYSBzbWFsbCBzY29wZSBjYW5ub3QgcmFpc2UuCiAgICAgICAgIiIiCiAgICAgICAgcSA9IGwyX25vcm1hbGl6ZShxdWVyaWVzKSBpZiBzZWxmLm5vcm1hbGl6ZWQgZWxzZSBucC5hc2FycmF5KHF1ZXJpZXMsIGR0eXBlPW5wLmZsb2F0MzIpCiAgICAgICAgcSA9IG5wLmFzY29udGlndW91c2FycmF5KHEsIGR0eXBlPW5wLmZsb2F0MzIpCiAgICAgICAgayA9IG1pbihpbnQodG9wX2spLCBsZW4oc2VsZi5ldmlkZW5jZV9pZHMpKQogICAgICAgIGlmIGsgPT0gMDoKICAgICAgICAgICAgcmV0dXJuIFtbXSBmb3IgXyBpbiByYW5nZShxLnNoYXBlWzBdKV0sIG5wLnplcm9zKChxLnNoYXBlWzBdLCAwKSwgZHR5cGU9bnAuZmxvYXQzMikKCiAgICAgICAgaWYgc2VsZi51c2VfZmFpc3MgYW5kIHNlbGYuX2luZGV4IGlzIG5vdCBOb25lOgogICAgICAgICAgICBzY29yZXMsIGlkeCA9IHNlbGYuX2luZGV4LnNlYXJjaChxLCBrKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNpbXMgPSBxIEAgc2VsZi52ZWN0b3JzLlQKICAgICAgICAgICAgIyBhcmdzb3J0IG9uIC1zaW1zIGlzIHN0YWJsZSwgc28gZXF1YWwgc2NvcmVzIGtlZXAgaW5kZXggb3JkZXIuCiAgICAgICAgICAgIGlkeCA9IG5wLmFyZ3NvcnQoLXNpbXMsIGF4aXM9MSwga2luZD0ic3RhYmxlIilbOiwgOmtdCiAgICAgICAgICAgIHNjb3JlcyA9IG5wLnRha2VfYWxvbmdfYXhpcyhzaW1zLCBpZHgsIGF4aXM9MSkKCiAgICAgICAgaWRzID0gW1tzZWxmLmV2aWRlbmNlX2lkc1tqXSBmb3IgaiBpbiByb3cgaWYgaiA+PSAwXSBmb3Igcm93IGluIGlkeF0KICAgICAgICByZXR1cm4gaWRzLCBucC5hc2FycmF5KHNjb3JlcywgZHR5cGU9bnAuZmxvYXQzMikKCiAgICBkZWYgc2F2ZShzZWxmLCBvdXRfZGlyOiBQYXRoKSAtPiBkaWN0W3N0ciwgc3RyXToKICAgICAgICBvdXRfZGlyID0gUGF0aChvdXRfZGlyKQogICAgICAgIG91dF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIHZlY190bXAgPSBvdXRfZGlyIC8gInZlY3RvcnMudG1wLm5weSIKICAgICAgICBucC5zYXZlKHZlY190bXAsIHNlbGYudmVjdG9ycykKICAgICAgICB2ZWNfdG1wLnJlcGxhY2Uob3V0X2RpciAvICJ2ZWN0b3JzLm5weSIpCiAgICAgICAgbWV0YV90bXAgPSBvdXRfZGlyIC8gImluZGV4X21ldGEudG1wLmpzb24iCiAgICAgICAgbWV0YV90bXAud3JpdGVfdGV4dChqc29uLmR1bXBzKHsKICAgICAgICAgICAgImV2aWRlbmNlX2lkcyI6IHNlbGYuZXZpZGVuY2VfaWRzLCAiZGltIjogc2VsZi5kaW0sCiAgICAgICAgICAgICJub3JtYWxpemVkIjogc2VsZi5ub3JtYWxpemVkLCAibiI6IGxlbihzZWxmLmV2aWRlbmNlX2lkcyksCiAgICAgICAgICAgICJiYWNrZW5kIjogImZhaXNzIiBpZiBzZWxmLnVzZV9mYWlzcyBlbHNlICJudW1weSIsCiAgICAgICAgfSwgZW5zdXJlX2FzY2lpPUZhbHNlKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgICAgICBtZXRhX3RtcC5yZXBsYWNlKG91dF9kaXIgLyAiaW5kZXhfbWV0YS5qc29uIikKICAgICAgICByZXR1cm4geyJ2ZWN0b3JzIjogc3RyKG91dF9kaXIgLyAidmVjdG9ycy5ucHkiKSwKICAgICAgICAgICAgICAgICJtZXRhIjogc3RyKG91dF9kaXIgLyAiaW5kZXhfbWV0YS5qc29uIil9CgogICAgQGNsYXNzbWV0aG9kCiAgICBkZWYgbG9hZChjbHMsIG91dF9kaXI6IFBhdGgsIHVzZV9mYWlzczogYm9vbCB8IE5vbmUgPSBOb25lKSAtPiAiRXZpZGVuY2VJbmRleCI6CiAgICAgICAgb3V0X2RpciA9IFBhdGgob3V0X2RpcikKICAgICAgICBtZXRhID0ganNvbi5sb2Fkcygob3V0X2RpciAvICJpbmRleF9tZXRhLmpzb24iKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgdmVjdG9ycyA9IG5wLmxvYWQob3V0X2RpciAvICJ2ZWN0b3JzLm5weSIpCiAgICAgICAgcmV0dXJuIGNscyhtZXRhWyJldmlkZW5jZV9pZHMiXSwgdmVjdG9ycywKICAgICAgICAgICAgICAgICAgIG5vcm1hbGl6ZT1ib29sKG1ldGEuZ2V0KCJub3JtYWxpemVkIiwgVHJ1ZSkpLAogICAgICAgICAgICAgICAgICAgdmVjdG9yc19wcmVub3JtYWxpemVkPVRydWUsICAgIyBzdG9yZWQgdmVjdG9ycyBhbHJlYWR5IHVuaXQtbGVuZ3RoCiAgICAgICAgICAgICAgICAgICB1c2VfZmFpc3M9dXNlX2ZhaXNzKQoKCmRlZiBicnV0ZV9mb3JjZV9zZWFyY2gocXVlcmllczogbnAubmRhcnJheSwgdmVjdG9yczogbnAubmRhcnJheSwKICAgICAgICAgICAgICAgICAgICAgICBldmlkZW5jZV9pZHM6IFNlcXVlbmNlW3N0cl0sIHRvcF9rOiBpbnQKICAgICAgICAgICAgICAgICAgICAgICApIC0+IHR1cGxlW2xpc3RbbGlzdFtzdHJdXSwgbnAubmRhcnJheV06CiAgICAiIiJSZWZlcmVuY2UgaW1wbGVtZW50YXRpb24gdXNlZCBvbmx5IGJ5IHRlc3RzIHRvIHByb3ZlIHRoZSBpbmRleCBleGFjdC4iIiIKICAgIHEsIHYgPSBsMl9ub3JtYWxpemUocXVlcmllcyksIGwyX25vcm1hbGl6ZSh2ZWN0b3JzKQogICAgc2ltcyA9IHEgQCB2LlQKICAgIGsgPSBtaW4odG9wX2ssIGxlbihldmlkZW5jZV9pZHMpKQogICAgaWR4ID0gbnAuYXJnc29ydCgtc2ltcywgYXhpcz0xLCBraW5kPSJzdGFibGUiKVs6LCA6a10KICAgIHNjb3JlcyA9IG5wLnRha2VfYWxvbmdfYXhpcyhzaW1zLCBpZHgsIGF4aXM9MSkKICAgIHJldHVybiBbW2V2aWRlbmNlX2lkc1tqXSBmb3IgaiBpbiByb3ddIGZvciByb3cgaW4gaWR4XSwgc2NvcmVzCg==", "sha256": "f62a796985541622cccbb787d1caf113a8509cb8c7ec2ee7981c53cc11751fe6", "size": 6034}, "src/retrieval/bm25_baseline.py": {"b64": "IiIiTW9kdWxlIDYg4oCUIEJNMjUgbGV4aWNhbCByZXRyaWV2YWwgYmFzZWxpbmUuCgpXaHkgYSBsZXhpY2FsIGJhc2VsaW5lIG1hdHRlcnMgaGVyZTogaWYgQk0yNSBtYXRjaGVzIG9yIGJlYXRzIExhQlNFLCB0aGUKZGVuc2UgcmV0cmlldmVyIGlzIG5vdCBlYXJuaW5nIGl0cyBwbGFjZSBpbiB0aGUgcGlwZWxpbmUsIGFuZCByZXBvcnRpbmcgb25seQp0aGUgZGVuc2UgbnVtYmVycyB3b3VsZCBvdmVyc3RhdGUgdGhlIGNvbnRyaWJ1dGlvbi4gVGhpcyBpcyB0aGUgaG9uZXN0CmNvbnRyb2wgZm9yIHRoZSByZXRyaWV2YWwgc3RhZ2UuCgpUYW1pbC1zYWZlIHRva2VuaXNhdGlvbi4gYHJhbmtfYm0yNWAgYW5kIG1vc3QgQk0yNSB0dXRvcmlhbHMgc3BsaXQgb24Kd2hpdGVzcGFjZSBhZnRlciBgLmxvd2VyKClgLCB3aGljaCBpcyB3cm9uZyBmb3IgdGhpcyBjb3JwdXMgdHdpY2Ugb3ZlcjoKVGFtaWwgaGFzIG5vIGNhc2UsIGFuZCBUYW1pbCBwdW5jdHVhdGlvbi9aV05KIGhhbmRsaW5nIGRpZmZlcnMgZnJvbSBMYXRpbi4KVG9rZW5pc2F0aW9uIGhlcmUgdXNlcyBhIFVuaWNvZGUgd29yZC1jaGFyYWN0ZXIgcmVnZXgsIGxvd2VyY2FzZXMgTGF0aW4gb25seQoobWlycm9yaW5nIE1vZHVsZSAyIGFuZCBgY2xhaW1fc2VsZWN0aW9uLm5vcm1hbGl6ZV9jbGFpbWApLCBhbmQgbmV2ZXIgYXBwbGllcwpORktDIC0tIHdoaWNoIGNhbiBjb2xsYXBzZSBsaW5ndWlzdGljYWxseSBkaXN0aW5jdCBUYW1pbCBjaGFyYWN0ZXJzLgoKSW1wbGVtZW50ZWQgZGlyZWN0bHkgcmF0aGVyIHRoYW4gdmlhIGByYW5rX2JtMjVgIHNvIHRoZSBwcm9qZWN0IGdhaW5zIG5vIG5ldwpkZXBlbmRlbmN5IGFuZCB0aGUgc2NvcmluZyBpcyBhdWRpdGFibGUuIGB0ZXN0cy90ZXN0X3JldHJpZXZhbC5weWAgY2hlY2tzIHRoZQpzY29yZXMgYWdhaW5zdCBoYW5kLWNvbXB1dGVkIEJNMjUgdmFsdWVzLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBtYXRoCmltcG9ydCByZQppbXBvcnQgdW5pY29kZWRhdGEKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgQ291bnRlcgpmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBJdGVyYWJsZSwgU2VxdWVuY2UKCl9UQU1JTF9MTywgX1RBTUlMX0hJID0gIuCugCIsICLgr78iCgojIGBcd2AgYWxvbmUgaXMgV1JPTkcgZm9yIFRhbWlsLiBQeXRob24ncyBgXHdgIGZvbGxvd3Mgc3RyLmlzYWxudW0oKSwgd2hpY2ggaXMKIyBGYWxzZSBmb3IgY29tYmluaW5nIG1hcmtzIChjYXRlZ29yaWVzIE1uL01jKSAtLSBhbmQgVGFtaWwgdm93ZWwgc2lnbnMgYW5kIHRoZQojIHZpcmFtYSBhcmUgZXhhY3RseSB0aGF0LiBNZWFzdXJlZDogciJcdysiIHRva2VuaXNlcwojICLgrqTgrr7grpXgr43grpXgr4HgrqTgrrLgr40iIGFzIFsn4K6kJywn4K6VJywn4K6VJywn4K6k4K6yJ10sIHNoYXR0ZXJpbmcgdGhlIHdvcmQgYXQgZXZlcnkgbWF0cmEgYW5kCiMgc2lsZW50bHkgZHJvcHBpbmcgVSswQkJFLCBVKzBCQkYsIFUrMEJDMSwgVSswQkMyIGFuZCBVKzBCQ0QuIEJNMjUgb3ZlciBUYW1pbAojIHdvdWxkIHRoZW4gbWF0Y2ggb24gbWVhbmluZ2xlc3MgY29uc29uYW50IGZyYWdtZW50cy4KIyBUaGUgVGFtaWwgYmxvY2sgYW5kIGdlbmVyYWwgY29tYmluaW5nIGRpYWNyaXRpY3MgYXJlIHRoZXJlZm9yZSBhZGRlZAojIGV4cGxpY2l0bHkgc28gYSB3b3JkIHN1cnZpdmVzIGFzIG9uZSB0b2tlbi4KX1dPUkRfUkUgPSByZS5jb21waWxlKHIiW1x34K6ALeCvv8yALc2v4oCM4oCNXSsiLCByZS5VTklDT0RFKQoKCmRlZiB0b2tlbml6ZSh0ZXh0OiBzdHIpIC0+IGxpc3Rbc3RyXToKICAgICIiIlVuaWNvZGUgd29yZCB0b2tlbnM7IExhdGluIGxvd2VyY2FzZWQsIFRhbWlsIGxlZnQgdW50b3VjaGVkLiIiIgogICAgaWYgbm90IHRleHQ6CiAgICAgICAgcmV0dXJuIFtdCiAgICBzID0gdW5pY29kZWRhdGEubm9ybWFsaXplKCJORkMiLCB0ZXh0KSAgICAgICMgTkZDIG9ubHksIG5ldmVyIE5GS0MKICAgIG91dCA9IFtdCiAgICBmb3IgdG9rIGluIF9XT1JEX1JFLmZpbmRhbGwocyk6CiAgICAgICAgIyBMb3dlcmNhc2luZyBpcyBhIG5vLW9wIGZvciBUYW1pbCBidXQgd291bGQgYmUgd3JvbmcgdG8gc2tpcCBmb3IgTGF0aW4uCiAgICAgICAgb3V0LmFwcGVuZCgiIi5qb2luKGNoIGlmIF9UQU1JTF9MTyA8PSBjaCA8PSBfVEFNSUxfSEkgZWxzZSBjaC5sb3dlcigpIGZvciBjaCBpbiB0b2spKQogICAgcmV0dXJuIG91dAoKCmNsYXNzIEJNMjU6CiAgICAiIiJPa2FwaSBCTTI1IHdpdGggdGhlIHN0YW5kYXJkIElERiB2YXJpYW50IHVzZWQgYnkgUm9iZXJ0c29uIGV0IGFsLgoKICAgIElERiBpcyBmbG9vcmVkIGF0IHplcm8uIFdpdGhvdXQgdGhlIGZsb29yLCBhIHRlcm0gYXBwZWFyaW5nIGluIG1vcmUgdGhhbgogICAgaGFsZiB0aGUgY29ycHVzIGdldHMgYSBuZWdhdGl2ZSBJREYgYW5kIGEgZG9jdW1lbnQgY2FuIGJlICpwZW5hbGlzZWQqIGZvcgogICAgY29udGFpbmluZyBhIHF1ZXJ5IHRlcm0sIHdoaWNoIHByb2R1Y2VzIG5vbnNlbnNpY2FsIHJhbmtpbmdzIG9uIHNtYWxsCiAgICBldmlkZW5jZSBwb29scyBsaWtlIFgtRkFDVCdzIDYsOTY4IGl0ZW1zLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNvcnB1c190b2tlbnM6IFNlcXVlbmNlW1NlcXVlbmNlW3N0cl1dLCBrMTogZmxvYXQgPSAxLjUsCiAgICAgICAgICAgICAgICAgYjogZmxvYXQgPSAwLjc1KSAtPiBOb25lOgogICAgICAgIHNlbGYuazEsIHNlbGYuYiA9IGZsb2F0KGsxKSwgZmxvYXQoYikKICAgICAgICBzZWxmLmNvcnB1c190b2tlbnMgPSBbbGlzdChkKSBmb3IgZCBpbiBjb3JwdXNfdG9rZW5zXQogICAgICAgIHNlbGYubl9kb2NzID0gbGVuKHNlbGYuY29ycHVzX3Rva2VucykKICAgICAgICBzZWxmLmRvY19sZW4gPSBbbGVuKGQpIGZvciBkIGluIHNlbGYuY29ycHVzX3Rva2Vuc10KICAgICAgICBzZWxmLmF2Z19sZW4gPSAoc3VtKHNlbGYuZG9jX2xlbikgLyBzZWxmLm5fZG9jcykgaWYgc2VsZi5uX2RvY3MgZWxzZSAwLjAKICAgICAgICBzZWxmLnRmOiBsaXN0W0NvdW50ZXJdID0gW0NvdW50ZXIoZCkgZm9yIGQgaW4gc2VsZi5jb3JwdXNfdG9rZW5zXQogICAgICAgIGRmOiBDb3VudGVyID0gQ291bnRlcigpCiAgICAgICAgZm9yIHQgaW4gc2VsZi50ZjoKICAgICAgICAgICAgZGYudXBkYXRlKHQua2V5cygpKQogICAgICAgIHNlbGYuZGYgPSBkZgogICAgICAgIHNlbGYuaWRmID0gewogICAgICAgICAgICB0ZXJtOiBtYXgoMC4wLCBtYXRoLmxvZygoc2VsZi5uX2RvY3MgLSBuICsgMC41KSAvIChuICsgMC41KSArIDEuMCkpCiAgICAgICAgICAgIGZvciB0ZXJtLCBuIGluIGRmLml0ZW1zKCkKICAgICAgICB9CgogICAgZGVmIHNjb3JlKHNlbGYsIHF1ZXJ5X3Rva2VuczogU2VxdWVuY2Vbc3RyXSwgZG9jX2luZGV4OiBpbnQpIC0+IGZsb2F0OgogICAgICAgIGlmIG5vdCBzZWxmLm5fZG9jczoKICAgICAgICAgICAgcmV0dXJuIDAuMAogICAgICAgIHRmLCBkbCA9IHNlbGYudGZbZG9jX2luZGV4XSwgc2VsZi5kb2NfbGVuW2RvY19pbmRleF0KICAgICAgICBkZW5vbV9ub3JtID0gc2VsZi5rMSAqICgxIC0gc2VsZi5iICsgc2VsZi5iICogKGRsIC8gc2VsZi5hdmdfbGVuIGlmIHNlbGYuYXZnX2xlbiBlbHNlIDAuMCkpCiAgICAgICAgdG90YWwgPSAwLjAKICAgICAgICBmb3IgdGVybSBpbiBxdWVyeV90b2tlbnM6CiAgICAgICAgICAgIGYgPSB0Zi5nZXQodGVybSwgMCkKICAgICAgICAgICAgaWYgbm90IGY6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICB0b3RhbCArPSBzZWxmLmlkZi5nZXQodGVybSwgMC4wKSAqIChmICogKHNlbGYuazEgKyAxKSkgLyAoZiArIGRlbm9tX25vcm0pCiAgICAgICAgcmV0dXJuIHRvdGFsCgogICAgZGVmIHNjb3JlcyhzZWxmLCBxdWVyeV90b2tlbnM6IFNlcXVlbmNlW3N0cl0pIC0+IGxpc3RbZmxvYXRdOgogICAgICAgIHJldHVybiBbc2VsZi5zY29yZShxdWVyeV90b2tlbnMsIGkpIGZvciBpIGluIHJhbmdlKHNlbGYubl9kb2NzKV0KCgpjbGFzcyBCTTI1UmV0cmlldmVyOgogICAgIiIiQk0yNSBvdmVyIGFuIGV2aWRlbmNlIHBvb2wgcmVzdHJpY3RlZCB0byBhIHBoYXNlJ3MgcGVybWl0dGVkIGlkcy4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgZXZpZGVuY2VfaWRzOiBTZXF1ZW5jZVtzdHJdLCB0ZXh0czogU2VxdWVuY2Vbc3RyXSwKICAgICAgICAgICAgICAgICBrMTogZmxvYXQgPSAxLjUsIGI6IGZsb2F0ID0gMC43NSkgLT4gTm9uZToKICAgICAgICBpZiBsZW4oZXZpZGVuY2VfaWRzKSAhPSBsZW4odGV4dHMpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYie2xlbihldmlkZW5jZV9pZHMpfSBpZHMgYnV0IHtsZW4odGV4dHMpfSB0ZXh0cyIpCiAgICAgICAgc2VsZi5ldmlkZW5jZV9pZHMgPSBsaXN0KGV2aWRlbmNlX2lkcykKICAgICAgICBzZWxmLmJtMjUgPSBCTTI1KFt0b2tlbml6ZSh0KSBmb3IgdCBpbiB0ZXh0c10sIGsxPWsxLCBiPWIpCgogICAgZGVmIF9fbGVuX18oc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBsZW4oc2VsZi5ldmlkZW5jZV9pZHMpCgogICAgZGVmIHJldHJpZXZlKHNlbGYsIHF1ZXJ5OiBzdHIsIHRvcF9rOiBpbnQpIC0+IHR1cGxlW2xpc3Rbc3RyXSwgbGlzdFtmbG9hdF1dOgogICAgICAgICIiIlRvcC1rIGlkcywgdGllcyBicm9rZW4gYnkgaW5kZXggb3JkZXIgc28gcmVzdWx0cyBhcmUgZGV0ZXJtaW5pc3RpYy4iIiIKICAgICAgICBzY29yZXMgPSBzZWxmLmJtMjUuc2NvcmVzKHRva2VuaXplKHF1ZXJ5KSkKICAgICAgICBrID0gbWluKGludCh0b3BfayksIGxlbihzZWxmLmV2aWRlbmNlX2lkcykpCiAgICAgICAgb3JkZXIgPSBzb3J0ZWQocmFuZ2UobGVuKHNjb3JlcykpLCBrZXk9bGFtYmRhIGk6ICgtc2NvcmVzW2ldLCBpKSlbOmtdCiAgICAgICAgcmV0dXJuIFtzZWxmLmV2aWRlbmNlX2lkc1tpXSBmb3IgaSBpbiBvcmRlcl0sIFtzY29yZXNbaV0gZm9yIGkgaW4gb3JkZXJdCgogICAgZGVmIHJldHJpZXZlX21hbnkoc2VsZiwgcXVlcmllczogSXRlcmFibGVbc3RyXSwgdG9wX2s6IGludAogICAgICAgICAgICAgICAgICAgICAgKSAtPiBsaXN0W3R1cGxlW2xpc3Rbc3RyXSwgbGlzdFtmbG9hdF1dXToKICAgICAgICByZXR1cm4gW3NlbGYucmV0cmlldmUocSwgdG9wX2spIGZvciBxIGluIHF1ZXJpZXNdCg==", "sha256": "eeb2e66f12a051ff670bd4cc255ae9efd4e1ca888cdbaf77764ab6fecf80c79b", "size": 5170}, "src/retrieval/index_scope.py": {"b64": "IiIiTW9kdWxlIDYg4oCUIHRyYW5zbGF0ZSBNb2R1bGUgNSdzIGNsYWltLWxldmVsIGluZGV4IHNjb3BlIGludG8gZXZpZGVuY2UgaWRzLgoKVGhpcyBpcyB0aGUgc2luZ2xlIG1vc3QgZGFuZ2Vyb3VzIGZpbGUgaW4gTW9kdWxlIDYuIE1vZHVsZSA1J3MKYGJ1aWxkX2luZGV4X3Njb3BlKClgIGFuc3dlcnMgIndoaWNoIENMQUlNUyBtYXkgYSBnaXZlbiBwaGFzZSBzZWU/IiwgYnV0IGEKcmV0cmlldmFsIGluZGV4IHN0b3JlcyBFVklERU5DRS4gVGhlIHRyYW5zbGF0aW9uIGJldHdlZW4gdGhlIHR3byBpcyB3aGVyZSBhCmxlYWthZ2UgYnVnIHdvdWxkIGhpZGUsIGJlY2F1c2Ugbm90aGluZyBkb3duc3RyZWFtIHdvdWxkIGxvb2sgd3Jvbmc6IHRoZQppbmRleCB3b3VsZCBzaW1wbHkgYmUgYSBsaXR0bGUgdG9vIHJpY2gsIGFuZCBldmVyeSByZXRyaWV2YWwgbnVtYmVyIHdvdWxkIGJlCm9wdGltaXN0aWMuCgpUaGUgcnVsZSBpbXBsZW1lbnRlZCBoZXJlOgoKICAgIHBlcm1pdHRlZCBldmlkZW5jZSBmb3IgcGhhc2UgUAogICAgICAgID0gdW5pb24gb2YgdGhlIGV2aWRlbmNlIGNpdGVkIGJ5IHRoZSBjbGFpbXMgcGVybWl0dGVkIGluIHBoYXNlIFAKCkV2aWRlbmNlIGNpdGVkIE9OTFkgYnkgb3V0LW9mLXNjb3BlIGNsYWltcyBpcyBleGNsdWRlZC4gRXZpZGVuY2UgY2l0ZWQgYnkKYm90aCBhbiBpbi1zY29wZSBhbmQgYW4gb3V0LW9mLXNjb3BlIGNsYWltIGlzIElOQ0xVREVEIC0tIGl0IGlzIGxlZ2l0aW1hdGVseQpyZWFjaGFibGUgZnJvbSB0aGUgaW4tc2NvcGUgc2lkZSwgYW5kIGV4Y2x1ZGluZyBpdCB3b3VsZCBkaXN0b3J0IHRoZSBjb3JwdXMuClRoYXQgYXN5bW1ldHJ5IGlzIGRlbGliZXJhdGUgYW5kIGlzIGFzc2VydGVkIGJ5IHRlc3RzLgoKTW9kdWxlIDUgaXMgcmVhZC1vbmx5IGhlcmU6IGBidWlsZF9pbmRleF9zY29wZSgpYCBpcyBpbXBvcnRlZCBhbmQgdXNlZAphcy1pcywgbmV2ZXIgcmVpbXBsZW1lbnRlZCwgc28gdGhlIHR3byBjYW4gbmV2ZXIgZHJpZnQuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBJdGVyYWJsZQoKZnJvbSBzcmMuZmFjdF92ZXJpZmljYXRpb24ubGVha2FnZV9nYXRlIGltcG9ydCBidWlsZF9pbmRleF9zY29wZQoKCkBkYXRhY2xhc3MKY2xhc3MgRXZpZGVuY2VTY29wZToKICAgICIiIldoaWNoIGV2aWRlbmNlIGEgcGhhc2UgbWF5IGluZGV4LCBhbmQgdGhlIGF1ZGl0IHRyYWlsIGZvciB3aHkuIiIiCgogICAgcGhhc2U6IHN0cgogICAgYWxsb3dlZF9zcGxpdHM6IGxpc3Rbc3RyXQogICAgY2xhaW1faW5kaWNlczogbGlzdFtpbnRdCiAgICBldmlkZW5jZV9pZHM6IHNldFtzdHJdCiAgICBuX2NsYWltc19pbl9zY29wZTogaW50ID0gMAogICAgbl9jbGFpbXNfb3V0X29mX3Njb3BlOiBpbnQgPSAwCiAgICBuX2V2aWRlbmNlX2luX3Njb3BlOiBpbnQgPSAwCiAgICBuX2V2aWRlbmNlX2V4Y2x1ZGVkOiBpbnQgPSAwCiAgICBuX2V2aWRlbmNlX3NoYXJlZF93aXRoX2V4Y2x1ZGVkX2NsYWltczogaW50ID0gMAogICAgc3RhdHM6IGRpY3Rbc3RyLCBBbnldID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWRpY3QpCgogICAgZGVmIGNvbnRhaW5zKHNlbGYsIGV2aWRlbmNlX2lkOiBzdHIpIC0+IGJvb2w6CiAgICAgICAgcmV0dXJuIGV2aWRlbmNlX2lkIGluIHNlbGYuZXZpZGVuY2VfaWRzCgoKZGVmIGNsYWltX2V2aWRlbmNlX2lkcyhjbGFpbTogZGljdFtzdHIsIEFueV0pIC0+IGxpc3Rbc3RyXToKICAgICIiIkV2aWRlbmNlIGlkcyBjaXRlZCBieSBvbmUgY2xhaW0gcmVjb3JkLCBkZWZlbnNpdmVseS4KCiAgICBNb2R1bGUgNSB3cml0ZXMgYGV2aWRlbmNlYCBhcyBhIGxpc3Qgb2YgZXZpZGVuY2UtaWQgc3RyaW5ncy4gQSBjbGFpbSB3aXRoCiAgICBubyBnb2xkIGV2aWRlbmNlIChGRVZFUidzIE5PVCBFTk9VR0ggSU5GTykgeWllbGRzIGFuIGVtcHR5IGxpc3QsIHdoaWNoIGlzCiAgICBjb3JyZWN0IGFuZCBub3QgYW4gZXJyb3IuCiAgICAiIiIKICAgIGV2ID0gY2xhaW0uZ2V0KCJldmlkZW5jZSIpCiAgICBpZiBub3QgZXY6CiAgICAgICAgcmV0dXJuIFtdCiAgICBpZiBpc2luc3RhbmNlKGV2LCBzdHIpOgogICAgICAgIHJldHVybiBbZXZdCiAgICByZXR1cm4gW3N0cihlKSBmb3IgZSBpbiBldiBpZiBlXQoKCmRlZiBidWlsZF9ldmlkZW5jZV9zY29wZSgKICAgIGNsYWltczogbGlzdFtkaWN0W3N0ciwgQW55XV0sCiAgICBzcGxpdHM6IGRpY3Rbc3RyLCBsaXN0W2ludF1dLAogICAgcGhhc2U6IHN0ciwKICAgIHNjb3BlX2NvbmZpZzogZGljdFtzdHIsIGxpc3Rbc3RyXV0sCikgLT4gRXZpZGVuY2VTY29wZToKICAgICIiIkV2aWRlbmNlIHBlcm1pdHRlZCBmb3IgYHBoYXNlYCwgZGVyaXZlZCBmcm9tIE1vZHVsZSA1J3MgY2xhaW0gc2NvcGUuCgogICAgUmFpc2VzIG9uIGFuIHVua25vd24gcGhhc2Ugb3IgYW4gdW5rbm93biBzcGxpdCBuYW1lIC0tIGRlbGVnYXRlZCB0bwogICAgYGJ1aWxkX2luZGV4X3Njb3BlKClgLCBzbyB0aGUgZXJyb3IgdGV4dCBzdGF5cyBjb25zaXN0ZW50IHdpdGggTW9kdWxlIDUuCiAgICAiIiIKICAgIGNsYWltX2lkeCA9IGJ1aWxkX2luZGV4X3Njb3BlKHNwbGl0cywgcGhhc2UsIHNjb3BlX2NvbmZpZykKICAgIGluX3Njb3BlID0gc2V0KGNsYWltX2lkeCkKCiAgICBwZXJtaXR0ZWQ6IHNldFtzdHJdID0gc2V0KCkKICAgIGZvciBpIGluIGNsYWltX2lkeDoKICAgICAgICBpZiBpIDwgMCBvciBpID49IGxlbihjbGFpbXMpOgogICAgICAgICAgICByYWlzZSBJbmRleEVycm9yKGYiY2xhaW0gaW5kZXgge2l9IG91dCBvZiByYW5nZSBmb3Ige2xlbihjbGFpbXMpfSBjbGFpbXMiKQogICAgICAgIHBlcm1pdHRlZC51cGRhdGUoY2xhaW1fZXZpZGVuY2VfaWRzKGNsYWltc1tpXSkpCgogICAgb3V0X29mX3Njb3BlX29ubHk6IHNldFtzdHJdID0gc2V0KCkKICAgIHNoYXJlZCA9IDAKICAgIGZvciBpLCBjIGluIGVudW1lcmF0ZShjbGFpbXMpOgogICAgICAgIGlmIGkgaW4gaW5fc2NvcGU6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgZm9yIGVpZCBpbiBjbGFpbV9ldmlkZW5jZV9pZHMoYyk6CiAgICAgICAgICAgIGlmIGVpZCBpbiBwZXJtaXR0ZWQ6CiAgICAgICAgICAgICAgICBzaGFyZWQgKz0gMQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgb3V0X29mX3Njb3BlX29ubHkuYWRkKGVpZCkKCiAgICBzY29wZSA9IEV2aWRlbmNlU2NvcGUoCiAgICAgICAgcGhhc2U9cGhhc2UsCiAgICAgICAgYWxsb3dlZF9zcGxpdHM9bGlzdChzY29wZV9jb25maWcuZ2V0KHBoYXNlLCBbXSkpLAogICAgICAgIGNsYWltX2luZGljZXM9Y2xhaW1faWR4LAogICAgICAgIGV2aWRlbmNlX2lkcz1wZXJtaXR0ZWQsCiAgICAgICAgbl9jbGFpbXNfaW5fc2NvcGU9bGVuKGNsYWltX2lkeCksCiAgICAgICAgbl9jbGFpbXNfb3V0X29mX3Njb3BlPWxlbihjbGFpbXMpIC0gbGVuKGNsYWltX2lkeCksCiAgICAgICAgbl9ldmlkZW5jZV9pbl9zY29wZT1sZW4ocGVybWl0dGVkKSwKICAgICAgICBuX2V2aWRlbmNlX2V4Y2x1ZGVkPWxlbihvdXRfb2Zfc2NvcGVfb25seSksCiAgICAgICAgbl9ldmlkZW5jZV9zaGFyZWRfd2l0aF9leGNsdWRlZF9jbGFpbXM9c2hhcmVkLAogICAgKQogICAgc2NvcGUuc3RhdHMgPSB7CiAgICAgICAgInBoYXNlIjogcGhhc2UsCiAgICAgICAgImFsbG93ZWRfc3BsaXRzIjogc2NvcGUuYWxsb3dlZF9zcGxpdHMsCiAgICAgICAgIm5fY2xhaW1zX2luX3Njb3BlIjogc2NvcGUubl9jbGFpbXNfaW5fc2NvcGUsCiAgICAgICAgIm5fY2xhaW1zX291dF9vZl9zY29wZSI6IHNjb3BlLm5fY2xhaW1zX291dF9vZl9zY29wZSwKICAgICAgICAibl9ldmlkZW5jZV9pbl9zY29wZSI6IHNjb3BlLm5fZXZpZGVuY2VfaW5fc2NvcGUsCiAgICAgICAgIm5fZXZpZGVuY2VfZXhjbHVkZWRfYXNfb3V0X29mX3Njb3BlX29ubHkiOiBzY29wZS5uX2V2aWRlbmNlX2V4Y2x1ZGVkLAogICAgICAgICJuX2NpdGF0aW9uc19mcm9tX2V4Y2x1ZGVkX2NsYWltc190b19pbl9zY29wZV9ldmlkZW5jZSI6IHNoYXJlZCwKICAgICAgICAicnVsZSI6ICgiZXZpZGVuY2UgcGVybWl0dGVkID0gdW5pb24gb2YgZXZpZGVuY2UgY2l0ZWQgYnkgcGVybWl0dGVkIGNsYWltczsgIgogICAgICAgICAgICAgICAgICJldmlkZW5jZSBjaXRlZCBvbmx5IGJ5IG91dC1vZi1zY29wZSBjbGFpbXMgaXMgZXhjbHVkZWQiKSwKICAgIH0KICAgIHJldHVybiBzY29wZQoKCmRlZiBhc3NlcnRfaW5kZXhfd2l0aGluX3Njb3BlKGluZGV4ZWRfaWRzOiBJdGVyYWJsZVtzdHJdLCBzY29wZTogRXZpZGVuY2VTY29wZSkgLT4gTm9uZToKICAgICIiIkhhcmQgZ2F0ZTogZmFpbCBsb3VkbHkgaWYgdGhlIGJ1aWx0IGluZGV4IGV4Y2VlZHMgaXRzIHNjb3BlLgoKICAgIENhbGxlZCBhZnRlciBpbmRleCBjb25zdHJ1Y3Rpb24sIG5vdCBiZWZvcmUsIHNvIGl0IGF1ZGl0cyB3aGF0IHdhcwogICAgYWN0dWFsbHkgaW5kZXhlZCByYXRoZXIgdGhhbiB3aGF0IHdhcyBpbnRlbmRlZC4KICAgICIiIgogICAgZXh0cmEgPSBzb3J0ZWQoc2V0KGluZGV4ZWRfaWRzKSAtIHNjb3BlLmV2aWRlbmNlX2lkcykKICAgIGlmIGV4dHJhOgogICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKAogICAgICAgICAgICBmIkxFQUtBR0U6IHtsZW4oZXh0cmEpfSBldmlkZW5jZSBpdGVtKHMpIGluIHRoZSB7c2NvcGUucGhhc2Uhcn0gaW5kZXggYXJlICIKICAgICAgICAgICAgZiJvdXRzaWRlIGl0cyBwZXJtaXR0ZWQgc2NvcGUgKGFsbG93ZWQgc3BsaXRzIHtzY29wZS5hbGxvd2VkX3NwbGl0c30pLiAiCiAgICAgICAgICAgIGYiRmlyc3Qgb2ZmZW5kZXJzOiB7ZXh0cmFbOjVdfSIpCgoKZGVmIHZlcmlmeV9waGFzZV9pc29sYXRpb24oCiAgICBjbGFpbXM6IGxpc3RbZGljdFtzdHIsIEFueV1dLAogICAgc3BsaXRzOiBkaWN0W3N0ciwgbGlzdFtpbnRdXSwKICAgIHNjb3BlX2NvbmZpZzogZGljdFtzdHIsIGxpc3Rbc3RyXV0sCikgLT4gZGljdFtzdHIsIEFueV06CiAgICAiIiJJbmRlcGVuZGVudCByZS1jaGVjayB0aGF0IGVhY2ggcGhhc2UgZXhjbHVkZXMgd2hhdCBpdCBtdXN0IGV4Y2x1ZGUuCgogICAgUmVjb21wdXRlZCBmcm9tIHRoZSBjbGFpbXMgcmF0aGVyIHRoYW4gZnJvbSB0aGUgc2NvcGVzLCBzbyBhIGJ1ZyBpbgogICAgYGJ1aWxkX2V2aWRlbmNlX3Njb3BlKClgIGNhbm5vdCBoaWRlIGJlaGluZCBpdHMgb3duIGJvb2trZWVwaW5nLgogICAgIiIiCiAgICBjaGVja3M6IGxpc3RbZGljdFtzdHIsIEFueV1dID0gW10KCiAgICBkZWYgc3BsaXRfZXhjbHVzaXZlX2V2aWRlbmNlKHRhcmdldDogc3RyLCBvdGhlcnM6IGxpc3Rbc3RyXSkgLT4gc2V0W3N0cl06CiAgICAgICAgIiIiRXZpZGVuY2UgY2l0ZWQgYnkgYHRhcmdldGAgYW5kIGJ5IG5vIHNwbGl0IGluIGBvdGhlcnNgLiIiIgogICAgICAgIHRndDogc2V0W3N0cl0gPSBzZXQoKQogICAgICAgIGZvciBpIGluIHNwbGl0cy5nZXQodGFyZ2V0LCBbXSk6CiAgICAgICAgICAgIHRndC51cGRhdGUoY2xhaW1fZXZpZGVuY2VfaWRzKGNsYWltc1tpXSkpCiAgICAgICAgZWxzZXdoZXJlOiBzZXRbc3RyXSA9IHNldCgpCiAgICAgICAgZm9yIHMgaW4gb3RoZXJzOgogICAgICAgICAgICBmb3IgaSBpbiBzcGxpdHMuZ2V0KHMsIFtdKToKICAgICAgICAgICAgICAgIGVsc2V3aGVyZS51cGRhdGUoY2xhaW1fZXZpZGVuY2VfaWRzKGNsYWltc1tpXSkpCiAgICAgICAgcmV0dXJuIHRndCAtIGVsc2V3aGVyZQoKICAgIGRlZiBhZGQobmFtZTogc3RyLCBwYXNzZWQ6IGJvb2wsIGRldGFpbDogc3RyKSAtPiBOb25lOgogICAgICAgIGNoZWNrcy5hcHBlbmQoeyJuYW1lIjogbmFtZSwgInBhc3NlZCI6IGJvb2wocGFzc2VkKSwgImRldGFpbCI6IGRldGFpbH0pCgogICAgcHJlc2VudCA9IFtzIGZvciBzIGluICgidHJhaW4iLCAidmFsaWRhdGlvbiIsICJ0ZXN0IikgaWYgcyBpbiBzcGxpdHNdCgogICAgaWYgInRyYWluIiBpbiBzcGxpdHM6CiAgICAgICAgdHJhaW5fc2NvcGUgPSBidWlsZF9ldmlkZW5jZV9zY29wZShjbGFpbXMsIHNwbGl0cywgInRyYWluIiwgc2NvcGVfY29uZmlnKQogICAgICAgIGZvciBob2xkb3V0IGluICgidmFsaWRhdGlvbiIsICJ0ZXN0Iik6CiAgICAgICAgICAgIGlmIGhvbGRvdXQgbm90IGluIHNwbGl0czoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGV4Y2wgPSBzcGxpdF9leGNsdXNpdmVfZXZpZGVuY2UoaG9sZG91dCwgW3MgZm9yIHMgaW4gcHJlc2VudCBpZiBzICE9IGhvbGRvdXRdKQogICAgICAgICAgICBiYWQgPSBleGNsICYgdHJhaW5fc2NvcGUuZXZpZGVuY2VfaWRzCiAgICAgICAgICAgIGFkZChmInRyYWluX2luZGV4X2V4Y2x1ZGVzX3tob2xkb3V0fV9leGNsdXNpdmVfZXZpZGVuY2UiLCBub3QgYmFkLAogICAgICAgICAgICAgICAgZiJ7bGVuKGV4Y2wpfSB7aG9sZG91dH0tZXhjbHVzaXZlIGV2aWRlbmNlIGl0ZW0ocyk7IHtsZW4oYmFkKX0gbGVha2VkIGludG8gdHJhaW4iCiAgICAgICAgICAgICAgICArIChmIiBlLmcuIHtzb3J0ZWQoYmFkKVs6M119IiBpZiBiYWQgZWxzZSAiIikpCgogICAgaWYgInZhbGlkYXRpb24iIGluIHNwbGl0cyBhbmQgInRlc3QiIGluIHNwbGl0czoKICAgICAgICB2YWxfc2NvcGUgPSBidWlsZF9ldmlkZW5jZV9zY29wZShjbGFpbXMsIHNwbGl0cywgInZhbGlkYXRpb24iLCBzY29wZV9jb25maWcpCiAgICAgICAgZXhjbCA9IHNwbGl0X2V4Y2x1c2l2ZV9ldmlkZW5jZSgidGVzdCIsIFtzIGZvciBzIGluIHByZXNlbnQgaWYgcyAhPSAidGVzdCJdKQogICAgICAgIGJhZCA9IGV4Y2wgJiB2YWxfc2NvcGUuZXZpZGVuY2VfaWRzCiAgICAgICAgYWRkKCJ2YWxpZGF0aW9uX2luZGV4X2V4Y2x1ZGVzX3Rlc3RfZXhjbHVzaXZlX2V2aWRlbmNlIiwgbm90IGJhZCwKICAgICAgICAgICAgZiJ7bGVuKGV4Y2wpfSB0ZXN0LWV4Y2x1c2l2ZSBldmlkZW5jZSBpdGVtKHMpOyB7bGVuKGJhZCl9IGxlYWtlZCBpbnRvIHZhbGlkYXRpb24iCiAgICAgICAgICAgICsgKGYiIGUuZy4ge3NvcnRlZChiYWQpWzozXX0iIGlmIGJhZCBlbHNlICIiKSkKCiAgICBpZiAidGVzdCIgaW4gc3BsaXRzOgogICAgICAgIHRlc3Rfc2NvcGUgPSBidWlsZF9ldmlkZW5jZV9zY29wZShjbGFpbXMsIHNwbGl0cywgInRlc3QiLCBzY29wZV9jb25maWcpCiAgICAgICAgYWxsX2lkczogc2V0W3N0cl0gPSBzZXQoKQogICAgICAgIGZvciBjIGluIGNsYWltczoKICAgICAgICAgICAgYWxsX2lkcy51cGRhdGUoY2xhaW1fZXZpZGVuY2VfaWRzKGMpKQogICAgICAgIGFkZCgidGVzdF9pbmRleF93aXRoaW5fY29ycHVzIiwgdGVzdF9zY29wZS5ldmlkZW5jZV9pZHMgPD0gYWxsX2lkcywKICAgICAgICAgICAgZiJ7bGVuKHRlc3Rfc2NvcGUuZXZpZGVuY2VfaWRzKX0gZXZpZGVuY2UgaXRlbShzKSBpbmRleGVkIGF0IHRlc3QgcGhhc2UiKQoKICAgIHJldHVybiB7CiAgICAgICAgInBhc3NlZCI6IGFsbChjWyJwYXNzZWQiXSBmb3IgYyBpbiBjaGVja3MpLAogICAgICAgICJuX2NoZWNrcyI6IGxlbihjaGVja3MpLAogICAgICAgICJuX3Bhc3NlZCI6IHN1bSgxIGZvciBjIGluIGNoZWNrcyBpZiBjWyJwYXNzZWQiXSksCiAgICAgICAgImNoZWNrcyI6IGNoZWNrcywKICAgIH0K", "sha256": "d21cda79fc41a5c32bbe7d1e3b261156d7b208758382e677af9ca97e5b55f5cf", "size": 7653}, "src/retrieval/retrieval_metrics.py": {"b64": "IiIiTW9kdWxlIDYg4oCUIHJldHJpZXZhbCBtZXRyaWNzIGFnYWluc3QgUkVBTCBnb2xkIGV2aWRlbmNlLgoKRGVsaWJlcmF0ZWx5IHNlcGFyYXRlIGZyb20gYHNyYy9ldmFsdWF0aW9uL3JldHJpZXZhbF9tZXRyaWNzLnB5YC4gVGhhdCBmaWxlCm9wZXJhdGlvbmFsaXNlcyByZXRyaWV2YWwgYXMgYSBzYW1lLWNsYXNzIG5lYXJlc3QtbmVpZ2hib3VyIHByb3h5IGFuZCBzYXlzIHNvCmluIGl0cyBkb2NzdHJpbmc6ICJ0aGlzIHByb2plY3QncyBkYXRhIGRvZXNuJ3QgaGF2ZSBhbiBleHBsaWNpdCBjbGFpbS9ldmlkZW5jZQpyZXRyaWV2YWwgc3RydWN0dXJlIi4gTW9kdWxlIDUgaGFzIHNpbmNlIGJ1aWx0IGV4YWN0bHkgdGhhdCBzdHJ1Y3R1cmUsIHNvIHRoZQpwcm94eSBpcyBubyBsb25nZXIgdGhlIGhvbmVzdCBtZWFzdXJlbWVudC4gVGhlIG9sZCBmaWxlIGlzIGxlZnQgdW50b3VjaGVkCihNb2R1bGUgNCByZXN1bHRzIGRlcGVuZCBvbiBpdCk7IHRoaXMgb25lIGlzIHdoYXQgTW9kdWxlIDYgcmVwb3J0cy4KCkdvbGQgcmVsZXZhbmNlIGlzIGJpbmFyeSBhbmQgY29tZXMgZnJvbSBNb2R1bGUgNSdzIGBjbGFpbS5ldmlkZW5jZWAgbGlzdC4KCkNsYWltcyB3aXRoIE5PIGdvbGQgZXZpZGVuY2UgLS0gbm90YWJseSBGRVZFUidzIE5PVCBFTk9VR0ggSU5GTyBjbGFpbXMsIDQyNwpvZiB0aGVtIGluIHRoZSBmcm96ZW4gY29ycHVzIC0tIGFyZSBleGNsdWRlZCBmcm9tIGV2ZXJ5IHJlY2FsbCBkZW5vbWluYXRvci4KSW5jbHVkaW5nIHRoZW0gd291bGQgc2lsZW50bHkgZGVwcmVzcyBSZWNhbGxAayB3aXRoIGEgdGVybSBubyByZXRyaWV2ZXIgY2FuCmV2ZXIgc2F0aXNmeS4gVGhleSBhcmUgY291bnRlZCBhbmQgcmVwb3J0ZWQgaW5zdGVhZCBvZiBiZWluZyBkcm9wcGVkIHF1aWV0bHkuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG1hdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueSwgSXRlcmFibGUsIFNlcXVlbmNlCgoKZGVmIHJlY2FsbF9hdF9rKHJldHJpZXZlZDogU2VxdWVuY2Vbc3RyXSwgZ29sZDogSXRlcmFibGVbc3RyXSwgazogaW50KSAtPiBmbG9hdDoKICAgICIiIkZyYWN0aW9uIG9mIHRoaXMgY2xhaW0ncyBnb2xkIGV2aWRlbmNlIGZvdW5kIGluIHRoZSB0b3AgYGtgLgoKICAgIFBlci1jbGFpbSByZWNhbGwsIG5vdCBoaXQtcmF0ZTogYSBjbGFpbSB3aXRoIDMgZ29sZCBpdGVtcyBvZiB3aGljaCAyIGFyZQogICAgcmV0cmlldmVkIHNjb3JlcyAyLzMuIE1hY3JvLWF2ZXJhZ2luZyB0aGVzZSBnaXZlcyBlcXVhbCB3ZWlnaHQgdG8gZXZlcnkKICAgIGNsYWltIHJlZ2FyZGxlc3Mgb2YgaG93IG11Y2ggZXZpZGVuY2UgaXQgaGFwcGVucyB0byBjaXRlLgogICAgIiIiCiAgICBnb2xkX3NldCA9IHNldChnb2xkKQogICAgaWYgbm90IGdvbGRfc2V0OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInJlY2FsbF9hdF9rIGNhbGxlZCBvbiBhIGNsYWltIHdpdGggbm8gZ29sZCBldmlkZW5jZTsgIgogICAgICAgICAgICAgICAgICAgICAgICAgInN1Y2ggY2xhaW1zIG11c3QgYmUgZXhjbHVkZWQgYmVmb3JlIGF2ZXJhZ2luZyIpCiAgICBoaXRzID0gc3VtKDEgZm9yIGUgaW4gcmV0cmlldmVkWzprXSBpZiBlIGluIGdvbGRfc2V0KQogICAgcmV0dXJuIGhpdHMgLyBsZW4oZ29sZF9zZXQpCgoKZGVmIHJlY2lwcm9jYWxfcmFuayhyZXRyaWV2ZWQ6IFNlcXVlbmNlW3N0cl0sIGdvbGQ6IEl0ZXJhYmxlW3N0cl0pIC0+IGZsb2F0OgogICAgIiIiMSAvIHJhbmsgb2YgdGhlIEZJUlNUIGdvbGQgaXRlbTsgMC4wIGlmIG5vbmUgcmV0cmlldmVkLiBSYW5rcyBhcmUgMS1iYXNlZC4iIiIKICAgIGdvbGRfc2V0ID0gc2V0KGdvbGQpCiAgICBmb3IgcG9zLCBlIGluIGVudW1lcmF0ZShyZXRyaWV2ZWQsIHN0YXJ0PTEpOgogICAgICAgIGlmIGUgaW4gZ29sZF9zZXQ6CiAgICAgICAgICAgIHJldHVybiAxLjAgLyBwb3MKICAgIHJldHVybiAwLjAKCgpkZWYgZGNnKHJlbGV2YW5jZXM6IFNlcXVlbmNlW2Zsb2F0XSkgLT4gZmxvYXQ6CiAgICAiIiJEaXNjb3VudGVkIGN1bXVsYXRpdmUgZ2FpbiwgbG9nMihyYW5rKzEpIGRpc2NvdW50LCAxLWJhc2VkIHJhbmtzLiIiIgogICAgcmV0dXJuIHN1bShyZWwgLyBtYXRoLmxvZzIoaSArIDEpIGZvciBpLCByZWwgaW4gZW51bWVyYXRlKHJlbGV2YW5jZXMsIHN0YXJ0PTEpKQoKCmRlZiBuZGNnX2F0X2socmV0cmlldmVkOiBTZXF1ZW5jZVtzdHJdLCBnb2xkOiBJdGVyYWJsZVtzdHJdLCBrOiBpbnQpIC0+IGZsb2F0OgogICAgIiIibkRDR0BrIHdpdGggYmluYXJ5IHJlbGV2YW5jZS4KCiAgICBUaGUgaWRlYWwgcmFua2luZyBwbGFjZXMgbWluKHxnb2xkfCwgaykgcmVsZXZhbnQgaXRlbXMgZmlyc3QsIHNvIGEgY2xhaW0KICAgIHdpdGggbW9yZSBnb2xkIGl0ZW1zIHRoYW4gYGtgIGlzIG5vdCBwZW5hbGlzZWQgZm9yIHRoZSBpbXBvc3NpYmxlLgogICAgIiIiCiAgICBnb2xkX3NldCA9IHNldChnb2xkKQogICAgaWYgbm90IGdvbGRfc2V0OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm5kY2dfYXRfayBjYWxsZWQgb24gYSBjbGFpbSB3aXRoIG5vIGdvbGQgZXZpZGVuY2UiKQogICAgZ2FpbnMgPSBbMS4wIGlmIGUgaW4gZ29sZF9zZXQgZWxzZSAwLjAgZm9yIGUgaW4gcmV0cmlldmVkWzprXV0KICAgIGlkZWFsID0gWzEuMF0gKiBtaW4obGVuKGdvbGRfc2V0KSwgaykKICAgIGlkY2cgPSBkY2coaWRlYWwpCiAgICByZXR1cm4gZGNnKGdhaW5zKSAvIGlkY2cgaWYgaWRjZyA+IDAgZWxzZSAwLjAKCgpkZWYgZXZhbHVhdGVfcmV0cmlldmFsKAogICAgcmVzdWx0czogZGljdFtzdHIsIFNlcXVlbmNlW3N0cl1dLAogICAgZ29sZDogZGljdFtzdHIsIFNlcXVlbmNlW3N0cl1dLAogICAga192YWx1ZXM6IFNlcXVlbmNlW2ludF0gPSAoMSwgNSwgMTAsIDUwKSwKICAgIG5kY2dfazogaW50ID0gMTAsCiAgICBsYW5ndWFnZXM6IGRpY3Rbc3RyLCBzdHJdIHwgTm9uZSA9IE5vbmUsCikgLT4gZGljdFtzdHIsIEFueV06CiAgICAiIiJNYWNyby1hdmVyYWdlZCByZXRyaWV2YWwgbWV0cmljcyBvdmVyIGNsYWltcyB0aGF0IGhhdmUgZ29sZCBldmlkZW5jZS4KCiAgICBgcmVzdWx0c2AgbWFwcyBjbGFpbV9pZCAtPiByYW5rZWQgZXZpZGVuY2UgaWRzIChiZXN0IGZpcnN0KS4KICAgIGBnb2xkYCAgICBtYXBzIGNsYWltX2lkIC0+IGdvbGQgZXZpZGVuY2UgaWRzLgoKICAgIEV2ZXJ5IGNsYWltIGluIGByZXN1bHRzYCBpcyBhY2NvdW50ZWQgZm9yIGluIGV4YWN0bHkgb25lIG9mOgogICAgc2NvcmVkLCBleGNsdWRlZF9ub19nb2xkLCBvciBtaXNzaW5nX2Zyb21fZ29sZC4KICAgICIiIgogICAgc2NvcmVkOiBsaXN0W3N0cl0gPSBbXQogICAgbm9fZ29sZDogbGlzdFtzdHJdID0gW10KICAgIG1pc3NpbmdfZ29sZF9lbnRyeTogbGlzdFtzdHJdID0gW10KCiAgICBmb3IgY2lkIGluIHJlc3VsdHM6CiAgICAgICAgaWYgY2lkIG5vdCBpbiBnb2xkOgogICAgICAgICAgICBtaXNzaW5nX2dvbGRfZW50cnkuYXBwZW5kKGNpZCkKICAgICAgICBlbGlmIG5vdCBnb2xkW2NpZF06CiAgICAgICAgICAgIG5vX2dvbGQuYXBwZW5kKGNpZCkKICAgICAgICBlbHNlOgogICAgICAgICAgICBzY29yZWQuYXBwZW5kKGNpZCkKCiAgICBwZXJfY2xhaW06IGRpY3Rbc3RyLCBkaWN0W3N0ciwgZmxvYXRdXSA9IHt9CiAgICBmb3IgY2lkIGluIHNjb3JlZDoKICAgICAgICByLCBnID0gcmVzdWx0c1tjaWRdLCBnb2xkW2NpZF0KICAgICAgICBtOiBkaWN0W3N0ciwgZmxvYXRdID0ge2YicmVjYWxsQHtrfSI6IHJlY2FsbF9hdF9rKHIsIGcsIGspIGZvciBrIGluIGtfdmFsdWVzfQogICAgICAgIG1bIm1yciJdID0gcmVjaXByb2NhbF9yYW5rKHIsIGcpCiAgICAgICAgbVtmIm5kY2dAe25kY2dfa30iXSA9IG5kY2dfYXRfayhyLCBnLCBuZGNnX2spCiAgICAgICAgcGVyX2NsYWltW2NpZF0gPSBtCgogICAgZGVmIG1hY3JvKG1ldHJpYzogc3RyLCBzdWJzZXQ6IFNlcXVlbmNlW3N0cl0pIC0+IGZsb2F0OgogICAgICAgIHZhbHMgPSBbcGVyX2NsYWltW2NdW21ldHJpY10gZm9yIGMgaW4gc3Vic2V0IGlmIGMgaW4gcGVyX2NsYWltXQogICAgICAgIHJldHVybiBzdW0odmFscykgLyBsZW4odmFscykgaWYgdmFscyBlbHNlIDAuMAoKICAgIG5hbWVzID0gW2YicmVjYWxsQHtrfSIgZm9yIGsgaW4ga192YWx1ZXNdICsgWyJtcnIiLCBmIm5kY2dAe25kY2dfa30iXQogICAgb3V0OiBkaWN0W3N0ciwgQW55XSA9IHsKICAgICAgICAib3ZlcmFsbCI6IHttOiBtYWNybyhtLCBzY29yZWQpIGZvciBtIGluIG5hbWVzfSwKICAgICAgICAiYWNjb3VudGluZyI6IHsKICAgICAgICAgICAgIm5fcXVlcmllcyI6IGxlbihyZXN1bHRzKSwKICAgICAgICAgICAgIm5fc2NvcmVkIjogbGVuKHNjb3JlZCksCiAgICAgICAgICAgICJuX2V4Y2x1ZGVkX25vX2dvbGRfZXZpZGVuY2UiOiBsZW4obm9fZ29sZCksCiAgICAgICAgICAgICJuX21pc3NpbmdfZnJvbV9nb2xkX21hcCI6IGxlbihtaXNzaW5nX2dvbGRfZW50cnkpLAogICAgICAgICAgICAiYmFsYW5jZXMiOiBsZW4ocmVzdWx0cykgPT0gbGVuKHNjb3JlZCkgKyBsZW4obm9fZ29sZCkgKyBsZW4obWlzc2luZ19nb2xkX2VudHJ5KSwKICAgICAgICAgICAgIm5vdGUiOiAoIkNsYWltcyB3aXRoIG5vIGdvbGQgZXZpZGVuY2UgKGUuZy4gRkVWRVIgTk9UIEVOT1VHSCBJTkZPKSBhcmUgIgogICAgICAgICAgICAgICAgICAgICAiZXhjbHVkZWQgZnJvbSByZWNhbGwgZGVub21pbmF0b3JzIGFuZCBjb3VudGVkIGhlcmUuIFRoZXkgYXJlICIKICAgICAgICAgICAgICAgICAgICAgIm5vdCBzaWxlbnRseSBkcm9wcGVkLiIpLAogICAgICAgIH0sCiAgICAgICAgImtfdmFsdWVzIjogbGlzdChrX3ZhbHVlcyksCiAgICAgICAgIm5kY2dfayI6IG5kY2dfaywKICAgIH0KCiAgICBpZiBsYW5ndWFnZXM6CiAgICAgICAgYnlfbGFuZzogZGljdFtzdHIsIGRpY3Rbc3RyLCBmbG9hdF1dID0ge30KICAgICAgICBjb3VudHM6IGRpY3Rbc3RyLCBpbnRdID0ge30KICAgICAgICBmb3IgbGFuZyBpbiBzb3J0ZWQoc2V0KGxhbmd1YWdlcy52YWx1ZXMoKSkpOgogICAgICAgICAgICBzdWJzZXQgPSBbYyBmb3IgYyBpbiBzY29yZWQgaWYgbGFuZ3VhZ2VzLmdldChjKSA9PSBsYW5nXQogICAgICAgICAgICBjb3VudHNbbGFuZ10gPSBsZW4oc3Vic2V0KQogICAgICAgICAgICBpZiBzdWJzZXQ6CiAgICAgICAgICAgICAgICBieV9sYW5nW2xhbmddID0ge206IG1hY3JvKG0sIHN1YnNldCkgZm9yIG0gaW4gbmFtZXN9CiAgICAgICAgb3V0WyJwZXJfbGFuZ3VhZ2UiXSA9IGJ5X2xhbmcKICAgICAgICBvdXRbInBlcl9sYW5ndWFnZV9jb3VudHMiXSA9IGNvdW50cwoKICAgIG91dFsicGVyX2NsYWltIl0gPSBwZXJfY2xhaW0KICAgIHJldHVybiBvdXQK", "sha256": "d6aee73d785d521ce3f5b2549f79e925e84bfd2ab5d0cc4e2aac8ca38672c527", "size": 5373}, "src/retrieval/reranker.py": {"b64": "IiIiTW9kdWxlIDYg4oCUIFhMTS1SIGNyb3NzLWVuY29kZXIgcmVyYW5raW5nLgoKQSBiaS1lbmNvZGVyIHNjb3JlcyBjbGFpbSBhbmQgZXZpZGVuY2UgaW5kZXBlbmRlbnRseSwgc28gaXQgbmV2ZXIgc2VlcyBob3cKdGhleSBpbnRlcmFjdC4gQSBjcm9zcy1lbmNvZGVyIHJlYWRzIGBbY2xhaW07IGV2aWRlbmNlXWAgam9pbnRseSBhbmQgY2FuCmNhdGNoIG5lZ2F0aW9uLCBlbnRpdHkgc3dhcHMgYW5kIGhlZGdpbmcgLS0gZXhhY3RseSB0aGUgZGlzdGluY3Rpb25zIHRoYXQKbWF0dGVyIGZvciBmYWN0IHZlcmlmaWNhdGlvbi4gSXQgaXMgZmFyIHRvbyBzbG93IHRvIHNjb3JlIHRoZSB3aG9sZSBwb29sLApoZW5jZSB0aGUgdHdvLXN0YWdlIGRlc2lnbjogcmV0cmlldmUgNTAgY2hlYXBseSwgcmVyYW5rIHRob3NlIDUwIGNhcmVmdWxseS4KCkhhcmQgaW52YXJpYW50OiByZXJhbmtpbmcgaXMgYSBQRVJNVVRBVElPTiBvZiB0aGUgY2FuZGlkYXRlIHNldC4gSXQgbWF5CnJlb3JkZXIgYW5kIGl0IG1heSB0cnVuY2F0ZSB0byB0b3AtbiwgYnV0IGl0IG11c3QgbmV2ZXIgaW50cm9kdWNlIGFuCmV2aWRlbmNlIGlkIHRoZSByZXRyaWV2ZXIgZGlkIG5vdCBwcm9wb3NlLiBBIHZpb2xhdGlvbiB3b3VsZCBzaWxlbnRseSB0dXJuCnJldHJpZXZhbCBmYWlsdXJlcyBpbnRvIHJldHJpZXZhbCBzdWNjZXNzZXMgYW5kIGluZmxhdGUgZXZlcnkgZG93bnN0cmVhbQpudW1iZXIuIGByZXJhbmtfY2FuZGlkYXRlcygpYCBlbmZvcmNlcyBpdCBpbiBjb2RlLCBhbmQKYHRlc3RzL3Rlc3RfcmV0cmlldmFsLnB5YCBuZWdhdGl2ZS10ZXN0cyB0aGUgZ3VhcmQuCgpgc2NvcmVfZm5gIGlzIGluamVjdGVkIHNvIHRoZSB3aG9sZSBwYXRoIGlzIHRlc3RhYmxlIHdpdGhvdXQgZG93bmxvYWRpbmcKWExNLVIuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIENhbGxhYmxlLCBTZXF1ZW5jZQoKaW1wb3J0IG51bXB5IGFzIG5wCgoKY2xhc3MgQ2FuZGlkYXRlU2V0VmlvbGF0aW9uKEFzc2VydGlvbkVycm9yKToKICAgICIiIlJhaXNlZCB3aGVuIHJlcmFua2luZyByZXR1cm5zIGFuIGlkIHRoYXQgd2FzIG5vdCBhIGNhbmRpZGF0ZS4iIiIKCgpkZWYgYXNzZXJ0X3N1YnNldF9vZl9jYW5kaWRhdGVzKHJlcmFua2VkOiBTZXF1ZW5jZVtzdHJdLCBjYW5kaWRhdGVzOiBTZXF1ZW5jZVtzdHJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNsYWltX2lkOiBzdHIgPSAiIikgLT4gTm9uZToKICAgICIiIlJlcmFua2luZyBtdXN0IGJlIGEgcGVybXV0YXRpb24tYW5kLXRydW5jYXRpb24gb2YgdGhlIGNhbmRpZGF0ZSBzZXQuCgogICAgRW5mb3JjZWQgb24gdGhlIE9VVFBVVCByYXRoZXIgdGhhbiB0cnVzdGVkIGZyb20gdGhlIGltcGxlbWVudGF0aW9uLCBzbyBhCiAgICBjdXN0b20gc2NvcmVyIG9yIGEgc3ViY2xhc3MgY2Fubm90IHF1aWV0bHkgd2lkZW4gdGhlIHBvb2wuIEludmVudGluZwogICAgZXZpZGVuY2Ugd291bGQgdHVybiByZXRyaWV2YWwgbWlzc2VzIGludG8gYXBwYXJlbnQgaGl0cyBhbmQgaW5mbGF0ZSBldmVyeQogICAgZG93bnN0cmVhbSBtZXRyaWMuCiAgICAiIiIKICAgIGV4dHJhID0gc2V0KHJlcmFua2VkKSAtIHNldChjYW5kaWRhdGVzKQogICAgaWYgZXh0cmE6CiAgICAgICAgcmFpc2UgQ2FuZGlkYXRlU2V0VmlvbGF0aW9uKAogICAgICAgICAgICBmInJlcmFua2VyIGludHJvZHVjZWQge2xlbihleHRyYSl9IGV2aWRlbmNlIGlkKHMpIGFic2VudCBmcm9tIHRoZSBjYW5kaWRhdGUgIgogICAgICAgICAgICBmInNldHtmJyBmb3IgY2xhaW0ge2NsYWltX2lkfScgaWYgY2xhaW1faWQgZWxzZSAnJ306IHtzb3J0ZWQoZXh0cmEpWzo1XX0iKQogICAgaWYgbGVuKHNldChyZXJhbmtlZCkpICE9IGxlbihyZXJhbmtlZCk6CiAgICAgICAgcmFpc2UgQ2FuZGlkYXRlU2V0VmlvbGF0aW9uKAogICAgICAgICAgICBmInJlcmFua2VyIHJldHVybmVkIGR1cGxpY2F0ZSBldmlkZW5jZSBpZHMiCiAgICAgICAgICAgIGYie2YnIGZvciBjbGFpbSB7Y2xhaW1faWR9JyBpZiBjbGFpbV9pZCBlbHNlICcnfSIpCgoKZGVmIHJlcmFua19jYW5kaWRhdGVzKGNhbmRpZGF0ZXM6IFNlcXVlbmNlW3N0cl0sIHNjb3JlczogU2VxdWVuY2VbZmxvYXRdLAogICAgICAgICAgICAgICAgICAgICAgdG9wX246IGludCkgLT4gdHVwbGVbbGlzdFtzdHJdLCBsaXN0W2Zsb2F0XV06CiAgICAiIiJTb3J0IGNhbmRpZGF0ZXMgYnkgc2NvcmUgKGRlc2MpIGFuZCBrZWVwIGB0b3BfbmAuCgogICAgVGllcyBicmVhayBieSBvcmlnaW5hbCBjYW5kaWRhdGUgb3JkZXIsIHNvIHRoZSBvdXRwdXQgaXMgZGV0ZXJtaW5pc3RpYwogICAgZm9yIGVxdWFsIHNjb3JlcyAtLSBpbXBvcnRhbnQgYmVjYXVzZSBhIGNyb3NzLWVuY29kZXIgY2FuIGVhc2lseSBlbWl0CiAgICBpZGVudGljYWwgbG9naXRzIGZvciBuZWFyLWlkZW50aWNhbCBldmlkZW5jZSBzZW50ZW5jZXMuCiAgICAiIiIKICAgIGlmIGxlbihjYW5kaWRhdGVzKSAhPSBsZW4oc2NvcmVzKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYie2xlbihjYW5kaWRhdGVzKX0gY2FuZGlkYXRlcyBidXQge2xlbihzY29yZXMpfSBzY29yZXMiKQogICAgb3JkZXIgPSBzb3J0ZWQocmFuZ2UobGVuKGNhbmRpZGF0ZXMpKSwga2V5PWxhbWJkYSBpOiAoLWZsb2F0KHNjb3Jlc1tpXSksIGkpKQogICAga2VlcCA9IG9yZGVyWzogbWluKGludCh0b3BfbiksIGxlbihvcmRlcikpXQogICAgb3V0X2lkcyA9IFtjYW5kaWRhdGVzW2ldIGZvciBpIGluIGtlZXBdCiAgICBvdXRfc2NvcmVzID0gW2Zsb2F0KHNjb3Jlc1tpXSkgZm9yIGkgaW4ga2VlcF0KCiAgICBleHRyYSA9IHNldChvdXRfaWRzKSAtIHNldChjYW5kaWRhdGVzKQogICAgaWYgZXh0cmE6CiAgICAgICAgcmFpc2UgQ2FuZGlkYXRlU2V0VmlvbGF0aW9uKAogICAgICAgICAgICBmInJlcmFua2VyIHByb2R1Y2VkIHtsZW4oZXh0cmEpfSBpZChzKSBhYnNlbnQgZnJvbSB0aGUgY2FuZGlkYXRlIHNldDogIgogICAgICAgICAgICBmIntzb3J0ZWQoZXh0cmEpWzo1XX0iKQogICAgaWYgbGVuKHNldChvdXRfaWRzKSkgIT0gbGVuKG91dF9pZHMpOgogICAgICAgIHJhaXNlIENhbmRpZGF0ZVNldFZpb2xhdGlvbigicmVyYW5rZXIgcmV0dXJuZWQgZHVwbGljYXRlIGV2aWRlbmNlIGlkcyIpCiAgICByZXR1cm4gb3V0X2lkcywgb3V0X3Njb3JlcwoKCmNsYXNzIENyb3NzRW5jb2RlclJlcmFua2VyOgogICAgIiIiUmVzdW1hYmxlIHJlcmFua2luZyBvdmVyIHBlci1jbGFpbSBjYW5kaWRhdGUgbGlzdHMuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHNjb3JlX2ZuOiBDYWxsYWJsZVtbU2VxdWVuY2VbdHVwbGVbc3RyLCBzdHJdXV0sIG5wLm5kYXJyYXldLAogICAgICAgICAgICAgICAgIHRvcF9uOiBpbnQgPSA1LCBiYXRjaF9zaXplOiBpbnQgPSAzMikgLT4gTm9uZToKICAgICAgICBzZWxmLnNjb3JlX2ZuID0gc2NvcmVfZm4KICAgICAgICBzZWxmLnRvcF9uID0gaW50KHRvcF9uKQogICAgICAgIHNlbGYuYmF0Y2hfc2l6ZSA9IGludChiYXRjaF9zaXplKQoKICAgIGRlZiByZXJhbmtfb25lKHNlbGYsIGNsYWltX3RleHQ6IHN0ciwgY2FuZGlkYXRlX2lkczogU2VxdWVuY2Vbc3RyXSwKICAgICAgICAgICAgICAgICAgIGV2aWRlbmNlX3RleHRzOiBkaWN0W3N0ciwgc3RyXSkgLT4gdHVwbGVbbGlzdFtzdHJdLCBsaXN0W2Zsb2F0XV06CiAgICAgICAgaWYgbm90IGNhbmRpZGF0ZV9pZHM6CiAgICAgICAgICAgIHJldHVybiBbXSwgW10KICAgICAgICBwYWlycyA9IFsoY2xhaW1fdGV4dCwgZXZpZGVuY2VfdGV4dHMuZ2V0KGUsICIiKSkgZm9yIGUgaW4gY2FuZGlkYXRlX2lkc10KICAgICAgICBzY29yZXM6IGxpc3RbZmxvYXRdID0gW10KICAgICAgICBmb3IgaSBpbiByYW5nZSgwLCBsZW4ocGFpcnMpLCBzZWxmLmJhdGNoX3NpemUpOgogICAgICAgICAgICBzY29yZXMuZXh0ZW5kKG5wLmFzYXJyYXkoc2VsZi5zY29yZV9mbihwYWlyc1tpOmkgKyBzZWxmLmJhdGNoX3NpemVdKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGR0eXBlPW5wLmZsb2F0MzIpLnJhdmVsKCkudG9saXN0KCkpCiAgICAgICAgaWYgbGVuKHNjb3JlcykgIT0gbGVuKGNhbmRpZGF0ZV9pZHMpOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJzY29yZV9mbiByZXR1cm5lZCB7bGVuKHNjb3Jlcyl9IHNjb3JlcyBmb3IgIgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZiJ7bGVuKGNhbmRpZGF0ZV9pZHMpfSBjYW5kaWRhdGVzIikKICAgICAgICByZXR1cm4gcmVyYW5rX2NhbmRpZGF0ZXMoY2FuZGlkYXRlX2lkcywgc2NvcmVzLCBzZWxmLnRvcF9uKQoKICAgIGRlZiByZXJhbmtfYWxsKHNlbGYsIGNsYWltczogZGljdFtzdHIsIHN0cl0sIGNhbmRpZGF0ZXM6IGRpY3Rbc3RyLCBTZXF1ZW5jZVtzdHJdXSwKICAgICAgICAgICAgICAgICAgIGV2aWRlbmNlX3RleHRzOiBkaWN0W3N0ciwgc3RyXSwgb3V0X2RpcjogUGF0aCwKICAgICAgICAgICAgICAgICAgIHNoYXJkX3NpemU6IGludCA9IDEwMDAsCiAgICAgICAgICAgICAgICAgICBsb2c6IENhbGxhYmxlW1tzdHJdLCBOb25lXSA9IHByaW50KSAtPiBkaWN0W3N0ciwgQW55XToKICAgICAgICAiIiJSZXJhbmsgZXZlcnkgY2xhaW0sIHNoYXJkaW5nICsgY2hlY2twb2ludGluZyBhcyB3ZSBnby4iIiIKICAgICAgICBvdXRfZGlyID0gUGF0aChvdXRfZGlyKQogICAgICAgIG91dF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIG1hbmlmZXN0X3BhdGggPSBvdXRfZGlyIC8gIm1hbmlmZXN0Lmpzb24iCiAgICAgICAgbWFuOiBkaWN0W3N0ciwgQW55XSA9IHt9CiAgICAgICAgaWYgbWFuaWZlc3RfcGF0aC5pc19maWxlKCk6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIG1hbiA9IGpzb24ubG9hZHMobWFuaWZlc3RfcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgICAgIGV4Y2VwdCBqc29uLkpTT05EZWNvZGVFcnJvcjoKICAgICAgICAgICAgICAgIG1hbiA9IHt9CgogICAgICAgIGNsYWltX2lkcyA9IHNvcnRlZChjYW5kaWRhdGVzKSAgICAgICAgICAjIGZpeGVkIG9yZGVyIC0+IHJlc3VtYWJsZSBzaGFyZHMKICAgICAgICBuX3NoYXJkcyA9IChsZW4oY2xhaW1faWRzKSArIHNoYXJkX3NpemUgLSAxKSAvLyBzaGFyZF9zaXplIGlmIGNsYWltX2lkcyBlbHNlIDAKICAgICAgICBkb25lID0gc2V0KG1hbi5nZXQoImNvbXBsZXRlZF9zaGFyZHMiLCBbXSkpCiAgICAgICAgZm9yIGkgaW4gcmFuZ2Uobl9zaGFyZHMpOgogICAgICAgICAgICBpZiAob3V0X2RpciAvIGYicmVyYW5rX3tpOjA1ZH0uanNvbiIpLmlzX2ZpbGUoKToKICAgICAgICAgICAgICAgIGRvbmUuYWRkKGkpCgogICAgICAgIG1hbi51cGRhdGUoeyJuX2NsYWltcyI6IGxlbihjbGFpbV9pZHMpLCAic2hhcmRfc2l6ZSI6IHNoYXJkX3NpemUsCiAgICAgICAgICAgICAgICAgICAgIm5fc2hhcmRzIjogbl9zaGFyZHMsICJ0b3BfbiI6IHNlbGYudG9wX24sCiAgICAgICAgICAgICAgICAgICAgImNvbXBsZXRlZF9zaGFyZHMiOiBzb3J0ZWQoZG9uZSksICJzdGF0dXMiOiAiaW5fcHJvZ3Jlc3MifSkKICAgICAgICBtYW5pZmVzdF9wYXRoLndyaXRlX3RleHQoanNvbi5kdW1wcyhtYW4sIGluZGVudD0yKSwgZW5jb2Rpbmc9InV0Zi04IikKCiAgICAgICAgdDAgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgZm9yIGkgaW4gcmFuZ2Uobl9zaGFyZHMpOgogICAgICAgICAgICBpZiBpIGluIGRvbmU6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBsbywgaGkgPSBpICogc2hhcmRfc2l6ZSwgbWluKChpICsgMSkgKiBzaGFyZF9zaXplLCBsZW4oY2xhaW1faWRzKSkKICAgICAgICAgICAgc2hhcmQ6IGRpY3Rbc3RyLCBBbnldID0ge30KICAgICAgICAgICAgZm9yIGNpZCBpbiBjbGFpbV9pZHNbbG86aGldOgogICAgICAgICAgICAgICAgaWRzLCBzY29yZXMgPSBzZWxmLnJlcmFua19vbmUoY2xhaW1zLmdldChjaWQsICIiKSwgY2FuZGlkYXRlc1tjaWRdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZXZpZGVuY2VfdGV4dHMpCiAgICAgICAgICAgICAgICAjIEVuZm9yY2UgdGhlIGludmFyaWFudCBvbiB3aGF0ZXZlciByZXJhbmtfb25lIHJldHVybmVkLiBBCiAgICAgICAgICAgICAgICAjIHN1YmNsYXNzIG9yIGEgY3VzdG9tIHNjb3JlciBjb3VsZCBicmVhayBpdDsgdGhlIHBpcGVsaW5lIG11c3QKICAgICAgICAgICAgICAgICMgbm90IHRydXN0IHRoZSByZXJhbmtlcidzIG93biBib29ra2VlcGluZy4KICAgICAgICAgICAgICAgIGFzc2VydF9zdWJzZXRfb2ZfY2FuZGlkYXRlcyhpZHMsIGNhbmRpZGF0ZXNbY2lkXSwgY2lkKQogICAgICAgICAgICAgICAgc2hhcmRbY2lkXSA9IHsiZXZpZGVuY2VfaWRzIjogaWRzLCAic2NvcmVzIjogc2NvcmVzfQogICAgICAgICAgICB0bXAgPSBvdXRfZGlyIC8gZiJyZXJhbmtfe2k6MDVkfS50bXAuanNvbiIKICAgICAgICAgICAgdG1wLndyaXRlX3RleHQoanNvbi5kdW1wcyhzaGFyZCwgZW5zdXJlX2FzY2lpPUZhbHNlKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgICAgICAgICAgdG1wLnJlcGxhY2Uob3V0X2RpciAvIGYicmVyYW5rX3tpOjA1ZH0uanNvbiIpCiAgICAgICAgICAgIGRvbmUuYWRkKGkpCiAgICAgICAgICAgIG1hblsiY29tcGxldGVkX3NoYXJkcyJdID0gc29ydGVkKGRvbmUpCiAgICAgICAgICAgIG1hbmlmZXN0X3BhdGgud3JpdGVfdGV4dChqc29uLmR1bXBzKG1hbiwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQogICAgICAgICAgICBsb2coZiIgIHJlcmFuayBzaGFyZCB7aSArIDF9L3tuX3NoYXJkc30gW3tsb306e2hpfV0iKQoKICAgICAgICBtYW5bInN0YXR1cyJdID0gImNvbXBsZXRlIgogICAgICAgIG1hblsidG90YWxfc2Vjb25kcyJdID0gcm91bmQodGltZS5wZXJmX2NvdW50ZXIoKSAtIHQwLCAyKQogICAgICAgIG1hbmlmZXN0X3BhdGgud3JpdGVfdGV4dChqc29uLmR1bXBzKG1hbiwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQogICAgICAgIHJldHVybiBtYW4KCgpkZWYgbG9hZF9yZXJhbmtlZChvdXRfZGlyOiBQYXRoKSAtPiBkaWN0W3N0ciwgZGljdFtzdHIsIEFueV1dOgogICAgb3V0X2RpciA9IFBhdGgob3V0X2RpcikKICAgIG1lcmdlZDogZGljdFtzdHIsIGRpY3Rbc3RyLCBBbnldXSA9IHt9CiAgICBmb3IgcCBpbiBzb3J0ZWQob3V0X2Rpci5nbG9iKCJyZXJhbmtfKi5qc29uIikpOgogICAgICAgIGlmIHAubmFtZS5lbmRzd2l0aCgiLnRtcC5qc29uIik6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgbWVyZ2VkLnVwZGF0ZShqc29uLmxvYWRzKHAucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKSkKICAgIHJldHVybiBtZXJnZWQKCgpkZWYgYnVpbGRfY3Jvc3NfZW5jb2Rlcl9zY29yZV9mbihtb2RlbF9uYW1lOiBzdHIgPSAieGxtLXJvYmVydGEtYmFzZSIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1heF9sZW5ndGg6IGludCA9IDI1NiwgZGV2aWNlOiBzdHIgfCBOb25lID0gTm9uZQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICApIC0+IENhbGxhYmxlW1tTZXF1ZW5jZVt0dXBsZVtzdHIsIHN0cl1dXSwgbnAubmRhcnJheV06CiAgICAiIiJMYXppbHkgYnVpbGQgYSByZWxldmFuY2Ugc2NvcmVyIGZyb20gYW4gWExNLVIgc2VxdWVuY2UgY2xhc3NpZmllci4KCiAgICBJbXBvcnRlZCBpbnNpZGUgdGhlIGZ1bmN0aW9uIHNvIHRoZSBtb2R1bGUgc3RheXMgaW1wb3J0YWJsZSB3aXRob3V0CiAgICB0cmFuc2Zvcm1lcnMgYW5kIHdpdGhvdXQgZG93bmxvYWRpbmcgd2VpZ2h0cy4gQW4gdW50cmFpbmVkIGhlYWQgZ2l2ZXMKICAgIGFyYml0cmFyeSBzY29yZXMgLS0gdGhpcyBpcyB3aXJlZCBmb3IgYSBmaW5lLXR1bmVkIGNoZWNrcG9pbnQsIGFuZCB0aGUKICAgIENvbGFiIG5vdGVib29rIHNheXMgc28gYXQgdGhlIHBvaW50IG9mIHVzZS4KICAgICIiIgogICAgaW1wb3J0IHRvcmNoICAjIGxvY2FsIGltcG9ydHMgb24gcHVycG9zZQogICAgZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IEF1dG9Nb2RlbEZvclNlcXVlbmNlQ2xhc3NpZmljYXRpb24sIEF1dG9Ub2tlbml6ZXIKCiAgICB0b2sgPSBBdXRvVG9rZW5pemVyLmZyb21fcHJldHJhaW5lZChtb2RlbF9uYW1lKQogICAgbW9kZWwgPSBBdXRvTW9kZWxGb3JTZXF1ZW5jZUNsYXNzaWZpY2F0aW9uLmZyb21fcHJldHJhaW5lZChtb2RlbF9uYW1lLCBudW1fbGFiZWxzPTEpCiAgICBkZXYgPSBkZXZpY2Ugb3IgKCJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpCiAgICBtb2RlbC50byhkZXYpLmV2YWwoKQoKICAgIEB0b3JjaC5ub19ncmFkKCkKICAgIGRlZiBfc2NvcmUocGFpcnM6IFNlcXVlbmNlW3R1cGxlW3N0ciwgc3RyXV0pIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgZW5jID0gdG9rKFtwWzBdIGZvciBwIGluIHBhaXJzXSwgW3BbMV0gZm9yIHAgaW4gcGFpcnNdLCBwYWRkaW5nPVRydWUsCiAgICAgICAgICAgICAgICAgIHRydW5jYXRpb249VHJ1ZSwgbWF4X2xlbmd0aD1tYXhfbGVuZ3RoLCByZXR1cm5fdGVuc29ycz0icHQiKS50byhkZXYpCiAgICAgICAgcmV0dXJuIG1vZGVsKCoqZW5jKS5sb2dpdHMuc3F1ZWV6ZSgtMSkuZmxvYXQoKS5jcHUoKS5udW1weSgpCgogICAgcmV0dXJuIF9zY29yZQo=", "sha256": "c4732974ec2ec670ae3db23fae2d16e4dbb5fa88baf7f6c46632098c907d68df", "size": 8771}, "src/verification/__init__.py": {"b64": "IiIiTW9kdWxlIDYg4oCUIGZhY3QgdmVyaWZpY2F0aW9uLgoKRkVWRVIgYW5kIFgtRkFDVCBhcmUga2VwdCBhcyBTRVBBUkFURSB0YXNrcyB3aXRoIHNlcGFyYXRlIGxhYmVsIHNwYWNlcyBhbmQKc2VwYXJhdGUgcmVwb3J0ZWQgcmVzdWx0cy4gVGhlIFhMTS1SIGVuY29kZXIgaXMgc2hhcmVkIHNvIGNyb3NzLWxpbmd1YWwKdHJhbnNmZXIgaXMgbWVhc3VyYWJsZTsgdGhlIGNsYXNzaWZpY2F0aW9uIGhlYWQgaXMgcmVwbGFjZWQsIG5ldmVyIHJldXNlZC4KCiAgICBkYXRhc2V0ICAgICAgICAgIGxhYmVsIHNwYWNlcywgKGNsYWltLCBldmlkZW5jZSkgcGFpcmluZywgc3RyYXRpZmllZCBzdWJzZXRzCiAgICB2ZXJpZmllcl9tb2RlbCAgIFhMTS1SICsgcmVwbGFjZWFibGUgaGVhZAogICAgdHJhaW5lciAgICAgICAgICByZXN1bWFibGUgdHJhaW5pbmcsIGNoZWNrcG9pbnRzIGluY2wuIFJORyBzdGF0ZQogICAgY2FsaWJyYXRpb24gICAgICBFQ0UsIHJlbGlhYmlsaXR5IGRhdGEsIHRlbXBlcmF0dXJlIHNjYWxpbmcKICAgIGFibGF0aW9ucyAgICAgICAgdGhlIHNpeCBjb25maWd1cmVkIGFibGF0aW9ucwoiIiIKCmZyb20gc3JjLnZlcmlmaWNhdGlvbi5jYWxpYnJhdGlvbiBpbXBvcnQgKCAgIyBub3FhOiBGNDAxCiAgICBjYWxpYnJhdGVfYW5kX2V2YWx1YXRlLAogICAgZXhwZWN0ZWRfY2FsaWJyYXRpb25fZXJyb3IsCiAgICBmaXRfdGVtcGVyYXR1cmUsCiAgICByZWxpYWJpbGl0eV9kYXRhLAogICAgc29mdG1heCwKKQpmcm9tIHNyYy52ZXJpZmljYXRpb24uZGF0YXNldCBpbXBvcnQgKCAgIyBub3FhOiBGNDAxCiAgICBGRVZFUl9MQUJFTFMsCiAgICBGRVZFUl9TUEFDRSwKICAgIExBQkVMX1NQQUNFUywKICAgIFhfRkFDVF9MQUJFTFMsCiAgICBYX0ZBQ1RfU1BBQ0UsCiAgICBMYWJlbFNwYWNlLAogICAgVmVyaWZpZXJFeGFtcGxlLAogICAgYXNzZXJ0X2xhYmVsX3NwYWNlc19kaXNqb2ludCwKICAgIGJ1aWxkX2V4YW1wbGVzLAogICAgZ2V0X2xhYmVsX3NwYWNlLAogICAgc3RyYXRpZmllZF9zdWJzZXQsCikKCl9fYWxsX18gPSBbCiAgICAiRkVWRVJfTEFCRUxTIiwgIlhfRkFDVF9MQUJFTFMiLCAiRkVWRVJfU1BBQ0UiLCAiWF9GQUNUX1NQQUNFIiwgIkxBQkVMX1NQQUNFUyIsCiAgICAiTGFiZWxTcGFjZSIsICJWZXJpZmllckV4YW1wbGUiLCAiYnVpbGRfZXhhbXBsZXMiLCAiZ2V0X2xhYmVsX3NwYWNlIiwKICAgICJzdHJhdGlmaWVkX3N1YnNldCIsICJhc3NlcnRfbGFiZWxfc3BhY2VzX2Rpc2pvaW50IiwKICAgICJleHBlY3RlZF9jYWxpYnJhdGlvbl9lcnJvciIsICJyZWxpYWJpbGl0eV9kYXRhIiwgImZpdF90ZW1wZXJhdHVyZSIsCiAgICAiY2FsaWJyYXRlX2FuZF9ldmFsdWF0ZSIsICJzb2Z0bWF4IiwKXQo=", "sha256": "c3b5ad89e7f5eb8aaee5fe6195a2733a996b65567e505ccef676c351e15ccc00", "size": 1373}, "src/verification/dataset.py": {"b64": "IiIiTW9kdWxlIDYg4oCUIHZlcmlmaWVyIGRhdGFzZXQ6IGxhYmVsIHNwYWNlcyBhbmQgKGNsYWltLCBldmlkZW5jZSkgcGFpcmluZy4KClRIRSBMQUJFTC1TUEFDRSBSVUxFLCBlbmZvcmNlZCBpbiBjb2RlIHJhdGhlciB0aGFuIGxlZnQgdG8gZGlzY2lwbGluZToKCiAgICBGRVZFUiAgOiBTVVBQT1JUUyAvIFJFRlVURVMgLyBOT1QgRU5PVUdIIElORk8gICAodGV4dHVhbCBlbnRhaWxtZW50KQogICAgWC1GQUNUIDogZmFsc2UgLyB0cnVlIC8gcGFydGx5IHRydWUvbWlzbGVhZGluZyAgKGZhY3QtY2hlY2tlciB2ZXJhY2l0eSkKClRoZXNlIGFyZSBkaWZmZXJlbnQgdGFza3Mgb3ZlciBkaWZmZXJlbnQgbGFiZWwgc2VtYW50aWNzLiAicGFydGx5CnRydWUvbWlzbGVhZGluZyIgaGFzIG5vIEZFVkVSIGNvdW50ZXJwYXJ0OyBmb3JjaW5nIGl0IGludG8gU1VQUE9SVFMgb3IKUkVGVVRFUyB3b3VsZCBmYWJyaWNhdGUgYSBqdWRnZW1lbnQgdGhlIGFubm90YXRvciBuZXZlciBtYWRlLiBTbzoKCiAgKiBubyBtYXBwaW5nIGZ1bmN0aW9uIGJldHdlZW4gdGhlIHR3byBzcGFjZXMgZXhpc3RzIGluIHRoaXMgZmlsZTsKICAqIGBMYWJlbFNwYWNlYCByZWZ1c2VzIHRvIGVuY29kZSBhIGxhYmVsIGl0IGRvZXMgbm90IG93bjsKICAqIFgtRkFDVCBsYWJlbHMgYXJlIHN0b3JlZCBleGFjdGx5IGFzIHB1Ymxpc2hlZCwgaW5jbHVkaW5nIHRoZSBsaXRlcmFsCiAgICBzdHJpbmcgInBhcnRseSB0cnVlL21pc2xlYWRpbmciOwogICogdGhlIGVuY29kZXIgaXMgc2hhcmVkIGJldHdlZW4gdGFza3Mgc28gRkVWRVIgLT4gVGFtaWwgdHJhbnNmZXIgaXMKICAgIG1lYXN1cmFibGUsIGJ1dCB0aGUgY2xhc3NpZmljYXRpb24gSEVBRCBpcyByZXBsYWNlZCwgbmV2ZXIgcmV1c2VkLgoKUmVwb3J0ZWQgc2VwYXJhdGVseSwgYWx3YXlzLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcywgZmllbGQKZnJvbSB0eXBpbmcgaW1wb3J0IEFueSwgSXRlcmFibGUsIFNlcXVlbmNlCgpGRVZFUl9MQUJFTFMgPSBbIlNVUFBPUlRTIiwgIlJFRlVURVMiLCAiTk9UIEVOT1VHSCBJTkZPIl0KWF9GQUNUX0xBQkVMUyA9IFsiZmFsc2UiLCAidHJ1ZSIsICJwYXJ0bHkgdHJ1ZS9taXNsZWFkaW5nIl0KCgpjbGFzcyBMYWJlbFNwYWNlOgogICAgIiIiQSBjbG9zZWQgbGFiZWwgdm9jYWJ1bGFyeS4gVW5rbm93biBsYWJlbHMgcmFpc2UgaW5zdGVhZCBvZiBkZWZhdWx0aW5nLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBuYW1lOiBzdHIsIGxhYmVsczogU2VxdWVuY2Vbc3RyXSwgdGFzazogc3RyKSAtPiBOb25lOgogICAgICAgIHNlbGYubmFtZSA9IG5hbWUKICAgICAgICBzZWxmLnRhc2sgPSB0YXNrCiAgICAgICAgc2VsZi5sYWJlbHMgPSBsaXN0KGxhYmVscykKICAgICAgICBzZWxmLl90b19pZCA9IHtsYWI6IGkgZm9yIGksIGxhYiBpbiBlbnVtZXJhdGUoc2VsZi5sYWJlbHMpfQogICAgICAgICMgVG9sZXJhdGUgY2FzZS93aGl0ZXNwYWNlIG5vaXNlIGluIHRoZSBzb3VyY2UsIG5ldmVyIHNlbWFudGljIGRyaWZ0LgogICAgICAgIHNlbGYuX25vcm0gPSB7bGFiLnN0cmlwKCkubG93ZXIoKTogaSBmb3IgaSwgbGFiIGluIGVudW1lcmF0ZShzZWxmLmxhYmVscyl9CgogICAgZGVmIF9fbGVuX18oc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBsZW4oc2VsZi5sYWJlbHMpCgogICAgZGVmIGVuY29kZShzZWxmLCBsYWJlbDogQW55KSAtPiBpbnQ6CiAgICAgICAgaWYgbGFiZWwgaXMgTm9uZToKICAgICAgICAgICAgcmFpc2UgS2V5RXJyb3IoZiJ7c2VsZi5uYW1lfTogbWlzc2luZyBsYWJlbCIpCiAgICAgICAgcyA9IHN0cihsYWJlbCkKICAgICAgICBpZiBzIGluIHNlbGYuX3RvX2lkOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fdG9faWRbc10KICAgICAgICBrZXkgPSBzLnN0cmlwKCkubG93ZXIoKQogICAgICAgIGlmIGtleSBpbiBzZWxmLl9ub3JtOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fbm9ybVtrZXldCiAgICAgICAgcmFpc2UgS2V5RXJyb3IoCiAgICAgICAgICAgIGYie3NlbGYubmFtZX06IHVua25vd24gbGFiZWwge3Mhcn0uIEtub3duOiB7c2VsZi5sYWJlbHN9LiAiCiAgICAgICAgICAgIGYiTGFiZWxzIGFyZSBuZXZlciByZW1hcHBlZCBhY3Jvc3MgZGF0YXNldHM7IGlmIHRoaXMgbGFiZWwgY2FtZSBmcm9tICIKICAgICAgICAgICAgZiJhbm90aGVyIGNvcnB1cywgaXQgYmVsb25ncyB0byBhIGRpZmZlcmVudCBsYWJlbCBzcGFjZS4iKQoKICAgIGRlZiBkZWNvZGUoc2VsZiwgaWR4OiBpbnQpIC0+IHN0cjoKICAgICAgICBpZiBub3QgMCA8PSBpbnQoaWR4KSA8IGxlbihzZWxmLmxhYmVscyk6CiAgICAgICAgICAgIHJhaXNlIEluZGV4RXJyb3IoZiJ7c2VsZi5uYW1lfTogbGFiZWwgaWQge2lkeH0gb3V0c2lkZSAwLi57bGVuKHNlbGYubGFiZWxzKS0xfSIpCiAgICAgICAgcmV0dXJuIHNlbGYubGFiZWxzW2ludChpZHgpXQoKICAgIGRlZiBjYW5fZW5jb2RlKHNlbGYsIGxhYmVsOiBBbnkpIC0+IGJvb2w6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBzZWxmLmVuY29kZShsYWJlbCkKICAgICAgICAgICAgcmV0dXJuIFRydWUKICAgICAgICBleGNlcHQgS2V5RXJyb3I6CiAgICAgICAgICAgIHJldHVybiBGYWxzZQoKCkZFVkVSX1NQQUNFID0gTGFiZWxTcGFjZSgiZmV2ZXIiLCBGRVZFUl9MQUJFTFMsIHRhc2s9ImVudGFpbG1lbnQiKQpYX0ZBQ1RfU1BBQ0UgPSBMYWJlbFNwYWNlKCJ4X2ZhY3QiLCBYX0ZBQ1RfTEFCRUxTLCB0YXNrPSJ2ZXJhY2l0eSIpCkxBQkVMX1NQQUNFUyA9IHsiZmV2ZXIiOiBGRVZFUl9TUEFDRSwgInhfZmFjdCI6IFhfRkFDVF9TUEFDRX0KCgpkZWYgZ2V0X2xhYmVsX3NwYWNlKGRhdGFzZXQ6IHN0cikgLT4gTGFiZWxTcGFjZToKICAgIGlmIGRhdGFzZXQgbm90IGluIExBQkVMX1NQQUNFUzoKICAgICAgICByYWlzZSBLZXlFcnJvcihmIm5vIGxhYmVsIHNwYWNlIGZvciBkYXRhc2V0IHtkYXRhc2V0IXJ9OyBrbm93bjoge3NvcnRlZChMQUJFTF9TUEFDRVMpfSIpCiAgICByZXR1cm4gTEFCRUxfU1BBQ0VTW2RhdGFzZXRdCgoKZGVmIGFzc2VydF9sYWJlbF9zcGFjZXNfZGlzam9pbnQoKSAtPiBOb25lOgogICAgIiIiRkVWRVIgYW5kIFgtRkFDVCBsYWJlbCBzdHJpbmdzIG11c3Qgbm90IG92ZXJsYXAuCgogICAgSWYgdGhleSBldmVyIGRpZCwgYSBtaXMtcm91dGVkIHJlY29yZCBjb3VsZCBiZSBzaWxlbnRseSBlbmNvZGVkIGFnYWluc3QKICAgIHRoZSB3cm9uZyBzcGFjZSBhbmQgbm8gZXJyb3Igd291bGQgc3VyZmFjZS4KICAgICIiIgogICAgb3ZlcmxhcCA9IHNldChGRVZFUl9MQUJFTFMpICYgc2V0KFhfRkFDVF9MQUJFTFMpCiAgICBpZiBvdmVybGFwOgogICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKGYibGFiZWwgc3BhY2VzIG92ZXJsYXA6IHtzb3J0ZWQob3ZlcmxhcCl9IikKCgpAZGF0YWNsYXNzCmNsYXNzIFZlcmlmaWVyRXhhbXBsZToKICAgIGNsYWltX2lkOiBzdHIKICAgIGNsYWltX3RleHQ6IHN0cgogICAgZXZpZGVuY2VfdGV4dHM6IGxpc3Rbc3RyXQogICAgbGFiZWxfaWQ6IGludAogICAgbGFiZWxfc3RyOiBzdHIKICAgIGRhdGFzZXQ6IHN0cgogICAgbGFuZ3VhZ2U6IHN0cgogICAgc3BsaXQ6IHN0cgogICAgbWV0YWRhdGE6IGRpY3Rbc3RyLCBBbnldID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWRpY3QpCgogICAgZGVmIHRvX3RleHRfcGFpcihzZWxmLCBzZXBhcmF0b3I6IHN0ciA9ICIgPC9zPiAiLCBtYXhfZXZpZGVuY2U6IGludCA9IDUpIC0+IHR1cGxlW3N0ciwgc3RyXToKICAgICAgICAiIiIoY2xhaW0sIGpvaW5lZCBldmlkZW5jZSkgZm9yIGEgY3Jvc3MtZW5jb2RlciB0b2tlbml6ZXIuCgogICAgICAgIEV2aWRlbmNlIGlzIGpvaW5lZCBpbiBSQU5LIE9SREVSOiB0aGUgcmVyYW5rZXIncyBiZXN0IGNhbmRpZGF0ZSBjb21lcwogICAgICAgIGZpcnN0LCBzbyB0cnVuY2F0aW9uIGF0IG1heF9sZW5ndGggZHJvcHMgdGhlIGxlYXN0IHJlbGV2YW50IHRleHQuCiAgICAgICAgIiIiCiAgICAgICAgcmV0dXJuIHNlbGYuY2xhaW1fdGV4dCwgc2VwYXJhdG9yLmpvaW4oc2VsZi5ldmlkZW5jZV90ZXh0c1s6bWF4X2V2aWRlbmNlXSkKCgpkZWYgYnVpbGRfZXhhbXBsZXMoCiAgICBjbGFpbXM6IFNlcXVlbmNlW2RpY3Rbc3RyLCBBbnldXSwKICAgIGV2aWRlbmNlX3RleHRzOiBkaWN0W3N0ciwgc3RyXSwKICAgIHJldHJpZXZlZDogZGljdFtzdHIsIFNlcXVlbmNlW3N0cl1dIHwgTm9uZSwKICAgIGRhdGFzZXQ6IHN0ciwKICAgIHNwbGl0X29mOiBkaWN0W2ludCwgc3RyXSwKICAgIG1heF9ldmlkZW5jZTogaW50ID0gNSwKICAgIHVzZV9nb2xkX2V2aWRlbmNlOiBib29sID0gRmFsc2UsCiAgICBjbGFpbV9vbmx5OiBib29sID0gRmFsc2UsCikgLT4gdHVwbGVbbGlzdFtWZXJpZmllckV4YW1wbGVdLCBkaWN0W3N0ciwgQW55XV06CiAgICAiIiJBc3NlbWJsZSB2ZXJpZmllciBleGFtcGxlcywgYWNjb3VudGluZyBmb3IgZXZlcnkgaW5wdXQgY2xhaW0uCgogICAgYHVzZV9nb2xkX2V2aWRlbmNlPVRydWVgIGlzIHRoZSBvcmFjbGUgYWJsYXRpb24gLS0gYW4gdXBwZXIgYm91bmQsIG5ldmVyCiAgICB0aGUgcHJpbWFyeSBwYXRoIChgZ29sZF9ldmlkZW5jZV9pbl9wcmltYXJ5X3BhdGg6IGZhbHNlYCBpbiBNb2R1bGUgNSdzCiAgICBjb25maWcpLiBgY2xhaW1fb25seT1UcnVlYCBpcyB0aGUgYWJsYXRpb24gdGhhdCBpc29sYXRlcyB3aGV0aGVyIGV2aWRlbmNlCiAgICBjb250cmlidXRlcyBhdCBhbGwuCiAgICAiIiIKICAgIHNwYWNlID0gZ2V0X2xhYmVsX3NwYWNlKGRhdGFzZXQpCiAgICBvdXQ6IGxpc3RbVmVyaWZpZXJFeGFtcGxlXSA9IFtdCiAgICBzdGF0cyA9IHsibl9pbnB1dCI6IGxlbihjbGFpbXMpLCAibl9idWlsdCI6IDAsICJuX2Ryb3BwZWRfdW5rbm93bl9sYWJlbCI6IDAsCiAgICAgICAgICAgICAibl9kcm9wcGVkX25vX3NwbGl0IjogMCwgIm5fd2l0aF9ub19ldmlkZW5jZSI6IDAsCiAgICAgICAgICAgICAiYnlfbGFiZWwiOiB7fSwgImJ5X3NwbGl0Ijoge30sICJldmlkZW5jZV9zb3VyY2UiOgogICAgICAgICAgICAgICAgICJjbGFpbV9vbmx5IiBpZiBjbGFpbV9vbmx5IGVsc2UgKCJnb2xkIiBpZiB1c2VfZ29sZF9ldmlkZW5jZSBlbHNlICJyZXRyaWV2ZWQiKX0KCiAgICBmb3IgaSwgYyBpbiBlbnVtZXJhdGUoY2xhaW1zKToKICAgICAgICBzcGxpdCA9IHNwbGl0X29mLmdldChpKQogICAgICAgIGlmIHNwbGl0IGlzIE5vbmU6CiAgICAgICAgICAgIHN0YXRzWyJuX2Ryb3BwZWRfbm9fc3BsaXQiXSArPSAxCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgdHJ5OgogICAgICAgICAgICBsaWQgPSBzcGFjZS5lbmNvZGUoYy5nZXQoImxhYmVsIikpCiAgICAgICAgZXhjZXB0IEtleUVycm9yOgogICAgICAgICAgICBzdGF0c1sibl9kcm9wcGVkX3Vua25vd25fbGFiZWwiXSArPSAxCiAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgIGNpZCA9IGMuZ2V0KCJjbGFpbV9pZCIsIHN0cihpKSkKICAgICAgICBpZiBjbGFpbV9vbmx5OgogICAgICAgICAgICBldl9pZHM6IFNlcXVlbmNlW3N0cl0gPSBbXQogICAgICAgIGVsaWYgdXNlX2dvbGRfZXZpZGVuY2U6CiAgICAgICAgICAgIGV2X2lkcyA9IGMuZ2V0KCJldmlkZW5jZSIpIG9yIFtdCiAgICAgICAgZWxzZToKICAgICAgICAgICAgZXZfaWRzID0gKHJldHJpZXZlZCBvciB7fSkuZ2V0KGNpZCwgW10pCiAgICAgICAgdGV4dHMgPSBbZXZpZGVuY2VfdGV4dHNbZV0gZm9yIGUgaW4gbGlzdChldl9pZHMpWzptYXhfZXZpZGVuY2VdIGlmIGUgaW4gZXZpZGVuY2VfdGV4dHNdCiAgICAgICAgaWYgbm90IHRleHRzIGFuZCBub3QgY2xhaW1fb25seToKICAgICAgICAgICAgc3RhdHNbIm5fd2l0aF9ub19ldmlkZW5jZSJdICs9IDEKCiAgICAgICAgbGFiID0gc3BhY2UuZGVjb2RlKGxpZCkKICAgICAgICBzdGF0c1siYnlfbGFiZWwiXVtsYWJdID0gc3RhdHNbImJ5X2xhYmVsIl0uZ2V0KGxhYiwgMCkgKyAxCiAgICAgICAgc3RhdHNbImJ5X3NwbGl0Il1bc3BsaXRdID0gc3RhdHNbImJ5X3NwbGl0Il0uZ2V0KHNwbGl0LCAwKSArIDEKICAgICAgICBvdXQuYXBwZW5kKFZlcmlmaWVyRXhhbXBsZSgKICAgICAgICAgICAgY2xhaW1faWQ9Y2lkLCBjbGFpbV90ZXh0PWMuZ2V0KCJ0ZXh0IiwgIiIpLCBldmlkZW5jZV90ZXh0cz10ZXh0cywKICAgICAgICAgICAgbGFiZWxfaWQ9bGlkLCBsYWJlbF9zdHI9bGFiLCBkYXRhc2V0PWRhdGFzZXQsCiAgICAgICAgICAgIGxhbmd1YWdlPXN0cihjLmdldCgibGFuZ3VhZ2UiLCAidW5rbm93biIpKSwgc3BsaXQ9c3BsaXQsCiAgICAgICAgICAgIG1ldGFkYXRhPXsibl9ldmlkZW5jZSI6IGxlbih0ZXh0cyksCiAgICAgICAgICAgICAgICAgICAgICAib2ZmaWNpYWxfc3BsaXQiOiBjLmdldCgibWV0YWRhdGEiLCB7fSkuZ2V0KCJvZmZpY2lhbF9zcGxpdCIpfSkpCiAgICAgICAgc3RhdHNbIm5fYnVpbHQiXSArPSAxCgogICAgc3RhdHNbImFjY291bnRpbmdfYmFsYW5jZXMiXSA9ICgKICAgICAgICBzdGF0c1sibl9pbnB1dCJdID09IHN0YXRzWyJuX2J1aWx0Il0gKyBzdGF0c1sibl9kcm9wcGVkX3Vua25vd25fbGFiZWwiXQogICAgICAgICsgc3RhdHNbIm5fZHJvcHBlZF9ub19zcGxpdCJdKQogICAgcmV0dXJuIG91dCwgc3RhdHMKCgpkZWYgc3RyYXRpZmllZF9zdWJzZXQoZXhhbXBsZXM6IFNlcXVlbmNlW1ZlcmlmaWVyRXhhbXBsZV0sIHNpemU6IGludCB8IE5vbmUsCiAgICAgICAgICAgICAgICAgICAgICBzZWVkOiBpbnQgPSA0MikgLT4gbGlzdFtWZXJpZmllckV4YW1wbGVdOgogICAgIiIiTGFiZWwtc3RyYXRpZmllZCBzdWJzYW1wbGUsIGRldGVybWluaXN0aWMgZm9yIGEgZ2l2ZW4gc2VlZC4KCiAgICBVc2VkIHRvIGN1dCBGRVZFUidzIDIyOCwwOTAgdHJhaW5pbmcgY2xhaW1zIHRvIGEgc2l6ZSBmcmVlIENvbGFiIGNhbgogICAgYWN0dWFsbHkgdHJhaW4gb24uIFN0cmF0aWZ5aW5nIGtlZXBzIHRoZSBsYWJlbCBkaXN0cmlidXRpb24gaW50YWN0LCBzbyB0aGUKICAgIHJlZHVjdGlvbiBjaGFuZ2VzIHNjYWxlIGJ1dCBub3QgdGhlIHRhc2sgYmFsYW5jZSAtLSBhbmQgaXQgaXMgcmVwb3J0ZWQgYXMKICAgIGEgc2NvcGUgbGltaXRhdGlvbiByYXRoZXIgdGhhbiBwcmVzZW50ZWQgYXMgYSBmdWxsLWNvcnB1cyByZXN1bHQuCiAgICAiIiIKICAgIGltcG9ydCByYW5kb20KCiAgICBpZiBzaXplIGlzIE5vbmUgb3Igc2l6ZSA+PSBsZW4oZXhhbXBsZXMpOgogICAgICAgIHJldHVybiBsaXN0KGV4YW1wbGVzKQogICAgYnlfbGFiZWw6IGRpY3RbaW50LCBsaXN0W1ZlcmlmaWVyRXhhbXBsZV1dID0ge30KICAgIGZvciBlIGluIGV4YW1wbGVzOgogICAgICAgIGJ5X2xhYmVsLnNldGRlZmF1bHQoZS5sYWJlbF9pZCwgW10pLmFwcGVuZChlKQoKICAgIHJuZyA9IHJhbmRvbS5SYW5kb20oc2VlZCkKICAgIHRvdGFsID0gbGVuKGV4YW1wbGVzKQogICAgb3V0OiBsaXN0W1ZlcmlmaWVyRXhhbXBsZV0gPSBbXQogICAgZm9yIGxpZCBpbiBzb3J0ZWQoYnlfbGFiZWwpOgogICAgICAgIGdyb3VwID0gc29ydGVkKGJ5X2xhYmVsW2xpZF0sIGtleT1sYW1iZGEgeDogeC5jbGFpbV9pZCkgICAjIHN0YWJsZSBpbnB1dCBvcmRlcgogICAgICAgIHRha2UgPSBtYXgoMSwgcm91bmQoc2l6ZSAqIGxlbihncm91cCkgLyB0b3RhbCkpCiAgICAgICAgcm5nLnNodWZmbGUoZ3JvdXApCiAgICAgICAgb3V0LmV4dGVuZChncm91cFs6dGFrZV0pCiAgICBvdXQuc29ydChrZXk9bGFtYmRhIHg6IHguY2xhaW1faWQpCiAgICBpZiBsZW4ob3V0KSA+IHNpemU6CiAgICAgICAgcm5nLnNodWZmbGUob3V0KQogICAgICAgIG91dCA9IHNvcnRlZChvdXRbOnNpemVdLCBrZXk9bGFtYmRhIHg6IHguY2xhaW1faWQpCiAgICByZXR1cm4gb3V0Cg==", "sha256": "22aa477a88acf2c73cb304c174e3112b8bd68a0ad0459f7f73494dd5f9e438a8", "size": 7894}, "src/verification/verifier_model.py": {"b64": "IiIiTW9kdWxlIDYg4oCUIFhMTS1SIHZlcmlmaWVyIHdpdGggYSByZXBsYWNlYWJsZSBjbGFzc2lmaWNhdGlvbiBoZWFkLgoKQXJjaGl0ZWN0dXJlOiBzaGFyZWQgWExNLVIgZW5jb2RlciArIGEgc21hbGwgTUxQIGhlYWQuCgpUaGUgaGVhZCBpcyBkZWxpYmVyYXRlbHkgc2VwYXJhYmxlIGZyb20gdGhlIGVuY29kZXIuIEZFVkVSIHByZXRyYWlucyB0aGUKZW5jb2RlciBvbiAzLXdheSBlbnRhaWxtZW50OyBYLUZBQ1QgdGhlbiBSRVBMQUNFUyB0aGUgaGVhZCB3aXRoIGEgZnJlc2hseQppbml0aWFsaXNlZCAzLXdheSB2ZXJhY2l0eSBoZWFkIGFuZCBmaW5lLXR1bmVzLiBCb3RoIHNwYWNlcyBoYXBwZW4gdG8gaGF2ZQp0aHJlZSBjbGFzc2VzLCB3aGljaCBtYWtlcyBpdCB0ZW1wdGluZyB0byBrZWVwIHRoZSBGRVZFUiBoZWFkIGFuZCAianVzdCBmaW5lLQp0dW5lIiAtLSB0aGF0IHdvdWxkIHNtdWdnbGUgRkVWRVIncyBlbnRhaWxtZW50IHNlbWFudGljcyBpbnRvIFgtRkFDVCdzCnZlcmFjaXR5IGxhYmVscyBhbmQgbWFrZSB0aGUgdHJhbnNmZXIgcmVzdWx0IHVuaW50ZXJwcmV0YWJsZS4gYHN3YXBfaGVhZCgpYAp0aGVyZWZvcmUgYWx3YXlzIHJlLWluaXRpYWxpc2VzLCBhbmQgcmVjb3JkcyB0aGF0IGl0IGRpZCBzby4KCktlZXBpbmcgdGhlIGVuY29kZXIgc2hhcmVkIGlzIHRoZSB3aG9sZSBwb2ludDogaXQgaXMgd2hhdCBtYWtlcwpGRVZFUiAtPiBUYW1pbCB0cmFuc2ZlciBtZWFzdXJhYmxlLCBhbmQgY29ubmVjdHMgdG8gTW9kdWxlIDQncyBmaW5kaW5nIHRoYXQKbmFpdmUgRW5nbGlzaC10by1UYW1pbCB0cmFuc2ZlciBpcyB3ZWFrICh6ZXJvLXNob3QgVGFtaWwgTWFjcm8tRjEgMC4zMjE2KS4KCnRvcmNoIGlzIGltcG9ydGVkIGxhemlseSBzbyB0aGlzIG1vZHVsZSBjYW4gYmUgaW1wb3J0ZWQsIGluc3BlY3RlZCBhbmQKcGFydGlhbGx5IHRlc3RlZCBvbiBhIG1hY2hpbmUgd2l0aCBubyB0b3JjaCBhbmQgbm8gZG93bmxvYWRlZCB3ZWlnaHRzLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwpmcm9tIHR5cGluZyBpbXBvcnQgQW55CgoKQGRhdGFjbGFzcwpjbGFzcyBWZXJpZmllckNvbmZpZzoKICAgIGVuY29kZXJfbmFtZTogc3RyID0gInhsbS1yb2JlcnRhLWJhc2UiCiAgICBudW1fbGFiZWxzOiBpbnQgPSAzCiAgICBoaWRkZW5fZGltOiBpbnQgPSA1MTIKICAgIGRyb3BvdXQ6IGZsb2F0ID0gMC4xCiAgICBtYXhfbGVuZ3RoOiBpbnQgPSAyNTYKICAgIGxhYmVsX3NwYWNlOiBzdHIgPSAiZmV2ZXIiCgogICAgZGVmIHRvX2RpY3Qoc2VsZikgLT4gZGljdFtzdHIsIEFueV06CiAgICAgICAgcmV0dXJuIHsiZW5jb2Rlcl9uYW1lIjogc2VsZi5lbmNvZGVyX25hbWUsICJudW1fbGFiZWxzIjogc2VsZi5udW1fbGFiZWxzLAogICAgICAgICAgICAgICAgImhpZGRlbl9kaW0iOiBzZWxmLmhpZGRlbl9kaW0sICJkcm9wb3V0Ijogc2VsZi5kcm9wb3V0LAogICAgICAgICAgICAgICAgIm1heF9sZW5ndGgiOiBzZWxmLm1heF9sZW5ndGgsICJsYWJlbF9zcGFjZSI6IHNlbGYubGFiZWxfc3BhY2V9CgoKZGVmIGJ1aWxkX3ZlcmlmaWVyKGNmZzogVmVyaWZpZXJDb25maWcpOgogICAgIiIiQ29uc3RydWN0IHRoZSBtb2RlbC4gSW1wb3J0cyB0b3JjaC90cmFuc2Zvcm1lcnMgb25seSB3aGVuIGNhbGxlZC4iIiIKICAgIGltcG9ydCB0b3JjaAogICAgaW1wb3J0IHRvcmNoLm5uIGFzIG5uCiAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b01vZGVsCgogICAgY2xhc3MgVmVyaWZpZXIobm4uTW9kdWxlKToKICAgICAgICBkZWYgX19pbml0X18oc2VsZiwgY29uZmlnOiBWZXJpZmllckNvbmZpZykgLT4gTm9uZToKICAgICAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgICAgIHNlbGYuY29uZmlnID0gY29uZmlnCiAgICAgICAgICAgIHNlbGYuZW5jb2RlciA9IEF1dG9Nb2RlbC5mcm9tX3ByZXRyYWluZWQoY29uZmlnLmVuY29kZXJfbmFtZSkKICAgICAgICAgICAgaGlkZGVuID0gc2VsZi5lbmNvZGVyLmNvbmZpZy5oaWRkZW5fc2l6ZQogICAgICAgICAgICBzZWxmLmhlYWQgPSBzZWxmLl9tYWtlX2hlYWQoaGlkZGVuLCBjb25maWcpCiAgICAgICAgICAgIHNlbGYubGFiZWxfc3BhY2UgPSBjb25maWcubGFiZWxfc3BhY2UKCiAgICAgICAgQHN0YXRpY21ldGhvZAogICAgICAgIGRlZiBfbWFrZV9oZWFkKGhpZGRlbjogaW50LCBjb25maWc6IFZlcmlmaWVyQ29uZmlnKSAtPiAibm4uTW9kdWxlIjoKICAgICAgICAgICAgcmV0dXJuIG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgICAgICBubi5Ecm9wb3V0KGNvbmZpZy5kcm9wb3V0KSwKICAgICAgICAgICAgICAgIG5uLkxpbmVhcihoaWRkZW4sIGNvbmZpZy5oaWRkZW5fZGltKSwKICAgICAgICAgICAgICAgIG5uLlRhbmgoKSwKICAgICAgICAgICAgICAgIG5uLkRyb3BvdXQoY29uZmlnLmRyb3BvdXQpLAogICAgICAgICAgICAgICAgbm4uTGluZWFyKGNvbmZpZy5oaWRkZW5fZGltLCBjb25maWcubnVtX2xhYmVscyksCiAgICAgICAgICAgICkKCiAgICAgICAgZGVmIGZvcndhcmQoc2VsZiwgaW5wdXRfaWRzLCBhdHRlbnRpb25fbWFzayk6CiAgICAgICAgICAgIG91dCA9IHNlbGYuZW5jb2RlcihpbnB1dF9pZHM9aW5wdXRfaWRzLCBhdHRlbnRpb25fbWFzaz1hdHRlbnRpb25fbWFzaykKICAgICAgICAgICAgIyBNYXNrLXdlaWdodGVkIG1lYW4gcG9vbGluZy4gTW9kdWxlIDQncyBhYmxhdGlvbiBzZWxlY3RlZCBtZWFuIG92ZXIKICAgICAgICAgICAgIyBDTFMgZm9yIHRoaXMgZW5jb2Rlciwgc28gdGhlIHNhbWUgY2hvaWNlIGlzIGtlcHQgZm9yIGNvbnNpc3RlbmN5LgogICAgICAgICAgICBtYXNrID0gYXR0ZW50aW9uX21hc2sudW5zcXVlZXplKC0xKS5mbG9hdCgpCiAgICAgICAgICAgIHN1bW1lZCA9IChvdXQubGFzdF9oaWRkZW5fc3RhdGUgKiBtYXNrKS5zdW0oZGltPTEpCiAgICAgICAgICAgIHBvb2xlZCA9IHN1bW1lZCAvIG1hc2suc3VtKGRpbT0xKS5jbGFtcChtaW49MWUtOSkKICAgICAgICAgICAgcmV0dXJuIHNlbGYuaGVhZChwb29sZWQpCgogICAgICAgIGRlZiBzd2FwX2hlYWQoc2VsZiwgbnVtX2xhYmVsczogaW50LCBsYWJlbF9zcGFjZTogc3RyLAogICAgICAgICAgICAgICAgICAgICAgaGlkZGVuX2RpbTogaW50IHwgTm9uZSA9IE5vbmUpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgICAgICAgICAiIiJSZXBsYWNlIHRoZSBoZWFkIGZvciBhIG5ldyBsYWJlbCBzcGFjZS4gQUxXQVlTIHJlLWluaXRpYWxpc2VzLgoKICAgICAgICAgICAgUmV0dXJucyBhIHJlY29yZCBvZiB0aGUgc3dhcCBzbyB0aGUgcmVwb3J0IGNhbiBzdGF0ZSBwbGFpbmx5IHRoYXQKICAgICAgICAgICAgbm8gRkVWRVIgaGVhZCB3ZWlnaHRzIHN1cnZpdmVkIGludG8gdGhlIFgtRkFDVCBtb2RlbC4KICAgICAgICAgICAgIiIiCiAgICAgICAgICAgIG9sZCA9IHsibGFiZWxfc3BhY2UiOiBzZWxmLmxhYmVsX3NwYWNlLCAibnVtX2xhYmVscyI6IHNlbGYuY29uZmlnLm51bV9sYWJlbHN9CiAgICAgICAgICAgIHNlbGYuY29uZmlnLm51bV9sYWJlbHMgPSBpbnQobnVtX2xhYmVscykKICAgICAgICAgICAgc2VsZi5jb25maWcubGFiZWxfc3BhY2UgPSBsYWJlbF9zcGFjZQogICAgICAgICAgICBpZiBoaWRkZW5fZGltIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgc2VsZi5jb25maWcuaGlkZGVuX2RpbSA9IGludChoaWRkZW5fZGltKQogICAgICAgICAgICBoaWRkZW4gPSBzZWxmLmVuY29kZXIuY29uZmlnLmhpZGRlbl9zaXplCiAgICAgICAgICAgIHNlbGYuaGVhZCA9IHNlbGYuX21ha2VfaGVhZChoaWRkZW4sIHNlbGYuY29uZmlnKQogICAgICAgICAgICByZXR1cm4geyJoZWFkX3JlaW5pdGlhbGlzZWQiOiBUcnVlLCAiZW5jb2Rlcl9yZXVzZWQiOiBUcnVlLAogICAgICAgICAgICAgICAgICAgICJmcm9tIjogb2xkLCAidG8iOiB7ImxhYmVsX3NwYWNlIjogbGFiZWxfc3BhY2UsICJudW1fbGFiZWxzIjogbnVtX2xhYmVsc30sCiAgICAgICAgICAgICAgICAgICAgIm5vdGUiOiAoIkVuY29kZXIgd2VpZ2h0cyBhcmUgY2FycmllZCBvdmVyICh0aGF0IGlzIHRoZSB0cmFuc2ZlciBiZWluZyAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIm1lYXN1cmVkKTsgaGVhZCB3ZWlnaHRzIGFyZSBkaXNjYXJkZWQgc28gRkVWRVIgZW50YWlsbWVudCAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInNlbWFudGljcyBjYW5ub3QgbGVhayBpbnRvIFgtRkFDVCB2ZXJhY2l0eSBsYWJlbHMuIil9CgogICAgcmV0dXJuIFZlcmlmaWVyKGNmZykKCgpkZWYgY291bnRfcGFyYW1ldGVycyhtb2RlbCkgLT4gZGljdFtzdHIsIGludF06CiAgICBlbmMgPSBzdW0ocC5udW1lbCgpIGZvciBwIGluIG1vZGVsLmVuY29kZXIucGFyYW1ldGVycygpKQogICAgaGVhZCA9IHN1bShwLm51bWVsKCkgZm9yIHAgaW4gbW9kZWwuaGVhZC5wYXJhbWV0ZXJzKCkpCiAgICByZXR1cm4geyJlbmNvZGVyIjogZW5jLCAiaGVhZCI6IGhlYWQsICJ0b3RhbCI6IGVuYyArIGhlYWR9Cg==", "sha256": "fd91405a702a17ad1ec24f7dbd020b7151ac199b4542d54d22d94e756bd32f50", "size": 4537}, "src/verification/trainer.py": {"b64": "IiIiTW9kdWxlIDYg4oCUIHJlc3VtYWJsZSB2ZXJpZmllciB0cmFpbmluZy4KCkJ1aWx0IGZvciBmcmVlIENvbGFiLCB3aGljaCBkaXNjb25uZWN0cy4gQSBjaGVja3BvaW50IHRoZXJlZm9yZSBzdG9yZXMKZXZlcnl0aGluZyBuZWVkZWQgdG8gY29udGludWUgKmlkZW50aWNhbGx5Kiwgbm90IGp1c3QgdGhlIHdlaWdodHM6CgogICAgbW9kZWwsIG9wdGltaXplciwgc2NoZWR1bGVyLCBlcG9jaCwgZ2xvYmFsX3N0ZXAsIFJORyBzdGF0ZQogICAgKHB5dGhvbiAvIG51bXB5IC8gdG9yY2ggLyBjdWRhKSwgYW5kIHRoZSBjb25maWcgdGhhdCBwcm9kdWNlZCBpdC4KCldpdGhvdXQgdGhlIFJORyBzdGF0ZSBhIHJlc3VtZWQgcnVuIHdvdWxkIHNlZSBhIGRpZmZlcmVudCBzaHVmZmxlIGFuZCB0YWtlIGEKZGlmZmVyZW50IHRyYWplY3RvcnksIHNvICJyZXN1bWUiIHdvdWxkIHNpbGVudGx5IG1lYW4gInJlc3RhcnQgZnJvbSB0aGVzZQp3ZWlnaHRzIiAtLSByZXByb2R1Y2libGUgb25seSBieSBhY2NpZGVudC4gYHRlc3RzL3Rlc3RfdmVyaWZpY2F0aW9uLnB5YAphc3NlcnRzIHRoYXQgYW4gaW50ZXJydXB0ZWQtdGhlbi1yZXN1bWVkIHJ1biBtYXRjaGVzIGFuIHVuaW50ZXJydXB0ZWQgb25lLgoKQ2hlY2twb2ludHMgYXJlIHdyaXR0ZW4gdGVtcC10aGVuLXJlbmFtZWQ7IGEgZGlzY29ubmVjdCBtaWQtd3JpdGUgbGVhdmVzIHRoZQpwcmV2aW91cyBnb29kIGNoZWNrcG9pbnQgaW50YWN0IHJhdGhlciB0aGFuIGEgdHJ1bmNhdGVkIGZpbGUuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IHJhbmRvbQppbXBvcnQgdGltZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBhc2RpY3QsIGRhdGFjbGFzcywgZmllbGQKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIENhbGxhYmxlLCBTZXF1ZW5jZQoKCkBkYXRhY2xhc3MKY2xhc3MgVHJhaW5TdGF0ZToKICAgIGVwb2NoOiBpbnQgPSAwCiAgICBnbG9iYWxfc3RlcDogaW50ID0gMAogICAgYmVzdF9tZXRyaWM6IGZsb2F0ID0gZmxvYXQoIi1pbmYiKQogICAgYmVzdF9zdGVwOiBpbnQgPSAwCiAgICBlcG9jaHNfd2l0aG91dF9pbXByb3ZlbWVudDogaW50ID0gMAogICAgaGlzdG9yeTogbGlzdFtkaWN0W3N0ciwgQW55XV0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKCiAgICBkZWYgdG9fZGljdChzZWxmKSAtPiBkaWN0W3N0ciwgQW55XToKICAgICAgICByZXR1cm4gYXNkaWN0KHNlbGYpCgogICAgQGNsYXNzbWV0aG9kCiAgICBkZWYgZnJvbV9kaWN0KGNscywgZDogZGljdFtzdHIsIEFueV0pIC0+ICJUcmFpblN0YXRlIjoKICAgICAgICByZXR1cm4gY2xzKCoqe2s6IHYgZm9yIGssIHYgaW4gZC5pdGVtcygpIGlmIGsgaW4gY2xzLl9fZGF0YWNsYXNzX2ZpZWxkc19ffSkKCgpkZWYgY2FwdHVyZV9ybmdfc3RhdGUoKSAtPiBkaWN0W3N0ciwgQW55XToKICAgICIiIlNuYXBzaG90IGV2ZXJ5IFJORyB0aGF0IGNhbiBhZmZlY3QgdGhlIHRyYWplY3RvcnkuIiIiCiAgICBzdGF0ZTogZGljdFtzdHIsIEFueV0gPSB7InB5dGhvbiI6IHJhbmRvbS5nZXRzdGF0ZSgpfQogICAgdHJ5OgogICAgICAgIGltcG9ydCBudW1weSBhcyBucAogICAgICAgIHN0YXRlWyJudW1weSJdID0gbnAucmFuZG9tLmdldF9zdGF0ZSgpCiAgICBleGNlcHQgRXhjZXB0aW9uOiAgIyBub3FhOiBCTEUwMDEKICAgICAgICBwYXNzCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHRvcmNoCiAgICAgICAgc3RhdGVbInRvcmNoIl0gPSB0b3JjaC5nZXRfcm5nX3N0YXRlKCkKICAgICAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICBzdGF0ZVsiY3VkYSJdID0gdG9yY2guY3VkYS5nZXRfcm5nX3N0YXRlX2FsbCgpCiAgICBleGNlcHQgRXhjZXB0aW9uOiAgIyBub3FhOiBCTEUwMDEKICAgICAgICBwYXNzCiAgICByZXR1cm4gc3RhdGUKCgpkZWYgcmVzdG9yZV9ybmdfc3RhdGUoc3RhdGU6IGRpY3Rbc3RyLCBBbnldKSAtPiBOb25lOgogICAgaWYgbm90IHN0YXRlOgogICAgICAgIHJldHVybgogICAgaWYgInB5dGhvbiIgaW4gc3RhdGU6CiAgICAgICAgcyA9IHN0YXRlWyJweXRob24iXQogICAgICAgIHJhbmRvbS5zZXRzdGF0ZSgoc1swXSwgdHVwbGUoc1sxXSksIHNbMl0pIGlmIGlzaW5zdGFuY2UocywgKGxpc3QsIHR1cGxlKSkgZWxzZSBzKQogICAgdHJ5OgogICAgICAgIGltcG9ydCBudW1weSBhcyBucAogICAgICAgIGlmICJudW1weSIgaW4gc3RhdGU6CiAgICAgICAgICAgIG5wLnJhbmRvbS5zZXRfc3RhdGUoc3RhdGVbIm51bXB5Il0pCiAgICBleGNlcHQgRXhjZXB0aW9uOiAgIyBub3FhOiBCTEUwMDEKICAgICAgICBwYXNzCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHRvcmNoCiAgICAgICAgaWYgInRvcmNoIiBpbiBzdGF0ZToKICAgICAgICAgICAgdG9yY2guc2V0X3JuZ19zdGF0ZShzdGF0ZVsidG9yY2giXSkKICAgICAgICBpZiAiY3VkYSIgaW4gc3RhdGUgYW5kIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgICAgIHRvcmNoLmN1ZGEuc2V0X3JuZ19zdGF0ZV9hbGwoc3RhdGVbImN1ZGEiXSkKICAgIGV4Y2VwdCBFeGNlcHRpb246ICAjIG5vcWE6IEJMRTAwMQogICAgICAgIHBhc3MKCgpkZWYgc2V0X3NlZWQoc2VlZDogaW50KSAtPiBOb25lOgogICAgcmFuZG9tLnNlZWQoc2VlZCkKICAgIHRyeToKICAgICAgICBpbXBvcnQgbnVtcHkgYXMgbnAKICAgICAgICBucC5yYW5kb20uc2VlZChzZWVkKQogICAgZXhjZXB0IEV4Y2VwdGlvbjogICMgbm9xYTogQkxFMDAxCiAgICAgICAgcGFzcwogICAgdHJ5OgogICAgICAgIGltcG9ydCB0b3JjaAogICAgICAgIHRvcmNoLm1hbnVhbF9zZWVkKHNlZWQpCiAgICAgICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICAgICAgdG9yY2guY3VkYS5tYW51YWxfc2VlZF9hbGwoc2VlZCkKICAgIGV4Y2VwdCBFeGNlcHRpb246ICAjIG5vcWE6IEJMRTAwMQogICAgICAgIHBhc3MKCgpjbGFzcyBDaGVja3BvaW50TWFuYWdlcjoKICAgICIiIkF0b21pYyBzYXZlL2xvYWQgb2YgZnVsbCB0cmFpbmluZyBzdGF0ZS4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgY2twdF9kaXI6IFBhdGgpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5kaXIgPSBQYXRoKGNrcHRfZGlyKQogICAgICAgIHNlbGYuZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBzZWxmLmxhdGVzdCA9IHNlbGYuZGlyIC8gImxhdGVzdC5wdCIKICAgICAgICBzZWxmLmJlc3QgPSBzZWxmLmRpciAvICJiZXN0LnB0IgogICAgICAgIHNlbGYubWV0YV9wYXRoID0gc2VsZi5kaXIgLyAiY2hlY2twb2ludF9tZXRhLmpzb24iCgogICAgZGVmIHNhdmUoc2VsZiwgbW9kZWwsIG9wdGltaXplciwgc2NoZWR1bGVyLCBzdGF0ZTogVHJhaW5TdGF0ZSwKICAgICAgICAgICAgIGNvbmZpZzogZGljdFtzdHIsIEFueV0sIGlzX2Jlc3Q6IGJvb2wgPSBGYWxzZSkgLT4gZGljdFtzdHIsIEFueV06CiAgICAgICAgaW1wb3J0IHRvcmNoCgogICAgICAgIHBheWxvYWQgPSB7CiAgICAgICAgICAgICJtb2RlbF9zdGF0ZSI6IG1vZGVsLnN0YXRlX2RpY3QoKSwKICAgICAgICAgICAgIm9wdGltaXplcl9zdGF0ZSI6IG9wdGltaXplci5zdGF0ZV9kaWN0KCkgaWYgb3B0aW1pemVyIGVsc2UgTm9uZSwKICAgICAgICAgICAgInNjaGVkdWxlcl9zdGF0ZSI6IHNjaGVkdWxlci5zdGF0ZV9kaWN0KCkgaWYgc2NoZWR1bGVyIGVsc2UgTm9uZSwKICAgICAgICAgICAgInRyYWluX3N0YXRlIjogc3RhdGUudG9fZGljdCgpLAogICAgICAgICAgICAicm5nX3N0YXRlIjogY2FwdHVyZV9ybmdfc3RhdGUoKSwKICAgICAgICAgICAgImNvbmZpZyI6IGNvbmZpZywKICAgICAgICB9CiAgICAgICAgdG1wID0gc2VsZi5kaXIgLyAibGF0ZXN0LnRtcC5wdCIKICAgICAgICB0b3JjaC5zYXZlKHBheWxvYWQsIHRtcCkKICAgICAgICB0bXAucmVwbGFjZShzZWxmLmxhdGVzdCkKICAgICAgICBpZiBpc19iZXN0OgogICAgICAgICAgICB0bXAyID0gc2VsZi5kaXIgLyAiYmVzdC50bXAucHQiCiAgICAgICAgICAgIHRvcmNoLnNhdmUocGF5bG9hZCwgdG1wMikKICAgICAgICAgICAgdG1wMi5yZXBsYWNlKHNlbGYuYmVzdCkKICAgICAgICBtZXRhID0geyJlcG9jaCI6IHN0YXRlLmVwb2NoLCAiZ2xvYmFsX3N0ZXAiOiBzdGF0ZS5nbG9iYWxfc3RlcCwKICAgICAgICAgICAgICAgICJiZXN0X21ldHJpYyI6IHN0YXRlLmJlc3RfbWV0cmljLCAiYmVzdF9zdGVwIjogc3RhdGUuYmVzdF9zdGVwLAogICAgICAgICAgICAgICAgImhhc19iZXN0Ijogc2VsZi5iZXN0LmlzX2ZpbGUoKSwgImNvbmZpZyI6IGNvbmZpZ30KICAgICAgICB0bXAzID0gc2VsZi5kaXIgLyAiY2hlY2twb2ludF9tZXRhLnRtcC5qc29uIgogICAgICAgIHRtcDMud3JpdGVfdGV4dChqc29uLmR1bXBzKG1ldGEsIGluZGVudD0yLCBkZWZhdWx0PXN0ciksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgdG1wMy5yZXBsYWNlKHNlbGYubWV0YV9wYXRoKQogICAgICAgIHJldHVybiBtZXRhCgogICAgZGVmIGxvYWQoc2VsZiwgbW9kZWwsIG9wdGltaXplcj1Ob25lLCBzY2hlZHVsZXI9Tm9uZSwgcGF0aDogUGF0aCB8IE5vbmUgPSBOb25lCiAgICAgICAgICAgICApIC0+IFRyYWluU3RhdGUgfCBOb25lOgogICAgICAgICMgRXhpc3RlbmNlIGlzIGNoZWNrZWQgQkVGT1JFIGltcG9ydGluZyB0b3JjaDogIm5vIGNoZWNrcG9pbnQgeWV0IiBpcwogICAgICAgICMgdGhlIG5vcm1hbCBmaXJzdC1ydW4gY2FzZSBhbmQgbXVzdCBub3QgcmVxdWlyZSBhIHRvcmNoIGluc3RhbGwuCiAgICAgICAgcCA9IFBhdGgocGF0aCkgaWYgcGF0aCBlbHNlIHNlbGYubGF0ZXN0CiAgICAgICAgaWYgbm90IHAuaXNfZmlsZSgpOgogICAgICAgICAgICByZXR1cm4gTm9uZQoKICAgICAgICBpbXBvcnQgdG9yY2gKCiAgICAgICAgcGF5bG9hZCA9IHRvcmNoLmxvYWQocCwgbWFwX2xvY2F0aW9uPSJjcHUiLCB3ZWlnaHRzX29ubHk9RmFsc2UpCiAgICAgICAgbW9kZWwubG9hZF9zdGF0ZV9kaWN0KHBheWxvYWRbIm1vZGVsX3N0YXRlIl0pCiAgICAgICAgaWYgb3B0aW1pemVyIGlzIG5vdCBOb25lIGFuZCBwYXlsb2FkLmdldCgib3B0aW1pemVyX3N0YXRlIik6CiAgICAgICAgICAgIG9wdGltaXplci5sb2FkX3N0YXRlX2RpY3QocGF5bG9hZFsib3B0aW1pemVyX3N0YXRlIl0pCiAgICAgICAgaWYgc2NoZWR1bGVyIGlzIG5vdCBOb25lIGFuZCBwYXlsb2FkLmdldCgic2NoZWR1bGVyX3N0YXRlIik6CiAgICAgICAgICAgIHNjaGVkdWxlci5sb2FkX3N0YXRlX2RpY3QocGF5bG9hZFsic2NoZWR1bGVyX3N0YXRlIl0pCiAgICAgICAgcmVzdG9yZV9ybmdfc3RhdGUocGF5bG9hZC5nZXQoInJuZ19zdGF0ZSIsIHt9KSkKICAgICAgICByZXR1cm4gVHJhaW5TdGF0ZS5mcm9tX2RpY3QocGF5bG9hZC5nZXQoInRyYWluX3N0YXRlIiwge30pKQoKICAgIGRlZiBleGlzdHMoc2VsZikgLT4gYm9vbDoKICAgICAgICByZXR1cm4gc2VsZi5sYXRlc3QuaXNfZmlsZSgpCgoKZGVmIG1hY3JvX2YxKHlfdHJ1ZTogU2VxdWVuY2VbaW50XSwgeV9wcmVkOiBTZXF1ZW5jZVtpbnRdLCBuX2NsYXNzZXM6IGludCkgLT4gZmxvYXQ6CiAgICAiIiJNYWNyby1GMSB3aXRob3V0IHNrbGVhcm4sIHNvIG1ldHJpY3Mgd29yayBpbiBhbnkgZW52aXJvbm1lbnQuCgogICAgQSBjbGFzcyB3aXRoIG5vIHRydWUgYW5kIG5vIHByZWRpY3RlZCBpbnN0YW5jZXMgY29udHJpYnV0ZXMgRjEgPSAwIGFuZCBpcwogICAgc3RpbGwgY291bnRlZCBpbiB0aGUgZGVub21pbmF0b3I7IHNpbGVudGx5IHNraXBwaW5nIGl0IHdvdWxkIGluZmxhdGUgdGhlCiAgICBzY29yZSBvbiB0aGUgcmFyZSBYLUZBQ1QgY2xhc3Nlcy4KICAgICIiIgogICAgZjFzID0gW10KICAgIGZvciBjIGluIHJhbmdlKG5fY2xhc3Nlcyk6CiAgICAgICAgdHAgPSBzdW0oMSBmb3IgdCwgcCBpbiB6aXAoeV90cnVlLCB5X3ByZWQpIGlmIHQgPT0gYyBhbmQgcCA9PSBjKQogICAgICAgIGZwID0gc3VtKDEgZm9yIHQsIHAgaW4gemlwKHlfdHJ1ZSwgeV9wcmVkKSBpZiB0ICE9IGMgYW5kIHAgPT0gYykKICAgICAgICBmbiA9IHN1bSgxIGZvciB0LCBwIGluIHppcCh5X3RydWUsIHlfcHJlZCkgaWYgdCA9PSBjIGFuZCBwICE9IGMpCiAgICAgICAgcHJlYyA9IHRwIC8gKHRwICsgZnApIGlmICh0cCArIGZwKSBlbHNlIDAuMAogICAgICAgIHJlYyA9IHRwIC8gKHRwICsgZm4pIGlmICh0cCArIGZuKSBlbHNlIDAuMAogICAgICAgIGYxcy5hcHBlbmQoMiAqIHByZWMgKiByZWMgLyAocHJlYyArIHJlYykgaWYgKHByZWMgKyByZWMpIGVsc2UgMC4wKQogICAgcmV0dXJuIHN1bShmMXMpIC8gbGVuKGYxcykgaWYgZjFzIGVsc2UgMC4wCgoKZGVmIGNsYXNzaWZpY2F0aW9uX3JlcG9ydCh5X3RydWU6IFNlcXVlbmNlW2ludF0sIHlfcHJlZDogU2VxdWVuY2VbaW50XSwKICAgICAgICAgICAgICAgICAgICAgICAgICBsYWJlbHM6IFNlcXVlbmNlW3N0cl0pIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgbiA9IGxlbihsYWJlbHMpCiAgICBhY2MgPSBzdW0oMSBmb3IgdCwgcCBpbiB6aXAoeV90cnVlLCB5X3ByZWQpIGlmIHQgPT0gcCkgLyBsZW4oeV90cnVlKSBpZiB5X3RydWUgZWxzZSAwLjAKICAgIHBlcjogZGljdFtzdHIsIGRpY3Rbc3RyLCBmbG9hdF1dID0ge30KICAgIHByZWNzLCByZWNzID0gW10sIFtdCiAgICBmb3IgYyBpbiByYW5nZShuKToKICAgICAgICB0cCA9IHN1bSgxIGZvciB0LCBwIGluIHppcCh5X3RydWUsIHlfcHJlZCkgaWYgdCA9PSBjIGFuZCBwID09IGMpCiAgICAgICAgZnAgPSBzdW0oMSBmb3IgdCwgcCBpbiB6aXAoeV90cnVlLCB5X3ByZWQpIGlmIHQgIT0gYyBhbmQgcCA9PSBjKQogICAgICAgIGZuID0gc3VtKDEgZm9yIHQsIHAgaW4gemlwKHlfdHJ1ZSwgeV9wcmVkKSBpZiB0ID09IGMgYW5kIHAgIT0gYykKICAgICAgICBwcmVjID0gdHAgLyAodHAgKyBmcCkgaWYgKHRwICsgZnApIGVsc2UgMC4wCiAgICAgICAgcmVjID0gdHAgLyAodHAgKyBmbikgaWYgKHRwICsgZm4pIGVsc2UgMC4wCiAgICAgICAgZjEgPSAyICogcHJlYyAqIHJlYyAvIChwcmVjICsgcmVjKSBpZiAocHJlYyArIHJlYykgZWxzZSAwLjAKICAgICAgICBwZXJbbGFiZWxzW2NdXSA9IHsicHJlY2lzaW9uIjogcHJlYywgInJlY2FsbCI6IHJlYywgImYxIjogZjEsCiAgICAgICAgICAgICAgICAgICAgICAgICAgInN1cHBvcnQiOiBzdW0oMSBmb3IgdCBpbiB5X3RydWUgaWYgdCA9PSBjKX0KICAgICAgICBwcmVjcy5hcHBlbmQocHJlYykKICAgICAgICByZWNzLmFwcGVuZChyZWMpCiAgICBjbSA9IFtbc3VtKDEgZm9yIHQsIHAgaW4gemlwKHlfdHJ1ZSwgeV9wcmVkKSBpZiB0ID09IGkgYW5kIHAgPT0gaikgZm9yIGogaW4gcmFuZ2UobildCiAgICAgICAgICBmb3IgaSBpbiByYW5nZShuKV0KICAgIHJldHVybiB7ImFjY3VyYWN5IjogYWNjLCAibWFjcm9fZjEiOiBtYWNyb19mMSh5X3RydWUsIHlfcHJlZCwgbiksCiAgICAgICAgICAgICJwcmVjaXNpb25fbWFjcm8iOiBzdW0ocHJlY3MpIC8gbiBpZiBuIGVsc2UgMC4wLAogICAgICAgICAgICAicmVjYWxsX21hY3JvIjogc3VtKHJlY3MpIC8gbiBpZiBuIGVsc2UgMC4wLAogICAgICAgICAgICAicGVyX2NsYXNzIjogcGVyLCAiY29uZnVzaW9uX21hdHJpeCI6IGNtLCAibGFiZWxzIjogbGlzdChsYWJlbHMpfQoKCmNsYXNzIFZlcmlmaWVyVHJhaW5lcjoKICAgICIiIlRyYWluaW5nIGxvb3Agd2l0aCBwZXItc3RlcCBjaGVja3BvaW50aW5nIGFuZCBlYXJseSBzdG9wcGluZy4KCiAgICBgc3RlcF9mbmAgYW5kIGBldmFsX2ZuYCBhcmUgaW5qZWN0ZWQgc28gdGhlIHJlc3VtZSBsb2dpYyBjYW4gYmUgdGVzdGVkCiAgICB3aXRob3V0IHRvcmNoLCBhIEdQVSwgb3IgYW55IGRvd25sb2FkZWQgd2VpZ2h0cy4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBja3B0X2RpcjogUGF0aCwgY29uZmlnOiBkaWN0W3N0ciwgQW55XSwKICAgICAgICAgICAgICAgICBjaGVja3BvaW50X2V2ZXJ5X3N0ZXBzOiBpbnQgPSA1MDAsCiAgICAgICAgICAgICAgICAgZWFybHlfc3RvcHBpbmdfcGF0aWVuY2U6IGludCA9IDIpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5ja3B0ID0gQ2hlY2twb2ludE1hbmFnZXIoY2twdF9kaXIpCiAgICAgICAgc2VsZi5jb25maWcgPSBjb25maWcKICAgICAgICBzZWxmLmNoZWNrcG9pbnRfZXZlcnlfc3RlcHMgPSBpbnQoY2hlY2twb2ludF9ldmVyeV9zdGVwcykKICAgICAgICBzZWxmLnBhdGllbmNlID0gaW50KGVhcmx5X3N0b3BwaW5nX3BhdGllbmNlKQoKICAgIGRlZiB0cmFpbihzZWxmLCBtb2RlbCwgb3B0aW1pemVyLCBzY2hlZHVsZXIsIGJhdGNoZXNfZm46IENhbGxhYmxlW1tpbnRdLCBTZXF1ZW5jZVtBbnldXSwKICAgICAgICAgICAgICBzdGVwX2ZuOiBDYWxsYWJsZVtbQW55LCBBbnldLCBmbG9hdF0sIGV2YWxfZm46IENhbGxhYmxlW1tBbnldLCBkaWN0W3N0ciwgQW55XV0sCiAgICAgICAgICAgICAgZXBvY2hzOiBpbnQsIHJlc3VtZTogYm9vbCA9IFRydWUsCiAgICAgICAgICAgICAgbG9nOiBDYWxsYWJsZVtbc3RyXSwgTm9uZV0gPSBwcmludCkgLT4gZGljdFtzdHIsIEFueV06CiAgICAgICAgc3RhdGUgPSBUcmFpblN0YXRlKCkKICAgICAgICBpZiByZXN1bWUgYW5kIHNlbGYuY2twdC5leGlzdHMoKToKICAgICAgICAgICAgbG9hZGVkID0gc2VsZi5ja3B0LmxvYWQobW9kZWwsIG9wdGltaXplciwgc2NoZWR1bGVyKQogICAgICAgICAgICBpZiBsb2FkZWQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICBzdGF0ZSA9IGxvYWRlZAogICAgICAgICAgICAgICAgbG9nKGYiICByZXN1bWVkIGF0IGVwb2NoIHtzdGF0ZS5lcG9jaH0sIHN0ZXAge3N0YXRlLmdsb2JhbF9zdGVwfSIpCgogICAgICAgIHQwID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIHN0b3BwZWRfZWFybHkgPSBGYWxzZQogICAgICAgIGZvciBlcG9jaCBpbiByYW5nZShzdGF0ZS5lcG9jaCwgaW50KGVwb2NocykpOgogICAgICAgICAgICBzdGF0ZS5lcG9jaCA9IGVwb2NoCiAgICAgICAgICAgIGZvciBiYXRjaCBpbiBiYXRjaGVzX2ZuKGVwb2NoKToKICAgICAgICAgICAgICAgIGxvc3MgPSBzdGVwX2ZuKG1vZGVsLCBiYXRjaCkKICAgICAgICAgICAgICAgIHN0YXRlLmdsb2JhbF9zdGVwICs9IDEKICAgICAgICAgICAgICAgIGlmIHNlbGYuY2hlY2twb2ludF9ldmVyeV9zdGVwcyBhbmQgXAogICAgICAgICAgICAgICAgICAgICAgICBzdGF0ZS5nbG9iYWxfc3RlcCAlIHNlbGYuY2hlY2twb2ludF9ldmVyeV9zdGVwcyA9PSAwOgogICAgICAgICAgICAgICAgICAgIHNlbGYuY2twdC5zYXZlKG1vZGVsLCBvcHRpbWl6ZXIsIHNjaGVkdWxlciwgc3RhdGUsIHNlbGYuY29uZmlnKQoKICAgICAgICAgICAgbWV0cmljcyA9IGV2YWxfZm4obW9kZWwpCiAgICAgICAgICAgIHNjb3JlID0gZmxvYXQobWV0cmljcy5nZXQoIm1hY3JvX2YxIiwgMC4wKSkKICAgICAgICAgICAgaW1wcm92ZWQgPSBzY29yZSA+IHN0YXRlLmJlc3RfbWV0cmljCiAgICAgICAgICAgIGlmIGltcHJvdmVkOgogICAgICAgICAgICAgICAgc3RhdGUuYmVzdF9tZXRyaWMsIHN0YXRlLmJlc3Rfc3RlcCA9IHNjb3JlLCBzdGF0ZS5nbG9iYWxfc3RlcAogICAgICAgICAgICAgICAgc3RhdGUuZXBvY2hzX3dpdGhvdXRfaW1wcm92ZW1lbnQgPSAwCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBzdGF0ZS5lcG9jaHNfd2l0aG91dF9pbXByb3ZlbWVudCArPSAxCiAgICAgICAgICAgIHN0YXRlLmhpc3RvcnkuYXBwZW5kKHsiZXBvY2giOiBlcG9jaCwgInN0ZXAiOiBzdGF0ZS5nbG9iYWxfc3RlcCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJsb3NzIjogZmxvYXQobG9zcyksICoqbWV0cmljc30pCiAgICAgICAgICAgIHNlbGYuY2twdC5zYXZlKG1vZGVsLCBvcHRpbWl6ZXIsIHNjaGVkdWxlciwgc3RhdGUsIHNlbGYuY29uZmlnLCBpc19iZXN0PWltcHJvdmVkKQogICAgICAgICAgICBsb2coZiIgIGVwb2NoIHtlcG9jaH06IG1hY3JvX2YxPXtzY29yZTouNGZ9IgogICAgICAgICAgICAgICAgZiJ7JyAoYmVzdCknIGlmIGltcHJvdmVkIGVsc2UgJyd9IikKCiAgICAgICAgICAgIGlmIHN0YXRlLmVwb2Noc193aXRob3V0X2ltcHJvdmVtZW50ID49IHNlbGYucGF0aWVuY2U6CiAgICAgICAgICAgICAgICBsb2coZiIgIGVhcmx5IHN0b3BwaW5nOiBubyBpbXByb3ZlbWVudCBmb3Ige3NlbGYucGF0aWVuY2V9IGVwb2NoKHMpIikKICAgICAgICAgICAgICAgIHN0b3BwZWRfZWFybHkgPSBUcnVlCiAgICAgICAgICAgICAgICBzdGF0ZS5lcG9jaCA9IGVwb2NoICsgMQogICAgICAgICAgICAgICAgc2VsZi5ja3B0LnNhdmUobW9kZWwsIG9wdGltaXplciwgc2NoZWR1bGVyLCBzdGF0ZSwgc2VsZi5jb25maWcpCiAgICAgICAgICAgICAgICBicmVhawogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHN0YXRlLmVwb2NoID0gaW50KGVwb2NocykKICAgICAgICAgICAgc2VsZi5ja3B0LnNhdmUobW9kZWwsIG9wdGltaXplciwgc2NoZWR1bGVyLCBzdGF0ZSwgc2VsZi5jb25maWcpCgogICAgICAgIHJldHVybiB7ImVwb2Noc19ydW4iOiBzdGF0ZS5lcG9jaCwgImdsb2JhbF9zdGVwIjogc3RhdGUuZ2xvYmFsX3N0ZXAsCiAgICAgICAgICAgICAgICAiYmVzdF9tZXRyaWMiOiBzdGF0ZS5iZXN0X21ldHJpYywgImJlc3Rfc3RlcCI6IHN0YXRlLmJlc3Rfc3RlcCwKICAgICAgICAgICAgICAgICJzdG9wcGVkX2Vhcmx5Ijogc3RvcHBlZF9lYXJseSwgImhpc3RvcnkiOiBzdGF0ZS5oaXN0b3J5LAogICAgICAgICAgICAgICAgInNlY29uZHMiOiByb3VuZCh0aW1lLnBlcmZfY291bnRlcigpIC0gdDAsIDIpfQo=", "sha256": "3b9eab58eaadecf91e410d018d77535dfbde3f186d49f051077abf351386fa64", "size": 10841}, "src/verification/calibration.py": {"b64": "IiIiTW9kdWxlIDYg4oCUIGNhbGlicmF0aW9uOiBFQ0UsIHJlbGlhYmlsaXR5IGRhdGEsIHRlbXBlcmF0dXJlIHNjYWxpbmcuCgpBIGZhY3QtdmVyaWZpY2F0aW9uIHN5c3RlbSB0aGF0IGlzIGNvbmZpZGVudGx5IHdyb25nIGlzIHdvcnNlIHRoYW4gb25lIHRoYXQKaXMgdW5zdXJlLCBiZWNhdXNlIGRvd25zdHJlYW0gdGhlIGNvbmZpZGVuY2UgaXMgd2hhdCBhIHJlYWRlciB3b3VsZCBhY3Qgb24uCkFjY3VyYWN5IGFsb25lIGNhbm5vdCBzaG93IHRoaXM7IEVDRSBjYW4uCgpFQ0UgaGVyZSBpcyB0aGUgc3RhbmRhcmQgKmNvbmZpZGVuY2UqIEVDRSAoR3VvIGV0IGFsLiAyMDE3KTogYmluIHByZWRpY3Rpb25zCmJ5IG1heCBzb2Z0bWF4IHByb2JhYmlsaXR5LCBhbmQgYXZlcmFnZSB8YWNjdXJhY3kgLSBjb25maWRlbmNlfCBwZXIgYmluLAp3ZWlnaHRlZCBieSBiaW4gcG9wdWxhdGlvbi4KClRlbXBlcmF0dXJlIHNjYWxpbmcgaXMgZml0dGVkIG9uIFZBTElEQVRJT04gb25seSBhbmQgdGhlbiBhcHBsaWVkIHRvIHRlc3QuCkZpdHRpbmcgaXQgb24gdGVzdCB3b3VsZCBiZSB0dW5pbmcgb24gdGhlIHRlc3Qgc2V0LCB3aGljaCB0aGUgcHJvamVjdCdzCnByb3RvY29sIGZvcmJpZHMgKCJ0ZXN0IGV2YWx1YXRlZCBvbmNlIikuIGBmaXRfdGVtcGVyYXR1cmUoKWAgcmVmdXNlcyBhCnNwbGl0IG5hbWVkICJ0ZXN0IiBhcyBhIGd1YXJkIGFnYWluc3QgZG9pbmcgdGhpcyBieSBhY2NpZGVudC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbWF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBTZXF1ZW5jZQoKaW1wb3J0IG51bXB5IGFzIG5wCgoKZGVmIHNvZnRtYXgobG9naXRzOiBucC5uZGFycmF5LCB0ZW1wZXJhdHVyZTogZmxvYXQgPSAxLjApIC0+IG5wLm5kYXJyYXk6CiAgICBpZiB0ZW1wZXJhdHVyZSA8PSAwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ0ZW1wZXJhdHVyZSBtdXN0IGJlID4gMCwgZ290IHt0ZW1wZXJhdHVyZX0iKQogICAgeiA9IG5wLmFzYXJyYXkobG9naXRzLCBkdHlwZT1ucC5mbG9hdDY0KSAvIGZsb2F0KHRlbXBlcmF0dXJlKQogICAgeiA9IHogLSB6Lm1heChheGlzPS0xLCBrZWVwZGltcz1UcnVlKSAgICAgICAgICAjIHN0YWJpbGl0eQogICAgZSA9IG5wLmV4cCh6KQogICAgcmV0dXJuIGUgLyBlLnN1bShheGlzPS0xLCBrZWVwZGltcz1UcnVlKQoKCmRlZiBleHBlY3RlZF9jYWxpYnJhdGlvbl9lcnJvcihwcm9iczogbnAubmRhcnJheSwgbGFiZWxzOiBTZXF1ZW5jZVtpbnRdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbl9iaW5zOiBpbnQgPSAxMCkgLT4gZGljdFtzdHIsIEFueV06CiAgICAiIiJDb25maWRlbmNlIEVDRSB3aXRoIGVxdWFsLXdpZHRoIGJpbnMgb3ZlciBbMCwgMV0uCgogICAgQmluIGVkZ2VzIGFyZSBoYWxmLW9wZW4gKGxvLCBoaV0gd2l0aCB0aGUgZmlyc3QgYmluIGNsb3NlZCBhdCAwLCBzbyBldmVyeQogICAgY29uZmlkZW5jZSBsYW5kcyBpbiBleGFjdGx5IG9uZSBiaW4gYW5kIHRoZSBiaW4gcG9wdWxhdGlvbnMgc3VtIHRvIE4uCiAgICAiIiIKICAgIHByb2JzID0gbnAuYXNhcnJheShwcm9icywgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIGlmIHByb2JzLm5kaW0gIT0gMjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYicHJvYnMgbXVzdCBiZSAobiwgbl9jbGFzc2VzKSwgZ290IHtwcm9icy5zaGFwZX0iKQogICAgbGFiZWxzID0gbnAuYXNhcnJheShsYWJlbHMsIGR0eXBlPWludCkKICAgIGlmIGxlbihsYWJlbHMpICE9IHByb2JzLnNoYXBlWzBdOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ7bGVuKGxhYmVscyl9IGxhYmVscyBidXQge3Byb2JzLnNoYXBlWzBdfSByb3dzIikKICAgIGlmIHByb2JzLnNoYXBlWzBdID09IDA6CiAgICAgICAgcmV0dXJuIHsiZWNlIjogMC4wLCAibiI6IDAsICJuX2JpbnMiOiBuX2JpbnMsICJiaW5zIjogW119CgogICAgY29uZiA9IHByb2JzLm1heChheGlzPTEpCiAgICBwcmVkID0gcHJvYnMuYXJnbWF4KGF4aXM9MSkKICAgIGNvcnJlY3QgPSAocHJlZCA9PSBsYWJlbHMpLmFzdHlwZShucC5mbG9hdDY0KQoKICAgIGVkZ2VzID0gbnAubGluc3BhY2UoMC4wLCAxLjAsIG5fYmlucyArIDEpCiAgICBiaW5zOiBsaXN0W2RpY3Rbc3RyLCBBbnldXSA9IFtdCiAgICBlY2UgPSAwLjAKICAgIG4gPSBsZW4obGFiZWxzKQogICAgZm9yIGkgaW4gcmFuZ2Uobl9iaW5zKToKICAgICAgICBsbywgaGkgPSBlZGdlc1tpXSwgZWRnZXNbaSArIDFdCiAgICAgICAgbWFzayA9IChjb25mID4gbG8pICYgKGNvbmYgPD0gaGkpIGlmIGkgPiAwIGVsc2UgKGNvbmYgPj0gbG8pICYgKGNvbmYgPD0gaGkpCiAgICAgICAgY291bnQgPSBpbnQobWFzay5zdW0oKSkKICAgICAgICBpZiBjb3VudCA9PSAwOgogICAgICAgICAgICBiaW5zLmFwcGVuZCh7ImJpbiI6IGksICJsbyI6IGZsb2F0KGxvKSwgImhpIjogZmxvYXQoaGkpLCAiY291bnQiOiAwLAogICAgICAgICAgICAgICAgICAgICAgICAgImFjY3VyYWN5IjogTm9uZSwgImNvbmZpZGVuY2UiOiBOb25lLCAiZ2FwIjogTm9uZX0pCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgYWNjID0gZmxvYXQoY29ycmVjdFttYXNrXS5tZWFuKCkpCiAgICAgICAgYXZnX2NvbmYgPSBmbG9hdChjb25mW21hc2tdLm1lYW4oKSkKICAgICAgICBnYXAgPSBhYnMoYWNjIC0gYXZnX2NvbmYpCiAgICAgICAgZWNlICs9IChjb3VudCAvIG4pICogZ2FwCiAgICAgICAgYmlucy5hcHBlbmQoeyJiaW4iOiBpLCAibG8iOiBmbG9hdChsbyksICJoaSI6IGZsb2F0KGhpKSwgImNvdW50IjogY291bnQsCiAgICAgICAgICAgICAgICAgICAgICJhY2N1cmFjeSI6IGFjYywgImNvbmZpZGVuY2UiOiBhdmdfY29uZiwgImdhcCI6IGdhcH0pCgogICAgcmV0dXJuIHsiZWNlIjogZmxvYXQoZWNlKSwgIm4iOiBuLCAibl9iaW5zIjogbl9iaW5zLCAiYmlucyI6IGJpbnMsCiAgICAgICAgICAgICJtZWFuX2NvbmZpZGVuY2UiOiBmbG9hdChjb25mLm1lYW4oKSksICJhY2N1cmFjeSI6IGZsb2F0KGNvcnJlY3QubWVhbigpKX0KCgpkZWYgcmVsaWFiaWxpdHlfZGF0YShwcm9iczogbnAubmRhcnJheSwgbGFiZWxzOiBTZXF1ZW5jZVtpbnRdLAogICAgICAgICAgICAgICAgICAgICBuX2JpbnM6IGludCA9IDEwKSAtPiBkaWN0W3N0ciwgbGlzdF06CiAgICAiIiJQb2ludHMgZm9yIGEgcmVsaWFiaWxpdHkgZGlhZ3JhbS4gRW1wdHkgYmlucyBhcmUgb21pdHRlZCwgbm90IHBsb3R0ZWQgYXQgemVyby4iIiIKICAgIHJlcyA9IGV4cGVjdGVkX2NhbGlicmF0aW9uX2Vycm9yKHByb2JzLCBsYWJlbHMsIG5fYmlucykKICAgIHB0cyA9IFtiIGZvciBiIGluIHJlc1siYmlucyJdIGlmIGJbImNvdW50Il0gPiAwXQogICAgcmV0dXJuIHsiY29uZmlkZW5jZSI6IFtiWyJjb25maWRlbmNlIl0gZm9yIGIgaW4gcHRzXSwKICAgICAgICAgICAgImFjY3VyYWN5IjogW2JbImFjY3VyYWN5Il0gZm9yIGIgaW4gcHRzXSwKICAgICAgICAgICAgImNvdW50cyI6IFtiWyJjb3VudCJdIGZvciBiIGluIHB0c10sCiAgICAgICAgICAgICJlY2UiOiByZXNbImVjZSJdfQoKCmRlZiBmaXRfdGVtcGVyYXR1cmUobG9naXRzOiBucC5uZGFycmF5LCBsYWJlbHM6IFNlcXVlbmNlW2ludF0sIHNwbGl0X25hbWU6IHN0ciA9ICJ2YWxpZGF0aW9uIiwKICAgICAgICAgICAgICAgICAgICBsbzogZmxvYXQgPSAwLjA1LCBoaTogZmxvYXQgPSAxMC4wLCB0b2w6IGZsb2F0ID0gMWUtNCkgLT4gZGljdFtzdHIsIEFueV06CiAgICAiIiJGaXQgYSBzaW5nbGUgdGVtcGVyYXR1cmUgYnkgbWluaW1pc2luZyBOTEwgdmlhIHRlcm5hcnkgc2VhcmNoLgoKICAgIFRoZSBOTEwgb2YgdGVtcGVyYXR1cmUgc2NhbGluZyBpcyBjb252ZXggaW4gbG9nLXRlbXBlcmF0dXJlLCBzbyB0ZXJuYXJ5CiAgICBzZWFyY2ggY29udmVyZ2VzIHdpdGhvdXQgbmVlZGluZyBhbiBvcHRpbWlzZXIgb3IgYSBncmFkaWVudCAtLSB3aGljaCBrZWVwcwogICAgdGhpcyBkZXRlcm1pbmlzdGljIGFuZCBkZXBlbmRlbmN5LWZyZWUuCiAgICAiIiIKICAgIGlmIHN0cihzcGxpdF9uYW1lKS5sb3dlcigpID09ICJ0ZXN0IjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAiUmVmdXNpbmcgdG8gZml0IHRlbXBlcmF0dXJlIG9uIHRoZSB0ZXN0IHNwbGl0OiB0aGF0IGlzIHR1bmluZyBvbiB0ZXN0LiAiCiAgICAgICAgICAgICJGaXQgb24gdmFsaWRhdGlvbiBhbmQgYXBwbHkgdGhlIGZpdHRlZCB2YWx1ZSB0byB0ZXN0LiIpCiAgICBsb2dpdHMgPSBucC5hc2FycmF5KGxvZ2l0cywgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIGxhYmVscyA9IG5wLmFzYXJyYXkobGFiZWxzLCBkdHlwZT1pbnQpCiAgICBpZiBsb2dpdHMuc2hhcGVbMF0gPT0gMDoKICAgICAgICByZXR1cm4geyJ0ZW1wZXJhdHVyZSI6IDEuMCwgIm5sbF9iZWZvcmUiOiAwLjAsICJubGxfYWZ0ZXIiOiAwLjAsICJzcGxpdCI6IHNwbGl0X25hbWV9CgogICAgZGVmIG5sbCh0OiBmbG9hdCkgLT4gZmxvYXQ6CiAgICAgICAgcCA9IHNvZnRtYXgobG9naXRzLCB0KQogICAgICAgIHJldHVybiBmbG9hdCgtbnAubG9nKG5wLmNsaXAocFtucC5hcmFuZ2UobGVuKGxhYmVscykpLCBsYWJlbHNdLCAxZS0xMiwgMS4wKSkubWVhbigpKQoKICAgIGEsIGIgPSBtYXRoLmxvZyhsbyksIG1hdGgubG9nKGhpKQogICAgd2hpbGUgYiAtIGEgPiB0b2w6CiAgICAgICAgbTEsIG0yID0gYSArIChiIC0gYSkgLyAzLjAsIGIgLSAoYiAtIGEpIC8gMy4wCiAgICAgICAgaWYgbmxsKG1hdGguZXhwKG0xKSkgPCBubGwobWF0aC5leHAobTIpKToKICAgICAgICAgICAgYiA9IG0yCiAgICAgICAgZWxzZToKICAgICAgICAgICAgYSA9IG0xCiAgICB0ID0gbWF0aC5leHAoKGEgKyBiKSAvIDIuMCkKICAgIHJldHVybiB7InRlbXBlcmF0dXJlIjogZmxvYXQodCksICJubGxfYmVmb3JlIjogbmxsKDEuMCksICJubGxfYWZ0ZXIiOiBubGwodCksCiAgICAgICAgICAgICJzcGxpdCI6IHNwbGl0X25hbWUsCiAgICAgICAgICAgICJub3RlIjogIkZpdHRlZCBvbiB2YWxpZGF0aW9uOyBhcHBseSB0aGlzIHZhbHVlIHVuY2hhbmdlZCB0byB0ZXN0LiJ9CgoKZGVmIGNhbGlicmF0ZV9hbmRfZXZhbHVhdGUobG9naXRzOiBucC5uZGFycmF5LCBsYWJlbHM6IFNlcXVlbmNlW2ludF0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgIHRlbXBlcmF0dXJlOiBmbG9hdCA9IDEuMCwgbl9iaW5zOiBpbnQgPSAxMCkgLT4gZGljdFtzdHIsIEFueV06CiAgICAiIiJFQ0UgYmVmb3JlIGFuZCBhZnRlciBhcHBseWluZyBhbiBhbHJlYWR5LWZpdHRlZCB0ZW1wZXJhdHVyZS4iIiIKICAgIGJlZm9yZSA9IGV4cGVjdGVkX2NhbGlicmF0aW9uX2Vycm9yKHNvZnRtYXgobG9naXRzLCAxLjApLCBsYWJlbHMsIG5fYmlucykKICAgIGFmdGVyID0gZXhwZWN0ZWRfY2FsaWJyYXRpb25fZXJyb3Ioc29mdG1heChsb2dpdHMsIHRlbXBlcmF0dXJlKSwgbGFiZWxzLCBuX2JpbnMpCiAgICByZXR1cm4gewogICAgICAgICJ0ZW1wZXJhdHVyZSI6IGZsb2F0KHRlbXBlcmF0dXJlKSwKICAgICAgICAiZWNlX2JlZm9yZSI6IGJlZm9yZVsiZWNlIl0sICJlY2VfYWZ0ZXIiOiBhZnRlclsiZWNlIl0sCiAgICAgICAgImFjY3VyYWN5IjogYmVmb3JlWyJhY2N1cmFjeSJdLCAgICMgdW5jaGFuZ2VkOiB0ZW1wZXJhdHVyZSBjYW5ub3QgYWx0ZXIgYXJnbWF4CiAgICAgICAgInJlbGlhYmlsaXR5X2JlZm9yZSI6IHJlbGlhYmlsaXR5X2RhdGEoc29mdG1heChsb2dpdHMsIDEuMCksIGxhYmVscywgbl9iaW5zKSwKICAgICAgICAicmVsaWFiaWxpdHlfYWZ0ZXIiOiByZWxpYWJpbGl0eV9kYXRhKHNvZnRtYXgobG9naXRzLCB0ZW1wZXJhdHVyZSksIGxhYmVscywgbl9iaW5zKSwKICAgICAgICAibm90ZSI6ICgiVGVtcGVyYXR1cmUgc2NhbGluZyBpcyBtb25vdG9uZSwgc28gYWNjdXJhY3kgYW5kIE1hY3JvLUYxIGFyZSAiCiAgICAgICAgICAgICAgICAgImlkZW50aWNhbCBiZWZvcmUgYW5kIGFmdGVyOyBvbmx5IGNvbmZpZGVuY2UgY2hhbmdlcy4iKSwKICAgIH0K", "sha256": "d5f5b1b75e06c656c2ebd3f162aa03f3cf5832f0be33e060b2dcd77ca9562049", "size": 6015}, "src/verification/ablations.py": {"b64": "IiIiTW9kdWxlIDYg4oCUIHRoZSBzaXggY29uZmlndXJlZCBhYmxhdGlvbnMuCgpFYWNoIGFibGF0aW9uIGlzb2xhdGVzIE9ORSBkZXNpZ24gZGVjaXNpb24gc28gdGhlIHBhcGVyIGNhbiBhdHRyaWJ1dGUgYW55CmdhaW4gdG8gYSBzcGVjaWZpYyBjYXVzZSByYXRoZXIgdGhhbiB0byB0aGUgcGlwZWxpbmUgYXMgYSB3aG9sZS4gRXZlcnkKYWJsYXRpb24gaXMgYSBwdXJlIHNwZWNpZmljYXRpb24gKGEgZGljdCkgcGx1cyBhIGRldGVybWluaXN0aWMgcnVubmVyLCBzbwpyZS1ydW5uaW5nIHdpdGggdGhlIHNhbWUgc2VlZCByZXByb2R1Y2VzIHRoZSBzYW1lIHJlc3VsdC4KCiAgICBibTI1X3ZzX2RlbnNlICAgICAgICAgICAgaXMgdGhlIGRlbnNlIHJldHJpZXZlciBlYXJuaW5nIGl0cyBwbGFjZT8KICAgIHRvcF9rX3ZhbHVlcyAgICAgICAgICAgICBob3cgbXVjaCBkb2VzIGNhbmRpZGF0ZSBkZXB0aCBtYXR0ZXI/CiAgICBjbGFpbV9vbmx5X3ZzX2V2aWRlbmNlICAgZG9lcyBldmlkZW5jZSBoZWxwIGF0IGFsbCwgb3IgaXMgdGhlIHZlcmlmaWVyCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAganVzdCBsZWFybmluZyBjbGFpbS1zaWRlIHByaW9ycz8KICAgIGVuZ2xpc2hfb25seV92c19tdWx0aSAgICBkb2VzIG11bHRpbGluZ3VhbCBldmlkZW5jZSBoZWxwIFRhbWlsPwogICAgcmV0cmlldmFsX29ubHlfdnNfcmVyYW5rIGRvZXMgdGhlIGV4cGVuc2l2ZSBjcm9zcy1lbmNvZGVyIHBheSBmb3IgaXRzZWxmPwogICAgb3JhY2xlX2dvbGRfZXZpZGVuY2UgICAgIHVwcGVyIGJvdW5kIHdpdGggcGVyZmVjdCByZXRyaWV2YWwKCmBjbGFpbV9vbmx5YCBpcyB0aGUgbW9zdCBpbXBvcnRhbnQgb2YgdGhlc2UuIElmIGEgY2xhaW0tb25seSB2ZXJpZmllciBzY29yZXMKY2xvc2UgdG8gdGhlIGZ1bGwgcGlwZWxpbmUsIHRoZSBzeXN0ZW0gaXMgbm90IHJlYWxseSB2ZXJpZnlpbmcgYWdhaW5zdApldmlkZW5jZSwgYW5kIHJlcG9ydGluZyB0aGUgZnVsbC1waXBlbGluZSBudW1iZXIgYXMgImZhY3QgdmVyaWZpY2F0aW9uIiB3b3VsZApvdmVyc3RhdGUgd2hhdCB3YXMgYnVpbHQuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBDYWxsYWJsZSwgU2VxdWVuY2UKCgpAZGF0YWNsYXNzCmNsYXNzIEFibGF0aW9uU3BlYzoKICAgIG5hbWU6IHN0cgogICAgcXVlc3Rpb246IHN0cgogICAgdmFyaWFudHM6IGxpc3RbZGljdFtzdHIsIEFueV1dCiAgICBtZXRyaWM6IHN0ciA9ICJtYWNyb19mMSIKICAgIHN0YWdlOiBzdHIgPSAidmVyaWZpY2F0aW9uIiAgICAgICAgICAjIG9yICJyZXRyaWV2YWwiCiAgICBub3Rlczogc3RyID0gIiIKCiAgICBkZWYgdG9fZGljdChzZWxmKSAtPiBkaWN0W3N0ciwgQW55XToKICAgICAgICByZXR1cm4geyJuYW1lIjogc2VsZi5uYW1lLCAicXVlc3Rpb24iOiBzZWxmLnF1ZXN0aW9uLCAidmFyaWFudHMiOiBzZWxmLnZhcmlhbnRzLAogICAgICAgICAgICAgICAgIm1ldHJpYyI6IHNlbGYubWV0cmljLCAic3RhZ2UiOiBzZWxmLnN0YWdlLCAibm90ZXMiOiBzZWxmLm5vdGVzfQoKCmRlZiBidWlsZF9hYmxhdGlvbl9zcGVjcyhjZmc6IGRpY3Rbc3RyLCBBbnldKSAtPiBsaXN0W0FibGF0aW9uU3BlY106CiAgICBhYiA9IGNmZy5nZXQoImFibGF0aW9ucyIsIHt9KQogICAgc3BlY3M6IGxpc3RbQWJsYXRpb25TcGVjXSA9IFtdCgogICAgaWYgYWIuZ2V0KCJibTI1X3ZzX2RlbnNlIiwgRmFsc2UpOgogICAgICAgIHNwZWNzLmFwcGVuZChBYmxhdGlvblNwZWMoCiAgICAgICAgICAgIG5hbWU9ImJtMjVfdnNfZGVuc2UiLAogICAgICAgICAgICBxdWVzdGlvbj0iRG9lcyBMYUJTRSBkZW5zZSByZXRyaWV2YWwgYmVhdCBhIEJNMjUgbGV4aWNhbCBiYXNlbGluZT8iLAogICAgICAgICAgICB2YXJpYW50cz1beyJyZXRyaWV2ZXIiOiAiYm0yNSJ9LCB7InJldHJpZXZlciI6ICJkZW5zZSJ9XSwKICAgICAgICAgICAgbWV0cmljPSJyZWNhbGxANSIsIHN0YWdlPSJyZXRyaWV2YWwiLAogICAgICAgICAgICBub3Rlcz0iSWYgQk0yNSBtYXRjaGVzIGRlbnNlLCB0aGUgZGVuc2UgcmV0cmlldmVyIGlzIG5vdCBqdXN0aWZpZWQuIikpCgogICAgaWYgYWIuZ2V0KCJ0b3Bfa192YWx1ZXMiKToKICAgICAgICBzcGVjcy5hcHBlbmQoQWJsYXRpb25TcGVjKAogICAgICAgICAgICBuYW1lPSJ0b3Bfa19zd2VlcCIsCiAgICAgICAgICAgIHF1ZXN0aW9uPSJIb3cgZG9lcyBjYW5kaWRhdGUgZGVwdGggYWZmZWN0IHJldHJpZXZhbCBxdWFsaXR5PyIsCiAgICAgICAgICAgIHZhcmlhbnRzPVt7InRvcF9rIjogaW50KGspfSBmb3IgayBpbiBhYlsidG9wX2tfdmFsdWVzIl1dLAogICAgICAgICAgICBtZXRyaWM9InJlY2FsbEBrIiwgc3RhZ2U9InJldHJpZXZhbCIsCiAgICAgICAgICAgIG5vdGVzPSJSZWNhbGwgaXMgbW9ub3RvbmUgbm9uLWRlY3JlYXNpbmcgaW4gazsgYSBmbGF0IGN1cnZlIG1lYW5zICIKICAgICAgICAgICAgICAgICAgImV4dHJhIGNhbmRpZGF0ZXMgYWRkIG5vdGhpbmcgYW5kIHRvcF9rIGNhbiBiZSBjdXQuIikpCgogICAgaWYgYWIuZ2V0KCJjbGFpbV9vbmx5X3ZzX2NsYWltX2V2aWRlbmNlIiwgRmFsc2UpOgogICAgICAgIHNwZWNzLmFwcGVuZChBYmxhdGlvblNwZWMoCiAgICAgICAgICAgIG5hbWU9ImNsYWltX29ubHlfdnNfY2xhaW1fZXZpZGVuY2UiLAogICAgICAgICAgICBxdWVzdGlvbj0iRG9lcyByZXRyaWV2ZWQgZXZpZGVuY2UgaW1wcm92ZSB2ZXJpZmljYXRpb24gYXQgYWxsPyIsCiAgICAgICAgICAgIHZhcmlhbnRzPVt7ImNsYWltX29ubHkiOiBUcnVlfSwgeyJjbGFpbV9vbmx5IjogRmFsc2V9XSwKICAgICAgICAgICAgbm90ZXM9IlRoZSBjcml0aWNhbCBjb250cm9sLiBBIHNtYWxsIGdhcCBtZWFucyB0aGUgdmVyaWZpZXIgaXMgIgogICAgICAgICAgICAgICAgICAiZXhwbG9pdGluZyBjbGFpbS1zaWRlIHByaW9ycyByYXRoZXIgdGhhbiB2ZXJpZnlpbmcuIikpCgogICAgaWYgYWIuZ2V0KCJlbmdsaXNoX29ubHlfdnNfbXVsdGlsaW5ndWFsX2V2aWRlbmNlIiwgRmFsc2UpOgogICAgICAgIHNwZWNzLmFwcGVuZChBYmxhdGlvblNwZWMoCiAgICAgICAgICAgIG5hbWU9ImVuZ2xpc2hfb25seV92c19tdWx0aWxpbmd1YWxfZXZpZGVuY2UiLAogICAgICAgICAgICBxdWVzdGlvbj0iRG9lcyBtdWx0aWxpbmd1YWwgZXZpZGVuY2UgaGVscCBUYW1pbCBjbGFpbXM/IiwKICAgICAgICAgICAgdmFyaWFudHM9W3siZXZpZGVuY2VfbGFuZ3VhZ2VzIjogWyJlbiJdfSwKICAgICAgICAgICAgICAgICAgICAgIHsiZXZpZGVuY2VfbGFuZ3VhZ2VzIjogWyJlbiIsICJ0YSJdfV0sCiAgICAgICAgICAgIG5vdGVzPSJSZWxldmFudCB0byB0aGUgbG93LXJlc291cmNlIGNsYWltIG9mIHRoaXMgcHJvamVjdC4iKSkKCiAgICBpZiBhYi5nZXQoInJldHJpZXZhbF9vbmx5X3ZzX3JldHJpZXZhbF9yZXJhbmsiLCBGYWxzZSk6CiAgICAgICAgc3BlY3MuYXBwZW5kKEFibGF0aW9uU3BlYygKICAgICAgICAgICAgbmFtZT0icmV0cmlldmFsX29ubHlfdnNfcmV0cmlldmFsX3JlcmFuayIsCiAgICAgICAgICAgIHF1ZXN0aW9uPSJEb2VzIGNyb3NzLWVuY29kZXIgcmVyYW5raW5nIGp1c3RpZnkgaXRzIGNvbXB1dGUgY29zdD8iLAogICAgICAgICAgICB2YXJpYW50cz1beyJyZXJhbmsiOiBGYWxzZX0sIHsicmVyYW5rIjogVHJ1ZX1dLAogICAgICAgICAgICBub3Rlcz0iUmVyYW5raW5nIGRvbWluYXRlcyBNb2R1bGUgNidzIEdQVSBidWRnZXQ7IGlmIHRoZSBnYWluIGlzICIKICAgICAgICAgICAgICAgICAgInNtYWxsIGl0IHNob3VsZCBiZSByZXBvcnRlZCBhcyBub3QgY29zdC1lZmZlY3RpdmUuIikpCgogICAgaWYgYWIuZ2V0KCJvcmFjbGVfZ29sZF9ldmlkZW5jZSIsIEZhbHNlKToKICAgICAgICBzcGVjcy5hcHBlbmQoQWJsYXRpb25TcGVjKAogICAgICAgICAgICBuYW1lPSJvcmFjbGVfZ29sZF9ldmlkZW5jZSIsCiAgICAgICAgICAgIHF1ZXN0aW9uPSJXaGF0IGlzIHRoZSBjZWlsaW5nIGlmIHJldHJpZXZhbCB3ZXJlIHBlcmZlY3Q/IiwKICAgICAgICAgICAgdmFyaWFudHM9W3sidXNlX2dvbGRfZXZpZGVuY2UiOiBGYWxzZX0sIHsidXNlX2dvbGRfZXZpZGVuY2UiOiBUcnVlfV0sCiAgICAgICAgICAgIG5vdGVzPSJVcHBlciBib3VuZCBvbmx5LiBNb2R1bGUgNSBjb25maWcgc2V0cyAiCiAgICAgICAgICAgICAgICAgICJnb2xkX2V2aWRlbmNlX2luX3ByaW1hcnlfcGF0aDogZmFsc2UsIHNvIHRoZSBvcmFjbGUgaXMgbmV2ZXIgIgogICAgICAgICAgICAgICAgICAidGhlIGhlYWRsaW5lIHJlc3VsdC4iKSkKCiAgICByZXR1cm4gc3BlY3MKCgpkZWYgcnVuX2FibGF0aW9uKHNwZWM6IEFibGF0aW9uU3BlYywgcnVubmVyOiBDYWxsYWJsZVtbZGljdFtzdHIsIEFueV1dLCBkaWN0W3N0ciwgQW55XV0sCiAgICAgICAgICAgICAgICAgc2VlZDogaW50ID0gNDIpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgIiIiRXhlY3V0ZSBldmVyeSB2YXJpYW50IHRocm91Z2ggYHJ1bm5lcmAgYW5kIHN1bW1hcmlzZS4KCiAgICBgcnVubmVyYCBtYXBzIGEgdmFyaWFudCBkaWN0IHRvIGEgbWV0cmljcyBkaWN0LiBJbmplY3RpbmcgaXQga2VlcHMgdGhpcwogICAgdGVzdGFibGUgYW5kIGtlZXBzIHRoZSBhYmxhdGlvbiBmcmFtZXdvcmsgaW5kZXBlbmRlbnQgb2Ygd2hldGhlciB0aGUKICAgIHZhcmlhbnQgaXMgYSByZXRyaWV2YWwgb3IgYSB2ZXJpZmljYXRpb24gZXhwZXJpbWVudC4KICAgICIiIgogICAgcmVzdWx0cyA9IFtdCiAgICBmb3IgdmFyaWFudCBpbiBzcGVjLnZhcmlhbnRzOgogICAgICAgIHBheWxvYWQgPSBkaWN0KHZhcmlhbnQpCiAgICAgICAgcGF5bG9hZFsic2VlZCJdID0gc2VlZAogICAgICAgIG1ldHJpY3MgPSBydW5uZXIocGF5bG9hZCkKICAgICAgICByZXN1bHRzLmFwcGVuZCh7InZhcmlhbnQiOiB2YXJpYW50LCAibWV0cmljcyI6IG1ldHJpY3N9KQoKICAgIHZhbHVlcyA9IFsoclsidmFyaWFudCJdLCByWyJtZXRyaWNzIl0uZ2V0KHNwZWMubWV0cmljKSkgZm9yIHIgaW4gcmVzdWx0c10KICAgIHNjb3JlZCA9IFsodiwgbSkgZm9yIHYsIG0gaW4gdmFsdWVzIGlmIGlzaW5zdGFuY2UobSwgKGludCwgZmxvYXQpKV0KICAgIGJlc3QgPSBtYXgoc2NvcmVkLCBrZXk9bGFtYmRhIHZtOiB2bVsxXSkgaWYgc2NvcmVkIGVsc2UgKE5vbmUsIE5vbmUpCiAgICB3b3JzdCA9IG1pbihzY29yZWQsIGtleT1sYW1iZGEgdm06IHZtWzFdKSBpZiBzY29yZWQgZWxzZSAoTm9uZSwgTm9uZSkKICAgIGRlbHRhID0gKGJlc3RbMV0gLSB3b3JzdFsxXSkgaWYgc2NvcmVkIGFuZCBsZW4oc2NvcmVkKSA+IDEgZWxzZSAwLjAKCiAgICByZXR1cm4geyJuYW1lIjogc3BlYy5uYW1lLCAicXVlc3Rpb24iOiBzcGVjLnF1ZXN0aW9uLCAibWV0cmljIjogc3BlYy5tZXRyaWMsCiAgICAgICAgICAgICJzdGFnZSI6IHNwZWMuc3RhZ2UsICJub3RlcyI6IHNwZWMubm90ZXMsICJzZWVkIjogc2VlZCwKICAgICAgICAgICAgInJlc3VsdHMiOiByZXN1bHRzLCAiYmVzdF92YXJpYW50IjogYmVzdFswXSwgImJlc3RfdmFsdWUiOiBiZXN0WzFdLAogICAgICAgICAgICAid29yc3RfdmFyaWFudCI6IHdvcnN0WzBdLCAid29yc3RfdmFsdWUiOiB3b3JzdFsxXSwKICAgICAgICAgICAgImRlbHRhIjogZGVsdGEsCiAgICAgICAgICAgICJpbnRlcnByZXRhdGlvbiI6ICgiQSBuZWFyLXplcm8gZGVsdGEgbWVhbnMgdGhpcyBkZXNpZ24gY2hvaWNlIGRvZXMgbm90ICIKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJtYXR0ZXIgZm9yIHRoaXMgY29ycHVzIGFuZCBzaG91bGQgYmUgcmVwb3J0ZWQgYXMgc3VjaCwgIgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIm5vdCBvbWl0dGVkLiIpfQoKCmRlZiBydW5fYWxsX2FibGF0aW9ucyhjZmc6IGRpY3Rbc3RyLCBBbnldLAogICAgICAgICAgICAgICAgICAgICAgcnVubmVyczogZGljdFtzdHIsIENhbGxhYmxlW1tkaWN0W3N0ciwgQW55XV0sIGRpY3Rbc3RyLCBBbnldXV0sCiAgICAgICAgICAgICAgICAgICAgICBzZWVkOiBpbnQgPSA0MikgLT4gZGljdFtzdHIsIEFueV06CiAgICAiIiJSdW4gZXZlcnkgY29uZmlndXJlZCBhYmxhdGlvbiB0aGF0IGhhcyBhIHJlZ2lzdGVyZWQgcnVubmVyLiIiIgogICAgc3BlY3MgPSBidWlsZF9hYmxhdGlvbl9zcGVjcyhjZmcpCiAgICBvdXQ6IGRpY3Rbc3RyLCBBbnldID0geyJzZWVkIjogc2VlZCwgIm5fc3BlY3MiOiBsZW4oc3BlY3MpLCAiYWJsYXRpb25zIjoge30sICJza2lwcGVkIjogW119CiAgICBmb3Igc3BlYyBpbiBzcGVjczoKICAgICAgICBydW5uZXIgPSBydW5uZXJzLmdldChzcGVjLm5hbWUpCiAgICAgICAgaWYgcnVubmVyIGlzIE5vbmU6CiAgICAgICAgICAgIG91dFsic2tpcHBlZCJdLmFwcGVuZCh7Im5hbWUiOiBzcGVjLm5hbWUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInJlYXNvbiI6ICJubyBydW5uZXIgcmVnaXN0ZXJlZCBmb3IgdGhpcyBhYmxhdGlvbiJ9KQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIG91dFsiYWJsYXRpb25zIl1bc3BlYy5uYW1lXSA9IHJ1bl9hYmxhdGlvbihzcGVjLCBydW5uZXIsIHNlZWQ9c2VlZCkKICAgIHJldHVybiBvdXQK", "sha256": "cbaf2ae5c879ff0e5c78df0d9423d0bcbdc1cba229b7b373c017e76098cf5452", "size": 6642}, "scripts/m6_common.py": {"b64": "IiIiTW9kdWxlIDYg4oCUIHNoYXJlZCBzdGFnZSBwbHVtYmluZy4KCkV2ZXJ5IG02Xyogc2NyaXB0IGxvYWRzIHRoZSBzYW1lIGNvcnB1cywgd3JpdGVzIHRoZSBzYW1lIG1hbmlmZXN0IHNoYXBlLCBhbmQKdmVyaWZpZXMgTW9kdWxlIDUgaXMgdW50b3VjaGVkLiBDZW50cmFsaXNpbmcgdGhhdCBoZXJlIGtlZXBzIHRoZSBuaW5lIHN0YWdlCnNjcmlwdHMgc2hvcnQgYW5kIHN0b3BzIHRoZW0gZHJpZnRpbmcgYXBhcnQuCgpNT0RVTEUgNSBJUyBSRUFELU9OTFkuIGBsb2FkX2NvcnB1cygpYCBvcGVucyBmaWxlcyBpbiByZWFkIG1vZGUgb25seSwgYW5kCmBtb2R1bGU1X2ZpbmdlcnByaW50KClgIGxldHMgYW55IHN0YWdlIHByb3ZlIG5vdGhpbmcgdW5kZXIgTW9kdWxlIDUncwpkaXJlY3RvcmllcyBjaGFuZ2VkIHdoaWxlIGl0IHJhbi4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgc3lzCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgpQUk9KRUNUX1JPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1sxXQppZiBzdHIoUFJPSkVDVF9ST09UKSBub3QgaW4gc3lzLnBhdGg6CiAgICBzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFBST0pFQ1RfUk9PVCkpCgpmcm9tIHNyYy51dGlscy5jb25maWdfbG9hZGVyIGltcG9ydCBsb2FkX3lhbWwgICMgbm9xYTogRTQwMgoKTTVfUkVBRE9OTFkgPSBbCiAgICAiZGF0YXNldHMvZmFjdF92ZXJpZmljYXRpb24iLAogICAgInNyYy9mYWN0X3ZlcmlmaWNhdGlvbiIsCiAgICAiY29uZmlncy9tb2R1bGU1X2ZhY3RfdmVyaWZpY2F0aW9uLnlhbWwiLApdCgoKZGVmIGxvYWRfY29uZmlnKHBhdGg6IHN0ciB8IFBhdGggfCBOb25lID0gTm9uZSkgLT4gZGljdFtzdHIsIEFueV06CiAgICBwID0gUGF0aChwYXRoKSBpZiBwYXRoIGVsc2UgUFJPSkVDVF9ST09UIC8gImNvbmZpZ3MiIC8gIm1vZHVsZTZfcmV0cmlldmFsX3ZlcmlmaWNhdGlvbi55YW1sIgogICAgcmV0dXJuIGxvYWRfeWFtbChwKQoKCmRlZiBzdGFnZV9kaXIoY2ZnOiBkaWN0W3N0ciwgQW55XSwgc3RhZ2U6IHN0ciwgcm9vdDogUGF0aCB8IE5vbmUgPSBOb25lKSAtPiBQYXRoOgogICAgZCA9IChyb290IG9yIFBST0pFQ1RfUk9PVCkgLyBjZmdbIm91dHB1dCJdWyJyb290Il0gLyBzdGFnZQogICAgZC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICByZXR1cm4gZAoKCmRlZiBsb2FkX2pzb25sKHBhdGg6IFBhdGgpIC0+IGxpc3RbZGljdFtzdHIsIEFueV1dOgogICAgcm93cyA9IFtdCiAgICB3aXRoIFBhdGgocGF0aCkub3BlbihlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIGZvciBsbiwgbGluZSBpbiBlbnVtZXJhdGUoZiwgMSk6CiAgICAgICAgICAgIGxpbmUgPSBsaW5lLnN0cmlwKCkKICAgICAgICAgICAgaWYgbGluZToKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICByb3dzLmFwcGVuZChqc29uLmxvYWRzKGxpbmUpKQogICAgICAgICAgICAgICAgZXhjZXB0IGpzb24uSlNPTkRlY29kZUVycm9yIGFzIGU6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImNvcnJ1cHQgSlNPTkwge3BhdGh9Ontsbn06IHtlfSIpIGZyb20gZQogICAgcmV0dXJuIHJvd3MKCgpkZWYgbG9hZF9jb3JwdXMoY2ZnOiBkaWN0W3N0ciwgQW55XSwgZGF0YXNldDogc3RyLCByb290OiBQYXRoIHwgTm9uZSA9IE5vbmUKICAgICAgICAgICAgICAgICkgLT4gZGljdFtzdHIsIEFueV06CiAgICAiIiJSZWFkIG9uZSBNb2R1bGUgNSBjb3JwdXMuIFJlYWQtb25seSwgYWx3YXlzLiIiIgogICAgYmFzZSA9IChyb290IG9yIFBST0pFQ1RfUk9PVCkgLyBjZmdbImlucHV0cyJdWyJjb3JwdXNfZGlyIl0gLyBkYXRhc2V0CiAgICBjbGFpbXNfcCA9IGJhc2UgLyBjZmdbImlucHV0cyJdWyJjbGFpbXNfZmlsZW5hbWUiXQogICAgZXZfcCA9IGJhc2UgLyBjZmdbImlucHV0cyJdWyJldmlkZW5jZV9maWxlbmFtZSJdCiAgICBzcF9wID0gYmFzZSAvIGNmZ1siaW5wdXRzIl1bInNwbGl0c19maWxlbmFtZSJdCiAgICBmb3IgcCBpbiAoY2xhaW1zX3AsIGV2X3AsIHNwX3ApOgogICAgICAgIGlmIG5vdCBwLmlzX2ZpbGUoKToKICAgICAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJNb2R1bGUgNSBvdXRwdXQgbWlzc2luZzoge3B9LiBSdW4gTW9kdWxlIDUgZmlyc3QuIikKCiAgICBjbGFpbXMgPSBsb2FkX2pzb25sKGNsYWltc19wKQogICAgZXZpZGVuY2UgPSBsb2FkX2pzb25sKGV2X3ApCiAgICBzcGxpdHNfZG9jID0ganNvbi5sb2FkcyhzcF9wLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgIHNwbGl0cyA9IHNwbGl0c19kb2MuZ2V0KCJzcGxpdHMiLCB7fSkKCiAgICBzcGxpdF9vZjogZGljdFtpbnQsIHN0cl0gPSB7fQogICAgZm9yIHMsIGlkeHMgaW4gc3BsaXRzLml0ZW1zKCk6CiAgICAgICAgZm9yIGkgaW4gaWR4czoKICAgICAgICAgICAgc3BsaXRfb2ZbaV0gPSBzCgogICAgcmV0dXJuIHsKICAgICAgICAiZGF0YXNldCI6IGRhdGFzZXQsCiAgICAgICAgImNsYWltcyI6IGNsYWltcywKICAgICAgICAiZXZpZGVuY2UiOiBldmlkZW5jZSwKICAgICAgICAiZXZpZGVuY2VfdGV4dHMiOiB7ZVsiZXZpZGVuY2VfaWQiXTogZS5nZXQoInRleHQiLCAiIikgZm9yIGUgaW4gZXZpZGVuY2V9LAogICAgICAgICJzcGxpdHMiOiBzcGxpdHMsCiAgICAgICAgInNwbGl0X29mIjogc3BsaXRfb2YsCiAgICAgICAgImNsYWltX2lkX29mIjoge2k6IGMuZ2V0KCJjbGFpbV9pZCIsIHN0cihpKSkgZm9yIGksIGMgaW4gZW51bWVyYXRlKGNsYWltcyl9LAogICAgICAgICJpbmRleF9vZl9jbGFpbV9pZCI6IHtjLmdldCgiY2xhaW1faWQiLCBzdHIoaSkpOiBpIGZvciBpLCBjIGluIGVudW1lcmF0ZShjbGFpbXMpfSwKICAgICAgICAicGF0aHMiOiB7ImNsYWltcyI6IHN0cihjbGFpbXNfcCksICJldmlkZW5jZSI6IHN0cihldl9wKSwgInNwbGl0cyI6IHN0cihzcF9wKX0sCiAgICB9CgoKZGVmIGdvbGRfZXZpZGVuY2VfbWFwKGNvcnB1czogZGljdFtzdHIsIEFueV0pIC0+IGRpY3Rbc3RyLCBsaXN0W3N0cl1dOgogICAgcmV0dXJuIHtjLmdldCgiY2xhaW1faWQiLCBzdHIoaSkpOiBsaXN0KGMuZ2V0KCJldmlkZW5jZSIpIG9yIFtdKQogICAgICAgICAgICBmb3IgaSwgYyBpbiBlbnVtZXJhdGUoY29ycHVzWyJjbGFpbXMiXSl9CgoKZGVmIG1vZHVsZTVfZmluZ2VycHJpbnQocm9vdDogUGF0aCB8IE5vbmUgPSBOb25lKSAtPiBkaWN0W3N0ciwgQW55XToKICAgICIiIlNpemUrY291bnQgc2lnbmF0dXJlIG9mIGV2ZXJ5IE1vZHVsZSA1IHBhdGgsIGZvciBiZWZvcmUvYWZ0ZXIgY29tcGFyaXNvbi4iIiIKICAgIGJhc2UgPSByb290IG9yIFBST0pFQ1RfUk9PVAogICAgb3V0OiBkaWN0W3N0ciwgQW55XSA9IHt9CiAgICBmb3IgcmVsIGluIE01X1JFQURPTkxZOgogICAgICAgIHAgPSBiYXNlIC8gcmVsCiAgICAgICAgaWYgbm90IHAuZXhpc3RzKCk6CiAgICAgICAgICAgIG91dFtyZWxdID0gTm9uZQogICAgICAgIGVsaWYgcC5pc19maWxlKCk6CiAgICAgICAgICAgIG91dFtyZWxdID0geyJieXRlcyI6IHAuc3RhdCgpLnN0X3NpemUsCiAgICAgICAgICAgICAgICAgICAgICAgICJzaGEyNTYiOiBoYXNobGliLnNoYTI1NihwLnJlYWRfYnl0ZXMoKSkuaGV4ZGlnZXN0KClbOjE2XX0KICAgICAgICBlbHNlOgogICAgICAgICAgICBmaWxlcyA9IHNvcnRlZChmIGZvciBmIGluIHAucmdsb2IoIioiKSBpZiBmLmlzX2ZpbGUoKSBhbmQgIl9fcHljYWNoZV9fIiBub3QgaW4gc3RyKGYpKQogICAgICAgICAgICBvdXRbcmVsXSA9IHsibl9maWxlcyI6IGxlbihmaWxlcyksCiAgICAgICAgICAgICAgICAgICAgICAgICJ0b3RhbF9ieXRlcyI6IHN1bShmLnN0YXQoKS5zdF9zaXplIGZvciBmIGluIGZpbGVzKX0KICAgIHJldHVybiBvdXQKCgpkZWYgYXNzZXJ0X21vZHVsZTVfdW50b3VjaGVkKGJlZm9yZTogZGljdFtzdHIsIEFueV0sIHJvb3Q6IFBhdGggfCBOb25lID0gTm9uZSkgLT4gTm9uZToKICAgIGFmdGVyID0gbW9kdWxlNV9maW5nZXJwcmludChyb290KQogICAgY2hhbmdlZCA9IHtrOiB7ImJlZm9yZSI6IGJlZm9yZS5nZXQoayksICJhZnRlciI6IGFmdGVyLmdldChrKX0KICAgICAgICAgICAgICAgZm9yIGsgaW4gYWZ0ZXIgaWYgYmVmb3JlLmdldChrKSAhPSBhZnRlci5nZXQoayl9CiAgICBpZiBjaGFuZ2VkOgogICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKGYiTU9EVUxFIDUgV0FTIE1PRElGSUVEOiB7anNvbi5kdW1wcyhjaGFuZ2VkLCBpbmRlbnQ9Mil9IikKCgpkZWYgd3JpdGVfbWFuaWZlc3Qoc3RhZ2VfcGF0aDogUGF0aCwgcGF5bG9hZDogZGljdFtzdHIsIEFueV0pIC0+IFBhdGg6CiAgICBwYXlsb2FkID0gZGljdChwYXlsb2FkKQogICAgcGF5bG9hZC5zZXRkZWZhdWx0KCJ3cml0dGVuX2F0IiwgdGltZS5zdHJmdGltZSgiJVktJW0tJWRUJUg6JU06JVMiKSkKICAgIHRtcCA9IHN0YWdlX3BhdGggLyAibWFuaWZlc3QudG1wLmpzb24iCiAgICB0bXAud3JpdGVfdGV4dChqc29uLmR1bXBzKHBheWxvYWQsIGluZGVudD0yLCBlbnN1cmVfYXNjaWk9RmFsc2UsIGRlZmF1bHQ9c3RyKSwKICAgICAgICAgICAgICAgICAgIGVuY29kaW5nPSJ1dGYtOCIpCiAgICB0bXAucmVwbGFjZShzdGFnZV9wYXRoIC8gIm1hbmlmZXN0Lmpzb24iKQogICAgcmV0dXJuIHN0YWdlX3BhdGggLyAibWFuaWZlc3QuanNvbiIKCgpkZWYgcmVhZF9tYW5pZmVzdChzdGFnZV9wYXRoOiBQYXRoKSAtPiBkaWN0W3N0ciwgQW55XToKICAgIHAgPSBQYXRoKHN0YWdlX3BhdGgpIC8gIm1hbmlmZXN0Lmpzb24iCiAgICBpZiBub3QgcC5pc19maWxlKCk6CiAgICAgICAgcmV0dXJuIHt9CiAgICB0cnk6CiAgICAgICAgcmV0dXJuIGpzb24ubG9hZHMocC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBleGNlcHQganNvbi5KU09ORGVjb2RlRXJyb3I6CiAgICAgICAgcmV0dXJuIHt9CgoKZGVmIHdyaXRlX2pzb24ocGF0aDogUGF0aCwgcGF5bG9hZDogQW55KSAtPiBQYXRoOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHRtcCA9IHBhdGgud2l0aF9zdWZmaXgoIi50bXAuanNvbiIpCiAgICB0bXAud3JpdGVfdGV4dChqc29uLmR1bXBzKHBheWxvYWQsIGluZGVudD0yLCBlbnN1cmVfYXNjaWk9RmFsc2UsIGRlZmF1bHQ9c3RyKSwKICAgICAgICAgICAgICAgICAgIGVuY29kaW5nPSJ1dGYtOCIpCiAgICB0bXAucmVwbGFjZShwYXRoKQogICAgcmV0dXJuIHBhdGgKCgpkZWYgY29uZmlnX2hhc2goY2ZnOiBkaWN0W3N0ciwgQW55XSwga2V5czogbGlzdFtzdHJdIHwgTm9uZSA9IE5vbmUpIC0+IHN0cjoKICAgICIiIkhhc2ggb2YgdGhlIGNvbmZpZyBzbGljZSBhIHN0YWdlIGRlcGVuZHMgb24sIHNvIGEgY2hhbmdlZCBzZXR0aW5nIGlzCiAgICB2aXNpYmxlIGluIHRoZSBtYW5pZmVzdCByYXRoZXIgdGhhbiBzaWxlbnRseSByZXVzaW5nIHN0YWxlIHNoYXJkcy4iIiIKICAgIHN1YiA9IHtrOiBjZmcuZ2V0KGspIGZvciBrIGluIChrZXlzIG9yIHNvcnRlZChjZmcpKX0KICAgIHJldHVybiBoYXNobGliLnNoYTI1Nihqc29uLmR1bXBzKHN1Yiwgc29ydF9rZXlzPVRydWUsIGRlZmF1bHQ9c3RyKS5lbmNvZGUoKSkuaGV4ZGlnZXN0KClbOjE2XQoKCmRlZiBkZXRlY3RfZGV2aWNlKCkgLT4gc3RyOgogICAgdHJ5OgogICAgICAgIGltcG9ydCB0b3JjaAogICAgICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgICAgIHJldHVybiAiY3VkYSIKICAgIGV4Y2VwdCBFeGNlcHRpb246ICAjIG5vcWE6IEJMRTAwMQogICAgICAgIHBhc3MKICAgIHJldHVybiAiY3B1IgoKCmRlZiBiYW5uZXIoc3RhZ2U6IHN0ciwgY2ZnOiBkaWN0W3N0ciwgQW55XSkgLT4gTm9uZToKICAgIHByaW50KCI9IiAqIDcwKQogICAgcHJpbnQoZiJNT0RVTEUgNiDigJQge3N0YWdlfSIpCiAgICBwcmludCgiPSIgKiA3MCkKICAgIHByaW50KGYiICBkZXZpY2U6IHtkZXRlY3RfZGV2aWNlKCl9IikK", "sha256": "e6cd0f42a0141a1091886c89d4db6b1462ac030f5df6e4c43af4fbeb2978d0fc", "size": 6096}, "scripts/m6_01_encode_evidence.py": {"b64": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uCiIiIk1vZHVsZSA2IFN0YWdlIDEg4oCUIGVuY29kZSB0aGUgZXZpZGVuY2UgcG9vbCB3aXRoIExhQlNFLiBHUFUuIFJlc3VtYWJsZS4KCkVuY29kZXMgb25seSB0aGUgZXZpZGVuY2UgcGVybWl0dGVkIGZvciB0aGUgcmVxdWVzdGVkIHBoYXNlLCBkZXJpdmVkIGZyb20KTW9kdWxlIDUncyBidWlsZF9pbmRleF9zY29wZSgpLiBFbmNvZGluZyB0aGUgd2hvbGUgcG9vbCBhbmQgZmlsdGVyaW5nIGxhdGVyCndvdWxkIHdvcmssIGJ1dCBzY29waW5nIGhlcmUgbWVhbnMgYW4gb3V0LW9mLXNjb3BlIGl0ZW0gbmV2ZXIgZXZlbiBnZXRzIGEKdmVjdG9yIC0tIHRoZSBsZWFrYWdlIGd1YXJhbnRlZSBob2xkcyBieSBjb25zdHJ1Y3Rpb24sIG5vdCBieSBkaXNjaXBsaW5lLgoKU2hhcmRzIG9mIDEwLDAwMCwgYXRvbWljIHRlbXAtdGhlbi1yZW5hbWUsIG1hbmlmZXN0LWRyaXZlbiByZXN1bWUuCgogICAgcHl0aG9uIHNjcmlwdHMvbTZfMDFfZW5jb2RlX2V2aWRlbmNlLnB5IC0tZGF0YXNldCBmZXZlciAtLXBoYXNlIHRlc3QKIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgc3lzCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1sxXSkpCgpmcm9tIHNjcmlwdHMubTZfY29tbW9uIGltcG9ydCAoICAjIG5vcWE6IEU0MDIKICAgIFBST0pFQ1RfUk9PVCwgYXNzZXJ0X21vZHVsZTVfdW50b3VjaGVkLCBiYW5uZXIsIGNvbmZpZ19oYXNoLCBkZXRlY3RfZGV2aWNlLAogICAgbG9hZF9jb25maWcsIGxvYWRfY29ycHVzLCBtb2R1bGU1X2ZpbmdlcnByaW50LCBzdGFnZV9kaXIsIHdyaXRlX21hbmlmZXN0KQpmcm9tIHNyYy5yZXRyaWV2YWwuZW5jb2RlcnMgaW1wb3J0ICggICMgbm9xYTogRTQwMgogICAgTG93TWVtb3J5U3RvcCwgU2hhcmRlZEVuY29kZXIsIGF2YWlsYWJsZV9tYiwgYnVpbGRfbGFic2VfZW5jb2RlX2ZuKQpmcm9tIHNyYy5yZXRyaWV2YWwuaW5kZXhfc2NvcGUgaW1wb3J0IGJ1aWxkX2V2aWRlbmNlX3Njb3BlICAjIG5vcWE6IEU0MDIKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJFbmNvZGUgc2NvcGVkIGV2aWRlbmNlIHdpdGggTGFCU0UuIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1kYXRhc2V0IiwgcmVxdWlyZWQ9VHJ1ZSwgY2hvaWNlcz1bImZldmVyIiwgInhfZmFjdCJdKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXBoYXNlIiwgZGVmYXVsdD0idGVzdCIsIGNob2ljZXM9WyJ0cmFpbiIsICJ2YWxpZGF0aW9uIiwgInRlc3QiXSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1jb25maWciLCBkZWZhdWx0PU5vbmUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcHJvamVjdC1yb290IiwgZGVmYXVsdD1Ob25lKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWRyeS1ydW4iLCBhY3Rpb249InN0b3JlX3RydWUiLAogICAgICAgICAgICAgICAgICAgIGhlbHA9IlJlcG9ydCB3aGF0IHdvdWxkIGJlIGVuY29kZWQ7IGRvd25sb2FkIG5vIG1vZGVsLiIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbW9kZWwiLCBkZWZhdWx0PU5vbmUsCiAgICAgICAgICAgICAgICAgICAgaGVscD0iT3ZlcnJpZGUgdGhlIGJpLWVuY29kZXIuIEEgc21hbGxlciBtb2RlbCAoZS5nLiAiCiAgICAgICAgICAgICAgICAgICAgICAgICAic2VudGVuY2UtdHJhbnNmb3JtZXJzL3BhcmFwaHJhc2UtbXVsdGlsaW5ndWFsLU1pbmlMTS1MMTItdjIpICIKICAgICAgICAgICAgICAgICAgICAgICAgICJjdXRzIFJBTSB+NHggaWYgTGFCU0UgaXMgYmVpbmcgT09NLWtpbGxlZC4iKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWJhdGNoLXNpemUiLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW1heC1zZXEtbGVuZ3RoIiwgdHlwZT1pbnQsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zaGFyZC1zaXplIiwgdHlwZT1pbnQsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgICBoZWxwPSJTbWFsbGVyIHNoYXJkcyBjaGVja3BvaW50IG1vcmUgb2Z0ZW4gYW5kIGNhcCBwZWFrIFJBTS4iKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW1pbi1hdmFpbGFibGUtbWIiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTEyMDAuMCwKICAgICAgICAgICAgICAgICAgICBoZWxwPSJTdG9wIGNsZWFubHkgaWYgc3lzdGVtIFJBTSBkcm9wcyBiZWxvdyB0aGlzLCBpbnN0ZWFkIG9mICIKICAgICAgICAgICAgICAgICAgICAgICAgICJiZWluZyBPT00ta2lsbGVkIHdpdGggbm8gdHJhY2ViYWNrLiAwIGRpc2FibGVzLiIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbnVtLXRocmVhZHMiLCB0eXBlPWludCwgZGVmYXVsdD0yLAogICAgICAgICAgICAgICAgICAgIGhlbHA9InRvcmNoIENQVSB0aHJlYWRzLiBFYWNoIGNhcnJpZXMgaXRzIG93biB3b3Jrc3BhY2UsIHNvIGFuICIKICAgICAgICAgICAgICAgICAgICAgICAgICJ1bmJvdW5kZWQgY291bnQgbXVsdGlwbGllcyBwZWFrIFJBTSBvbiBhIHNtYWxsIGNvbnRhaW5lci4iKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQoKICAgIHJvb3QgPSBQYXRoKGFyZ3MucHJvamVjdF9yb290KSBpZiBhcmdzLnByb2plY3Rfcm9vdCBlbHNlIFBST0pFQ1RfUk9PVAogICAgY2ZnID0gbG9hZF9jb25maWcoYXJncy5jb25maWcpCiAgICBiYW5uZXIoZiJTdGFnZSAxIGVuY29kZSBldmlkZW5jZSBbe2FyZ3MuZGF0YXNldH0ve2FyZ3MucGhhc2V9XSIsIGNmZykKICAgIGJlZm9yZSA9IG1vZHVsZTVfZmluZ2VycHJpbnQocm9vdCkKCiAgICBjb3JwdXMgPSBsb2FkX2NvcnB1cyhjZmcsIGFyZ3MuZGF0YXNldCwgcm9vdD1yb290KQogICAgc2NvcGUgPSBidWlsZF9ldmlkZW5jZV9zY29wZShjb3JwdXNbImNsYWltcyJdLCBjb3JwdXNbInNwbGl0cyJdLCBhcmdzLnBoYXNlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjZmdbImxlYWthZ2UiXVsiaW5kZXhfc2NvcGUiXSkKICAgIGV2X2lkcyA9IHNvcnRlZChzY29wZS5ldmlkZW5jZV9pZHMpICAgICAgICAgICAgICAgICAjIHNvcnRlZCA9PiBkZXRlcm1pbmlzdGljIHNoYXJkcwogICAgZXZfdGV4dHMgPSBjb3JwdXNbImV2aWRlbmNlX3RleHRzIl0KICAgICMgVmlld3MsIG5vdCBhIHNlY29uZCBmdWxsIGNvcHkgb2YgdGhlIGNvcnB1cyB0ZXh0IChyZXF1aXJlbWVudDogbmV2ZXIgaG9sZAogICAgIyB0aGUgd2hvbGUgcG9vbCB0d2ljZSkuIHRleHRfZm4gbWF0ZXJpYWxpc2VzIG9uZSBzaGFyZCBhdCBhIHRpbWUuCiAgICB0ZXh0cyA9IFtldl90ZXh0cy5nZXQoZSwgIiIpIGZvciBlIGluIGV2X2lkc10KCiAgICBkZWYgdGV4dF9mbihsbzogaW50LCBoaTogaW50KSAtPiBsaXN0W3N0cl06CiAgICAgICAgcmV0dXJuIFtldl90ZXh0cy5nZXQoZSwgIiIpIGZvciBlIGluIGV2X2lkc1tsbzpoaV1dCgogICAgcHJpbnQoZiIgIGNsYWltcyBpbiBzY29wZSAgIDoge3Njb3BlLm5fY2xhaW1zX2luX3Njb3BlOix9IikKICAgIHByaW50KGYiICBldmlkZW5jZSBpbiBzY29wZSA6IHtsZW4oZXZfaWRzKTosfSIpCiAgICBwcmludChmIiAgZXZpZGVuY2UgZXhjbHVkZWQgOiB7c2NvcGUubl9ldmlkZW5jZV9leGNsdWRlZDosfSAob3V0LW9mLXNjb3BlIG9ubHkpIikKCiAgICBvdXQgPSBzdGFnZV9kaXIoY2ZnLCAiZW5jb2RlX2V2aWRlbmNlIiwgcm9vdCkgLyBmInthcmdzLmRhdGFzZXR9X3thcmdzLnBoYXNlfSIKICAgIG91dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgaWYgYXJncy5kcnlfcnVuOgogICAgICAgIHdyaXRlX21hbmlmZXN0KG91dCwgeyJzdGFnZSI6ICJlbmNvZGVfZXZpZGVuY2UiLCAiZGF0YXNldCI6IGFyZ3MuZGF0YXNldCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAicGhhc2UiOiBhcmdzLnBoYXNlLCAic3RhdHVzIjogImRyeV9ydW4iLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICJuX2V2aWRlbmNlIjogbGVuKGV2X2lkcyksICJzY29wZSI6IHNjb3BlLnN0YXRzfSkKICAgICAgICBwcmludCgiICBEUlkgUlVOOiBubyBtb2RlbCBsb2FkZWQsIG5vIHZlY3RvcnMgd3JpdHRlbi4iKQogICAgICAgIGFzc2VydF9tb2R1bGU1X3VudG91Y2hlZChiZWZvcmUsIHJvb3QpCiAgICAgICAgcmV0dXJuCgogICAgciA9IGNmZ1sicmV0cmlldmFsIl0KICAgIGRldmljZSA9IGRldGVjdF9kZXZpY2UoKQogICAgIyBDUFUtc2FmZSBkZWZhdWx0cy4gVGhlIGNvbmZpZyB2YWx1ZXMgKGJhdGNoIDY0IC8gc2VxIDI1NiAvIHNoYXJkIDEwMDAwKSBhcmUKICAgICMgc2l6ZWQgZm9yIGEgR1BVOyBvbiBDUFUgdGhhdCBzaGFyZCBzaXplIG1lYW5zIHRoZSB3aG9sZSA2LDk2OC1pdGVtIHBvb2wgZ29lcwogICAgIyB0aHJvdWdoIG9uZSBlbmNvZGUoKSBjYWxsLCBhbmQgdGhlIHBlYWsgaXMgd2hhdCBnZXRzIHRoZSBwcm9jZXNzIE9PTS1raWxsZWQuCiAgICBjcHUgPSBkZXZpY2UgPT0gImNwdSIKICAgIG1vZGVsX25hbWUgPSBhcmdzLm1vZGVsIG9yIHJbImJpX2VuY29kZXIiXQogICAgYmF0Y2ggPSBpbnQoYXJncy5iYXRjaF9zaXplIG9yICg4IGlmIGNwdSBlbHNlIHJbImVuY29kZV9iYXRjaF9zaXplIl0pKQogICAgc2VxID0gaW50KGFyZ3MubWF4X3NlcV9sZW5ndGggb3IgKDEyOCBpZiBjcHUgZWxzZSByWyJtYXhfc2VxX2xlbmd0aCJdKSkKICAgIHNoYXJkID0gaW50KGFyZ3Muc2hhcmRfc2l6ZSBvciAoNTAwIGlmIGNwdSBlbHNlIHJbInNoYXJkX3NpemUiXSkpCiAgICBzZXR0aW5ncyA9IHsibW9kZWwiOiBtb2RlbF9uYW1lLCAiYmF0Y2hfc2l6ZSI6IGJhdGNoLCAibWF4X3NlcV9sZW5ndGgiOiBzZXF9CgogICAgcHJpbnQoZiIgIG1vZGVsICAgICAgOiB7bW9kZWxfbmFtZX0iKQogICAgcHJpbnQoZiIgIGJhdGNoPXtiYXRjaH0gIHNlcT17c2VxfSAgc2hhcmQ9e3NoYXJkfSAgdGhyZWFkcz17YXJncy5udW1fdGhyZWFkc30iCiAgICAgICAgICArICgiICAgW0NQVS1zYWZlIGRlZmF1bHRzXSIgaWYgY3B1IGFuZCBub3QgKGFyZ3MuYmF0Y2hfc2l6ZSBvciBhcmdzLm1heF9zZXFfbGVuZ3RoCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgb3IgYXJncy5zaGFyZF9zaXplKSBlbHNlICIiKSkKICAgIHByaW50KGYiICBSQU0gZmxvb3IgIDoge2FyZ3MubWluX2F2YWlsYWJsZV9tYjouMGZ9IE1CICIKICAgICAgICAgIGYiKGF2YWlsYWJsZSBub3cge2F2YWlsYWJsZV9tYigpOi4wZn0gTUIpIikKICAgIG5fc2hhcmRzID0gKGxlbihldl9pZHMpICsgc2hhcmQgLSAxKSAvLyBzaGFyZAogICAgcHJpbnQoZiIgIHNoYXJkcyAgICAgOiB7bl9zaGFyZHN9ICAoe2xlbihldl9pZHMpOix9IGl0ZW1zKSIpCgogICAgZW5jID0gU2hhcmRlZEVuY29kZXIob3V0LCBzaGFyZF9zaXplPXNoYXJkLAogICAgICAgICAgICAgICAgICAgICAgICAgZW5jb2RlX2ZuPWJ1aWxkX2xhYnNlX2VuY29kZV9mbigKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtb2RlbF9uYW1lLCBiYXRjaF9zaXplPWJhdGNoLCBtYXhfc2VxX2xlbmd0aD1zZXEsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZGV2aWNlPWRldmljZSwgbnVtX3RocmVhZHM9aW50KGFyZ3MubnVtX3RocmVhZHMpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIGxvZz1wcmludCksCiAgICAgICAgICAgICAgICAgICAgICAgICBtb2RlbF9uYW1lPW1vZGVsX25hbWUpCgogICAgdDAgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICByYW1fbG8gPSBhdmFpbGFibGVfbWIoKQogICAgc3RvcHBlZCA9IE5vbmUKICAgIHRyeToKICAgICAgICBtYW4gPSBlbmMuZW5jb2RlKHRleHRzLCBpZHM9ZXZfaWRzLCBtaW5fYXZhaWxhYmxlX21iPWZsb2F0KGFyZ3MubWluX2F2YWlsYWJsZV9tYiksCiAgICAgICAgICAgICAgICAgICAgICAgICBzZXR0aW5ncz1zZXR0aW5ncywgdGV4dF9mbj10ZXh0X2ZuKQogICAgZXhjZXB0IExvd01lbW9yeVN0b3AgYXMgZXhjOgogICAgICAgIHN0b3BwZWQgPSBzdHIoZXhjKQogICAgICAgIG1hbiA9IGVuYy5sb2FkX21hbmlmZXN0KCkKCiAgICBtYW4udXBkYXRlKHsic3RhZ2UiOiAiZW5jb2RlX2V2aWRlbmNlIiwgImRhdGFzZXQiOiBhcmdzLmRhdGFzZXQsICJwaGFzZSI6IGFyZ3MucGhhc2UsCiAgICAgICAgICAgICAgICAic2NvcGUiOiBzY29wZS5zdGF0cywgInNldHRpbmdzIjogc2V0dGluZ3MsICJkZXZpY2UiOiBkZXZpY2UsCiAgICAgICAgICAgICAgICAiY29uZmlnX2hhc2giOiBjb25maWdfaGFzaChjZmcsIFsicmV0cmlldmFsIiwgImxlYWthZ2UiXSl9KQogICAgd3JpdGVfbWFuaWZlc3Qob3V0LCBtYW4pCiAgICBhc3NlcnRfbW9kdWxlNV91bnRvdWNoZWQoYmVmb3JlLCByb290KQoKICAgIGRvbmUgPSBzb3J0ZWQobWFuLmdldCgiY29tcGxldGVkX3NoYXJkcyIsIFtdKSkKICAgIHJlbWFpbmluZyA9IFtpIGZvciBpIGluIHJhbmdlKG5fc2hhcmRzKSBpZiBpIG5vdCBpbiBzZXQoZG9uZSldCiAgICB3cml0dGVuID0gc3VtKDEgZm9yIGkgaW4gcmFuZ2Uobl9zaGFyZHMpIGlmIChvdXQgLyBmInNoYXJkX3tpOjA1ZH0ubnB5IikuaXNfZmlsZSgpKQogICAgcmFtX25vdyA9IGF2YWlsYWJsZV9tYigpCiAgICBwcmludCgiXG4iICsgIi0iICogNjIpCiAgICBwcmludCgiU1RBR0UgMWEgU1VNTUFSWSIpCiAgICBwcmludCgiLSIgKiA2MikKICAgIHByaW50KGYiICBjb21wbGV0ZWQgc2hhcmRzICAgOiB7bGVuKGRvbmUpfS97bl9zaGFyZHN9IikKICAgIHByaW50KGYiICByZW1haW5pbmcgc2hhcmRzICAgOiB7bGVuKHJlbWFpbmluZyl9IgogICAgICAgICAgKyAoZiIgIChuZXh0OiB7cmVtYWluaW5nWzBdfSkiIGlmIHJlbWFpbmluZyBlbHNlICIiKSkKICAgIHByaW50KGYiICBlbWJlZGRpbmdzIHdyaXR0ZW4gOiB7d3JpdHRlbn0gc2hhcmQgZmlsZShzKSIpCiAgICBwcmludChmIiAgZWxhcHNlZCAgICAgICAgICAgIDoge3RpbWUucGVyZl9jb3VudGVyKCkgLSB0MDouMWZ9cyIpCiAgICBwcmludChmIiAgUkFNIGF2YWlsYWJsZSAgICAgIDogc3RhcnQge3JhbV9sbzouMGZ9IE1CIC0+IG5vdyB7cmFtX25vdzouMGZ9IE1CIgogICAgICAgICAgZiIgIChsb3dlc3Qgc2VlbiB7bWFuLmdldCgnbWluX2F2YWlsYWJsZV9tYl9vYnNlcnZlZCcpIG9yIHJhbV9ub3c6LjBmfSBNQikiKQogICAgcHJpbnQoZiIgIHN0YXR1cyAgICAgICAgICAgICA6IHttYW4uZ2V0KCdzdGF0dXMnKX0iKQogICAgcHJpbnQoZiIgIHJlc3VtZS1zYWZlICAgICAgICA6IFlFUyAtIHJlLXJ1biB0aGUgaWRlbnRpY2FsIGNvbW1hbmQ7ICIKICAgICAgICAgIGYiY29tcGxldGVkIHNoYXJkcyBhcmUgc2tpcHBlZCIpCiAgICBwcmludChmIiAgb3V0cHV0ICAgICAgICAgICAgIDoge291dH0iKQogICAgcHJpbnQoIiAgTW9kdWxlIDUgdW50b3VjaGVkIDogVHJ1ZSIpCiAgICBpZiBzdG9wcGVkOgogICAgICAgIHByaW50KCJcbiAgU1RPUFBFRCBFQVJMWSAoY2xlYW4sIG5vdCBhIGNyYXNoKToiKQogICAgICAgIHByaW50KGYiICB7c3RvcHBlZH0iKQogICAgICAgIHN5cy5leGl0KDIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=", "sha256": "b5561ae0073260b80a43532f056149e2b64b5b682f31b7880bbc77d74c31d7d4", "size": 7634}, "scripts/m6_02_build_index.py": {"b64": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uCiIiIk1vZHVsZSA2IFN0YWdlIDIg4oCUIGJ1aWxkIHRoZSBGQUlTUyBmbGF0LUlQIGluZGV4LiBDUFUuIEZhc3QuCgpMb2FkcyB0aGUgc2hhcmRzIGZyb20gU3RhZ2UgMSwgYnVpbGRzIGFuIGV4YWN0IGlubmVyLXByb2R1Y3QgaW5kZXggb3ZlcgpMMi1ub3JtYWxpc2VkIHZlY3RvcnMgKD09IGNvc2luZSksIGFuZCB0aGVuIEFVRElUUyB0aGUgYnVpbHQgaW5kZXggYWdhaW5zdAp0aGUgcGhhc2Ugc2NvcGUuIFRoZSBhdWRpdCBpcyBkZWxpYmVyYXRlbHkgYWZ0ZXIgY29uc3RydWN0aW9uOiBpdCBjaGVja3Mgd2hhdAp3YXMgYWN0dWFsbHkgaW5kZXhlZCwgbm90IHdoYXQgd2FzIGludGVuZGVkLgoKICAgIHB5dGhvbiBzY3JpcHRzL202XzAyX2J1aWxkX2luZGV4LnB5IC0tZGF0YXNldCBmZXZlciAtLXBoYXNlIHRlc3QKIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1sxXSkpCgpmcm9tIHNjcmlwdHMubTZfY29tbW9uIGltcG9ydCAoICAjIG5vcWE6IEU0MDIKICAgIFBST0pFQ1RfUk9PVCwgYXNzZXJ0X21vZHVsZTVfdW50b3VjaGVkLCBiYW5uZXIsIGxvYWRfY29uZmlnLCBsb2FkX2NvcnB1cywKICAgIG1vZHVsZTVfZmluZ2VycHJpbnQsIHJlYWRfbWFuaWZlc3QsIHN0YWdlX2Rpciwgd3JpdGVfbWFuaWZlc3QpCmZyb20gc3JjLnJldHJpZXZhbC5lbmNvZGVycyBpbXBvcnQgU2hhcmRlZEVuY29kZXIgICMgbm9xYTogRTQwMgpmcm9tIHNyYy5yZXRyaWV2YWwuZmFpc3NfaW5kZXggaW1wb3J0IEV2aWRlbmNlSW5kZXgsIGZhaXNzX2F2YWlsYWJsZSAgIyBub3FhOiBFNDAyCmZyb20gc3JjLnJldHJpZXZhbC5pbmRleF9zY29wZSBpbXBvcnQgKCAgIyBub3FhOiBFNDAyCiAgICBhc3NlcnRfaW5kZXhfd2l0aGluX3Njb3BlLCBidWlsZF9ldmlkZW5jZV9zY29wZSkKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJCdWlsZCB0aGUgZXZpZGVuY2UgaW5kZXguIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1kYXRhc2V0IiwgcmVxdWlyZWQ9VHJ1ZSwgY2hvaWNlcz1bImZldmVyIiwgInhfZmFjdCJdKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXBoYXNlIiwgZGVmYXVsdD0idGVzdCIsIGNob2ljZXM9WyJ0cmFpbiIsICJ2YWxpZGF0aW9uIiwgInRlc3QiXSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1jb25maWciLCBkZWZhdWx0PU5vbmUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcHJvamVjdC1yb290IiwgZGVmYXVsdD1Ob25lKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQoKICAgIHJvb3QgPSBQYXRoKGFyZ3MucHJvamVjdF9yb290KSBpZiBhcmdzLnByb2plY3Rfcm9vdCBlbHNlIFBST0pFQ1RfUk9PVAogICAgY2ZnID0gbG9hZF9jb25maWcoYXJncy5jb25maWcpCiAgICBiYW5uZXIoZiJTdGFnZSAyIGJ1aWxkIGluZGV4IFt7YXJncy5kYXRhc2V0fS97YXJncy5waGFzZX1dIiwgY2ZnKQogICAgcHJpbnQoZiIgIGZhaXNzIGF2YWlsYWJsZToge2ZhaXNzX2F2YWlsYWJsZSgpfSAobnVtcHkgZmFsbGJhY2sgaXMgZXhhY3QgZWl0aGVyIHdheSkiKQogICAgYmVmb3JlID0gbW9kdWxlNV9maW5nZXJwcmludChyb290KQoKICAgIHNyYyA9IHN0YWdlX2RpcihjZmcsICJlbmNvZGVfZXZpZGVuY2UiLCByb290KSAvIGYie2FyZ3MuZGF0YXNldH1fe2FyZ3MucGhhc2V9IgogICAgbWFuID0gcmVhZF9tYW5pZmVzdChzcmMpCiAgICBpZiBtYW4uZ2V0KCJzdGF0dXMiKSAhPSAiY29tcGxldGUiOgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoZiJTdGFnZSAxIG5vdCBjb21wbGV0ZSBmb3Ige2FyZ3MuZGF0YXNldH0ve2FyZ3MucGhhc2V9ICIKICAgICAgICAgICAgICAgICAgICAgICAgIGYiKHN0YXR1cz17bWFuLmdldCgnc3RhdHVzJykhcn0pLiBSdW4gbTZfMDEgZmlyc3QuIikKCiAgICBlbmMgPSBTaGFyZGVkRW5jb2RlcihzcmMsIHNoYXJkX3NpemU9aW50KG1hbi5nZXQoInNoYXJkX3NpemUiLCAxMDAwMCkpKQogICAgdmVjdG9ycyA9IGVuYy5sb2FkX2FsbCgpCiAgICBldl9pZHMgPSBlbmMubG9hZF9pZHMoKQogICAgaWYgbGVuKGV2X2lkcykgIT0gdmVjdG9ycy5zaGFwZVswXToKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KGYie2xlbihldl9pZHMpfSBpZHMgYnV0IHt2ZWN0b3JzLnNoYXBlWzBdfSB2ZWN0b3JzIikKCiAgICBpbmRleCA9IEV2aWRlbmNlSW5kZXgoZXZfaWRzLCB2ZWN0b3JzLCBub3JtYWxpemU9Ym9vbChjZmdbInJldHJpZXZhbCJdWyJub3JtYWxpemUiXSkpCgogICAgY29ycHVzID0gbG9hZF9jb3JwdXMoY2ZnLCBhcmdzLmRhdGFzZXQsIHJvb3Q9cm9vdCkKICAgIHNjb3BlID0gYnVpbGRfZXZpZGVuY2Vfc2NvcGUoY29ycHVzWyJjbGFpbXMiXSwgY29ycHVzWyJzcGxpdHMiXSwgYXJncy5waGFzZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY2ZnWyJsZWFrYWdlIl1bImluZGV4X3Njb3BlIl0pCiAgICBhc3NlcnRfaW5kZXhfd2l0aGluX3Njb3BlKGluZGV4LmV2aWRlbmNlX2lkcywgc2NvcGUpICAgIyByYWlzZXMgb24gbGVha2FnZQoKICAgIG91dCA9IHN0YWdlX2RpcihjZmcsICJidWlsZF9pbmRleCIsIHJvb3QpIC8gZiJ7YXJncy5kYXRhc2V0fV97YXJncy5waGFzZX0iCiAgICBwYXRocyA9IGluZGV4LnNhdmUob3V0KQogICAgd3JpdGVfbWFuaWZlc3Qob3V0LCB7InN0YWdlIjogImJ1aWxkX2luZGV4IiwgImRhdGFzZXQiOiBhcmdzLmRhdGFzZXQsCiAgICAgICAgICAgICAgICAgICAgICAgICAicGhhc2UiOiBhcmdzLnBoYXNlLCAic3RhdHVzIjogImNvbXBsZXRlIiwKICAgICAgICAgICAgICAgICAgICAgICAgICJuX2V2aWRlbmNlIjogbGVuKGluZGV4KSwgImRpbSI6IGluZGV4LmRpbSwKICAgICAgICAgICAgICAgICAgICAgICAgICJiYWNrZW5kIjogImZhaXNzIiBpZiBpbmRleC51c2VfZmFpc3MgZWxzZSAibnVtcHkiLAogICAgICAgICAgICAgICAgICAgICAgICAgInNjb3BlX2F1ZGl0IjogIlBBU1MiLCAic2NvcGUiOiBzY29wZS5zdGF0cywgInBhdGhzIjogcGF0aHN9KQoKICAgIGFzc2VydF9tb2R1bGU1X3VudG91Y2hlZChiZWZvcmUsIHJvb3QpCiAgICBwcmludChmIiAgaW5kZXhlZCB7bGVuKGluZGV4KTosfSBldmlkZW5jZSB2ZWN0b3JzIChkaW0ge2luZGV4LmRpbX0pIikKICAgIHByaW50KGYiICBzY29wZSBhdWRpdDogUEFTUyAgLT4gIHtvdXR9IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==", "sha256": "ef49945c040149f320ccc91066e66453e4962a8a495546d4b31df2ffc7fa77b3", "size": 3346}, "scripts/m6_03_encode_claims.py": {"b64": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uCiIiIk1vZHVsZSA2IFN0YWdlIDMg4oCUIGVuY29kZSBjbGFpbXMgKHF1ZXJpZXMpIHdpdGggTGFCU0UuIEdQVS4gUmVzdW1hYmxlLgoKT25seSB0aGUgY2xhaW1zIHRoYXQgd2lsbCBhY3R1YWxseSBiZSBxdWVyaWVkIGFyZSBlbmNvZGVkOiB0aGUgY29uZmlndXJlZApldmFsdWF0aW9uIHNwbGl0cywgcGx1cyBvcHRpb25hbGx5IHRyYWluIHdoZW4gYnVpbGRpbmcgdmVyaWZpZXIgaW5wdXRzLgoKICAgIHB5dGhvbiBzY3JpcHRzL202XzAzX2VuY29kZV9jbGFpbXMucHkgLS1kYXRhc2V0IGZldmVyIC0tc3BsaXQgdGVzdAoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBzeXMKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzFdKSkKCmZyb20gc2NyaXB0cy5tNl9jb21tb24gaW1wb3J0ICggICMgbm9xYTogRTQwMgogICAgUFJPSkVDVF9ST09ULCBhc3NlcnRfbW9kdWxlNV91bnRvdWNoZWQsIGJhbm5lciwgY29uZmlnX2hhc2gsIGRldGVjdF9kZXZpY2UsCiAgICBsb2FkX2NvbmZpZywgbG9hZF9jb3JwdXMsIG1vZHVsZTVfZmluZ2VycHJpbnQsIHN0YWdlX2Rpciwgd3JpdGVfbWFuaWZlc3QpCmZyb20gc3JjLnJldHJpZXZhbC5lbmNvZGVycyBpbXBvcnQgU2hhcmRlZEVuY29kZXIsIGJ1aWxkX2xhYnNlX2VuY29kZV9mbiAgIyBub3FhOiBFNDAyCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0iRW5jb2RlIGNsYWltcyB3aXRoIExhQlNFLiIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZGF0YXNldCIsIHJlcXVpcmVkPVRydWUsIGNob2ljZXM9WyJmZXZlciIsICJ4X2ZhY3QiXSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zcGxpdCIsIHJlcXVpcmVkPVRydWUsIGNob2ljZXM9WyJ0cmFpbiIsICJ2YWxpZGF0aW9uIiwgInRlc3QiXSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1jb25maWciLCBkZWZhdWx0PU5vbmUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcHJvamVjdC1yb290IiwgZGVmYXVsdD1Ob25lKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWRyeS1ydW4iLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW1vZGVsIiwgZGVmYXVsdD1Ob25lLAogICAgICAgICAgICAgICAgICAgIGhlbHA9Ik92ZXJyaWRlIHRoZSBiaS1lbmNvZGVyIChzbWFsbGVyIG1vZGVsID0gbGVzcyBSQU0pLiIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYmF0Y2gtc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PU5vbmUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbWF4LXNlcS1sZW5ndGgiLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNoYXJkLXNpemUiLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQoKICAgIHJvb3QgPSBQYXRoKGFyZ3MucHJvamVjdF9yb290KSBpZiBhcmdzLnByb2plY3Rfcm9vdCBlbHNlIFBST0pFQ1RfUk9PVAogICAgY2ZnID0gbG9hZF9jb25maWcoYXJncy5jb25maWcpCiAgICBiYW5uZXIoZiJTdGFnZSAzIGVuY29kZSBjbGFpbXMgW3thcmdzLmRhdGFzZXR9L3thcmdzLnNwbGl0fV0iLCBjZmcpCiAgICBiZWZvcmUgPSBtb2R1bGU1X2ZpbmdlcnByaW50KHJvb3QpCgogICAgY29ycHVzID0gbG9hZF9jb3JwdXMoY2ZnLCBhcmdzLmRhdGFzZXQsIHJvb3Q9cm9vdCkKICAgIGlkeHMgPSBzb3J0ZWQoY29ycHVzWyJzcGxpdHMiXS5nZXQoYXJncy5zcGxpdCwgW10pKQogICAgaWYgbm90IGlkeHM6CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdChmInNwbGl0IHthcmdzLnNwbGl0IXJ9IGlzIGVtcHR5IGZvciB7YXJncy5kYXRhc2V0fSIpCiAgICBjbGFpbV9pZHMgPSBbY29ycHVzWyJjbGFpbV9pZF9vZiJdW2ldIGZvciBpIGluIGlkeHNdCiAgICB0ZXh0cyA9IFtjb3JwdXNbImNsYWltcyJdW2ldLmdldCgidGV4dCIsICIiKSBmb3IgaSBpbiBpZHhzXQogICAgcHJpbnQoZiIgIGNsYWltcyB0byBlbmNvZGU6IHtsZW4odGV4dHMpOix9IikKCiAgICBvdXQgPSBzdGFnZV9kaXIoY2ZnLCAiZW5jb2RlX2NsYWltcyIsIHJvb3QpIC8gZiJ7YXJncy5kYXRhc2V0fV97YXJncy5zcGxpdH0iCiAgICBvdXQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQoKICAgIGlmIGFyZ3MuZHJ5X3J1bjoKICAgICAgICB3cml0ZV9tYW5pZmVzdChvdXQsIHsic3RhZ2UiOiAiZW5jb2RlX2NsYWltcyIsICJkYXRhc2V0IjogYXJncy5kYXRhc2V0LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICJzcGxpdCI6IGFyZ3Muc3BsaXQsICJzdGF0dXMiOiAiZHJ5X3J1biIsICJuX2NsYWltcyI6IGxlbih0ZXh0cyl9KQogICAgICAgIHByaW50KCIgIERSWSBSVU46IG5vIG1vZGVsIGxvYWRlZC4iKQogICAgICAgIGFzc2VydF9tb2R1bGU1X3VudG91Y2hlZChiZWZvcmUsIHJvb3QpCiAgICAgICAgcmV0dXJuCgogICAgciA9IGNmZ1sicmV0cmlldmFsIl0KICAgIG1vZGVsX25hbWUgPSBhcmdzLm1vZGVsIG9yIHJbImJpX2VuY29kZXIiXQogICAgYmF0Y2ggPSBpbnQoYXJncy5iYXRjaF9zaXplIG9yIHJbImVuY29kZV9iYXRjaF9zaXplIl0pCiAgICBzZXEgPSBpbnQoYXJncy5tYXhfc2VxX2xlbmd0aCBvciByWyJtYXhfc2VxX2xlbmd0aCJdKQogICAgc2hhcmQgPSBpbnQoYXJncy5zaGFyZF9zaXplIG9yIHJbInNoYXJkX3NpemUiXSkKICAgIHByaW50KGYiICBtb2RlbD17bW9kZWxfbmFtZX0gYmF0Y2g9e2JhdGNofSBzZXE9e3NlcX0gc2hhcmQ9e3NoYXJkfSIpCiAgICBlbmMgPSBTaGFyZGVkRW5jb2RlcihvdXQsIHNoYXJkX3NpemU9c2hhcmQsCiAgICAgICAgICAgICAgICAgICAgICAgICBlbmNvZGVfZm49YnVpbGRfbGFic2VfZW5jb2RlX2ZuKAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1vZGVsX25hbWUsIGJhdGNoX3NpemU9YmF0Y2gsIG1heF9zZXFfbGVuZ3RoPXNlcSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBkZXZpY2U9ZGV0ZWN0X2RldmljZSgpKSwKICAgICAgICAgICAgICAgICAgICAgICAgIG1vZGVsX25hbWU9bW9kZWxfbmFtZSkKICAgIG1hbiA9IGVuYy5lbmNvZGUodGV4dHMsIGlkcz1jbGFpbV9pZHMpCiAgICBtYW4udXBkYXRlKHsic3RhZ2UiOiAiZW5jb2RlX2NsYWltcyIsICJkYXRhc2V0IjogYXJncy5kYXRhc2V0LCAic3BsaXQiOiBhcmdzLnNwbGl0LAogICAgICAgICAgICAgICAgImNvbmZpZ19oYXNoIjogY29uZmlnX2hhc2goY2ZnLCBbInJldHJpZXZhbCJdKX0pCiAgICB3cml0ZV9tYW5pZmVzdChvdXQsIG1hbikKICAgIGFzc2VydF9tb2R1bGU1X3VudG91Y2hlZChiZWZvcmUsIHJvb3QpCiAgICBwcmludChmIiAgd3JvdGUge21hblsnbl9zaGFyZHMnXX0gc2hhcmQocykgLT4ge291dH0iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK", "sha256": "8e6d9902e4aec4e72bb16119facf986958957ed4e43ffb6e69411b7243c73d4d", "size": 3507}, "scripts/m6_04_retrieve.py": {"b64": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uCiIiIk1vZHVsZSA2IFN0YWdlIDQg4oCUIGRlbnNlIGFuZC9vciBCTTI1IHJldHJpZXZhbCArIHJlYWwgZ29sZCBtZXRyaWNzLiBDUFUuCgpSdW5zIHRoZSB0b3AtNTAgc2VhcmNoLCB0aGVuIHNjb3JlcyBpdCBhZ2FpbnN0IE1vZHVsZSA1J3MgZ29sZCBldmlkZW5jZS4KQ2xhaW1zIHdpdGggbm8gZ29sZCBldmlkZW5jZSAtLSBGRVZFUidzIE5PVCBFTk9VR0ggSU5GTyBhbW9uZyB0aGVtIC0tIGFyZQpleGNsdWRlZCBmcm9tIHJlY2FsbCBkZW5vbWluYXRvcnMgYW5kIGNvdW50ZWQgZXhwbGljaXRseSwgbmV2ZXIgZHJvcHBlZApzaWxlbnRseS4KCiAgICBweXRob24gc2NyaXB0cy9tNl8wNF9yZXRyaWV2ZS5weSAtLWRhdGFzZXQgZmV2ZXIgLS1zcGxpdCB0ZXN0IC0tcmV0cmlldmVyIGRlbnNlCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IHN5cwppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMV0pKQoKZnJvbSBzY3JpcHRzLm02X2NvbW1vbiBpbXBvcnQgKCAgIyBub3FhOiBFNDAyCiAgICBQUk9KRUNUX1JPT1QsIGFzc2VydF9tb2R1bGU1X3VudG91Y2hlZCwgYmFubmVyLCBnb2xkX2V2aWRlbmNlX21hcCwgbG9hZF9jb25maWcsCiAgICBsb2FkX2NvcnB1cywgbW9kdWxlNV9maW5nZXJwcmludCwgcmVhZF9tYW5pZmVzdCwgc3RhZ2VfZGlyLCB3cml0ZV9qc29uLCB3cml0ZV9tYW5pZmVzdCkKZnJvbSBzcmMucmV0cmlldmFsLmJtMjVfYmFzZWxpbmUgaW1wb3J0IEJNMjVSZXRyaWV2ZXIgICMgbm9xYTogRTQwMgpmcm9tIHNyYy5yZXRyaWV2YWwuZW5jb2RlcnMgaW1wb3J0IFNoYXJkZWRFbmNvZGVyICAjIG5vcWE6IEU0MDIKZnJvbSBzcmMucmV0cmlldmFsLmZhaXNzX2luZGV4IGltcG9ydCBFdmlkZW5jZUluZGV4ICAjIG5vcWE6IEU0MDIKZnJvbSBzcmMucmV0cmlldmFsLmluZGV4X3Njb3BlIGltcG9ydCAoICAjIG5vcWE6IEU0MDIKICAgIGFzc2VydF9pbmRleF93aXRoaW5fc2NvcGUsIGJ1aWxkX2V2aWRlbmNlX3Njb3BlKQpmcm9tIHNyYy5yZXRyaWV2YWwucmV0cmlldmFsX21ldHJpY3MgaW1wb3J0IGV2YWx1YXRlX3JldHJpZXZhbCAgIyBub3FhOiBFNDAyCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0iUmV0cmlldmUgY2FuZGlkYXRlcyBhbmQgc2NvcmUgdGhlbS4iKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWRhdGFzZXQiLCByZXF1aXJlZD1UcnVlLCBjaG9pY2VzPVsiZmV2ZXIiLCAieF9mYWN0Il0pCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc3BsaXQiLCByZXF1aXJlZD1UcnVlLCBjaG9pY2VzPVsidHJhaW4iLCAidmFsaWRhdGlvbiIsICJ0ZXN0Il0pCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcmV0cmlldmVyIiwgZGVmYXVsdD0iZGVuc2UiLCBjaG9pY2VzPVsiZGVuc2UiLCAiYm0yNSIsICJib3RoIl0pCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tdG9wLWsiLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWNvbmZpZyIsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1wcm9qZWN0LXJvb3QiLCBkZWZhdWx0PU5vbmUpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCgogICAgcm9vdCA9IFBhdGgoYXJncy5wcm9qZWN0X3Jvb3QpIGlmIGFyZ3MucHJvamVjdF9yb290IGVsc2UgUFJPSkVDVF9ST09UCiAgICBjZmcgPSBsb2FkX2NvbmZpZyhhcmdzLmNvbmZpZykKICAgIHRvcF9rID0gaW50KGFyZ3MudG9wX2sgb3IgY2ZnWyJyZXRyaWV2YWwiXVsidG9wX2tfY2FuZGlkYXRlcyJdKQogICAgcGhhc2UgPSBhcmdzLnNwbGl0CiAgICBiYW5uZXIoZiJTdGFnZSA0IHJldHJpZXZlIFt7YXJncy5kYXRhc2V0fS97YXJncy5zcGxpdH1dIHRvcC17dG9wX2t9IiwgY2ZnKQogICAgYmVmb3JlID0gbW9kdWxlNV9maW5nZXJwcmludChyb290KQoKICAgIGNvcnB1cyA9IGxvYWRfY29ycHVzKGNmZywgYXJncy5kYXRhc2V0LCByb290PXJvb3QpCiAgICBzY29wZSA9IGJ1aWxkX2V2aWRlbmNlX3Njb3BlKGNvcnB1c1siY2xhaW1zIl0sIGNvcnB1c1sic3BsaXRzIl0sIHBoYXNlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjZmdbImxlYWthZ2UiXVsiaW5kZXhfc2NvcGUiXSkKICAgIGlkeHMgPSBzb3J0ZWQoY29ycHVzWyJzcGxpdHMiXS5nZXQoYXJncy5zcGxpdCwgW10pKQogICAgY2xhaW1faWRzID0gW2NvcnB1c1siY2xhaW1faWRfb2YiXVtpXSBmb3IgaSBpbiBpZHhzXQogICAgY2xhaW1fdGV4dHMgPSBbY29ycHVzWyJjbGFpbXMiXVtpXS5nZXQoInRleHQiLCAiIikgZm9yIGkgaW4gaWR4c10KICAgIGdvbGQgPSBnb2xkX2V2aWRlbmNlX21hcChjb3JwdXMpCiAgICBsYW5ncyA9IHtjb3JwdXNbImNsYWltX2lkX29mIl1baV06IHN0cihjb3JwdXNbImNsYWltcyJdW2ldLmdldCgibGFuZ3VhZ2UiLCAidW5rbm93biIpKQogICAgICAgICAgICAgZm9yIGkgaW4gaWR4c30KCiAgICBvdXQgPSBzdGFnZV9kaXIoY2ZnLCAicmV0cmlldmUiLCByb290KSAvIGYie2FyZ3MuZGF0YXNldH1fe2FyZ3Muc3BsaXR9IgogICAgb3V0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHJlc3VsdHM6IGRpY3Rbc3RyLCBkaWN0XSA9IHt9CgogICAgaWYgYXJncy5yZXRyaWV2ZXIgaW4gKCJkZW5zZSIsICJib3RoIik6CiAgICAgICAgaWR4X2RpciA9IHN0YWdlX2RpcihjZmcsICJidWlsZF9pbmRleCIsIHJvb3QpIC8gZiJ7YXJncy5kYXRhc2V0fV97cGhhc2V9IgogICAgICAgIGlmIG5vdCAoaWR4X2RpciAvICJpbmRleF9tZXRhLmpzb24iKS5pc19maWxlKCk6CiAgICAgICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoZiJubyBpbmRleCBhdCB7aWR4X2Rpcn07IHJ1biBtNl8wMiBmaXJzdCIpCiAgICAgICAgaW5kZXggPSBFdmlkZW5jZUluZGV4LmxvYWQoaWR4X2RpcikKICAgICAgICBhc3NlcnRfaW5kZXhfd2l0aGluX3Njb3BlKGluZGV4LmV2aWRlbmNlX2lkcywgc2NvcGUpCgogICAgICAgIHFfZGlyID0gc3RhZ2VfZGlyKGNmZywgImVuY29kZV9jbGFpbXMiLCByb290KSAvIGYie2FyZ3MuZGF0YXNldH1fe2FyZ3Muc3BsaXR9IgogICAgICAgIGlmIHJlYWRfbWFuaWZlc3QocV9kaXIpLmdldCgic3RhdHVzIikgIT0gImNvbXBsZXRlIjoKICAgICAgICAgICAgcmFpc2UgU3lzdGVtRXhpdChmImNsYWltIGVtYmVkZGluZ3MgaW5jb21wbGV0ZSBhdCB7cV9kaXJ9OyBydW4gbTZfMDMgZmlyc3QiKQogICAgICAgIHFfZW5jID0gU2hhcmRlZEVuY29kZXIocV9kaXIpCiAgICAgICAgcXVlcmllcyA9IHFfZW5jLmxvYWRfYWxsKCkKICAgICAgICBxX2lkcyA9IHFfZW5jLmxvYWRfaWRzKCkKICAgICAgICBpZiBxX2lkcyAhPSBjbGFpbV9pZHM6CiAgICAgICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoImNsYWltIGVtYmVkZGluZyBpZHMgZG8gbm90IG1hdGNoIHRoZSBzcGxpdDsgcmUtcnVuIG02XzAzIikKCiAgICAgICAgdDAgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgcmFua2VkLCBzY29yZXMgPSBpbmRleC5zZWFyY2gocXVlcmllcywgdG9wX2spCiAgICAgICAgZGVuc2UgPSB7Y2lkOiBsaXN0KHIpIGZvciBjaWQsIHIgaW4gemlwKGNsYWltX2lkcywgcmFua2VkKX0KICAgICAgICB3cml0ZV9qc29uKG91dCAvICJkZW5zZV9jYW5kaWRhdGVzLmpzb24iLCBkZW5zZSkKICAgICAgICBtID0gZXZhbHVhdGVfcmV0cmlldmFsKGRlbnNlLCBnb2xkLCB0dXBsZShrIGZvciBrIGluICgxLCA1LCAxMCwgdG9wX2spKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG5kY2dfaz1pbnQoY2ZnWyJldmFsdWF0aW9uIl1bIm5kY2dfayJdKSwgbGFuZ3VhZ2VzPWxhbmdzKQogICAgICAgIG0ucG9wKCJwZXJfY2xhaW0iLCBOb25lKQogICAgICAgIHdyaXRlX2pzb24ob3V0IC8gImRlbnNlX21ldHJpY3MuanNvbiIsIG0pCiAgICAgICAgcmVzdWx0c1siZGVuc2UiXSA9IHsic2Vjb25kcyI6IHJvdW5kKHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MCwgMiksICJtZXRyaWNzIjogbX0KICAgICAgICBwcmludChmIiAgZGVuc2UgIDoge21bJ292ZXJhbGwnXX0iKQogICAgICAgIHByaW50KGYiICAgICAgICAgICBzY29yZWQge21bJ2FjY291bnRpbmcnXVsnbl9zY29yZWQnXTosfSwgIgogICAgICAgICAgICAgIGYiZXhjbHVkZWQgKG5vIGdvbGQpIHttWydhY2NvdW50aW5nJ11bJ25fZXhjbHVkZWRfbm9fZ29sZF9ldmlkZW5jZSddOix9IikKCiAgICBpZiBhcmdzLnJldHJpZXZlciBpbiAoImJtMjUiLCAiYm90aCIpOgogICAgICAgIGV2X2lkcyA9IHNvcnRlZChzY29wZS5ldmlkZW5jZV9pZHMpCiAgICAgICAgdDAgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgYm0gPSBCTTI1UmV0cmlldmVyKGV2X2lkcywgW2NvcnB1c1siZXZpZGVuY2VfdGV4dHMiXS5nZXQoZSwgIiIpIGZvciBlIGluIGV2X2lkc10sCiAgICAgICAgICAgICAgICAgICAgICAgICAgIGsxPWZsb2F0KGNmZ1siYm0yNSJdWyJrMSJdKSwgYj1mbG9hdChjZmdbImJtMjUiXVsiYiJdKSkKICAgICAgICBsZXggPSB7Y2lkOiBibS5yZXRyaWV2ZSh0eHQsIHRvcF9rKVswXSBmb3IgY2lkLCB0eHQgaW4gemlwKGNsYWltX2lkcywgY2xhaW1fdGV4dHMpfQogICAgICAgIHdyaXRlX2pzb24ob3V0IC8gImJtMjVfY2FuZGlkYXRlcy5qc29uIiwgbGV4KQogICAgICAgIG0gPSBldmFsdWF0ZV9yZXRyaWV2YWwobGV4LCBnb2xkLCB0dXBsZShrIGZvciBrIGluICgxLCA1LCAxMCwgdG9wX2spKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG5kY2dfaz1pbnQoY2ZnWyJldmFsdWF0aW9uIl1bIm5kY2dfayJdKSwgbGFuZ3VhZ2VzPWxhbmdzKQogICAgICAgIG0ucG9wKCJwZXJfY2xhaW0iLCBOb25lKQogICAgICAgIHdyaXRlX2pzb24ob3V0IC8gImJtMjVfbWV0cmljcy5qc29uIiwgbSkKICAgICAgICByZXN1bHRzWyJibTI1Il0gPSB7InNlY29uZHMiOiByb3VuZCh0aW1lLnBlcmZfY291bnRlcigpIC0gdDAsIDIpLCAibWV0cmljcyI6IG19CiAgICAgICAgcHJpbnQoZiIgIGJtMjUgICA6IHttWydvdmVyYWxsJ119IikKCiAgICB3cml0ZV9tYW5pZmVzdChvdXQsIHsic3RhZ2UiOiAicmV0cmlldmUiLCAiZGF0YXNldCI6IGFyZ3MuZGF0YXNldCwgInNwbGl0IjogYXJncy5zcGxpdCwKICAgICAgICAgICAgICAgICAgICAgICAgICJwaGFzZSI6IHBoYXNlLCAidG9wX2siOiB0b3BfaywgInN0YXR1cyI6ICJjb21wbGV0ZSIsCiAgICAgICAgICAgICAgICAgICAgICAgICAicmV0cmlldmVycyI6IHNvcnRlZChyZXN1bHRzKSwgInNjb3BlIjogc2NvcGUuc3RhdHMsCiAgICAgICAgICAgICAgICAgICAgICAgICAic3VtbWFyeSI6IHtrOiB2WyJtZXRyaWNzIl1bIm92ZXJhbGwiXSBmb3IgaywgdiBpbiByZXN1bHRzLml0ZW1zKCl9fSkKICAgIGFzc2VydF9tb2R1bGU1X3VudG91Y2hlZChiZWZvcmUsIHJvb3QpCiAgICBwcmludChmIlxuICB3cm90ZSAtPiB7b3V0fSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=", "sha256": "11b98584a9eba54ed682362674af4774fcee877918b3da85bac5071cca4585d7", "size": 5594}, "scripts/m6_05_rerank.py": {"b64": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uCiIiIk1vZHVsZSA2IFN0YWdlIDUg4oCUIGNyb3NzLWVuY29kZXIgcmVyYW5raW5nLiBHUFUuIFJlc3VtYWJsZS4KClJlcmFua3MgdGhlIHRvcC01MCBjYW5kaWRhdGVzIGRvd24gdG8gdG9wLTUuIFRoZSBjYW5kaWRhdGUgc2V0IGlzIG5ldmVyCndpZGVuZWQ6IGByZXJhbmtfYWxsKClgIGFzc2VydHMgdGhlIG91dHB1dCBpcyBhIHN1YnNldCBvZiB3aGF0IHJldHJpZXZhbApwcm9wb3NlZCwgYmVjYXVzZSBpbnZlbnRpbmcgZXZpZGVuY2Ugd291bGQgdHVybiByZXRyaWV2YWwgbWlzc2VzIGludG8KYXBwYXJlbnQgaGl0cyBhbmQgaW5mbGF0ZSBldmVyeSBkb3duc3RyZWFtIG51bWJlci4KClNlcXVlbmNlIGxlbmd0aCBpcyAyNTYsIG5vdCB0aGUgb3JpZ2luYWxseSBjb25maWd1cmVkIDUxMiwgYW5kIG9ubHkgdGhlCmV2YWx1YXRpb24gc3BsaXRzIHBsdXMgYSBkb2N1bWVudGVkIHRyYWluIHN1YnNldCBhcmUgcmVyYW5rZWQuIEJvdGggYXJlCmZyZWUtQ29sYWIgY29tcHV0ZSByZWR1Y3Rpb25zIGFuZCBhcmUgcmVjb3JkZWQgaW4gdGhlIG1hbmlmZXN0LgoKICAgIHB5dGhvbiBzY3JpcHRzL202XzA1X3JlcmFuay5weSAtLWRhdGFzZXQgZmV2ZXIgLS1zcGxpdCB0ZXN0CiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMV0pKQoKZnJvbSBzY3JpcHRzLm02X2NvbW1vbiBpbXBvcnQgKCAgIyBub3FhOiBFNDAyCiAgICBQUk9KRUNUX1JPT1QsIGFzc2VydF9tb2R1bGU1X3VudG91Y2hlZCwgYmFubmVyLCBkZXRlY3RfZGV2aWNlLCBnb2xkX2V2aWRlbmNlX21hcCwKICAgIGxvYWRfY29uZmlnLCBsb2FkX2NvcnB1cywgbW9kdWxlNV9maW5nZXJwcmludCwgc3RhZ2VfZGlyLCB3cml0ZV9qc29uLCB3cml0ZV9tYW5pZmVzdCkKZnJvbSBzcmMucmV0cmlldmFsLnJlcmFua2VyIGltcG9ydCAoICAjIG5vcWE6IEU0MDIKICAgIENyb3NzRW5jb2RlclJlcmFua2VyLCBidWlsZF9jcm9zc19lbmNvZGVyX3Njb3JlX2ZuLCBsb2FkX3JlcmFua2VkKQpmcm9tIHNyYy5yZXRyaWV2YWwucmV0cmlldmFsX21ldHJpY3MgaW1wb3J0IGV2YWx1YXRlX3JldHJpZXZhbCAgIyBub3FhOiBFNDAyCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0iUmVyYW5rIHJldHJpZXZlZCBjYW5kaWRhdGVzLiIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZGF0YXNldCIsIHJlcXVpcmVkPVRydWUsIGNob2ljZXM9WyJmZXZlciIsICJ4X2ZhY3QiXSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zcGxpdCIsIHJlcXVpcmVkPVRydWUsIGNob2ljZXM9WyJ0cmFpbiIsICJ2YWxpZGF0aW9uIiwgInRlc3QiXSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1jYW5kaWRhdGVzIiwgZGVmYXVsdD0iZGVuc2UiLCBjaG9pY2VzPVsiZGVuc2UiLCAiYm0yNSJdKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWNoZWNrcG9pbnQiLCBkZWZhdWx0PU5vbmUsCiAgICAgICAgICAgICAgICAgICAgaGVscD0iRmluZS10dW5lZCBjcm9zcy1lbmNvZGVyLiBXaXRob3V0IG9uZSB0aGUgc2NvcmVzIGFyZSAiCiAgICAgICAgICAgICAgICAgICAgICAgICAiZnJvbSBhbiB1bnRyYWluZWQgaGVhZCBhbmQgYXJlIE5PVCBtZWFuaW5nZnVsLiIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbGltaXQiLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lLAogICAgICAgICAgICAgICAgICAgIGhlbHA9IkNhcCBjbGFpbXMgcmVyYW5rZWQgKHRyYWluIHN1YnNldCkuIFJlY29yZGVkIGluIHRoZSBtYW5pZmVzdC4iKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWNvbmZpZyIsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1wcm9qZWN0LXJvb3QiLCBkZWZhdWx0PU5vbmUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZHJ5LXJ1biIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCgogICAgcm9vdCA9IFBhdGgoYXJncy5wcm9qZWN0X3Jvb3QpIGlmIGFyZ3MucHJvamVjdF9yb290IGVsc2UgUFJPSkVDVF9ST09UCiAgICBjZmcgPSBsb2FkX2NvbmZpZyhhcmdzLmNvbmZpZykKICAgIHJyID0gY2ZnWyJyZXJhbmtpbmciXQogICAgYmFubmVyKGYiU3RhZ2UgNSByZXJhbmsgW3thcmdzLmRhdGFzZXR9L3thcmdzLnNwbGl0fV0iLCBjZmcpCiAgICBiZWZvcmUgPSBtb2R1bGU1X2ZpbmdlcnByaW50KHJvb3QpCgogICAgY2FuZF9wYXRoID0gKHN0YWdlX2RpcihjZmcsICJyZXRyaWV2ZSIsIHJvb3QpIC8gZiJ7YXJncy5kYXRhc2V0fV97YXJncy5zcGxpdH0iCiAgICAgICAgICAgICAgICAgLyBmInthcmdzLmNhbmRpZGF0ZXN9X2NhbmRpZGF0ZXMuanNvbiIpCiAgICBpZiBub3QgY2FuZF9wYXRoLmlzX2ZpbGUoKToKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KGYibm8gY2FuZGlkYXRlcyBhdCB7Y2FuZF9wYXRofTsgcnVuIG02XzA0IGZpcnN0IikKICAgIGNhbmRpZGF0ZXMgPSBqc29uLmxvYWRzKGNhbmRfcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCgogICAgY29ycHVzID0gbG9hZF9jb3JwdXMoY2ZnLCBhcmdzLmRhdGFzZXQsIHJvb3Q9cm9vdCkKICAgIGNsYWltcyA9IHtjb3JwdXNbImNsYWltX2lkX29mIl1baV06IGNvcnB1c1siY2xhaW1zIl1baV0uZ2V0KCJ0ZXh0IiwgIiIpCiAgICAgICAgICAgICAgZm9yIGkgaW4gY29ycHVzWyJzcGxpdHMiXS5nZXQoYXJncy5zcGxpdCwgW10pfQoKICAgIGxpbWl0ID0gYXJncy5saW1pdAogICAgaWYgbGltaXQgaXMgTm9uZSBhbmQgYXJncy5zcGxpdCA9PSAidHJhaW4iOgogICAgICAgIGxpbWl0ID0gaW50KHJyLmdldCgidHJhaW5fc3Vic2V0X3NpemUiKSBvciAwKSBvciBOb25lCiAgICBpZiBsaW1pdDoKICAgICAgICBrZWVwID0gc29ydGVkKGNhbmRpZGF0ZXMpWzpsaW1pdF0gICAgICAgICAgIyBkZXRlcm1pbmlzdGljIHN1YnNldAogICAgICAgIGNhbmRpZGF0ZXMgPSB7azogY2FuZGlkYXRlc1trXSBmb3IgayBpbiBrZWVwfQogICAgcHJpbnQoZiIgIGNsYWltcyB0byByZXJhbms6IHtsZW4oY2FuZGlkYXRlcyk6LH0iCiAgICAgICAgICArIChmIiAgKGxpbWl0ZWQgZnJvbSB0aGUgZnVsbCBzcGxpdCkiIGlmIGxpbWl0IGVsc2UgIiIpKQoKICAgIG91dCA9IHN0YWdlX2RpcihjZmcsICJyZXJhbmsiLCByb290KSAvIGYie2FyZ3MuZGF0YXNldH1fe2FyZ3Muc3BsaXR9IgogICAgb3V0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICBpZiBhcmdzLmRyeV9ydW46CiAgICAgICAgd3JpdGVfbWFuaWZlc3Qob3V0LCB7InN0YWdlIjogInJlcmFuayIsICJkYXRhc2V0IjogYXJncy5kYXRhc2V0LCAic3BsaXQiOiBhcmdzLnNwbGl0LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICJzdGF0dXMiOiAiZHJ5X3J1biIsICJuX2NsYWltcyI6IGxlbihjYW5kaWRhdGVzKX0pCiAgICAgICAgcHJpbnQoIiAgRFJZIFJVTjogbm8gbW9kZWwgbG9hZGVkLiIpCiAgICAgICAgYXNzZXJ0X21vZHVsZTVfdW50b3VjaGVkKGJlZm9yZSwgcm9vdCkKICAgICAgICByZXR1cm4KCiAgICBpZiBub3QgYXJncy5jaGVja3BvaW50OgogICAgICAgIHByaW50KCIgIFdBUk5JTkc6IG5vIC0tY2hlY2twb2ludCBnaXZlbi4gQW4gdW50cmFpbmVkIGNyb3NzLWVuY29kZXIgaGVhZCAiCiAgICAgICAgICAgICAgInByb2R1Y2VzIGFyYml0cmFyeSBzY29yZXM7IHRoZXNlIHJlc3VsdHMgYXJlIG5vdCByZXBvcnRhYmxlLiIpCgogICAgcmVyYW5rZXIgPSBDcm9zc0VuY29kZXJSZXJhbmtlcigKICAgICAgICBzY29yZV9mbj1idWlsZF9jcm9zc19lbmNvZGVyX3Njb3JlX2ZuKAogICAgICAgICAgICBhcmdzLmNoZWNrcG9pbnQgb3IgcnJbImNyb3NzX2VuY29kZXIiXSwgbWF4X2xlbmd0aD1pbnQocnJbIm1heF9sZW5ndGgiXSksCiAgICAgICAgICAgIGRldmljZT1kZXRlY3RfZGV2aWNlKCkpLAogICAgICAgIHRvcF9uPWludChyclsidG9wX25fYWZ0ZXJfcmVyYW5rIl0pLCBiYXRjaF9zaXplPWludChyclsiYmF0Y2hfc2l6ZSJdKSkKICAgIG1hbiA9IHJlcmFua2VyLnJlcmFua19hbGwoY2xhaW1zLCBjYW5kaWRhdGVzLCBjb3JwdXNbImV2aWRlbmNlX3RleHRzIl0sIG91dCkKCiAgICBtZXJnZWQgPSBsb2FkX3JlcmFua2VkKG91dCkKICAgIHRvcDUgPSB7Y2lkOiB2WyJldmlkZW5jZV9pZHMiXSBmb3IgY2lkLCB2IGluIG1lcmdlZC5pdGVtcygpfQogICAgd3JpdGVfanNvbihvdXQgLyAicmVyYW5rZWRfdG9wNS5qc29uIiwgdG9wNSkKCiAgICBnb2xkID0gZ29sZF9ldmlkZW5jZV9tYXAoY29ycHVzKQogICAgbGFuZ3MgPSB7Y29ycHVzWyJjbGFpbV9pZF9vZiJdW2ldOiBzdHIoY29ycHVzWyJjbGFpbXMiXVtpXS5nZXQoImxhbmd1YWdlIiwgInVua25vd24iKSkKICAgICAgICAgICAgIGZvciBpIGluIGNvcnB1c1sic3BsaXRzIl0uZ2V0KGFyZ3Muc3BsaXQsIFtdKX0KICAgIG0gPSBldmFsdWF0ZV9yZXRyaWV2YWwodG9wNSwgZ29sZCwgKDEsIDUpLCBuZGNnX2s9NSwgbGFuZ3VhZ2VzPWxhbmdzKQogICAgbS5wb3AoInBlcl9jbGFpbSIsIE5vbmUpCiAgICB3cml0ZV9qc29uKG91dCAvICJyZXJhbmtfbWV0cmljcy5qc29uIiwgbSkKCiAgICBtYW4udXBkYXRlKHsic3RhZ2UiOiAicmVyYW5rIiwgImRhdGFzZXQiOiBhcmdzLmRhdGFzZXQsICJzcGxpdCI6IGFyZ3Muc3BsaXQsCiAgICAgICAgICAgICAgICAiY2FuZGlkYXRlc19mcm9tIjogYXJncy5jYW5kaWRhdGVzLCAibWF4X2xlbmd0aCI6IHJyWyJtYXhfbGVuZ3RoIl0sCiAgICAgICAgICAgICAgICAiY2hlY2twb2ludCI6IGFyZ3MuY2hlY2twb2ludCwgImxpbWl0ZWRfdG8iOiBsaW1pdCwKICAgICAgICAgICAgICAgICJjb21wdXRlX25vdGUiOiAibWF4X2xlbmd0aCAyNTYgKGNvbmZpZyBvcmlnaW5hbCA1MTIpOyAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInRyYWluIHJlcmFua2luZyBsaW1pdGVkIHRvIGEgZG9jdW1lbnRlZCBzdWJzZXQiLAogICAgICAgICAgICAgICAgIm1ldHJpY3MiOiBtWyJvdmVyYWxsIl0sICJzdGF0dXMiOiAiY29tcGxldGUifSkKICAgIHdyaXRlX21hbmlmZXN0KG91dCwgbWFuKQogICAgYXNzZXJ0X21vZHVsZTVfdW50b3VjaGVkKGJlZm9yZSwgcm9vdCkKICAgIHByaW50KGYiICByZXJhbmsgbWV0cmljczoge21bJ292ZXJhbGwnXX0iKQogICAgcHJpbnQoZiIgIHdyb3RlIC0+IHtvdXR9IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==", "sha256": "8b17c0a3a768666e0c58fe5b53f0bab78fcd33f337ca6e84d686a5e761cceea7", "size": 5341}, "scripts/m6_06_train_verifier.py": {"b64": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uCiIiIk1vZHVsZSA2IFN0YWdlIDYg4oCUIHRyYWluIHRoZSB2ZXJpZmllci4gR1BVLiBSZXN1bWFibGUuCgpUd28gZGlzdGluY3QgdGFza3MsIG5ldmVyIG1lcmdlZDoKCiAgLS1kYXRhc2V0IGZldmVyICAgMy13YXkgRU5UQUlMTUVOVCAgIFNVUFBPUlRTIC8gUkVGVVRFUyAvIE5PVCBFTk9VR0ggSU5GTwogIC0tZGF0YXNldCB4X2ZhY3QgIDMtd2F5IFZFUkFDSVRZICAgICBmYWxzZSAvIHRydWUgLyBwYXJ0bHkgdHJ1ZS9taXNsZWFkaW5nCgpYLUZBQ1QgdHJhaW5pbmcgc3RhcnRzIGZyb20gdGhlIEZFVkVSLXByZXRyYWluZWQgRU5DT0RFUiB2aWEKYC0taW5pdC1mcm9tYCwgYW5kIGFsd2F5cyBSRVBMQUNFUyB0aGUgY2xhc3NpZmljYXRpb24gaGVhZC4gUmV1c2luZyB0aGUgRkVWRVIKaGVhZCB3b3VsZCBzbXVnZ2xlIGVudGFpbG1lbnQgc2VtYW50aWNzIGludG8gdmVyYWNpdHkgbGFiZWxzIGFuZCBtYWtlIHRoZQp0cmFuc2ZlciBudW1iZXIgdW5pbnRlcnByZXRhYmxlLgoKQ29tcHV0ZSBzY29wZSAoZnJlZSBDb2xhYik6IEZFVkVSIGlzIHRyYWluZWQgb24gYSBzdHJhdGlmaWVkIDUwLDAwMC1jbGFpbQpzdWJzZXQgZm9yIDMgZXBvY2hzIGF0IG1heF9sZW5ndGggMjU2LCBub3QgMjI4LDA5MCB4IDEwIGF0IDUxMi4gUmVjb3JkZWQgaW4KdGhlIG1hbmlmZXN0IGFuZCByZXBvcnRlZCBhcyBhIGxpbWl0YXRpb24uCgogICAgcHl0aG9uIHNjcmlwdHMvbTZfMDZfdHJhaW5fdmVyaWZpZXIucHkgLS1kYXRhc2V0IGZldmVyCiAgICBweXRob24gc2NyaXB0cy9tNl8wNl90cmFpbl92ZXJpZmllci5weSAtLWRhdGFzZXQgeF9mYWN0IC0taW5pdC1mcm9tIG1vZGVscy9tb2R1bGU2L2ZldmVyCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMV0pKQoKZnJvbSBzY3JpcHRzLm02X2NvbW1vbiBpbXBvcnQgKCAgIyBub3FhOiBFNDAyCiAgICBQUk9KRUNUX1JPT1QsIGFzc2VydF9tb2R1bGU1X3VudG91Y2hlZCwgYmFubmVyLCBkZXRlY3RfZGV2aWNlLCBsb2FkX2NvbmZpZywKICAgIGxvYWRfY29ycHVzLCBtb2R1bGU1X2ZpbmdlcnByaW50LCBzdGFnZV9kaXIsIHdyaXRlX21hbmlmZXN0KQpmcm9tIHNyYy52ZXJpZmljYXRpb24uZGF0YXNldCBpbXBvcnQgKCAgIyBub3FhOiBFNDAyCiAgICBidWlsZF9leGFtcGxlcywgZ2V0X2xhYmVsX3NwYWNlLCBzdHJhdGlmaWVkX3N1YnNldCkKZnJvbSBzcmMudmVyaWZpY2F0aW9uLnRyYWluZXIgaW1wb3J0ICggICMgbm9xYTogRTQwMgogICAgQ2hlY2twb2ludE1hbmFnZXIsIFZlcmlmaWVyVHJhaW5lciwgY2xhc3NpZmljYXRpb25fcmVwb3J0LCBzZXRfc2VlZCkKZnJvbSBzcmMudmVyaWZpY2F0aW9uLnZlcmlmaWVyX21vZGVsIGltcG9ydCBWZXJpZmllckNvbmZpZywgYnVpbGRfdmVyaWZpZXIgICMgbm9xYTogRTQwMgoKCmRlZiBsb2FkX3JldHJpZXZlZChjZmcsIHJvb3QsIGRhdGFzZXQsIHNwbGl0KToKICAgICIiIlJlcmFua2VkIHRvcC01IGlmIHByZXNlbnQsIGVsc2UgZGVuc2UgY2FuZGlkYXRlcywgZWxzZSBub3RoaW5nLiIiIgogICAgcnIgPSBzdGFnZV9kaXIoY2ZnLCAicmVyYW5rIiwgcm9vdCkgLyBmIntkYXRhc2V0fV97c3BsaXR9IiAvICJyZXJhbmtlZF90b3A1Lmpzb24iCiAgICBpZiByci5pc19maWxlKCk6CiAgICAgICAgcmV0dXJuIGpzb24ubG9hZHMocnIucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKSwgInJlcmFua2VkX3RvcDUiCiAgICBkbiA9IHN0YWdlX2RpcihjZmcsICJyZXRyaWV2ZSIsIHJvb3QpIC8gZiJ7ZGF0YXNldH1fe3NwbGl0fSIgLyAiZGVuc2VfY2FuZGlkYXRlcy5qc29uIgogICAgaWYgZG4uaXNfZmlsZSgpOgogICAgICAgIHJldHVybiBqc29uLmxvYWRzKGRuLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSksICJkZW5zZV9jYW5kaWRhdGVzIgogICAgcmV0dXJuIHt9LCAibm9uZSIKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJUcmFpbiB0aGUgZmFjdCB2ZXJpZmllci4iKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWRhdGFzZXQiLCByZXF1aXJlZD1UcnVlLCBjaG9pY2VzPVsiZmV2ZXIiLCAieF9mYWN0Il0pCiAgICBhcC5hZGRfYXJndW1lbnQoIi0taW5pdC1mcm9tIiwgZGVmYXVsdD1Ob25lLAogICAgICAgICAgICAgICAgICAgIGhlbHA9IkNoZWNrcG9pbnQgZGlyIHdob3NlIEVOQ09ERVIgaXMgcmV1c2VkLiBUaGUgaGVhZCBpcyBhbHdheXMgIgogICAgICAgICAgICAgICAgICAgICAgICAgInJlLWluaXRpYWxpc2VkIGZvciB0aGUgbmV3IGxhYmVsIHNwYWNlLiIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tY2xhaW0tb25seSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsIGhlbHA9IkFibGF0aW9uOiBubyBldmlkZW5jZS4iKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW9yYWNsZSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsIGhlbHA9IkFibGF0aW9uOiBnb2xkIGV2aWRlbmNlLiIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZXBvY2hzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zZWVkIiwgdHlwZT1pbnQsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1jb25maWciLCBkZWZhdWx0PU5vbmUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcHJvamVjdC1yb290IiwgZGVmYXVsdD1Ob25lKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWRyeS1ydW4iLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQoKICAgIHJvb3QgPSBQYXRoKGFyZ3MucHJvamVjdF9yb290KSBpZiBhcmdzLnByb2plY3Rfcm9vdCBlbHNlIFBST0pFQ1RfUk9PVAogICAgY2ZnID0gbG9hZF9jb25maWcoYXJncy5jb25maWcpCiAgICB2ID0gY2ZnWyJ2ZXJpZmllciJdCiAgICBzZWVkID0gaW50KGFyZ3Muc2VlZCBvciBjZmdbInJlcHJvZHVjaWJpbGl0eSJdWyJzZWVkIl0pCiAgICBlcG9jaHMgPSBpbnQoYXJncy5lcG9jaHMgb3IgdlsiZXBvY2hzIl0pCiAgICBiYW5uZXIoZiJTdGFnZSA2IHRyYWluIHZlcmlmaWVyIFt7YXJncy5kYXRhc2V0fV0iLCBjZmcpCiAgICBiZWZvcmUgPSBtb2R1bGU1X2ZpbmdlcnByaW50KHJvb3QpCiAgICBzZXRfc2VlZChzZWVkKQoKICAgIHNwYWNlID0gZ2V0X2xhYmVsX3NwYWNlKGFyZ3MuZGF0YXNldCkKICAgIHByaW50KGYiICBsYWJlbCBzcGFjZSA6IHtzcGFjZS5uYW1lfSAoe3NwYWNlLnRhc2t9KSAtPiB7c3BhY2UubGFiZWxzfSIpCiAgICBwcmludChmIiAgTk9URTogRkVWRVIgYW5kIFgtRkFDVCBsYWJlbCBzcGFjZXMgYXJlIG5ldmVyIG1hcHBlZCBvbnRvIGVhY2ggb3RoZXIuIikKCiAgICBjb3JwdXMgPSBsb2FkX2NvcnB1cyhjZmcsIGFyZ3MuZGF0YXNldCwgcm9vdD1yb290KQogICAgcmV0cmlldmVkLCBzb3VyY2UgPSBsb2FkX3JldHJpZXZlZChjZmcsIHJvb3QsIGFyZ3MuZGF0YXNldCwgInRyYWluIikKICAgIHByaW50KGYiICBldmlkZW5jZSBzb3VyY2U6IHtzb3VyY2V9IikKCiAgICB0cmFpbl9leCwgc3RhdHMgPSBidWlsZF9leGFtcGxlcygKICAgICAgICBjb3JwdXNbImNsYWltcyJdLCBjb3JwdXNbImV2aWRlbmNlX3RleHRzIl0sIHJldHJpZXZlZCwgYXJncy5kYXRhc2V0LAogICAgICAgIGNvcnB1c1sic3BsaXRfb2YiXSwgbWF4X2V2aWRlbmNlPWludCh2WyJldmlkZW5jZV9wZXJfY2xhaW0iXSksCiAgICAgICAgdXNlX2dvbGRfZXZpZGVuY2U9Ym9vbChhcmdzLm9yYWNsZSksIGNsYWltX29ubHk9Ym9vbChhcmdzLmNsYWltX29ubHkpKQogICAgdHJhaW5fZXggPSBbZSBmb3IgZSBpbiB0cmFpbl9leCBpZiBlLnNwbGl0ID09ICJ0cmFpbiJdCiAgICB2YWxfZXggPSBbZSBmb3IgZSBpbiBidWlsZF9leGFtcGxlcygKICAgICAgICBjb3JwdXNbImNsYWltcyJdLCBjb3JwdXNbImV2aWRlbmNlX3RleHRzIl0sCiAgICAgICAgbG9hZF9yZXRyaWV2ZWQoY2ZnLCByb290LCBhcmdzLmRhdGFzZXQsICJ2YWxpZGF0aW9uIilbMF0sIGFyZ3MuZGF0YXNldCwKICAgICAgICBjb3JwdXNbInNwbGl0X29mIl0sIG1heF9ldmlkZW5jZT1pbnQodlsiZXZpZGVuY2VfcGVyX2NsYWltIl0pLAogICAgICAgIHVzZV9nb2xkX2V2aWRlbmNlPWJvb2woYXJncy5vcmFjbGUpLCBjbGFpbV9vbmx5PWJvb2woYXJncy5jbGFpbV9vbmx5KSlbMF0KICAgICAgICBpZiBlLnNwbGl0ID09ICJ2YWxpZGF0aW9uIl0KCiAgICBzdWJzZXRfc2l6ZSA9ICh2LmdldChhcmdzLmRhdGFzZXQpIG9yIHt9KS5nZXQoInRyYWluX3N1YnNldF9zaXplIikKICAgIG5fYmVmb3JlID0gbGVuKHRyYWluX2V4KQogICAgdHJhaW5fZXggPSBzdHJhdGlmaWVkX3N1YnNldCh0cmFpbl9leCwgc3Vic2V0X3NpemUsIHNlZWQ9c2VlZCkKICAgIHByaW50KGYiICB0cmFpbiBjbGFpbXM6IHtuX2JlZm9yZTosfSAtPiB7bGVuKHRyYWluX2V4KTosfSIKICAgICAgICAgICsgKGYiICAoc3RyYXRpZmllZCBzdWJzZXQsIGZyZWUtQ29sYWIgc2NvcGUpIiBpZiBzdWJzZXRfc2l6ZSBlbHNlICIiKSkKICAgIHByaW50KGYiICB2YWwgY2xhaW1zICA6IHtsZW4odmFsX2V4KTosfSIpCiAgICBwcmludChmIiAgYWNjb3VudGluZyBiYWxhbmNlczoge3N0YXRzWydhY2NvdW50aW5nX2JhbGFuY2VzJ119IikKCiAgICBvdXQgPSBzdGFnZV9kaXIoY2ZnLCAidmVyaWZpZXJfdHJhaW5pbmciLCByb290KSAvIGFyZ3MuZGF0YXNldAogICAgY2twdF9kaXIgPSByb290IC8gY2ZnWyJvdXRwdXQiXVsibW9kZWxzX2RpciJdIC8gYXJncy5kYXRhc2V0CiAgICBtYW5pZmVzdCA9IHsic3RhZ2UiOiAidmVyaWZpZXJfdHJhaW5pbmciLCAiZGF0YXNldCI6IGFyZ3MuZGF0YXNldCwKICAgICAgICAgICAgICAgICJsYWJlbF9zcGFjZSI6IHNwYWNlLmxhYmVscywgInRhc2siOiBzcGFjZS50YXNrLAogICAgICAgICAgICAgICAgIm5fdHJhaW4iOiBsZW4odHJhaW5fZXgpLCAibl90cmFpbl9iZWZvcmVfc3Vic2V0Ijogbl9iZWZvcmUsCiAgICAgICAgICAgICAgICAibl92YWwiOiBsZW4odmFsX2V4KSwgImVwb2NocyI6IGVwb2NocywgInNlZWQiOiBzZWVkLAogICAgICAgICAgICAgICAgImV2aWRlbmNlX3NvdXJjZSI6IHNvdXJjZSwgImNsYWltX29ubHkiOiBib29sKGFyZ3MuY2xhaW1fb25seSksCiAgICAgICAgICAgICAgICAib3JhY2xlX2dvbGRfZXZpZGVuY2UiOiBib29sKGFyZ3Mub3JhY2xlKSwKICAgICAgICAgICAgICAgICJleGFtcGxlX3N0YXRzIjogc3RhdHMsICJpbml0X2Zyb20iOiBhcmdzLmluaXRfZnJvbSwKICAgICAgICAgICAgICAgICJjb21wdXRlX3Njb3BlIjogY2ZnWyJjb21wdXRlX3Njb3BlIl1bInJlZHVjdGlvbnMiXSwKICAgICAgICAgICAgICAgICJkZXZpY2UiOiBkZXRlY3RfZGV2aWNlKCl9CgogICAgaWYgYXJncy5kcnlfcnVuOgogICAgICAgIG1hbmlmZXN0WyJzdGF0dXMiXSA9ICJkcnlfcnVuIgogICAgICAgIHdyaXRlX21hbmlmZXN0KG91dCwgbWFuaWZlc3QpCiAgICAgICAgcHJpbnQoIiAgRFJZIFJVTjogbm8gbW9kZWwgYnVpbHQsIG5vIHRyYWluaW5nIHBlcmZvcm1lZC4iKQogICAgICAgIGFzc2VydF9tb2R1bGU1X3VudG91Y2hlZChiZWZvcmUsIHJvb3QpCiAgICAgICAgcmV0dXJuCgogICAgaW1wb3J0IHRvcmNoCiAgICBmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFMb2FkZXIKICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvVG9rZW5pemVyLCBnZXRfbGluZWFyX3NjaGVkdWxlX3dpdGhfd2FybXVwCgogICAgdmNmZyA9IFZlcmlmaWVyQ29uZmlnKGVuY29kZXJfbmFtZT12WyJlbmNvZGVyIl0sIG51bV9sYWJlbHM9bGVuKHNwYWNlKSwKICAgICAgICAgICAgICAgICAgICAgICAgICBoaWRkZW5fZGltPWludCh2WyJoaWRkZW5fZGltIl0pLCBkcm9wb3V0PWZsb2F0KHZbImRyb3BvdXQiXSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgbWF4X2xlbmd0aD1pbnQodlsibWF4X2xlbmd0aCJdKSwgbGFiZWxfc3BhY2U9c3BhY2UubmFtZSkKICAgIG1vZGVsID0gYnVpbGRfdmVyaWZpZXIodmNmZykKCiAgICBpZiBhcmdzLmluaXRfZnJvbToKICAgICAgICBjbSA9IENoZWNrcG9pbnRNYW5hZ2VyKFBhdGgoYXJncy5pbml0X2Zyb20pKQogICAgICAgIHN0YXRlID0gY20ubG9hZChtb2RlbCkKICAgICAgICBzd2FwID0gbW9kZWwuc3dhcF9oZWFkKGxlbihzcGFjZSksIHNwYWNlLm5hbWUsIGludCh2WyJoaWRkZW5fZGltIl0pKQogICAgICAgIG1hbmlmZXN0WyJoZWFkX3N3YXAiXSA9IHN3YXAKICAgICAgICBwcmludChmIiAgbG9hZGVkIGVuY29kZXIgZnJvbSB7YXJncy5pbml0X2Zyb219OyBoZWFkIFJFLUlOSVRJQUxJU0VEICIKICAgICAgICAgICAgICBmIih7c3dhcFsnZnJvbSddfSAtPiB7c3dhcFsndG8nXX0pIikKCiAgICBkZXYgPSBkZXRlY3RfZGV2aWNlKCkKICAgIG1vZGVsLnRvKGRldikKICAgIHRvayA9IEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKHZbImVuY29kZXIiXSkKCiAgICBkZWYgY29sbGF0ZShiYXRjaCk6CiAgICAgICAgcGFpcnMgPSBbZS50b190ZXh0X3BhaXIobWF4X2V2aWRlbmNlPWludCh2WyJldmlkZW5jZV9wZXJfY2xhaW0iXSkpIGZvciBlIGluIGJhdGNoXQogICAgICAgIGVuYyA9IHRvayhbcFswXSBmb3IgcCBpbiBwYWlyc10sIFtwWzFdIGZvciBwIGluIHBhaXJzXSwgcGFkZGluZz1UcnVlLAogICAgICAgICAgICAgICAgICB0cnVuY2F0aW9uPVRydWUsIG1heF9sZW5ndGg9aW50KHZbIm1heF9sZW5ndGgiXSksIHJldHVybl90ZW5zb3JzPSJwdCIpCiAgICAgICAgZW5jWyJsYWJlbHMiXSA9IHRvcmNoLnRlbnNvcihbZS5sYWJlbF9pZCBmb3IgZSBpbiBiYXRjaF0pCiAgICAgICAgcmV0dXJuIGVuYwoKICAgIGcgPSB0b3JjaC5HZW5lcmF0b3IoKS5tYW51YWxfc2VlZChzZWVkKQogICAgdHJhaW5fZGwgPSBEYXRhTG9hZGVyKHRyYWluX2V4LCBiYXRjaF9zaXplPWludCh2WyJiYXRjaF9zaXplIl0pLCBzaHVmZmxlPVRydWUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgY29sbGF0ZV9mbj1jb2xsYXRlLCBnZW5lcmF0b3I9ZykKICAgIHZhbF9kbCA9IERhdGFMb2FkZXIodmFsX2V4LCBiYXRjaF9zaXplPWludCh2WyJiYXRjaF9zaXplIl0pLCBjb2xsYXRlX2ZuPWNvbGxhdGUpCgogICAgb3B0ID0gdG9yY2gub3B0aW0uQWRhbVcobW9kZWwucGFyYW1ldGVycygpLCBscj1mbG9hdCh2WyJsZWFybmluZ19yYXRlIl0pLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgd2VpZ2h0X2RlY2F5PWZsb2F0KHZbIndlaWdodF9kZWNheSJdKSkKICAgIHRvdGFsID0gbWF4KDEsIGxlbih0cmFpbl9kbCkgKiBlcG9jaHMpCiAgICBzY2hlZCA9IGdldF9saW5lYXJfc2NoZWR1bGVfd2l0aF93YXJtdXAoCiAgICAgICAgb3B0LCBpbnQodG90YWwgKiBmbG9hdCh2WyJ3YXJtdXBfcmF0aW8iXSkpLCB0b3RhbCkKCiAgICBkZWYgc3RlcF9mbihtLCBiYXRjaCk6CiAgICAgICAgbS50cmFpbigpCiAgICAgICAgYmF0Y2ggPSB7azogdC50byhkZXYpIGZvciBrLCB0IGluIGJhdGNoLml0ZW1zKCl9CiAgICAgICAgbGFiZWxzID0gYmF0Y2gucG9wKCJsYWJlbHMiKQogICAgICAgIGxvc3MgPSB0b3JjaC5ubi5mdW5jdGlvbmFsLmNyb3NzX2VudHJvcHkoCiAgICAgICAgICAgIG0oYmF0Y2hbImlucHV0X2lkcyJdLCBiYXRjaFsiYXR0ZW50aW9uX21hc2siXSksIGxhYmVscykKICAgICAgICBsb3NzLmJhY2t3YXJkKCkKICAgICAgICB0b3JjaC5ubi51dGlscy5jbGlwX2dyYWRfbm9ybV8obS5wYXJhbWV0ZXJzKCksIDEuMCkKICAgICAgICBvcHQuc3RlcCgpOyBzY2hlZC5zdGVwKCk7IG9wdC56ZXJvX2dyYWQoKQogICAgICAgIHJldHVybiBmbG9hdChsb3NzLml0ZW0oKSkKCiAgICBAdG9yY2gubm9fZ3JhZCgpCiAgICBkZWYgZXZhbF9mbihtKToKICAgICAgICBtLmV2YWwoKQogICAgICAgIHl0LCB5cCA9IFtdLCBbXQogICAgICAgIGZvciBiYXRjaCBpbiB2YWxfZGw6CiAgICAgICAgICAgIGJhdGNoID0ge2s6IHQudG8oZGV2KSBmb3IgaywgdCBpbiBiYXRjaC5pdGVtcygpfQogICAgICAgICAgICBsYWJlbHMgPSBiYXRjaC5wb3AoImxhYmVscyIpCiAgICAgICAgICAgIHByZWRzID0gbShiYXRjaFsiaW5wdXRfaWRzIl0sIGJhdGNoWyJhdHRlbnRpb25fbWFzayJdKS5hcmdtYXgoLTEpCiAgICAgICAgICAgIHl0LmV4dGVuZChsYWJlbHMuY3B1KCkudG9saXN0KCkpOyB5cC5leHRlbmQocHJlZHMuY3B1KCkudG9saXN0KCkpCiAgICAgICAgcmV0dXJuIGNsYXNzaWZpY2F0aW9uX3JlcG9ydCh5dCwgeXAsIHNwYWNlLmxhYmVscykKCiAgICB0cmFpbmVyID0gVmVyaWZpZXJUcmFpbmVyKGNrcHRfZGlyLCB7Kip2Y2ZnLnRvX2RpY3QoKSwgInNlZWQiOiBzZWVkLCAiZXBvY2hzIjogZXBvY2hzfSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY2hlY2twb2ludF9ldmVyeV9zdGVwcz1pbnQodlsiY2hlY2twb2ludF9ldmVyeV9zdGVwcyJdKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZWFybHlfc3RvcHBpbmdfcGF0aWVuY2U9aW50KHZbImVhcmx5X3N0b3BwaW5nX3BhdGllbmNlIl0pKQogICAgcmVzdWx0ID0gdHJhaW5lci50cmFpbihtb2RlbCwgb3B0LCBzY2hlZCwgbGFtYmRhIF9lOiB0cmFpbl9kbCwgc3RlcF9mbiwgZXZhbF9mbiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgZXBvY2hzPWVwb2NocywgcmVzdW1lPVRydWUpCiAgICBtYW5pZmVzdC51cGRhdGUoeyJzdGF0dXMiOiAiY29tcGxldGUiLCAidHJhaW5pbmciOiByZXN1bHQsCiAgICAgICAgICAgICAgICAgICAgICJjaGVja3BvaW50X2RpciI6IHN0cihja3B0X2Rpcil9KQogICAgd3JpdGVfbWFuaWZlc3Qob3V0LCBtYW5pZmVzdCkKICAgIGFzc2VydF9tb2R1bGU1X3VudG91Y2hlZChiZWZvcmUsIHJvb3QpCiAgICBwcmludChmIiAgYmVzdCB2YWwgbWFjcm9fZjE6IHtyZXN1bHRbJ2Jlc3RfbWV0cmljJ106LjRmfSIpCiAgICBwcmludChmIiAgY2hlY2twb2ludHMgLT4ge2NrcHRfZGlyfSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=", "sha256": "dd56104c3c3e9f7a5b4ae85c48e68a7ccef6343b736029e23a3a180da6420f44", "size": 9119}, "scripts/m6_07_evaluate_verifier.py": {"b64": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uCiIiIk1vZHVsZSA2IFN0YWdlIDcg4oCUIGV2YWx1YXRlICsgY2FsaWJyYXRlIHRoZSB2ZXJpZmllci4gR1BVIGZvciBpbmZlcmVuY2UuCgpUZW1wZXJhdHVyZSBpcyBGSVRURUQgT04gVkFMSURBVElPTiBhbmQgdGhlbiBhcHBsaWVkIHVuY2hhbmdlZCB0byB0ZXN0LgpGaXR0aW5nIG9uIHRlc3Qgd291bGQgYmUgdHVuaW5nIG9uIHRlc3QsIHdoaWNoIHRoZSBwcm9qZWN0J3MgcHJvdG9jb2wgZm9yYmlkcwooInRlc3QgZXZhbHVhdGVkIG9uY2UiKTsgYGZpdF90ZW1wZXJhdHVyZSgpYCByZWZ1c2VzIGEgc3BsaXQgbmFtZWQgInRlc3QiLgoKRkVWRVIgYW5kIFgtRkFDVCBhcmUgZXZhbHVhdGVkIGFuZCByZXBvcnRlZCBTRVBBUkFURUxZLCBhZ2FpbnN0IHRoZWlyIG93bgpsYWJlbCBzcGFjZXMuIFRoZXJlIGlzIG5vIGNvbWJpbmVkIHNjb3JlLCBiZWNhdXNlIHRoZXJlIGlzIG5vIHNoYXJlZCB0YXNrLgoKICAgIHB5dGhvbiBzY3JpcHRzL202XzA3X2V2YWx1YXRlX3ZlcmlmaWVyLnB5IC0tZGF0YXNldCBmZXZlciAtLXNwbGl0IHRlc3QKIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCgpzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzFdKSkKCmZyb20gc2NyaXB0cy5tNl9jb21tb24gaW1wb3J0ICggICMgbm9xYTogRTQwMgogICAgUFJPSkVDVF9ST09ULCBhc3NlcnRfbW9kdWxlNV91bnRvdWNoZWQsIGJhbm5lciwgZGV0ZWN0X2RldmljZSwgbG9hZF9jb25maWcsCiAgICBsb2FkX2NvcnB1cywgbW9kdWxlNV9maW5nZXJwcmludCwgc3RhZ2VfZGlyLCB3cml0ZV9qc29uLCB3cml0ZV9tYW5pZmVzdCkKZnJvbSBzY3JpcHRzLm02XzA2X3RyYWluX3ZlcmlmaWVyIGltcG9ydCBsb2FkX3JldHJpZXZlZCAgIyBub3FhOiBFNDAyCmZyb20gc3JjLnZlcmlmaWNhdGlvbi5jYWxpYnJhdGlvbiBpbXBvcnQgKCAgIyBub3FhOiBFNDAyCiAgICBjYWxpYnJhdGVfYW5kX2V2YWx1YXRlLCBleHBlY3RlZF9jYWxpYnJhdGlvbl9lcnJvciwgZml0X3RlbXBlcmF0dXJlLCBzb2Z0bWF4KQpmcm9tIHNyYy52ZXJpZmljYXRpb24uZGF0YXNldCBpbXBvcnQgYnVpbGRfZXhhbXBsZXMsIGdldF9sYWJlbF9zcGFjZSAgIyBub3FhOiBFNDAyCmZyb20gc3JjLnZlcmlmaWNhdGlvbi50cmFpbmVyIGltcG9ydCBDaGVja3BvaW50TWFuYWdlciwgY2xhc3NpZmljYXRpb25fcmVwb3J0ICAjIG5vcWE6IEU0MDIKZnJvbSBzcmMudmVyaWZpY2F0aW9uLnZlcmlmaWVyX21vZGVsIGltcG9ydCBWZXJpZmllckNvbmZpZywgYnVpbGRfdmVyaWZpZXIgICMgbm9xYTogRTQwMgoKCmRlZiBtYWluKCkgLT4gTm9uZToKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249IkV2YWx1YXRlIGFuZCBjYWxpYnJhdGUgdGhlIHZlcmlmaWVyLiIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZGF0YXNldCIsIHJlcXVpcmVkPVRydWUsIGNob2ljZXM9WyJmZXZlciIsICJ4X2ZhY3QiXSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zcGxpdCIsIGRlZmF1bHQ9InRlc3QiLCBjaG9pY2VzPVsidmFsaWRhdGlvbiIsICJ0ZXN0Il0pCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tY2hlY2twb2ludCIsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1jbGFpbS1vbmx5IiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1vcmFjbGUiLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWNvbmZpZyIsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1wcm9qZWN0LXJvb3QiLCBkZWZhdWx0PU5vbmUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZHJ5LXJ1biIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCgogICAgcm9vdCA9IFBhdGgoYXJncy5wcm9qZWN0X3Jvb3QpIGlmIGFyZ3MucHJvamVjdF9yb290IGVsc2UgUFJPSkVDVF9ST09UCiAgICBjZmcgPSBsb2FkX2NvbmZpZyhhcmdzLmNvbmZpZykKICAgIHYgPSBjZmdbInZlcmlmaWVyIl0KICAgIGJhbm5lcihmIlN0YWdlIDcgZXZhbHVhdGUgW3thcmdzLmRhdGFzZXR9L3thcmdzLnNwbGl0fV0iLCBjZmcpCiAgICBiZWZvcmUgPSBtb2R1bGU1X2ZpbmdlcnByaW50KHJvb3QpCgogICAgc3BhY2UgPSBnZXRfbGFiZWxfc3BhY2UoYXJncy5kYXRhc2V0KQogICAgY29ycHVzID0gbG9hZF9jb3JwdXMoY2ZnLCBhcmdzLmRhdGFzZXQsIHJvb3Q9cm9vdCkKICAgIG91dCA9IHN0YWdlX2RpcihjZmcsICJldmFsdWF0aW9uIiwgcm9vdCkgLyBmInthcmdzLmRhdGFzZXR9X3thcmdzLnNwbGl0fSIKICAgIG91dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgaWYgYXJncy5kcnlfcnVuOgogICAgICAgIHdyaXRlX21hbmlmZXN0KG91dCwgeyJzdGFnZSI6ICJldmFsdWF0aW9uIiwgImRhdGFzZXQiOiBhcmdzLmRhdGFzZXQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInNwbGl0IjogYXJncy5zcGxpdCwgInN0YXR1cyI6ICJkcnlfcnVuIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAibGFiZWxfc3BhY2UiOiBzcGFjZS5sYWJlbHN9KQogICAgICAgIHByaW50KCIgIERSWSBSVU46IG5vIG1vZGVsIGxvYWRlZC4iKQogICAgICAgIGFzc2VydF9tb2R1bGU1X3VudG91Y2hlZChiZWZvcmUsIHJvb3QpCiAgICAgICAgcmV0dXJuCgogICAgaW1wb3J0IHRvcmNoCiAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b1Rva2VuaXplcgoKICAgIGNrcHRfZGlyID0gUGF0aChhcmdzLmNoZWNrcG9pbnQgb3IgKHJvb3QgLyBjZmdbIm91dHB1dCJdWyJtb2RlbHNfZGlyIl0gLyBhcmdzLmRhdGFzZXQpKQogICAgdmNmZyA9IFZlcmlmaWVyQ29uZmlnKGVuY29kZXJfbmFtZT12WyJlbmNvZGVyIl0sIG51bV9sYWJlbHM9bGVuKHNwYWNlKSwKICAgICAgICAgICAgICAgICAgICAgICAgICBoaWRkZW5fZGltPWludCh2WyJoaWRkZW5fZGltIl0pLCBkcm9wb3V0PWZsb2F0KHZbImRyb3BvdXQiXSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgbWF4X2xlbmd0aD1pbnQodlsibWF4X2xlbmd0aCJdKSwgbGFiZWxfc3BhY2U9c3BhY2UubmFtZSkKICAgIG1vZGVsID0gYnVpbGRfdmVyaWZpZXIodmNmZykKICAgIGNtID0gQ2hlY2twb2ludE1hbmFnZXIoY2twdF9kaXIpCiAgICBpZiBjbS5sb2FkKG1vZGVsLCBwYXRoPWNtLmJlc3QgaWYgY20uYmVzdC5pc19maWxlKCkgZWxzZSBOb25lKSBpcyBOb25lOgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoZiJubyBjaGVja3BvaW50IGluIHtja3B0X2Rpcn07IHJ1biBtNl8wNiBmaXJzdCIpCiAgICBkZXYgPSBkZXRlY3RfZGV2aWNlKCkKICAgIG1vZGVsLnRvKGRldikuZXZhbCgpCiAgICB0b2sgPSBBdXRvVG9rZW5pemVyLmZyb21fcHJldHJhaW5lZCh2WyJlbmNvZGVyIl0pCgogICAgQHRvcmNoLm5vX2dyYWQoKQogICAgZGVmIGxvZ2l0c19mb3Ioc3BsaXQ6IHN0cik6CiAgICAgICAgZXggPSBbZSBmb3IgZSBpbiBidWlsZF9leGFtcGxlcygKICAgICAgICAgICAgY29ycHVzWyJjbGFpbXMiXSwgY29ycHVzWyJldmlkZW5jZV90ZXh0cyJdLAogICAgICAgICAgICBsb2FkX3JldHJpZXZlZChjZmcsIHJvb3QsIGFyZ3MuZGF0YXNldCwgc3BsaXQpWzBdLCBhcmdzLmRhdGFzZXQsCiAgICAgICAgICAgIGNvcnB1c1sic3BsaXRfb2YiXSwgbWF4X2V2aWRlbmNlPWludCh2WyJldmlkZW5jZV9wZXJfY2xhaW0iXSksCiAgICAgICAgICAgIHVzZV9nb2xkX2V2aWRlbmNlPWJvb2woYXJncy5vcmFjbGUpLCBjbGFpbV9vbmx5PWJvb2woYXJncy5jbGFpbV9vbmx5KSlbMF0KICAgICAgICAgICAgaWYgZS5zcGxpdCA9PSBzcGxpdF0KICAgICAgICBhbGxfbG9naXRzLCBsYWJlbHMsIGxhbmdzID0gW10sIFtdLCBbXQogICAgICAgIGJzID0gaW50KHZbImJhdGNoX3NpemUiXSkKICAgICAgICBmb3IgaSBpbiByYW5nZSgwLCBsZW4oZXgpLCBicyk6CiAgICAgICAgICAgIGNodW5rID0gZXhbaTppICsgYnNdCiAgICAgICAgICAgIHBhaXJzID0gW2UudG9fdGV4dF9wYWlyKG1heF9ldmlkZW5jZT1pbnQodlsiZXZpZGVuY2VfcGVyX2NsYWltIl0pKSBmb3IgZSBpbiBjaHVua10KICAgICAgICAgICAgZW5jID0gdG9rKFtwWzBdIGZvciBwIGluIHBhaXJzXSwgW3BbMV0gZm9yIHAgaW4gcGFpcnNdLCBwYWRkaW5nPVRydWUsCiAgICAgICAgICAgICAgICAgICAgICB0cnVuY2F0aW9uPVRydWUsIG1heF9sZW5ndGg9aW50KHZbIm1heF9sZW5ndGgiXSksCiAgICAgICAgICAgICAgICAgICAgICByZXR1cm5fdGVuc29ycz0icHQiKS50byhkZXYpCiAgICAgICAgICAgIGFsbF9sb2dpdHMuYXBwZW5kKG1vZGVsKGVuY1siaW5wdXRfaWRzIl0sIGVuY1siYXR0ZW50aW9uX21hc2siXSkuZmxvYXQoKS5jcHUoKS5udW1weSgpKQogICAgICAgICAgICBsYWJlbHMuZXh0ZW5kKGUubGFiZWxfaWQgZm9yIGUgaW4gY2h1bmspCiAgICAgICAgICAgIGxhbmdzLmV4dGVuZChlLmxhbmd1YWdlIGZvciBlIGluIGNodW5rKQogICAgICAgIHJldHVybiAobnAuY29uY2F0ZW5hdGUoYWxsX2xvZ2l0cykgaWYgYWxsX2xvZ2l0cyBlbHNlIG5wLnplcm9zKCgwLCBsZW4oc3BhY2UpKSksCiAgICAgICAgICAgICAgICBsYWJlbHMsIGxhbmdzKQoKICAgIHZhbF9sb2dpdHMsIHZhbF9sYWJlbHMsIF8gPSBsb2dpdHNfZm9yKCJ2YWxpZGF0aW9uIikKICAgIHRlbXAgPSBmaXRfdGVtcGVyYXR1cmUodmFsX2xvZ2l0cywgdmFsX2xhYmVscywgc3BsaXRfbmFtZT0idmFsaWRhdGlvbiIpCiAgICBwcmludChmIiAgdGVtcGVyYXR1cmUgZml0dGVkIG9uIFZBTElEQVRJT046IHt0ZW1wWyd0ZW1wZXJhdHVyZSddOi40Zn0gIgogICAgICAgICAgZiIobmxsIHt0ZW1wWydubGxfYmVmb3JlJ106LjRmfSAtPiB7dGVtcFsnbmxsX2FmdGVyJ106LjRmfSkiKQoKICAgIGxvZ2l0cywgbGFiZWxzLCBsYW5ncyA9IGxvZ2l0c19mb3IoYXJncy5zcGxpdCkKICAgIHByZWRzID0gc29mdG1heChsb2dpdHMsIHRlbXBbInRlbXBlcmF0dXJlIl0pLmFyZ21heCgxKS50b2xpc3QoKQogICAgcmVwb3J0ID0gY2xhc3NpZmljYXRpb25fcmVwb3J0KGxhYmVscywgcHJlZHMsIHNwYWNlLmxhYmVscykKICAgIGNhbGliID0gY2FsaWJyYXRlX2FuZF9ldmFsdWF0ZShsb2dpdHMsIGxhYmVscywgdGVtcFsidGVtcGVyYXR1cmUiXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBuX2JpbnM9aW50KGNmZ1siY2FsaWJyYXRpb24iXVsiZWNlX2JpbnMiXSkpCgogICAgcGVyX2xhbmcgPSB7fQogICAgZm9yIGxhbmcgaW4gc29ydGVkKHNldChsYW5ncykpOgogICAgICAgIHNlbCA9IFtpIGZvciBpLCB4IGluIGVudW1lcmF0ZShsYW5ncykgaWYgeCA9PSBsYW5nXQogICAgICAgIGlmIHNlbDoKICAgICAgICAgICAgcGVyX2xhbmdbbGFuZ10gPSBjbGFzc2lmaWNhdGlvbl9yZXBvcnQoW2xhYmVsc1tpXSBmb3IgaSBpbiBzZWxdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBbcHJlZHNbaV0gZm9yIGkgaW4gc2VsXSwgc3BhY2UubGFiZWxzKQoKICAgIHBheWxvYWQgPSB7ImRhdGFzZXQiOiBhcmdzLmRhdGFzZXQsICJzcGxpdCI6IGFyZ3Muc3BsaXQsICJ0YXNrIjogc3BhY2UudGFzaywKICAgICAgICAgICAgICAgImxhYmVsX3NwYWNlIjogc3BhY2UubGFiZWxzLCAibiI6IGxlbihsYWJlbHMpLAogICAgICAgICAgICAgICAibWV0cmljcyI6IHJlcG9ydCwgImNhbGlicmF0aW9uIjogY2FsaWIsICJ0ZW1wZXJhdHVyZSI6IHRlbXAsCiAgICAgICAgICAgICAgICJwZXJfbGFuZ3VhZ2UiOiBwZXJfbGFuZywKICAgICAgICAgICAgICAgImNsYWltX29ubHkiOiBib29sKGFyZ3MuY2xhaW1fb25seSksICJvcmFjbGUiOiBib29sKGFyZ3Mub3JhY2xlKSwKICAgICAgICAgICAgICAgIm5vdGUiOiAoIkZFVkVSIChlbnRhaWxtZW50KSBhbmQgWC1GQUNUICh2ZXJhY2l0eSkgYXJlIHNlcGFyYXRlIHRhc2tzICIKICAgICAgICAgICAgICAgICAgICAgICAgIndpdGggc2VwYXJhdGUgbGFiZWwgc3BhY2VzOyByZXN1bHRzIGFyZSBuZXZlciBjb21iaW5lZC4iKX0KICAgIHdyaXRlX2pzb24ob3V0IC8gInZlcmlmaWNhdGlvbl9tZXRyaWNzLmpzb24iLCBwYXlsb2FkKQogICAgd3JpdGVfbWFuaWZlc3Qob3V0LCB7InN0YWdlIjogImV2YWx1YXRpb24iLCAiZGF0YXNldCI6IGFyZ3MuZGF0YXNldCwKICAgICAgICAgICAgICAgICAgICAgICAgICJzcGxpdCI6IGFyZ3Muc3BsaXQsICJzdGF0dXMiOiAiY29tcGxldGUiLAogICAgICAgICAgICAgICAgICAgICAgICAgIm1hY3JvX2YxIjogcmVwb3J0WyJtYWNyb19mMSJdLCAiYWNjdXJhY3kiOiByZXBvcnRbImFjY3VyYWN5Il0sCiAgICAgICAgICAgICAgICAgICAgICAgICAiZWNlX2JlZm9yZSI6IGNhbGliWyJlY2VfYmVmb3JlIl0sICJlY2VfYWZ0ZXIiOiBjYWxpYlsiZWNlX2FmdGVyIl19KQoKICAgIGFzc2VydF9tb2R1bGU1X3VudG91Y2hlZChiZWZvcmUsIHJvb3QpCiAgICBwcmludChmIiAgYWNjdXJhY3kge3JlcG9ydFsnYWNjdXJhY3knXTouNGZ9ICBtYWNyby1GMSB7cmVwb3J0WydtYWNyb19mMSddOi40Zn0iKQogICAgcHJpbnQoZiIgIEVDRSB7Y2FsaWJbJ2VjZV9iZWZvcmUnXTouNGZ9IC0+IHtjYWxpYlsnZWNlX2FmdGVyJ106LjRmfSBhZnRlciBzY2FsaW5nIikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==", "sha256": "e021d32fea2a0e6d77d975065bb03bdae050e7269542b376982f5b301b571b96", "size": 6808}, "scripts/m6_08_run_ablations.py": {"b64": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uCiIiIk1vZHVsZSA2IFN0YWdlIDgg4oCUIHJ1biB0aGUgc2l4IGNvbmZpZ3VyZWQgYWJsYXRpb25zLiBNaXhlZCBDUFUvR1BVLgoKUmV0cmlldmFsIGFibGF0aW9ucyAoYm0yNV92c19kZW5zZSwgdG9wX2sgc3dlZXApIHJlYWQgdGhlIGFscmVhZHktY29tcHV0ZWQKU3RhZ2UgNCBvdXRwdXRzIGFuZCBhcmUgcHVyZSBDUFUuIFZlcmlmaWNhdGlvbiBhYmxhdGlvbnMgbmVlZCBhIHRyYWluZWQKY2hlY2twb2ludCBwZXIgdmFyaWFudCBhbmQgYXJlIHRoZXJlZm9yZSByZXBvcnRlZCBhcyBwZW5kaW5nIHVudGlsIHRob3NlIHJ1bnMKZXhpc3QgLS0gY2xhaW1pbmcgYSBudW1iZXIgd2l0aG91dCBydW5uaW5nIHRoZSB2YXJpYW50IHdvdWxkIGJlIGZhYnJpY2F0aW9uLgoKICAgIHB5dGhvbiBzY3JpcHRzL202XzA4X3J1bl9hYmxhdGlvbnMucHkgLS1kYXRhc2V0IGZldmVyIC0tc3BsaXQgdGVzdAoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBzeXMKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzFdKSkKCmZyb20gc2NyaXB0cy5tNl9jb21tb24gaW1wb3J0ICggICMgbm9xYTogRTQwMgogICAgUFJPSkVDVF9ST09ULCBhc3NlcnRfbW9kdWxlNV91bnRvdWNoZWQsIGJhbm5lciwgZ29sZF9ldmlkZW5jZV9tYXAsIGxvYWRfY29uZmlnLAogICAgbG9hZF9jb3JwdXMsIG1vZHVsZTVfZmluZ2VycHJpbnQsIHN0YWdlX2Rpciwgd3JpdGVfanNvbiwgd3JpdGVfbWFuaWZlc3QpCmZyb20gc3JjLnJldHJpZXZhbC5yZXRyaWV2YWxfbWV0cmljcyBpbXBvcnQgZXZhbHVhdGVfcmV0cmlldmFsICAjIG5vcWE6IEU0MDIKZnJvbSBzcmMudmVyaWZpY2F0aW9uLmFibGF0aW9ucyBpbXBvcnQgYnVpbGRfYWJsYXRpb25fc3BlY3MsIHJ1bl9hbGxfYWJsYXRpb25zICAjIG5vcWE6IEU0MDIKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJSdW4gTW9kdWxlIDYgYWJsYXRpb25zLiIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZGF0YXNldCIsIHJlcXVpcmVkPVRydWUsIGNob2ljZXM9WyJmZXZlciIsICJ4X2ZhY3QiXSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zcGxpdCIsIGRlZmF1bHQ9InRlc3QiLCBjaG9pY2VzPVsidmFsaWRhdGlvbiIsICJ0ZXN0Il0pCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tY29uZmlnIiwgZGVmYXVsdD1Ob25lKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXByb2plY3Qtcm9vdCIsIGRlZmF1bHQ9Tm9uZSkKICAgIGFyZ3MgPSBhcC5wYXJzZV9hcmdzKCkKCiAgICByb290ID0gUGF0aChhcmdzLnByb2plY3Rfcm9vdCkgaWYgYXJncy5wcm9qZWN0X3Jvb3QgZWxzZSBQUk9KRUNUX1JPT1QKICAgIGNmZyA9IGxvYWRfY29uZmlnKGFyZ3MuY29uZmlnKQogICAgYmFubmVyKGYiU3RhZ2UgOCBhYmxhdGlvbnMgW3thcmdzLmRhdGFzZXR9L3thcmdzLnNwbGl0fV0iLCBjZmcpCiAgICBiZWZvcmUgPSBtb2R1bGU1X2ZpbmdlcnByaW50KHJvb3QpCgogICAgY29ycHVzID0gbG9hZF9jb3JwdXMoY2ZnLCBhcmdzLmRhdGFzZXQsIHJvb3Q9cm9vdCkKICAgIGdvbGQgPSBnb2xkX2V2aWRlbmNlX21hcChjb3JwdXMpCiAgICByZXRfZGlyID0gc3RhZ2VfZGlyKGNmZywgInJldHJpZXZlIiwgcm9vdCkgLyBmInthcmdzLmRhdGFzZXR9X3thcmdzLnNwbGl0fSIKICAgIHJyX2RpciA9IHN0YWdlX2RpcihjZmcsICJyZXJhbmsiLCByb290KSAvIGYie2FyZ3MuZGF0YXNldH1fe2FyZ3Muc3BsaXR9IgoKICAgIGRlZiBfbG9hZChwOiBQYXRoKToKICAgICAgICByZXR1cm4ganNvbi5sb2FkcyhwLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkgaWYgcC5pc19maWxlKCkgZWxzZSBOb25lCgogICAgZGVuc2UgPSBfbG9hZChyZXRfZGlyIC8gImRlbnNlX2NhbmRpZGF0ZXMuanNvbiIpCiAgICBibTI1ID0gX2xvYWQocmV0X2RpciAvICJibTI1X2NhbmRpZGF0ZXMuanNvbiIpCiAgICByZXJhbmtlZCA9IF9sb2FkKHJyX2RpciAvICJyZXJhbmtlZF90b3A1Lmpzb24iKQoKICAgIGRlZiBzY29yZShjYW5kcywgayk6CiAgICAgICAgaWYgbm90IGNhbmRzOgogICAgICAgICAgICByZXR1cm4geyJlcnJvciI6ICJjYW5kaWRhdGVzIG5vdCBhdmFpbGFibGU7IHJ1biB0aGUgZWFybGllciBzdGFnZSJ9CiAgICAgICAgbSA9IGV2YWx1YXRlX3JldHJpZXZhbChjYW5kcywgZ29sZCwgKGssKSwgbmRjZ19rPW1pbihrLCAxMCkpCiAgICAgICAgcmV0dXJuIHtmInJlY2FsbEB7a30iOiBtWyJvdmVyYWxsIl1bZiJyZWNhbGxAe2t9Il0sICJtcnIiOiBtWyJvdmVyYWxsIl1bIm1yciJdLAogICAgICAgICAgICAgICAgIm5fc2NvcmVkIjogbVsiYWNjb3VudGluZyJdWyJuX3Njb3JlZCJdLAogICAgICAgICAgICAgICAgIm5fZXhjbHVkZWRfbm9fZ29sZCI6IG1bImFjY291bnRpbmciXVsibl9leGNsdWRlZF9ub19nb2xkX2V2aWRlbmNlIl19CgogICAgcnVubmVycyA9IHsKICAgICAgICAiYm0yNV92c19kZW5zZSI6IGxhbWJkYSB2OiAoCiAgICAgICAgICAgIHsqKnNjb3JlKGJtMjUgaWYgdlsicmV0cmlldmVyIl0gPT0gImJtMjUiIGVsc2UgZGVuc2UsIDUpLAogICAgICAgICAgICAgIm1hY3JvX2YxIjogTm9uZSwgInJlY2FsbEA1Ijogc2NvcmUoCiAgICAgICAgICAgICAgICAgYm0yNSBpZiB2WyJyZXRyaWV2ZXIiXSA9PSAiYm0yNSIgZWxzZSBkZW5zZSwgNSkuZ2V0KCJyZWNhbGxANSIpfSksCiAgICAgICAgInRvcF9rX3N3ZWVwIjogbGFtYmRhIHY6IHsicmVjYWxsQGsiOiBzY29yZShkZW5zZSwgaW50KHZbInRvcF9rIl0pKS5nZXQoCiAgICAgICAgICAgIGYicmVjYWxsQHtpbnQodlsndG9wX2snXSl9IiksICoqc2NvcmUoZGVuc2UsIGludCh2WyJ0b3BfayJdKSl9LAogICAgICAgICJyZXRyaWV2YWxfb25seV92c19yZXRyaWV2YWxfcmVyYW5rIjogbGFtYmRhIHY6IHNjb3JlKAogICAgICAgICAgICByZXJhbmtlZCBpZiB2WyJyZXJhbmsiXSBlbHNlIGRlbnNlLCA1KSwKICAgIH0KCiAgICBvdXQgPSBydW5fYWxsX2FibGF0aW9ucyhjZmcsIHJ1bm5lcnMsIHNlZWQ9aW50KGNmZ1sicmVwcm9kdWNpYmlsaXR5Il1bInNlZWQiXSkpCiAgICBmb3Igc3BlYyBpbiBidWlsZF9hYmxhdGlvbl9zcGVjcyhjZmcpOgogICAgICAgIGlmIHNwZWMubmFtZSBub3QgaW4gcnVubmVyczoKICAgICAgICAgICAgb3V0LnNldGRlZmF1bHQoInBlbmRpbmciLCBbXSkuYXBwZW5kKHsKICAgICAgICAgICAgICAgICJuYW1lIjogc3BlYy5uYW1lLCAicXVlc3Rpb24iOiBzcGVjLnF1ZXN0aW9uLAogICAgICAgICAgICAgICAgInJlYXNvbiI6ICJyZXF1aXJlcyBhIHRyYWluZWQgdmVyaWZpZXIgY2hlY2twb2ludCBwZXIgdmFyaWFudDsgIgogICAgICAgICAgICAgICAgICAgICAgICAgICJydW4gbTZfMDYvbTZfMDcgd2l0aCB0aGUgY29ycmVzcG9uZGluZyBmbGFncyJ9KQoKICAgIGQgPSBzdGFnZV9kaXIoY2ZnLCAiYWJsYXRpb25zIiwgcm9vdCkgLyBmInthcmdzLmRhdGFzZXR9X3thcmdzLnNwbGl0fSIKICAgIGQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgd3JpdGVfanNvbihkIC8gImFibGF0aW9uX3Jlc3VsdHMuanNvbiIsIG91dCkKICAgIHdyaXRlX21hbmlmZXN0KGQsIHsic3RhZ2UiOiAiYWJsYXRpb25zIiwgImRhdGFzZXQiOiBhcmdzLmRhdGFzZXQsICJzcGxpdCI6IGFyZ3Muc3BsaXQsCiAgICAgICAgICAgICAgICAgICAgICAgInN0YXR1cyI6ICJjb21wbGV0ZSIsICJuX3J1biI6IGxlbihvdXRbImFibGF0aW9ucyJdKSwKICAgICAgICAgICAgICAgICAgICAgICAibl9wZW5kaW5nIjogbGVuKG91dC5nZXQoInBlbmRpbmciLCBbXSkpLAogICAgICAgICAgICAgICAgICAgICAgICJuX3NraXBwZWQiOiBsZW4ob3V0LmdldCgic2tpcHBlZCIsIFtdKSl9KQoKICAgIGFzc2VydF9tb2R1bGU1X3VudG91Y2hlZChiZWZvcmUsIHJvb3QpCiAgICBmb3IgbmFtZSwgcmVzIGluIG91dFsiYWJsYXRpb25zIl0uaXRlbXMoKToKICAgICAgICBwcmludChmIiAge25hbWV9OiBiZXN0PXtyZXNbJ2Jlc3RfdmFyaWFudCddfSB2YWx1ZT17cmVzWydiZXN0X3ZhbHVlJ119ICIKICAgICAgICAgICAgICBmImRlbHRhPXtyZXNbJ2RlbHRhJ119IikKICAgIGZvciBwIGluIG91dC5nZXQoInBlbmRpbmciLCBbXSk6CiAgICAgICAgcHJpbnQoZiIgIFBFTkRJTkcge3BbJ25hbWUnXX06IHtwWydyZWFzb24nXX0iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK", "sha256": "21345aa3fceadde7959302817af09911dab6a74a99472c13adc16d67df95526c", "size": 4326}, "scripts/verify_module6_outputs.py": {"b64": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uCiIiIk1vZHVsZSA2IHZlcmlmaWVyIOKAlCBQQVNTL0ZBSUwgY2hlY2tzIG92ZXIgd2hhdGV2ZXIgTW9kdWxlIDYgaGFzIHByb2R1Y2VkLgoKRm9sbG93cyB0aGUgTW9kdWxlIDMvNC81IGNvbnZlbnRpb246IGluZGVwZW5kZW50IHJlLXZlcmlmaWNhdGlvbiB0aGF0CnJlY29tcHV0ZXMgZnJvbSB0aGUgYXJ0ZWZhY3RzIHJhdGhlciB0aGFuIHRydXN0aW5nIGVhY2ggc3RhZ2UncyBvd24gbWFuaWZlc3QsCmJlY2F1c2UgYSBzdGFnZSBidWcgYW5kIHRoYXQgc3RhZ2UncyBzZWxmLXJlcG9ydCB3b3VsZCBmYWlsIHRvZ2V0aGVyLgoKU3RhZ2VzIHRoYXQgaGF2ZSBub3QgcnVuIHlldCBhcmUgcmVwb3J0ZWQgU0tJUFBFRCwgbm90IFBBU1MuIEEgY2hlY2sgdGhhdApjYW5ub3QgZmFpbCBpcyB3b3JzZSB0aGFuIG5vIGNoZWNrLgoKICAgIHB5dGhvbiBzY3JpcHRzL3ZlcmlmeV9tb2R1bGU2X291dHB1dHMucHkgLS1kYXRhc2V0IGZldmVyIC0tc3BsaXQgdGVzdAoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgcmUKaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAoKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1sxXSkpCgpmcm9tIHNjcmlwdHMubTZfY29tbW9uIGltcG9ydCAoICAjIG5vcWE6IEU0MDIKICAgIFBST0pFQ1RfUk9PVCwgZ29sZF9ldmlkZW5jZV9tYXAsIGxvYWRfY29uZmlnLCBsb2FkX2NvcnB1cywgbW9kdWxlNV9maW5nZXJwcmludCwKICAgIHJlYWRfbWFuaWZlc3QsIHN0YWdlX2Rpciwgd3JpdGVfanNvbikKZnJvbSBzcmMucmV0cmlldmFsLmZhaXNzX2luZGV4IGltcG9ydCBFdmlkZW5jZUluZGV4LCBicnV0ZV9mb3JjZV9zZWFyY2ggICMgbm9xYTogRTQwMgpmcm9tIHNyYy5yZXRyaWV2YWwuaW5kZXhfc2NvcGUgaW1wb3J0ICggICMgbm9xYTogRTQwMgogICAgYnVpbGRfZXZpZGVuY2Vfc2NvcGUsIHZlcmlmeV9waGFzZV9pc29sYXRpb24pCmZyb20gc3JjLnJldHJpZXZhbC5yZXRyaWV2YWxfbWV0cmljcyBpbXBvcnQgZXZhbHVhdGVfcmV0cmlldmFsICAjIG5vcWE6IEU0MDIKZnJvbSBzcmMudmVyaWZpY2F0aW9uLmRhdGFzZXQgaW1wb3J0ICggICMgbm9xYTogRTQwMgogICAgRkVWRVJfTEFCRUxTLCBYX0ZBQ1RfTEFCRUxTLCBhc3NlcnRfbGFiZWxfc3BhY2VzX2Rpc2pvaW50LCBnZXRfbGFiZWxfc3BhY2UpCgpFVklERU5DRV9JRF9SRSA9IHJlLmNvbXBpbGUociJeW2Etel9dKzpldjpbMC05YS1mXXs2NH0kIikKCgpjbGFzcyBDaGVja2VyOgogICAgZGVmIF9faW5pdF9fKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5jaGVja3M6IGxpc3RbZGljdF0gPSBbXQoKICAgIGRlZiBhZGQoc2VsZiwgbmFtZTogc3RyLCBwYXNzZWQ6IGJvb2wgfCBOb25lLCBkZXRhaWw6IHN0ciA9ICIiKSAtPiBOb25lOgogICAgICAgIHN0YXR1cyA9ICJTS0lQIiBpZiBwYXNzZWQgaXMgTm9uZSBlbHNlICgiUEFTUyIgaWYgcGFzc2VkIGVsc2UgIkZBSUwiKQogICAgICAgIHNlbGYuY2hlY2tzLmFwcGVuZCh7Im5hbWUiOiBuYW1lLCAic3RhdHVzIjogc3RhdHVzLCAiZGV0YWlsIjogZGV0YWlsfSkKICAgICAgICBwcmludChmIiAge3N0YXR1czo0fSAge25hbWV9IiArIChmIiAgfCAge2RldGFpbH0iIGlmIGRldGFpbCBlbHNlICIiKSkKCiAgICBkZWYgc3VtbWFyeShzZWxmKSAtPiBkaWN0OgogICAgICAgIHAgPSBzdW0oMSBmb3IgYyBpbiBzZWxmLmNoZWNrcyBpZiBjWyJzdGF0dXMiXSA9PSAiUEFTUyIpCiAgICAgICAgZiA9IHN1bSgxIGZvciBjIGluIHNlbGYuY2hlY2tzIGlmIGNbInN0YXR1cyJdID09ICJGQUlMIikKICAgICAgICBzID0gc3VtKDEgZm9yIGMgaW4gc2VsZi5jaGVja3MgaWYgY1sic3RhdHVzIl0gPT0gIlNLSVAiKQogICAgICAgIHJldHVybiB7Im5fY2hlY2tzIjogbGVuKHNlbGYuY2hlY2tzKSwgInBhc3NlZCI6IHAsICJmYWlsZWQiOiBmLCAic2tpcHBlZCI6IHMsCiAgICAgICAgICAgICAgICAib3ZlcmFsbCI6ICJGQUlMIiBpZiBmIGVsc2UgKCJQQVNTIiBpZiBwIGVsc2UgIk5PVEhJTkdfVE9fQ0hFQ0siKSwKICAgICAgICAgICAgICAgICJjaGVja3MiOiBzZWxmLmNoZWNrc30KCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJWZXJpZnkgTW9kdWxlIDYgb3V0cHV0cy4iKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWRhdGFzZXQiLCByZXF1aXJlZD1UcnVlLCBjaG9pY2VzPVsiZmV2ZXIiLCAieF9mYWN0Il0pCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc3BsaXQiLCBkZWZhdWx0PSJ0ZXN0IiwgY2hvaWNlcz1bInZhbGlkYXRpb24iLCAidGVzdCJdKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWNvbmZpZyIsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1wcm9qZWN0LXJvb3QiLCBkZWZhdWx0PU5vbmUpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCgogICAgcm9vdCA9IFBhdGgoYXJncy5wcm9qZWN0X3Jvb3QpIGlmIGFyZ3MucHJvamVjdF9yb290IGVsc2UgUFJPSkVDVF9ST09UCiAgICBjZmcgPSBsb2FkX2NvbmZpZyhhcmdzLmNvbmZpZykKICAgIGNrID0gQ2hlY2tlcigpCiAgICBwcmludCgiPSIgKiA3MCkKICAgIHByaW50KGYiTU9EVUxFIDYgVkVSSUZJQ0FUSU9OIOKAlCB7YXJncy5kYXRhc2V0fS97YXJncy5zcGxpdH0iKQogICAgcHJpbnQoIj0iICogNzApCgogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIDEuIE1vZHVsZSA1IGludGFjdAogICAgY29ycHVzID0gbG9hZF9jb3JwdXMoY2ZnLCBhcmdzLmRhdGFzZXQsIHJvb3Q9cm9vdCkKICAgIGNrLmFkZCgibW9kdWxlNV9jb3JwdXNfcmVhZGFibGUiLAogICAgICAgICAgIGJvb2woY29ycHVzWyJjbGFpbXMiXSkgYW5kIGJvb2woY29ycHVzWyJldmlkZW5jZSJdKSwKICAgICAgICAgICBmIntsZW4oY29ycHVzWydjbGFpbXMnXSk6LH0gY2xhaW1zLCB7bGVuKGNvcnB1c1snZXZpZGVuY2UnXSk6LH0gZXZpZGVuY2UiKQoKICAgIGlkcyA9IFtlWyJldmlkZW5jZV9pZCJdIGZvciBlIGluIGNvcnB1c1siZXZpZGVuY2UiXV0KICAgIGNrLmFkZCgibW9kdWxlNV9ldmlkZW5jZV9pZHNfdW5pcXVlIiwgbGVuKHNldChpZHMpKSA9PSBsZW4oaWRzKSwKICAgICAgICAgICBmIntsZW4oaWRzKSAtIGxlbihzZXQoaWRzKSl9IGR1cGxpY2F0ZShzKSIpCiAgICBiYWQgPSBbaSBmb3IgaSBpbiBpZHNbOjUwMDBdIGlmIG5vdCBFVklERU5DRV9JRF9SRS5tYXRjaChpKV0KICAgIGNrLmFkZCgibW9kdWxlNV9ldmlkZW5jZV9pZHNfYXJlX3NoYTI1NiIsIG5vdCBiYWQsCiAgICAgICAgICAgZiJ7bGVuKGJhZCl9IG1hbGZvcm1lZCIgKyAoZiIsIGUuZy4ge2JhZFswXVs6NjBdfSIgaWYgYmFkIGVsc2UgIiIpKQoKICAgIGZsYXQgPSBbaSBmb3IgdiBpbiBjb3JwdXNbInNwbGl0cyJdLnZhbHVlcygpIGZvciBpIGluIHZdCiAgICBjay5hZGQoIm1vZHVsZTVfc3BsaXRzX2NvdmVyX2V2ZXJ5X2NsYWltX29uY2UiLAogICAgICAgICAgIHNvcnRlZChmbGF0KSA9PSBsaXN0KHJhbmdlKGxlbihjb3JwdXNbImNsYWltcyJdKSkpLAogICAgICAgICAgIGYie2xlbihmbGF0KX0gYXNzaWdubWVudHMgZm9yIHtsZW4oY29ycHVzWydjbGFpbXMnXSl9IGNsYWltcyIpCgogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gMi4gbGFiZWwgc3BhY2VzCiAgICB0cnk6CiAgICAgICAgYXNzZXJ0X2xhYmVsX3NwYWNlc19kaXNqb2ludCgpCiAgICAgICAgc3BhY2UgPSBnZXRfbGFiZWxfc3BhY2UoYXJncy5kYXRhc2V0KQogICAgICAgIGNrLmFkZCgibGFiZWxfc3BhY2VzX2Rpc2pvaW50X2FuZF91bm1hcHBlZCIsIFRydWUsCiAgICAgICAgICAgICAgIGYie3NwYWNlLm5hbWV9OiB7c3BhY2UubGFiZWxzfSIpCiAgICBleGNlcHQgQXNzZXJ0aW9uRXJyb3IgYXMgZToKICAgICAgICBjay5hZGQoImxhYmVsX3NwYWNlc19kaXNqb2ludF9hbmRfdW5tYXBwZWQiLCBGYWxzZSwgc3RyKGUpKQogICAgY2suYWRkKCJjb25maWdfZm9yYmlkc19sYWJlbF9tYXBwaW5nIiwKICAgICAgICAgICBjZmdbImxhYmVsX3NwYWNlcyJdWyJtYXBwaW5nX2JldHdlZW5fc3BhY2VzIl0gaXMgRmFsc2UpCiAgICBjay5hZGQoImZldmVyX2xhYmVsc19leGFjdCIsIGNmZ1sibGFiZWxfc3BhY2VzIl1bImZldmVyIl1bImxhYmVscyJdID09IEZFVkVSX0xBQkVMUykKICAgIGNrLmFkZCgieGZhY3RfbGFiZWxzX2V4YWN0X2FuZF9wcmVzZXJ2ZWQiLAogICAgICAgICAgIGNmZ1sibGFiZWxfc3BhY2VzIl1bInhfZmFjdCJdWyJsYWJlbHMiXSA9PSBYX0ZBQ1RfTEFCRUxTLAogICAgICAgICAgICIncGFydGx5IHRydWUvbWlzbGVhZGluZycga2VwdCB2ZXJiYXRpbSIpCgogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIDMuIGxlYWthZ2Ugc2NvcGluZwogICAgaXNvID0gdmVyaWZ5X3BoYXNlX2lzb2xhdGlvbihjb3JwdXNbImNsYWltcyJdLCBjb3JwdXNbInNwbGl0cyJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjZmdbImxlYWthZ2UiXVsiaW5kZXhfc2NvcGUiXSkKICAgIGNrLmFkZCgicGhhc2VfaXNvbGF0aW9uX2hvbGRzIiwgaXNvWyJwYXNzZWQiXSwKICAgICAgICAgICBmIntpc29bJ25fcGFzc2VkJ119L3tpc29bJ25fY2hlY2tzJ119IHNjb3BlIGNoZWNrcyIpCgogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gNC4gdGhlIGluZGV4CiAgICBpZHhfZGlyID0gc3RhZ2VfZGlyKGNmZywgImJ1aWxkX2luZGV4Iiwgcm9vdCkgLyBmInthcmdzLmRhdGFzZXR9X3thcmdzLnNwbGl0fSIKICAgIGlmIChpZHhfZGlyIC8gImluZGV4X21ldGEuanNvbiIpLmlzX2ZpbGUoKToKICAgICAgICBpbmRleCA9IEV2aWRlbmNlSW5kZXgubG9hZChpZHhfZGlyKQogICAgICAgIHNjb3BlID0gYnVpbGRfZXZpZGVuY2Vfc2NvcGUoY29ycHVzWyJjbGFpbXMiXSwgY29ycHVzWyJzcGxpdHMiXSwgYXJncy5zcGxpdCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNmZ1sibGVha2FnZSJdWyJpbmRleF9zY29wZSJdKQogICAgICAgIGV4dHJhID0gc2V0KGluZGV4LmV2aWRlbmNlX2lkcykgLSBzY29wZS5ldmlkZW5jZV9pZHMKICAgICAgICBjay5hZGQoImluZGV4X3dpdGhpbl9wZXJtaXR0ZWRfc2NvcGUiLCBub3QgZXh0cmEsCiAgICAgICAgICAgICAgIGYie2xlbihleHRyYSl9IG91dC1vZi1zY29wZSBpdGVtKHMpIgogICAgICAgICAgICAgICArIChmIiwgZS5nLiB7c29ydGVkKGV4dHJhKVs6M119IiBpZiBleHRyYSBlbHNlICIiKSkKICAgICAgICBjay5hZGQoImluZGV4X2lkc191bmlxdWUiLAogICAgICAgICAgICAgICBsZW4oc2V0KGluZGV4LmV2aWRlbmNlX2lkcykpID09IGxlbihpbmRleC5ldmlkZW5jZV9pZHMpKQogICAgICAgIG5vcm1zID0gbnAubGluYWxnLm5vcm0oaW5kZXgudmVjdG9yc1s6IG1pbig1MDAsIGxlbihpbmRleCkpXSwgYXhpcz0xKQogICAgICAgIGNrLmFkZCgiaW5kZXhfdmVjdG9yc19sMl9ub3JtYWxpc2VkIiwKICAgICAgICAgICAgICAgYm9vbChucC5hbGxjbG9zZShub3JtcywgMS4wLCBhdG9sPTFlLTMpKSwKICAgICAgICAgICAgICAgZiJtZWFuIG5vcm0ge2Zsb2F0KG5vcm1zLm1lYW4oKSk6LjZmfSIpCiAgICAgICAgbiA9IG1pbig2NCwgbGVuKGluZGV4KSkKICAgICAgICBpZiBuID49IDI6CiAgICAgICAgICAgIHEgPSBpbmRleC52ZWN0b3JzWzo1XQogICAgICAgICAgICBnb3QsIF8gPSBpbmRleC5zZWFyY2gocSwgbWluKDUsIG4pKQogICAgICAgICAgICBleHAsIF8gPSBicnV0ZV9mb3JjZV9zZWFyY2gocSwgaW5kZXgudmVjdG9ycywgaW5kZXguZXZpZGVuY2VfaWRzLCBtaW4oNSwgbikpCiAgICAgICAgICAgIGNrLmFkZCgiaW5kZXhfbWF0Y2hlc19icnV0ZV9mb3JjZV9jb3NpbmUiLCBnb3QgPT0gZXhwLAogICAgICAgICAgICAgICAgICAgImV4YWN0IHNlYXJjaCB2ZXJpZmllZCBvbiBhIHNhbXBsZSIpCiAgICBlbHNlOgogICAgICAgIGZvciBuYW1lIGluICgiaW5kZXhfd2l0aGluX3Blcm1pdHRlZF9zY29wZSIsICJpbmRleF9pZHNfdW5pcXVlIiwKICAgICAgICAgICAgICAgICAgICAgImluZGV4X3ZlY3RvcnNfbDJfbm9ybWFsaXNlZCIsICJpbmRleF9tYXRjaGVzX2JydXRlX2ZvcmNlX2Nvc2luZSIpOgogICAgICAgICAgICBjay5hZGQobmFtZSwgTm9uZSwgImluZGV4IG5vdCBidWlsdCB5ZXQgKHJ1biBtNl8wMikiKQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gNS4gcmV0cmlldmFsIHJ1bnMKICAgIHJldF9kaXIgPSBzdGFnZV9kaXIoY2ZnLCAicmV0cmlldmUiLCByb290KSAvIGYie2FyZ3MuZGF0YXNldH1fe2FyZ3Muc3BsaXR9IgogICAgZ29sZCA9IGdvbGRfZXZpZGVuY2VfbWFwKGNvcnB1cykKICAgIGNhbmRfcCA9IHJldF9kaXIgLyAiZGVuc2VfY2FuZGlkYXRlcy5qc29uIgogICAgaWYgY2FuZF9wLmlzX2ZpbGUoKToKICAgICAgICBjYW5kcyA9IGpzb24ubG9hZHMoY2FuZF9wLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICBzY29wZSA9IGJ1aWxkX2V2aWRlbmNlX3Njb3BlKGNvcnB1c1siY2xhaW1zIl0sIGNvcnB1c1sic3BsaXRzIl0sIGFyZ3Muc3BsaXQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjZmdbImxlYWthZ2UiXVsiaW5kZXhfc2NvcGUiXSkKICAgICAgICBvZmZlbmRlcnMgPSB7YzogW2UgZm9yIGUgaW4gdiBpZiBlIG5vdCBpbiBzY29wZS5ldmlkZW5jZV9pZHNdCiAgICAgICAgICAgICAgICAgICAgIGZvciBjLCB2IGluIGNhbmRzLml0ZW1zKCl9CiAgICAgICAgb2ZmZW5kZXJzID0ge2M6IHYgZm9yIGMsIHYgaW4gb2ZmZW5kZXJzLml0ZW1zKCkgaWYgdn0KICAgICAgICBjay5hZGQoInJldHJpZXZlZF9ldmlkZW5jZV93aXRoaW5fc2NvcGUiLCBub3Qgb2ZmZW5kZXJzLAogICAgICAgICAgICAgICBmIntsZW4ob2ZmZW5kZXJzKX0gY2xhaW0ocykgcmV0cmlldmVkIG91dC1vZi1zY29wZSBldmlkZW5jZSIpCgogICAgICAgIG0gPSBldmFsdWF0ZV9yZXRyaWV2YWwoY2FuZHMsIGdvbGQsICgxLCA1KSwgbmRjZ19rPTUpCiAgICAgICAgYWNjID0gbVsiYWNjb3VudGluZyJdCiAgICAgICAgY2suYWRkKCJyZXRyaWV2YWxfYWNjb3VudGluZ19iYWxhbmNlcyIsIGFjY1siYmFsYW5jZXMiXSwKICAgICAgICAgICAgICAgZiJ7YWNjWyduX3F1ZXJpZXMnXX0gPSB7YWNjWyduX3Njb3JlZCddfSBzY29yZWQgKyAiCiAgICAgICAgICAgICAgIGYie2FjY1snbl9leGNsdWRlZF9ub19nb2xkX2V2aWRlbmNlJ119IG5vLWdvbGQgKyAiCiAgICAgICAgICAgICAgIGYie2FjY1snbl9taXNzaW5nX2Zyb21fZ29sZF9tYXAnXX0gbWlzc2luZyIpCiAgICAgICAgY2suYWRkKCJub19nb2xkX2NsYWltc19leGNsdWRlZF9hbmRfY291bnRlZCIsCiAgICAgICAgICAgICAgIGFjY1sibl9leGNsdWRlZF9ub19nb2xkX2V2aWRlbmNlIl0gPj0gMCwKICAgICAgICAgICAgICAgZiJ7YWNjWyduX2V4Y2x1ZGVkX25vX2dvbGRfZXZpZGVuY2UnXX0gY2xhaW0ocykgd2l0aG91dCBnb2xkIGV2aWRlbmNlIikKICAgICAgICBjay5hZGQoInJlY2FsbF9tb25vdG9uZV9pbl9rIiwKICAgICAgICAgICAgICAgbVsib3ZlcmFsbCJdWyJyZWNhbGxAMSJdIDw9IG1bIm92ZXJhbGwiXVsicmVjYWxsQDUiXSArIDFlLTEyLAogICAgICAgICAgICAgICBmIlJAMSB7bVsnb3ZlcmFsbCddWydyZWNhbGxAMSddOi40Zn0gPD0gUkA1IHttWydvdmVyYWxsJ11bJ3JlY2FsbEA1J106LjRmfSIpCiAgICBlbHNlOgogICAgICAgIGZvciBuYW1lIGluICgicmV0cmlldmVkX2V2aWRlbmNlX3dpdGhpbl9zY29wZSIsICJyZXRyaWV2YWxfYWNjb3VudGluZ19iYWxhbmNlcyIsCiAgICAgICAgICAgICAgICAgICAgICJub19nb2xkX2NsYWltc19leGNsdWRlZF9hbmRfY291bnRlZCIsICJyZWNhbGxfbW9ub3RvbmVfaW5fayIpOgogICAgICAgICAgICBjay5hZGQobmFtZSwgTm9uZSwgInJldHJpZXZhbCBub3QgcnVuIHlldCAocnVuIG02XzA0KSIpCgogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIDYuIHJlcmFua2luZwogICAgcnJfcCA9IHN0YWdlX2RpcihjZmcsICJyZXJhbmsiLCByb290KSAvIGYie2FyZ3MuZGF0YXNldH1fe2FyZ3Muc3BsaXR9IiAvICJyZXJhbmtlZF90b3A1Lmpzb24iCiAgICBpZiBycl9wLmlzX2ZpbGUoKSBhbmQgY2FuZF9wLmlzX2ZpbGUoKToKICAgICAgICByciA9IGpzb24ubG9hZHMocnJfcC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgY2FuZHMgPSBqc29uLmxvYWRzKGNhbmRfcC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgaW52ZW50ZWQgPSB7Yzogc29ydGVkKHNldCh2KSAtIHNldChjYW5kcy5nZXQoYywgW10pKSkgZm9yIGMsIHYgaW4gcnIuaXRlbXMoKX0KICAgICAgICBpbnZlbnRlZCA9IHtjOiB2IGZvciBjLCB2IGluIGludmVudGVkLml0ZW1zKCkgaWYgdn0KICAgICAgICBjay5hZGQoInJlcmFua2VyX3ByZXNlcnZlZF9jYW5kaWRhdGVfc2V0Iiwgbm90IGludmVudGVkLAogICAgICAgICAgICAgICBmIntsZW4oaW52ZW50ZWQpfSBjbGFpbShzKSBnYWluZWQgZXZpZGVuY2Ugbm90IGluIHRoZWlyIGNhbmRpZGF0ZSBzZXQiKQogICAgICAgIHRvb19tYW55ID0ge2M6IGxlbih2KSBmb3IgYywgdiBpbiByci5pdGVtcygpCiAgICAgICAgICAgICAgICAgICAgaWYgbGVuKHYpID4gaW50KGNmZ1sicmVyYW5raW5nIl1bInRvcF9uX2FmdGVyX3JlcmFuayJdKX0KICAgICAgICBjay5hZGQoInJlcmFua2VyX3Jlc3BlY3RzX3RvcF9uIiwgbm90IHRvb19tYW55LCBmIntsZW4odG9vX21hbnkpfSBvdmVyLWxvbmcgbGlzdChzKSIpCiAgICBlbHNlOgogICAgICAgIGNrLmFkZCgicmVyYW5rZXJfcHJlc2VydmVkX2NhbmRpZGF0ZV9zZXQiLCBOb25lLCAicmVyYW5raW5nIG5vdCBydW4geWV0IChydW4gbTZfMDUpIikKICAgICAgICBjay5hZGQoInJlcmFua2VyX3Jlc3BlY3RzX3RvcF9uIiwgTm9uZSwgInJlcmFua2luZyBub3QgcnVuIHlldCIpCgogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSA3LiB2ZXJpZmljYXRpb24KICAgIGV2X3AgPSAoc3RhZ2VfZGlyKGNmZywgImV2YWx1YXRpb24iLCByb290KSAvIGYie2FyZ3MuZGF0YXNldH1fe2FyZ3Muc3BsaXR9IgogICAgICAgICAgICAvICJ2ZXJpZmljYXRpb25fbWV0cmljcy5qc29uIikKICAgIGlmIGV2X3AuaXNfZmlsZSgpOgogICAgICAgIHJlcyA9IGpzb24ubG9hZHMoZXZfcC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgY2suYWRkKCJ2ZXJpZmljYXRpb25fdXNlc19jb3JyZWN0X2xhYmVsX3NwYWNlIiwKICAgICAgICAgICAgICAgcmVzLmdldCgibGFiZWxfc3BhY2UiKSA9PSBnZXRfbGFiZWxfc3BhY2UoYXJncy5kYXRhc2V0KS5sYWJlbHMsCiAgICAgICAgICAgICAgIHN0cihyZXMuZ2V0KCJsYWJlbF9zcGFjZSIpKSkKICAgICAgICBjYWwgPSByZXMuZ2V0KCJjYWxpYnJhdGlvbiIsIHt9KQogICAgICAgIGNrLmFkZCgiZWNlX3JlcG9ydGVkIiwgImVjZV9iZWZvcmUiIGluIGNhbCBhbmQgImVjZV9hZnRlciIgaW4gY2FsLAogICAgICAgICAgICAgICBmIkVDRSB7Y2FsLmdldCgnZWNlX2JlZm9yZScpfSAtPiB7Y2FsLmdldCgnZWNlX2FmdGVyJyl9IikKICAgICAgICBjay5hZGQoInRlbXBlcmF0dXJlX2ZpdHRlZF9vbl92YWxpZGF0aW9uIiwKICAgICAgICAgICAgICAgcmVzLmdldCgidGVtcGVyYXR1cmUiLCB7fSkuZ2V0KCJzcGxpdCIpID09ICJ2YWxpZGF0aW9uIiwKICAgICAgICAgICAgICAgZiJmaXR0ZWQgb24ge3Jlcy5nZXQoJ3RlbXBlcmF0dXJlJywge30pLmdldCgnc3BsaXQnKSFyfSIpCiAgICAgICAgY20gPSByZXMuZ2V0KCJtZXRyaWNzIiwge30pLmdldCgiY29uZnVzaW9uX21hdHJpeCIpCiAgICAgICAgY2suYWRkKCJjb25mdXNpb25fbWF0cml4X3RvdGFsc19tYXRjaF9uIiwKICAgICAgICAgICAgICAgYm9vbChjbSkgYW5kIHN1bShzdW0ocikgZm9yIHIgaW4gY20pID09IHJlcy5nZXQoIm4iKSwKICAgICAgICAgICAgICAgZiJuPXtyZXMuZ2V0KCduJyl9IikKICAgIGVsc2U6CiAgICAgICAgZm9yIG5hbWUgaW4gKCJ2ZXJpZmljYXRpb25fdXNlc19jb3JyZWN0X2xhYmVsX3NwYWNlIiwgImVjZV9yZXBvcnRlZCIsCiAgICAgICAgICAgICAgICAgICAgICJ0ZW1wZXJhdHVyZV9maXR0ZWRfb25fdmFsaWRhdGlvbiIsICJjb25mdXNpb25fbWF0cml4X3RvdGFsc19tYXRjaF9uIik6CiAgICAgICAgICAgIGNrLmFkZChuYW1lLCBOb25lLCAidmVyaWZpZXIgbm90IGV2YWx1YXRlZCB5ZXQgKHJ1biBtNl8wNykiKQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSA4LiBjb21wdXRlLXNjb3BlIGhvbmVzdHkKICAgIGNzID0gY2ZnWyJjb21wdXRlX3Njb3BlIl0KICAgIGNrLmFkZCgiY29tcHV0ZV9yZWR1Y3Rpb25zX2RlY2xhcmVkIiwKICAgICAgICAgICBjc1siZnVsbF9zY2FsZV9yZXN1bHRzX2NsYWltZWQiXSBpcyBGYWxzZSBhbmQgbGVuKGNzWyJyZWR1Y3Rpb25zIl0pID49IDUsCiAgICAgICAgICAgZiJ7bGVuKGNzWydyZWR1Y3Rpb25zJ10pfSBkb2N1bWVudGVkIHJlZHVjdGlvbihzKSIpCgogICAgc3VtbWFyeSA9IGNrLnN1bW1hcnkoKQogICAgc3VtbWFyeVsibW9kdWxlNV9maW5nZXJwcmludCJdID0gbW9kdWxlNV9maW5nZXJwcmludChyb290KQogICAgb3V0ID0gc3RhZ2VfZGlyKGNmZywgIm1hbmlmZXN0cyIsIHJvb3QpIC8gZiJ2ZXJpZnlfe2FyZ3MuZGF0YXNldH1fe2FyZ3Muc3BsaXR9Lmpzb24iCiAgICB3cml0ZV9qc29uKG91dCwgc3VtbWFyeSkKCiAgICBwcmludCgiXG4iICsgIj0iICogNzApCiAgICBwcmludChmIk1PRFVMRSA2IFZFUklGSUNBVElPTjoge3N1bW1hcnlbJ292ZXJhbGwnXX0gICIKICAgICAgICAgIGYiKHtzdW1tYXJ5WydwYXNzZWQnXX0gcGFzcyAvIHtzdW1tYXJ5WydmYWlsZWQnXX0gZmFpbCAvIHtzdW1tYXJ5Wydza2lwcGVkJ119IHNraXApIikKICAgIHByaW50KGYid3JvdGUge291dH0iKQogICAgcHJpbnQoIj0iICogNzApCiAgICBzeXMuZXhpdCgxIGlmIHN1bW1hcnlbImZhaWxlZCJdIGVsc2UgMCkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==", "sha256": "12b26470affca9e5afadd7af900a08240a9601ef595387b9a98eb52d1637cf63", "size": 11371}, "tests/test_retrieval.py": {"b64": "...", "sha256": "bd276054cdc430cf4e609370f5f1a48c48b4394950d435bc102120f5a58c893a", "size": 19896}, "tests/test_verification.py": {"b64": "...", "sha256": "4dc3814ef5130c76b007ae318db8d53b0580581d2e3c194d22ca6473b35f9760", "size": 17225}}'''
print('payload', len(PAYLOAD_JSON), 'chars')

In [ ]:
import base64, hashlib, json, shutil, time
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")
PAYLOAD = json.loads(PAYLOAD_JSON)

stamp = time.strftime("%Y%m%d_%H%M%S")
backup = PROJECT / "backups" / f"module6_{stamp}"
n = 0
for rel in PAYLOAD:
    d = PROJECT / rel
    if d.exists():
        b = backup / rel
        b.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(d, b)
        n += 1
print("backed up", n, "existing file(s)")

for rel, m in PAYLOAD.items():
    d = PROJECT / rel
    d.parent.mkdir(parents=True, exist_ok=True)
    d.write_bytes(base64.b64decode(m["b64"]))
    got = hashlib.sha256(d.read_bytes()).hexdigest()
    ok = got == m["sha256"]
    print(("OK   " if ok else "FAIL "), rel, d.stat().st_size, "bytes")
    assert ok, f"hash mismatch {rel}"

for sub in ("src/retrieval", "src/verification", "scripts", "tests"):
    p = PROJECT / sub / "__pycache__"
    if p.exists():
        shutil.rmtree(p, ignore_errors=True)
print()
print("Module 6 files synced. Module 5 untouched.")

In [ ]:
import re, subprocess
from pathlib import Path
PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")

def run(cmd):
    r = subprocess.run(cmd, cwd=PROJECT, shell=True, capture_output=True, text=True)
    return r.returncode, r.stdout + r.stderr

rc, out = run("python -m py_compile " + " ".join([
    "src/retrieval/*.py", "src/verification/*.py", "scripts/m6_*.py",
    "scripts/verify_module6_outputs.py"]))
print("py_compile:", "PASS" if rc == 0 else "FAIL\n" + out[:400])

EXPECTED = {"tests/test_retrieval.py": 37, "tests/test_verification.py": 35,
            "tests/test_fact_verification.py": 49, "tests/test_module5_corpus_schema.py": 21,
            "tests/test_contrastive.py": 22, "tests/test_embeddings.py": 5}
bad = []
for t, exp in EXPECTED.items():
    rc, out = run("python " + t)
    m = None
    for line in out.splitlines():
        h = re.search(r"(\d+)/(\d+) passed", line)
        if h:
            m = h
    ok = bool(m) and rc == 0 and int(m.group(1)) == int(m.group(2)) == exp
    print(("PASS " if ok else "FAIL "), t, m.group(0) if m else out.strip()[-140:])
    if not ok:
        bad.append(t)
assert not bad, f"suites failed: {bad}"
print()
print("All suites green. Safe to proceed.")

In [ ]:
# Stage 0 - pipeline status: what is already done   [CPU]

import subprocess, sys, time
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")


def require_project():
    """Drive unmounts when the runtime dies. Without this check the failure
    surfaces as a bare FileNotFoundError from subprocess (it is the *cwd* that
    is missing, not the script), which is very hard to read."""
    if not PROJECT.exists():
        raise RuntimeError(
            "Google Drive is not mounted -- the runtime restarted or was "
            "disconnected.\n  Fix: re-run the MOUNT cell, then re-run this "
            "stage.\n  Nothing is lost: completed shards are skipped on resume.")


def device_report(needs_gpu, est_cpu=""):
    try:
        import torch
        gpu = torch.cuda.is_available()
    except Exception:
        gpu = False
    print("device:", "GPU" if gpu else "CPU")
    if needs_gpu and not gpu:
        print("  !! This stage is GPU-bound and no GPU is attached.")
        if est_cpu:
            print("  !! Estimated CPU time:", est_cpu)
        print("  !! Runtime > Change runtime type > GPU, then re-run.")
        print("  !! Long CPU jobs are a common cause of free-tier disconnection.")
    return gpu


def run_stage(cmd, needs_gpu=False, est_cpu=""):
    require_project()
    device_report(needs_gpu, est_cpu)
    print(">", " ".join(cmd))
    t0 = time.perf_counter()
    p = subprocess.Popen([sys.executable] + cmd, cwd=PROJECT, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    print("exit %s  (%.1fs)" % (p.returncode, time.perf_counter() - t0))
    if p.returncode != 0:
        raise RuntimeError("stage failed with exit %s - fix before continuing"
                           % p.returncode)

require_project()
import json
root = PROJECT / "outputs/module6"
if not root.exists():
    print("no Module 6 outputs yet")
else:
    for stage in sorted(p.name for p in root.iterdir() if p.is_dir()):
        for sub in sorted((root / stage).iterdir()):
            m = sub / "manifest.json"
            if m.is_file():
                d = json.loads(m.read_text())
                done = len(d.get("completed_shards", []))
                tot = d.get("n_shards", "?")
                print(f"  {stage}/{sub.name}: {d.get(chr(115)+chr(116)+chr(97)+chr(116)+chr(117)+chr(115))} "
                      f"shards {done}/{tot}")

In [ ]:
# Stage 1a - encode X-FACT test evidence on CPU   [CPU]
# Runs ONLY this stage and stops. It does not continue into FEVER.
import subprocess, sys, time
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")
if not PROJECT.exists():
    raise RuntimeError(
        "Google Drive is not mounted - the runtime restarted or was disconnected.\n"
        "  Fix: re-run the MOUNT cell, then re-run this cell.\n"
        "  Nothing is lost: completed shards are skipped on resume.")

cmd = [sys.executable, "scripts/m6_01_encode_evidence.py",
       "--dataset", "x_fact", "--phase", "test",
       "--batch-size", "8", "--max-seq-length", "128", "--shard-size", "500",
       "--num-threads", "2", "--min-available-mb", "1200"]
print(">", " ".join(cmd[1:]))
t0 = time.perf_counter()
p = subprocess.Popen(cmd, cwd=PROJECT, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end="", flush=True)
p.wait()
print("\nexit %s  (%.1fs wall clock)" % (p.returncode, time.perf_counter() - t0))
if p.returncode == 2:
    print("Stopped cleanly on the RAM floor. Completed shards are saved -")
    print("re-run this same cell to continue from where it stopped.")
elif p.returncode != 0:
    raise RuntimeError("Stage 1a failed with exit %s - see the log above" % p.returncode)
else:
    print("Stage 1a COMPLETE. Do not run Stage 1b yet.")


In [ ]:
# Stage 1b - encode FEVER evidence (67,388 items) -- GPU STRONGLY ADVISED   [GPU]

import subprocess, sys, time
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")


def require_project():
    """Drive unmounts when the runtime dies. Without this check the failure
    surfaces as a bare FileNotFoundError from subprocess (it is the *cwd* that
    is missing, not the script), which is very hard to read."""
    if not PROJECT.exists():
        raise RuntimeError(
            "Google Drive is not mounted -- the runtime restarted or was "
            "disconnected.\n  Fix: re-run the MOUNT cell, then re-run this "
            "stage.\n  Nothing is lost: completed shards are skipped on resume.")


def device_report(needs_gpu, est_cpu=""):
    try:
        import torch
        gpu = torch.cuda.is_available()
    except Exception:
        gpu = False
    print("device:", "GPU" if gpu else "CPU")
    if needs_gpu and not gpu:
        print("  !! This stage is GPU-bound and no GPU is attached.")
        if est_cpu:
            print("  !! Estimated CPU time:", est_cpu)
        print("  !! Runtime > Change runtime type > GPU, then re-run.")
        print("  !! Long CPU jobs are a common cause of free-tier disconnection.")
    return gpu


def run_stage(cmd, needs_gpu=False, est_cpu=""):
    require_project()
    device_report(needs_gpu, est_cpu)
    print(">", " ".join(cmd))
    t0 = time.perf_counter()
    p = subprocess.Popen([sys.executable] + cmd, cwd=PROJECT, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    print("exit %s  (%.1fs)" % (p.returncode, time.perf_counter() - t0))
    if p.returncode != 0:
        raise RuntimeError("stage failed with exit %s - fix before continuing"
                           % p.returncode)

# On CPU this is roughly 40-60 minutes and is a likely disconnection trigger.
# It is fully resumable: re-running skips completed shards.
run_stage(["scripts/m6_01_encode_evidence.py", "--dataset", "fever",
           "--phase", "test"], needs_gpu=True, est_cpu="~40-60 min on CPU")

In [ ]:
# Stage 2 - build FAISS indexes (both datasets)   [CPU]

import subprocess, sys, time
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")


def require_project():
    """Drive unmounts when the runtime dies. Without this check the failure
    surfaces as a bare FileNotFoundError from subprocess (it is the *cwd* that
    is missing, not the script), which is very hard to read."""
    if not PROJECT.exists():
        raise RuntimeError(
            "Google Drive is not mounted -- the runtime restarted or was "
            "disconnected.\n  Fix: re-run the MOUNT cell, then re-run this "
            "stage.\n  Nothing is lost: completed shards are skipped on resume.")


def device_report(needs_gpu, est_cpu=""):
    try:
        import torch
        gpu = torch.cuda.is_available()
    except Exception:
        gpu = False
    print("device:", "GPU" if gpu else "CPU")
    if needs_gpu and not gpu:
        print("  !! This stage is GPU-bound and no GPU is attached.")
        if est_cpu:
            print("  !! Estimated CPU time:", est_cpu)
        print("  !! Runtime > Change runtime type > GPU, then re-run.")
        print("  !! Long CPU jobs are a common cause of free-tier disconnection.")
    return gpu


def run_stage(cmd, needs_gpu=False, est_cpu=""):
    require_project()
    device_report(needs_gpu, est_cpu)
    print(">", " ".join(cmd))
    t0 = time.perf_counter()
    p = subprocess.Popen([sys.executable] + cmd, cwd=PROJECT, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    print("exit %s  (%.1fs)" % (p.returncode, time.perf_counter() - t0))
    if p.returncode != 0:
        raise RuntimeError("stage failed with exit %s - fix before continuing"
                           % p.returncode)

for ds in ["x_fact", "fever"]:
    run_stage(["scripts/m6_02_build_index.py", "--dataset", ds, "--phase", "test"])

In [ ]:
# Stage 3a - encode X-FACT claims   [GPU]

import subprocess, sys, time
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")


def require_project():
    """Drive unmounts when the runtime dies. Without this check the failure
    surfaces as a bare FileNotFoundError from subprocess (it is the *cwd* that
    is missing, not the script), which is very hard to read."""
    if not PROJECT.exists():
        raise RuntimeError(
            "Google Drive is not mounted -- the runtime restarted or was "
            "disconnected.\n  Fix: re-run the MOUNT cell, then re-run this "
            "stage.\n  Nothing is lost: completed shards are skipped on resume.")


def device_report(needs_gpu, est_cpu=""):
    try:
        import torch
        gpu = torch.cuda.is_available()
    except Exception:
        gpu = False
    print("device:", "GPU" if gpu else "CPU")
    if needs_gpu and not gpu:
        print("  !! This stage is GPU-bound and no GPU is attached.")
        if est_cpu:
            print("  !! Estimated CPU time:", est_cpu)
        print("  !! Runtime > Change runtime type > GPU, then re-run.")
        print("  !! Long CPU jobs are a common cause of free-tier disconnection.")
    return gpu


def run_stage(cmd, needs_gpu=False, est_cpu=""):
    require_project()
    device_report(needs_gpu, est_cpu)
    print(">", " ".join(cmd))
    t0 = time.perf_counter()
    p = subprocess.Popen([sys.executable] + cmd, cwd=PROJECT, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    print("exit %s  (%.1fs)" % (p.returncode, time.perf_counter() - t0))
    if p.returncode != 0:
        raise RuntimeError("stage failed with exit %s - fix before continuing"
                           % p.returncode)

for split in ["validation", "test"]:
    run_stage(["scripts/m6_03_encode_claims.py", "--dataset", "x_fact",
               "--split", split], needs_gpu=True, est_cpu="~1 min")

In [ ]:
# Stage 3b - encode FEVER claims (16k test + 16k val)   [GPU]

import subprocess, sys, time
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")


def require_project():
    """Drive unmounts when the runtime dies. Without this check the failure
    surfaces as a bare FileNotFoundError from subprocess (it is the *cwd* that
    is missing, not the script), which is very hard to read."""
    if not PROJECT.exists():
        raise RuntimeError(
            "Google Drive is not mounted -- the runtime restarted or was "
            "disconnected.\n  Fix: re-run the MOUNT cell, then re-run this "
            "stage.\n  Nothing is lost: completed shards are skipped on resume.")


def device_report(needs_gpu, est_cpu=""):
    try:
        import torch
        gpu = torch.cuda.is_available()
    except Exception:
        gpu = False
    print("device:", "GPU" if gpu else "CPU")
    if needs_gpu and not gpu:
        print("  !! This stage is GPU-bound and no GPU is attached.")
        if est_cpu:
            print("  !! Estimated CPU time:", est_cpu)
        print("  !! Runtime > Change runtime type > GPU, then re-run.")
        print("  !! Long CPU jobs are a common cause of free-tier disconnection.")
    return gpu


def run_stage(cmd, needs_gpu=False, est_cpu=""):
    require_project()
    device_report(needs_gpu, est_cpu)
    print(">", " ".join(cmd))
    t0 = time.perf_counter()
    p = subprocess.Popen([sys.executable] + cmd, cwd=PROJECT, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    print("exit %s  (%.1fs)" % (p.returncode, time.perf_counter() - t0))
    if p.returncode != 0:
        raise RuntimeError("stage failed with exit %s - fix before continuing"
                           % p.returncode)

for split in ["validation", "test"]:
    run_stage(["scripts/m6_03_encode_claims.py", "--dataset", "fever",
               "--split", split], needs_gpu=True, est_cpu="~20-30 min on CPU")

In [ ]:
# Stage 4 - retrieve (dense + BM25) and score -- FIRST REAL NUMBERS   [CPU]

import subprocess, sys, time
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")


def require_project():
    """Drive unmounts when the runtime dies. Without this check the failure
    surfaces as a bare FileNotFoundError from subprocess (it is the *cwd* that
    is missing, not the script), which is very hard to read."""
    if not PROJECT.exists():
        raise RuntimeError(
            "Google Drive is not mounted -- the runtime restarted or was "
            "disconnected.\n  Fix: re-run the MOUNT cell, then re-run this "
            "stage.\n  Nothing is lost: completed shards are skipped on resume.")


def device_report(needs_gpu, est_cpu=""):
    try:
        import torch
        gpu = torch.cuda.is_available()
    except Exception:
        gpu = False
    print("device:", "GPU" if gpu else "CPU")
    if needs_gpu and not gpu:
        print("  !! This stage is GPU-bound and no GPU is attached.")
        if est_cpu:
            print("  !! Estimated CPU time:", est_cpu)
        print("  !! Runtime > Change runtime type > GPU, then re-run.")
        print("  !! Long CPU jobs are a common cause of free-tier disconnection.")
    return gpu


def run_stage(cmd, needs_gpu=False, est_cpu=""):
    require_project()
    device_report(needs_gpu, est_cpu)
    print(">", " ".join(cmd))
    t0 = time.perf_counter()
    p = subprocess.Popen([sys.executable] + cmd, cwd=PROJECT, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    print("exit %s  (%.1fs)" % (p.returncode, time.perf_counter() - t0))
    if p.returncode != 0:
        raise RuntimeError("stage failed with exit %s - fix before continuing"
                           % p.returncode)

for ds in ["x_fact", "fever"]:
    for split in ["validation", "test"]:
        run_stage(["scripts/m6_04_retrieve.py", "--dataset", ds, "--split", split,
                   "--retriever", "both"])

In [ ]:
# Stage 9 - verify Module 6 outputs (safe to run at any point)   [CPU]

import subprocess, sys, time
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")


def require_project():
    """Drive unmounts when the runtime dies. Without this check the failure
    surfaces as a bare FileNotFoundError from subprocess (it is the *cwd* that
    is missing, not the script), which is very hard to read."""
    if not PROJECT.exists():
        raise RuntimeError(
            "Google Drive is not mounted -- the runtime restarted or was "
            "disconnected.\n  Fix: re-run the MOUNT cell, then re-run this "
            "stage.\n  Nothing is lost: completed shards are skipped on resume.")


def device_report(needs_gpu, est_cpu=""):
    try:
        import torch
        gpu = torch.cuda.is_available()
    except Exception:
        gpu = False
    print("device:", "GPU" if gpu else "CPU")
    if needs_gpu and not gpu:
        print("  !! This stage is GPU-bound and no GPU is attached.")
        if est_cpu:
            print("  !! Estimated CPU time:", est_cpu)
        print("  !! Runtime > Change runtime type > GPU, then re-run.")
        print("  !! Long CPU jobs are a common cause of free-tier disconnection.")
    return gpu


def run_stage(cmd, needs_gpu=False, est_cpu=""):
    require_project()
    device_report(needs_gpu, est_cpu)
    print(">", " ".join(cmd))
    t0 = time.perf_counter()
    p = subprocess.Popen([sys.executable] + cmd, cwd=PROJECT, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    print("exit %s  (%.1fs)" % (p.returncode, time.perf_counter() - t0))
    if p.returncode != 0:
        raise RuntimeError("stage failed with exit %s - fix before continuing"
                           % p.returncode)

for ds in ["x_fact", "fever"]:
    run_stage(["scripts/verify_module6_outputs.py", "--dataset", ds,
               "--split", "test"])

In [ ]:
# Stage 5 - cross-encoder rerank (needs a fine-tuned checkpoint)   [GPU]

import subprocess, sys, time
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")


def require_project():
    """Drive unmounts when the runtime dies. Without this check the failure
    surfaces as a bare FileNotFoundError from subprocess (it is the *cwd* that
    is missing, not the script), which is very hard to read."""
    if not PROJECT.exists():
        raise RuntimeError(
            "Google Drive is not mounted -- the runtime restarted or was "
            "disconnected.\n  Fix: re-run the MOUNT cell, then re-run this "
            "stage.\n  Nothing is lost: completed shards are skipped on resume.")


def device_report(needs_gpu, est_cpu=""):
    try:
        import torch
        gpu = torch.cuda.is_available()
    except Exception:
        gpu = False
    print("device:", "GPU" if gpu else "CPU")
    if needs_gpu and not gpu:
        print("  !! This stage is GPU-bound and no GPU is attached.")
        if est_cpu:
            print("  !! Estimated CPU time:", est_cpu)
        print("  !! Runtime > Change runtime type > GPU, then re-run.")
        print("  !! Long CPU jobs are a common cause of free-tier disconnection.")
    return gpu


def run_stage(cmd, needs_gpu=False, est_cpu=""):
    require_project()
    device_report(needs_gpu, est_cpu)
    print(">", " ".join(cmd))
    t0 = time.perf_counter()
    p = subprocess.Popen([sys.executable] + cmd, cwd=PROJECT, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    print("exit %s  (%.1fs)" % (p.returncode, time.perf_counter() - t0))
    if p.returncode != 0:
        raise RuntimeError("stage failed with exit %s - fix before continuing"
                           % p.returncode)

# Without --checkpoint the cross-encoder head is untrained and the scores
# are arbitrary. Do not report these numbers.
for ds in ["x_fact", "fever"]:
    run_stage(["scripts/m6_05_rerank.py", "--dataset", ds, "--split", "test"],
              needs_gpu=True, est_cpu="hours - do not run on CPU")

In [ ]:
# Stage 6a - train FEVER verifier (entailment)   [GPU]

import subprocess, sys, time
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")


def require_project():
    """Drive unmounts when the runtime dies. Without this check the failure
    surfaces as a bare FileNotFoundError from subprocess (it is the *cwd* that
    is missing, not the script), which is very hard to read."""
    if not PROJECT.exists():
        raise RuntimeError(
            "Google Drive is not mounted -- the runtime restarted or was "
            "disconnected.\n  Fix: re-run the MOUNT cell, then re-run this "
            "stage.\n  Nothing is lost: completed shards are skipped on resume.")


def device_report(needs_gpu, est_cpu=""):
    try:
        import torch
        gpu = torch.cuda.is_available()
    except Exception:
        gpu = False
    print("device:", "GPU" if gpu else "CPU")
    if needs_gpu and not gpu:
        print("  !! This stage is GPU-bound and no GPU is attached.")
        if est_cpu:
            print("  !! Estimated CPU time:", est_cpu)
        print("  !! Runtime > Change runtime type > GPU, then re-run.")
        print("  !! Long CPU jobs are a common cause of free-tier disconnection.")
    return gpu


def run_stage(cmd, needs_gpu=False, est_cpu=""):
    require_project()
    device_report(needs_gpu, est_cpu)
    print(">", " ".join(cmd))
    t0 = time.perf_counter()
    p = subprocess.Popen([sys.executable] + cmd, cwd=PROJECT, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    print("exit %s  (%.1fs)" % (p.returncode, time.perf_counter() - t0))
    if p.returncode != 0:
        raise RuntimeError("stage failed with exit %s - fix before continuing"
                           % p.returncode)

run_stage(["scripts/m6_06_train_verifier.py", "--dataset", "fever"],
          needs_gpu=True, est_cpu="many hours - do not run on CPU")

In [ ]:
# Stage 6b - transfer to X-FACT (veracity; head replaced)   [GPU]

import subprocess, sys, time
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")


def require_project():
    """Drive unmounts when the runtime dies. Without this check the failure
    surfaces as a bare FileNotFoundError from subprocess (it is the *cwd* that
    is missing, not the script), which is very hard to read."""
    if not PROJECT.exists():
        raise RuntimeError(
            "Google Drive is not mounted -- the runtime restarted or was "
            "disconnected.\n  Fix: re-run the MOUNT cell, then re-run this "
            "stage.\n  Nothing is lost: completed shards are skipped on resume.")


def device_report(needs_gpu, est_cpu=""):
    try:
        import torch
        gpu = torch.cuda.is_available()
    except Exception:
        gpu = False
    print("device:", "GPU" if gpu else "CPU")
    if needs_gpu and not gpu:
        print("  !! This stage is GPU-bound and no GPU is attached.")
        if est_cpu:
            print("  !! Estimated CPU time:", est_cpu)
        print("  !! Runtime > Change runtime type > GPU, then re-run.")
        print("  !! Long CPU jobs are a common cause of free-tier disconnection.")
    return gpu


def run_stage(cmd, needs_gpu=False, est_cpu=""):
    require_project()
    device_report(needs_gpu, est_cpu)
    print(">", " ".join(cmd))
    t0 = time.perf_counter()
    p = subprocess.Popen([sys.executable] + cmd, cwd=PROJECT, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    print("exit %s  (%.1fs)" % (p.returncode, time.perf_counter() - t0))
    if p.returncode != 0:
        raise RuntimeError("stage failed with exit %s - fix before continuing"
                           % p.returncode)

run_stage(["scripts/m6_06_train_verifier.py", "--dataset", "x_fact",
           "--init-from", "models/module6/fever"], needs_gpu=True)

In [ ]:
# Stage 7 - evaluate + calibrate (reported separately)   [GPU]

import subprocess, sys, time
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")


def require_project():
    """Drive unmounts when the runtime dies. Without this check the failure
    surfaces as a bare FileNotFoundError from subprocess (it is the *cwd* that
    is missing, not the script), which is very hard to read."""
    if not PROJECT.exists():
        raise RuntimeError(
            "Google Drive is not mounted -- the runtime restarted or was "
            "disconnected.\n  Fix: re-run the MOUNT cell, then re-run this "
            "stage.\n  Nothing is lost: completed shards are skipped on resume.")


def device_report(needs_gpu, est_cpu=""):
    try:
        import torch
        gpu = torch.cuda.is_available()
    except Exception:
        gpu = False
    print("device:", "GPU" if gpu else "CPU")
    if needs_gpu and not gpu:
        print("  !! This stage is GPU-bound and no GPU is attached.")
        if est_cpu:
            print("  !! Estimated CPU time:", est_cpu)
        print("  !! Runtime > Change runtime type > GPU, then re-run.")
        print("  !! Long CPU jobs are a common cause of free-tier disconnection.")
    return gpu


def run_stage(cmd, needs_gpu=False, est_cpu=""):
    require_project()
    device_report(needs_gpu, est_cpu)
    print(">", " ".join(cmd))
    t0 = time.perf_counter()
    p = subprocess.Popen([sys.executable] + cmd, cwd=PROJECT, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    print("exit %s  (%.1fs)" % (p.returncode, time.perf_counter() - t0))
    if p.returncode != 0:
        raise RuntimeError("stage failed with exit %s - fix before continuing"
                           % p.returncode)

for ds in ["fever", "x_fact"]:
    run_stage(["scripts/m6_07_evaluate_verifier.py", "--dataset", ds,
               "--split", "test"], needs_gpu=True)

In [ ]:
# Stage 8 - ablations   [CPU]

import subprocess, sys, time
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")


def require_project():
    """Drive unmounts when the runtime dies. Without this check the failure
    surfaces as a bare FileNotFoundError from subprocess (it is the *cwd* that
    is missing, not the script), which is very hard to read."""
    if not PROJECT.exists():
        raise RuntimeError(
            "Google Drive is not mounted -- the runtime restarted or was "
            "disconnected.\n  Fix: re-run the MOUNT cell, then re-run this "
            "stage.\n  Nothing is lost: completed shards are skipped on resume.")


def device_report(needs_gpu, est_cpu=""):
    try:
        import torch
        gpu = torch.cuda.is_available()
    except Exception:
        gpu = False
    print("device:", "GPU" if gpu else "CPU")
    if needs_gpu and not gpu:
        print("  !! This stage is GPU-bound and no GPU is attached.")
        if est_cpu:
            print("  !! Estimated CPU time:", est_cpu)
        print("  !! Runtime > Change runtime type > GPU, then re-run.")
        print("  !! Long CPU jobs are a common cause of free-tier disconnection.")
    return gpu


def run_stage(cmd, needs_gpu=False, est_cpu=""):
    require_project()
    device_report(needs_gpu, est_cpu)
    print(">", " ".join(cmd))
    t0 = time.perf_counter()
    p = subprocess.Popen([sys.executable] + cmd, cwd=PROJECT, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    print("exit %s  (%.1fs)" % (p.returncode, time.perf_counter() - t0))
    if p.returncode != 0:
        raise RuntimeError("stage failed with exit %s - fix before continuing"
                           % p.returncode)

for ds in ["fever", "x_fact"]:
    run_stage(["scripts/m6_08_run_ablations.py", "--dataset", ds, "--split", "test"])